# Paper 3 — Data Audit and Cohort Definition

## Reliable Structure–Processing–Performance Learning in Organic Photovoltaics

### Purpose

This notebook establishes the provenance, integrity, chemical identity,
target quality, dataset composition, overlap structure, and potential
sources of information leakage in the datasets used for this study.

The audit is performed before predictive modeling so that subsequent
training and validation cohorts are defined from scientifically justified
criteria rather than convenience.

### Primary datasets

1. OPV-DB
2. Wen–Zhang–Ma processing-aware OPV database

### Audit questions

- What information is actually contained in each released dataset?
- Are molecular identities and structures internally consistent?
- Are photovoltaic targets physically and internally consistent?
- How strongly is the literature concentrated around repeated donor,
  acceptor, and donor–acceptor systems?
- How complete are processing and device variables?
- Does requiring processing information introduce chemical selection bias?
- How much chemical and publication overlap would occur under conventional
  random train/test splitting?
- Which scientifically defensible cohorts should be used for subsequent
  modeling?

In [1]:
from pathlib import Path
import sys
import platform

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

OPVDB_DIR = RAW_DIR / "opvdb"
WEN_DIR = RAW_DIR / "opv-multi-tier-ml-database"

print("Notebook directory :", NOTEBOOK_DIR)
print("Project root       :", PROJECT_ROOT)
print("Raw data directory :", RAW_DIR)

print("\nDataset folders:")
print("OPV-DB exists      :", OPVDB_DIR.exists())
print("Wen/Ma exists      :", WEN_DIR.exists())

print("\nEnvironment:")
print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("OS     :", platform.platform())

Notebook directory : <PROJECT_ROOT>\notebooks
Project root       : <PROJECT_ROOT>
Raw data directory : <PROJECT_ROOT>\data\raw

Dataset folders:
OPV-DB exists      : True
Wen/Ma exists      : True

Environment:
Python : 3.11.15
Pandas : 3.0.5
NumPy  : 2.4.6
OS     : Windows-10-10.0.22000-SP0


## 1. Dataset Provenance, Integrity and Schema

### 1.1 Raw OPV-DB file inventory and integrity

Before inspecting or transforming the data, the released OPV-DB files are
inventoried and cryptographic SHA-256 hashes are recorded.

This provides a reproducible record of the exact raw files used in the
analysis and allows accidental modification of source data to be detected.

In [2]:
import hashlib

# Core OPV-DB files used in the study
opvdb_files = {
    "README": OPVDB_DIR / "README.md",
    "data_dictionary": OPVDB_DIR / "DATA_DICTIONARY.md",
    "third_party_attribution": OPVDB_DIR / "THIRD_PARTY_ATTRIBUTION.md",

    "materials_reference":
        OPVDB_DIR / "data" / "materials_reference.csv",

    "devices_full":
        OPVDB_DIR / "data" / "opv_devices_full.csv",

    "strict_performance":
        OPVDB_DIR / "data" / "opv_devices_strict_performance_benchmark.csv",

    "strict_molecular":
        OPVDB_DIR / "data" / "opv_devices_strict_molecular_benchmark.csv",

    "quality_rules":
        OPVDB_DIR / "validation" / "opv_quality_tier_rules.json",

    "quality_summary":
        OPVDB_DIR / "validation" / "opv_quality_tier_summary.csv",

    "field_coverage":
        OPVDB_DIR / "validation" / "opv_quality_tier_field_coverage.csv",

    "pce_consistency":
        OPVDB_DIR / "validation" / "opv_pce_consistency_records.csv",
}


def sha256_file(path, chunk_size=1024 * 1024):
    """Return SHA-256 checksum without loading the whole file into memory."""
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


inventory_rows = []

for label, path in opvdb_files.items():

    exists = path.exists()

    inventory_rows.append({
        "file": label,
        "filename": path.name,
        "exists": exists,
        "size_MB": round(path.stat().st_size / (1024 ** 2), 3)
                   if exists else np.nan,
        "sha256": sha256_file(path) if exists else None,
    })


opvdb_inventory = pd.DataFrame(inventory_rows)

opvdb_inventory

,file,filename,exists,size_MB,sha256
0,README,README.md,True,0.002,b9a931fd7497e4739fa60896fb04e8fd544eddf1fc3532...
1,data_dictionary,DATA_DICTIONARY.md,True,0.005,7fc8e21562e354c9b999893535dba56f9866d14aff5d2b...
2,third_party_attribution,THIRD_PARTY_ATTRIBUTION.md,True,0.001,e9f5d5042b2aa838107298ab651c1e08f2036e67eea0d1...
3,materials_reference,materials_reference.csv,True,0.829,2f1224dd8f70b510db9de455a298e5493f8fa99dc68347...
4,devices_full,opv_devices_full.csv,True,17.877,218e2034f895682505815c60ad42e14b0e24f1f1e11740...
5,strict_performance,opv_devices_strict_performance_benchmark.csv,True,14.746,d83c036854a2c1fdf1071ee53330232bfda9d119e4b9e3...
6,strict_molecular,opv_devices_strict_molecular_benchmark.csv,True,11.255,068c0137887c9108d6ab62771a750c84b7e34d8c4b32e4...
7,quality_rules,opv_quality_tier_rules.json,True,0.002,a6a13fe80f28a87552e6bd60b55820a334ebce5187066d...
8,quality_summary,opv_quality_tier_summary.csv,True,0.002,02e83da06cc2ed941fbc5d3ac8be9d303f819113cc08c3...
9,field_coverage,opv_quality_tier_field_coverage.csv,True,0.006,f9aec2942b8fb06c8dea1d37421e6caf7e537565b484a0...


### 1.2 Load released OPV-DB tables and verify record counts

The four released data tables are loaded directly from the untouched raw-data
directory. Their observed record counts are then compared with the record
counts reported by the OPV-DB authors in the accompanying validation summary.

This check verifies that the files were read correctly and that the local
copies correspond to the documented release before any cleaning or filtering
is performed.

In [3]:
# ------------------------------------------------------------------
# Load OPV-DB released tables
# ------------------------------------------------------------------

materials_ref = pd.read_csv(
    opvdb_files["materials_reference"],
    low_memory=False
)

devices_full = pd.read_csv(
    opvdb_files["devices_full"],
    low_memory=False
)

strict_performance = pd.read_csv(
    opvdb_files["strict_performance"],
    low_memory=False
)

strict_molecular = pd.read_csv(
    opvdb_files["strict_molecular"],
    low_memory=False
)

quality_summary = pd.read_csv(
    opvdb_files["quality_summary"],
    low_memory=False
)


# ------------------------------------------------------------------
# Compare observed record counts with author-reported counts
# ------------------------------------------------------------------

loaded_tables = {
    "full": devices_full,
    "strict_performance_benchmark": strict_performance,
    "strict_molecular_benchmark": strict_molecular,
}

expected_records = (
    quality_summary
    .set_index("tier")["records"]
    .to_dict()
)

count_check = []

for tier, df in loaded_tables.items():

    observed = len(df)
    expected = expected_records.get(tier)

    count_check.append({
        "tier": tier,
        "observed_records": observed,
        "author_reported_records": expected,
        "record_count_match": observed == expected,
        "columns": df.shape[1],
    })

count_check = pd.DataFrame(count_check)

print("Materials reference:")
print(f"  Records : {materials_ref.shape[0]:,}")
print(f"  Columns : {materials_ref.shape[1]:,}")

print("\nDevice-table integrity check:")
count_check

Materials reference:
  Records : 4,548
  Columns : 7

Device-table integrity check:


,tier,observed_records,author_reported_records,record_count_match,columns
0,full,38849,38849,True,37
1,strict_performance_benchmark,31360,31360,True,37
2,strict_molecular_benchmark,21720,21720,True,37


### 1.3 Schema and data-type audit

The released device tables are checked for schema consistency before any
analysis is performed.

This includes:

- column names and ordering,
- duplicated column names,
- unexpected unnamed/index columns,
- differences in fields between quality tiers,
- and inferred pandas data types.

Schema verification is necessary because inconsistencies at this stage can
silently affect downstream filtering, grouping, and model construction.

In [4]:
# ------------------------------------------------------------------
# Schema audit
# ------------------------------------------------------------------

def schema_summary(name, df):
    duplicated_cols = df.columns[df.columns.duplicated()].tolist()
    unnamed_cols = [
        col for col in df.columns
        if str(col).lower().startswith("unnamed")
    ]

    return {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_column_names": len(duplicated_cols),
        "unnamed_columns": len(unnamed_cols),
        "unique_column_names": df.columns.is_unique,
    }


schema_check = pd.DataFrame([
    schema_summary("full", devices_full),
    schema_summary("strict_performance", strict_performance),
    schema_summary("strict_molecular", strict_molecular),
    schema_summary("materials_reference", materials_ref),
])

print("Schema integrity:")
display(schema_check)


# ------------------------------------------------------------------
# Compare the three device-table schemas
# ------------------------------------------------------------------

full_cols = list(devices_full.columns)
perf_cols = list(strict_performance.columns)
mol_cols = list(strict_molecular.columns)

print("\nDevice-table schemas identical:")
print("Full vs strict performance :", full_cols == perf_cols)
print("Full vs strict molecular   :", full_cols == mol_cols)
print("Performance vs molecular   :", perf_cols == mol_cols)


# ------------------------------------------------------------------
# Display device-table fields and inferred dtypes
# ------------------------------------------------------------------

dtype_table = pd.DataFrame({
    "field": strict_molecular.columns,
    "dtype_full": [str(devices_full[c].dtype) for c in strict_molecular.columns],
    "dtype_strict_performance": [
        str(strict_performance[c].dtype)
        for c in strict_molecular.columns
    ],
    "dtype_strict_molecular": [
        str(strict_molecular[c].dtype)
        for c in strict_molecular.columns
    ],
})

display(dtype_table)

Schema integrity:


,table,rows,columns,duplicated_column_names,unnamed_columns,unique_column_names
0,full,38849,37,0,0,True
1,strict_performance,31360,37,0,0,True
2,strict_molecular,21720,37,0,0,True
3,materials_reference,4548,7,0,0,True



Device-table schemas identical:
Full vs strict performance : True
Full vs strict molecular   : True
Performance vs molecular   : True


,field,dtype_full,dtype_strict_performance,dtype_strict_molecular
0,id,int64,int64,int64
1,doi,str,str,str
2,doi_norm,str,str,str
3,donor,str,str,str
4,acceptor,str,str,str
5,donor_canonical,str,str,str
6,acceptor_canonical,str,str,str
7,donor_smiles,str,str,str
8,acceptor_smiles,str,str,str
9,voc,float64,float64,float64


### 1.4 Field completeness and information retention across quality tiers

Field completeness is independently calculated from the released device
tables and compared with the authors' accompanying field-coverage validation
file.

The purpose is to verify the documented release and to quantify how the
transition from the full archive to the strict performance and strict
molecular benchmarks changes the availability of molecular, device, and
processing information.

No missing values are imputed at this stage.

In [6]:
# ------------------------------------------------------------------
# Load author-reported field coverage
# ------------------------------------------------------------------

author_coverage = pd.read_csv(
    opvdb_files["field_coverage"],
    low_memory=False
)


# ------------------------------------------------------------------
# Independently calculate non-missing field coverage
# ------------------------------------------------------------------

tier_tables = {
    "full": devices_full,
    "strict_performance_benchmark": strict_performance,
    "strict_molecular_benchmark": strict_molecular,
}

coverage_rows = []

for tier, df in tier_tables.items():
    for field in df.columns:

        filled = int(df[field].notna().sum())
        percent = 100 * filled / len(df)

        coverage_rows.append({
            "tier": tier,
            "field": field,
            "observed_filled_records": filled,
            "observed_filled_percent": percent,
        })

observed_coverage = pd.DataFrame(coverage_rows)


# ------------------------------------------------------------------
# Compare our calculation with the release validation file
# ------------------------------------------------------------------

coverage_check = observed_coverage.merge(
    author_coverage[
        ["tier", "field", "filled_records", "filled_percent"]
    ],
    on=["tier", "field"],
    how="left"
)

coverage_check["record_count_match"] = (
    coverage_check["observed_filled_records"]
    == coverage_check["filled_records"]
)

coverage_check["percent_difference"] = (
    coverage_check["observed_filled_percent"]
    - coverage_check["filled_percent"]
).abs()


print("Coverage records checked :", len(coverage_check))
print(
    "Filled-count mismatches   :",
    (~coverage_check["record_count_match"]).sum()
)

print(
    "Maximum % difference      :",
    coverage_check["percent_difference"].max()
)



# Fields most relevant to this study
key_fields = [
    "doi_norm",
    "donor",
    "acceptor",
    "donor_smiles",
    "acceptor_smiles",
    "voc",
    "jsc",
    "ff",
    "pce",
    "d_a_ratio",
    "solvent",
    "additive",
    "additive_ratio",
    "active_layer_thickness",
    "annealing_temp",
    "device_structure",
    "device_type",
    "etl",
    "htl",
    "homo_d",
    "lumo_d",
    "eg_d",
    "homo_a",
    "lumo_a",
    "eg_a",
]

key_coverage = (
    observed_coverage[
        observed_coverage["field"].isin(key_fields)
    ]
    .pivot(
        index="field",
        columns="tier",
        values="observed_filled_percent"
    )
    .round(1)
)

key_coverage

Coverage records checked : 111
Filled-count mismatches   : 0
Maximum % difference      : 2.842170943040401e-14


tier,full,strict_molecular_benchmark,strict_performance_benchmark
field,,,
acceptor,97.1,100.0,100.0
acceptor_smiles,81.3,100.0,82.6
active_layer_thickness,27.0,30.3,26.9
additive,30.8,34.9,32.1
additive_ratio,19.5,23.3,20.2
annealing_temp,23.8,24.8,24.2
d_a_ratio,49.4,51.5,49.9
device_structure,82.1,82.9,83.2
device_type,75.4,77.1,76.8


### 1.5 Audit of donor–acceptor and additive ratio encodings

The donor–acceptor ratio (`d_a_ratio`) and additive ratio (`additive_ratio`)
are stored as text rather than numeric variables in the public release.

Before any numerical conversion is attempted, their raw encodings are
examined to determine whether the fields contain simple numbers, ratio
notation, units, ranges, categorical text, or other conventions.

The raw columns are not modified during this audit.

In [7]:
# ------------------------------------------------------------------
# Audit text-encoded ratio fields without modifying the raw data
# ------------------------------------------------------------------

ratio_fields = ["d_a_ratio", "additive_ratio"]


def inspect_text_numeric_field(df, field, tier_name):
    s = df[field]

    non_missing = s.dropna().astype(str).str.strip()

    # Temporary numeric conversion used ONLY to measure parseability
    numeric_test = pd.to_numeric(non_missing, errors="coerce")

    n_total = len(df)
    n_present = len(non_missing)
    n_numeric = int(numeric_test.notna().sum())
    n_non_numeric = int(numeric_test.isna().sum())

    return {
        "tier": tier_name,
        "field": field,
        "total_records": n_total,
        "filled_records": n_present,
        "unique_raw_values": non_missing.nunique(),
        "directly_numeric_records": n_numeric,
        "non_numeric_records": n_non_numeric,
        "directly_numeric_percent_of_filled":
            round(100 * n_numeric / n_present, 2) if n_present else np.nan,
    }


ratio_audit_rows = []

for tier_name, df in tier_tables.items():
    for field in ratio_fields:
        ratio_audit_rows.append(
            inspect_text_numeric_field(df, field, tier_name)
        )

ratio_audit = pd.DataFrame(ratio_audit_rows)

display(ratio_audit)



# ------------------------------------------------------------------
# Inspect raw value patterns in the strict molecular benchmark
# ------------------------------------------------------------------

for field in ratio_fields:

    print("\n" + "=" * 80)
    print(field)
    print("=" * 80)

    s = (
        strict_molecular[field]
        .dropna()
        .astype(str)
        .str.strip()
    )

    numeric_test = pd.to_numeric(s, errors="coerce")

    print(f"Filled records     : {len(s):,}")
    print(f"Unique raw values  : {s.nunique():,}")
    print(
        f"Directly numeric   : "
        f"{numeric_test.notna().sum():,} "
        f"({100 * numeric_test.notna().mean():.1f}%)"
    )

    print("\nMost frequent raw values:")
    display(
        s.value_counts()
        .head(20)
        .rename_axis("raw_value")
        .reset_index(name="count")
    )

    non_numeric = s[numeric_test.isna()]

    print("\nExample non-numeric encodings:")
    display(
        non_numeric.value_counts()
        .head(30)
        .rename_axis("raw_value")
        .reset_index(name="count")
    )

,tier,field,total_records,filled_records,unique_raw_values,directly_numeric_records,non_numeric_records,directly_numeric_percent_of_filled
0,full,d_a_ratio,38849,19201,705,22,19179,0.11
1,full,additive_ratio,38849,7565,742,172,7393,2.27
2,strict_performance_benchmark,d_a_ratio,31360,15652,640,16,15636,0.10
3,strict_performance_benchmark,additive_ratio,31360,6336,672,154,6182,2.43
4,strict_molecular_benchmark,d_a_ratio,21720,11183,527,13,11170,0.12
5,strict_molecular_benchmark,additive_ratio,21720,5050,616,125,4925,2.48



d_a_ratio
Filled records     : 11,183
Unique raw values  : 527
Directly numeric   : 13 (0.1%)

Most frequent raw values:


,raw_value,count
0,1:1,2863
1,1:1.2,1706
2,1:1.5,1524
3,1:2,991
4,1:3,383
5,1:4,250
6,1:0.8,224
7,1:1 w/w,200
8,1.5:1,183
9,2:1,166



Example non-numeric encodings:


,raw_value,count
0,1:1,2863
1,1:1.2,1706
2,1:1.5,1524
3,1:2,991
4,1:3,383
5,1:4,250
6,1:0.8,224
7,1:1 w/w,200
8,1.5:1,183
9,2:1,166



additive_ratio
Filled records     : 5,050
Unique raw values  : 616
Directly numeric   : 125 (2.5%)

Most frequent raw values:


,raw_value,count
0,0.5 vol%,496
1,0.5%,381
2,3 vol%,370
3,3%,357
4,1%,189
5,0.5% v/v,157
6,2%,133
7,1 vol%,97
8,3% v/v,95
9,2 vol%,93



Example non-numeric encodings:


,raw_value,count
0,0.5 vol%,496
1,0.5%,381
2,3 vol%,370
3,3%,357
4,1%,189
5,0.5% v/v,157
6,2%,133
7,1 vol%,97
8,3% v/v,95
9,2 vol%,93


### 1.6 Semantic classification of ratio encodings

The raw ratio fields use heterogeneous textual conventions.

For the donor–acceptor ratio, the audit determines how many records contain a
recoverable numeric `a:b` ratio and what qualifiers accompany that ratio.

For additive loading, units are explicitly classified because volume fraction,
weight fraction, concentration, and unspecified percentages are not
scientifically interchangeable.

No unit conversion or numerical harmonization is performed at this stage.

In [8]:
import re

# ================================================================
# D:A ratio encoding audit
# ================================================================

da_raw = (
    strict_molecular["d_a_ratio"]
    .dropna()
    .astype(str)
    .str.strip()
)

# Extract the first numeric a:b expression
da_extracted = da_raw.str.extract(
    r'(?P<donor_part>\d*\.?\d+)\s*:\s*(?P<acceptor_part>\d*\.?\d+)'
)

da_has_ratio = da_extracted.notna().all(axis=1)

# Text remaining after removing the numeric ratio itself
da_suffix = (
    da_raw
    .str.replace(
        r'\d*\.?\d+\s*:\s*\d*\.?\d+',
        '',
        regex=True
    )
    .str.strip()
    .str.lower()
)

print("D:A RATIO ENCODING")
print("=" * 70)
print(f"Filled records                : {len(da_raw):,}")
print(f"Numeric a:b ratio extractable : {da_has_ratio.sum():,}")
print(
    f"Extractable percentage        : "
    f"{100 * da_has_ratio.mean():.2f}%"
)

print("\nMost common residual qualifiers:")
display(
    da_suffix.value_counts()
    .head(30)
    .rename_axis("qualifier")
    .reset_index(name="count")
)

print("\nExamples with no extractable numeric a:b ratio:")
display(
    da_raw[~da_has_ratio]
    .value_counts()
    .head(40)
    .rename_axis("raw_value")
    .reset_index(name="count")
)



# ================================================================
# Additive-ratio unit audit
# ================================================================

add_raw = (
    strict_molecular["additive_ratio"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)


def classify_additive_unit(value):
    """Classify the reported additive-loading convention."""

    v = value.replace("μ", "u").replace("µ", "u")

    if re.search(r'mg\s*/\s*ml', v):
        return "mg/mL"

    if re.search(r'wt\s*%', v):
        return "wt%"

    if (
        re.search(r'vol\s*%', v)
        or re.search(r'v\s*/\s*v', v)
    ):
        return "vol% or v/v"

    if re.search(r'mol\s*%', v):
        return "mol%"

    if "%" in v:
        return "unspecified %"

    # plain numeric values
    if re.fullmatch(r'[+-]?\d*\.?\d+', v):
        return "unitless numeric"

    return "other / complex"


add_unit_class = add_raw.map(classify_additive_unit)

additive_unit_summary = (
    add_unit_class
    .value_counts()
    .rename_axis("unit_class")
    .reset_index(name="records")
)

additive_unit_summary["percent"] = (
    100 * additive_unit_summary["records"] / len(add_raw)
).round(2)

print("\nADDITIVE-RATIO ENCODING")
print("=" * 70)

display(additive_unit_summary)

print("\nExamples by unit class:")

for unit_class in additive_unit_summary["unit_class"]:
    print(f"\n--- {unit_class} ---")

    examples = (
        add_raw[add_unit_class == unit_class]
        .value_counts()
        .head(15)
        .rename_axis("raw_value")
        .reset_index(name="count")
    )

    display(examples)

D:A RATIO ENCODING
Filled records                : 11,183
Numeric a:b ratio extractable : 11,110
Extractable percentage        : 99.35%

Most common residual qualifiers:


,qualifier,count
0,,9719
1,w/w,583
2,wt%,124
3,wt/wt,122
4,", w/w",74
5,(w/w),48
6,:0,40
7,wt,31
8,by weight,27
9,wt:wt,25



Examples with no extractable numeric a:b ratio:


,raw_value,count
0,1/1.5,8
1,1/1.2 (w/w),6
2,1/1,5
3,1/2,5
4,1.25/1,4
5,1.5/1,3
6,5%,3
7,0%,2
8,10%,2
9,20%,2



ADDITIVE-RATIO ENCODING


...

,raw_value,count
0,3,26
1,0.5,18
2,1,13
3,5,9
4,0.6,7
5,1.5,6
6,2,6
7,0.3,6
8,0.1,4
9,4,3



--- mg/mL ---


,raw_value,count
0,10 mg/ml,37
1,1 mg/ml,12
2,15 mg/ml,7
3,5 mg/ml,7
4,12 mg/ml,6
5,20% v/v (2 mg/ml aqueous solution),4
6,8 mg/ml,4
7,0.01 mg/ml,3
8,0.02 mg/ml,3
9,0.04 mg/ml,3



--- mol% ---


,raw_value,count
0,5 mol%,3
1,0.25 mol %,2
2,0 mol%,1


### 1.7 Conservative recoverability of processing-ratio information

A numerical donor–acceptor ratio is retained only when the raw field contains
an unambiguous two-component ratio with no conflicting numerical information.

Entries containing additional component ratios, percentages, layer dimensions,
or other ambiguous numerical annotations are flagged rather than automatically
converted.

Additive-loading values are classified by measurement basis and are not
combined across physically different units.

In [9]:
# ------------------------------------------------------------------
# Conservative D:A ratio parser
# ------------------------------------------------------------------

weight_qualifiers = {
    "",
    "w/w",
    "(w/w)",
    ", w/w",
    ",w/w",
    "wt/wt",
    "(wt/wt)",
    "wt:wt",
    "w:w",
    "wt",
    "wt ratio",
    "mass ratio",
    "weight ratio",
    "by weight",
    "()",
}


def classify_da_ratio(value):
    """
    Conservatively classify D:A ratio strings.

    Returns:
        class
        donor_part
        acceptor_part
        acceptor_to_donor
    """
    if pd.isna(value):
        return pd.Series(
            ["missing", np.nan, np.nan, np.nan]
        )

    raw = str(value).strip().lower()

    # Match one simple a:b ratio
    match = re.fullmatch(
        r'\s*(\d*\.?\d+)\s*:\s*(\d*\.?\d+)'
        r'\s*(.*?)\s*',
        raw
    )

    if match:
        d = float(match.group(1))
        a = float(match.group(2))
        qualifier = match.group(3).strip()

        if qualifier in weight_qualifiers and d > 0:
            return pd.Series(
                ["safe_colon_ratio", d, a, a / d]
            )

        return pd.Series(
            ["ambiguous_colon_ratio", d, a, np.nan]
        )

    # Handle simple slash notation such as 1/1.5 or 2/1
    slash = re.fullmatch(
        r'\s*(\d*\.?\d+)\s*/\s*(\d*\.?\d+)'
        r'\s*(.*?)\s*',
        raw
    )

    if slash:
        d = float(slash.group(1))
        a = float(slash.group(2))
        qualifier = slash.group(3).strip()

        if qualifier in weight_qualifiers and d > 0:
            return pd.Series(
                ["safe_slash_ratio", d, a, a / d]
            )

        return pd.Series(
            ["ambiguous_slash_ratio", d, a, np.nan]
        )

    return pd.Series(
        ["unresolved", np.nan, np.nan, np.nan]
    )


da_parsed = strict_molecular["d_a_ratio"].apply(
    classify_da_ratio
)

da_parsed.columns = [
    "da_parse_class",
    "donor_part",
    "acceptor_part",
    "acceptor_to_donor_ratio",
]


da_parse_summary = (
    da_parsed["da_parse_class"]
    .value_counts(dropna=False)
    .rename_axis("parse_class")
    .reset_index(name="records")
)

da_parse_summary["percent_of_all_records"] = (
    100 * da_parse_summary["records"] / len(strict_molecular)
).round(2)

display(da_parse_summary)


safe_mask = da_parsed["da_parse_class"].isin(
    ["safe_colon_ratio", "safe_slash_ratio"]
)

print(
    f"Safely recoverable among filled D:A ratios: "
    f"{100 * safe_mask.sum() / strict_molecular['d_a_ratio'].notna().sum():.2f}%"
)

print("\nExamples not conservatively resolved:")

display(
    strict_molecular.loc[
        strict_molecular["d_a_ratio"].notna() & ~safe_mask,
        "d_a_ratio"
    ]
    .value_counts()
    .head(40)
    .rename_axis("raw_value")
    .reset_index(name="count")
)

,parse_class,records,percent_of_all_records
0,safe_colon_ratio,10680,49.17
1,missing,10537,48.51
2,ambiguous_colon_ratio,344,1.58
3,unresolved,118,0.54
4,safe_slash_ratio,41,0.19


Safely recoverable among filled D:A ratios: 95.87%

Examples not conservatively resolved:


,raw_value,count
0,1:1.5 wt%,31
1,1:1 wt%,25
2,1:1:0,20
3,1:1.2:0,16
4,1:0:1,13
5,1:2 wt%,12
6,1:0:1.2,11
7,1:1.2 wt%,10
8,P3HT:PCBM 10:8 (1.25:1 w/w),8
9,P3HT:PCBM = 1:0.8 (w/w),7


### 1.8 Additive-loading basis and recoverability

Additive loading is reported using several physically different conventions,
including volume fraction, weight fraction, solution concentration, molar
fraction, equivalents, and component ratios.

These quantities are not treated as interchangeable. The raw strings are
therefore classified by measurement basis before any numerical values are
extracted.

Records containing more than one measurement basis are explicitly identified
as mixed annotations rather than assigned to a single unit category.

In [10]:
# ------------------------------------------------------------------
# Improved additive-loading unit classification
# ------------------------------------------------------------------

def normalize_additive_text(value):
    """Normalize typography only; do not change scientific meaning."""
    if pd.isna(value):
        return None

    v = str(value).lower().strip()

    # Normalize Unicode characters commonly found in literature text
    v = (
        v.replace("μ", "u")
         .replace("µ", "u")
         .replace("−", "-")
         .replace("–", "-")
         .replace("—", "-")
    )

    # Collapse repeated whitespace
    v = re.sub(r"\s+", " ", v)

    return v


def classify_additive_basis(value):

    v = normalize_additive_text(value)

    if v is None:
        return "missing"

    # Detect scientifically distinct bases
    has_mgml = bool(
        re.search(
            r"mg\s*(?:/|\s)\s*ml(?:\s*(?:\^?\s*-?\s*1))?",
            v
        )
    )

    has_wt = bool(
        re.search(r"\bwt\.?\s*%", v)
    )

    has_vol = bool(
        re.search(r"\bvol\.?\s*%|\bv\s*/\s*v\b", v)
    )

    has_mol = bool(
        re.search(r"\bmol\.?\s*%", v)
    )

    has_equiv = bool(
        re.search(r"\beq\.?\b|\bequiv", v)
    )

    specific_bases = [
        has_mgml,
        has_wt,
        has_vol,
        has_mol,
        has_equiv,
    ]

    # More than one scientifically distinct unit basis
    if sum(specific_bases) > 1:
        return "mixed-unit annotation"

    if has_mgml:
        return "mg/mL"

    if has_wt:
        return "wt%"

    if has_vol:
        return "vol% or v/v"

    if has_mol:
        return "mol%"

    if has_equiv:
        return "equivalents"

    # Ratios such as additive:donor = 1:1
    if ":" in v and re.search(r"\d", v):
        return "component ratio"

    if "%" in v:
        return "unspecified %"

    if re.fullmatch(r"[+-]?\d*\.?\d+", v):
        return "unitless numeric"

    return "other / complex"


additive_raw = strict_molecular["additive_ratio"]

additive_basis = additive_raw.apply(classify_additive_basis)

additive_basis_summary = (
    additive_basis
    .value_counts(dropna=False)
    .rename_axis("loading_basis")
    .reset_index(name="records")
)

additive_basis_summary["percent_of_all_records"] = (
    100 * additive_basis_summary["records"] / len(strict_molecular)
).round(2)

additive_basis_summary["percent_of_filled_records"] = (
    100
    * additive_basis_summary["records"]
    / strict_molecular["additive_ratio"].notna().sum()
).round(2)

display(additive_basis_summary)



# ------------------------------------------------------------------
# Inspect examples from every populated measurement basis
# ------------------------------------------------------------------

for basis in additive_basis_summary["loading_basis"]:

    if basis == "missing":
        continue

    print("\n" + "=" * 75)
    print(basis)
    print("=" * 75)

    examples = (
        additive_raw[
            additive_basis == basis
        ]
        .dropna()
        .astype(str)
        .value_counts()
        .head(20)
        .rename_axis("raw_value")
        .reset_index(name="count")
    )

    display(examples)


    # ------------------------------------------------------------------
# Examine explicitly zero-reported additive loadings
# ------------------------------------------------------------------

def first_numeric_value(value):

    if pd.isna(value):
        return np.nan

    v = normalize_additive_text(value)

    match = re.search(r"[+-]?\d*\.?\d+", v)

    if match:
        return float(match.group())

    return np.nan


additive_numeric_candidate = additive_raw.apply(first_numeric_value)

zero_loading_mask = (
    additive_raw.notna()
    & additive_numeric_candidate.eq(0)
)

print(
    "Filled additive-loading records :",
    f"{additive_raw.notna().sum():,}"
)

print(
    "Explicit zero-valued loadings   :",
    f"{zero_loading_mask.sum():,}"
)

print("\nZero-valued records by measurement basis:")

display(
    additive_basis[zero_loading_mask]
    .value_counts()
    .rename_axis("loading_basis")
    .reset_index(name="records")
)

,loading_basis,records,percent_of_all_records,percent_of_filled_records
0,missing,16670,76.75,330.10
1,vol% or v/v,2317,10.67,45.88
2,unspecified %,1748,8.05,34.61
3,wt%,569,2.62,11.27
4,mg/mL,175,0.81,3.47
5,unitless numeric,125,0.58,2.48
6,component ratio,64,0.29,1.27
7,equivalents,19,0.09,0.38
8,other / complex,14,0.06,0.28
9,mixed-unit annotation,13,0.06,0.26



vol% or v/v


,raw_value,count
0,0.5 vol%,496
1,3 vol%,370
2,0.5% v/v,157
3,1 vol%,97
4,3% v/v,95
5,2 vol%,93
6,0.5 vol %,67
7,3 vol% DIO,65
8,0.25 vol%,55
9,0 vol%,48



unspecified %


,raw_value,count
0,0.5%,381
1,3%,357
2,1%,189
3,2%,133
4,5%,72
5,0.25%,50
6,0%,46
7,10%,36
8,0.2%,30
9,4%,30


...

,raw_value,count
0,20% v/v (2 mg/mL aqueous solution),4
1,5 mg mL−1 DIB; 0.3% v/v DIM,1
2,0.7 vol% CN + 3 wt% N2200,1
3,1 vol% CN + 3 wt% N2200,1
4,2 wt% CQDs; 40 vol.% IPA,1
5,0.8 wt%; 1.0 vol%,1
6,1.25 wt%; 1.0 vol%,1
7,1.7 wt%; 1.0 vol%,1
8,CN 0.25% v/v; DHT 10 wt%,1
9,5.0 wt % DBS + 0.5 vol % CN,1



mol%


,raw_value,count
0,5 mol%,3
1,0.25 mol %,2
2,0 mol%,1


Filled additive-loading records : 5,050
Explicit zero-valued loadings   : 158

Zero-valued records by measurement basis:


,loading_basis,records
0,vol% or v/v,63
1,unspecified %,55
2,wt%,36
3,mg/mL,2
4,component ratio,1
5,mol%,1


In [11]:
# ------------------------------------------------------------------
# Correct additive-basis summary
# ------------------------------------------------------------------

filled_additive_n = strict_molecular["additive_ratio"].notna().sum()

additive_basis_summary = (
    additive_basis
    .value_counts(dropna=False)
    .rename_axis("loading_basis")
    .reset_index(name="records")
)

additive_basis_summary["percent_of_all_records"] = (
    100 * additive_basis_summary["records"] / len(strict_molecular)
).round(2)

# Percentage among populated additive-loading records is meaningful
# only for non-missing categories.
additive_basis_summary["percent_of_filled_records"] = np.where(
    additive_basis_summary["loading_basis"].eq("missing"),
    np.nan,
    (
        100
        * additive_basis_summary["records"]
        / filled_additive_n
    ).round(2)
)

display(additive_basis_summary)

,loading_basis,records,percent_of_all_records,percent_of_filled_records
0,missing,16670,76.75,NaN
1,vol% or v/v,2317,10.67,45.88
2,unspecified %,1748,8.05,34.61
3,wt%,569,2.62,11.27
4,mg/mL,175,0.81,3.47
5,unitless numeric,125,0.58,2.48
6,component ratio,64,0.29,1.27
7,equivalents,19,0.09,0.38
8,other / complex,14,0.06,0.28
9,mixed-unit annotation,13,0.06,0.26


## 2. Chemical Identity Audit

### 2.1 Consistency of material labels, canonical labels, and SMILES

Reliable chemical generalization requires a defensible definition of material
identity.

The OPV-DB release contains three relevant identity layers:

1. literature-extracted material labels,
2. database-normalized canonical labels,
3. molecular SMILES representations.

The relationships among these layers are audited before molecular
canonicalization. Particular attention is given to:

- one raw name mapping to multiple canonical identities,
- one raw name mapping to multiple SMILES,
- several literature names mapping to one canonical material,
- one canonical material mapping to multiple molecular representations,
- and identical SMILES appearing under multiple canonical labels.

These cases can create hidden chemical overlap between training and test sets
if material names alone are used to define generalization.

In [12]:
# ------------------------------------------------------------------
# Material-identity mapping audit
# ------------------------------------------------------------------

identity_fields = {
    "donor": {
        "raw": "donor",
        "canonical": "donor_canonical",
        "smiles": "donor_smiles",
    },
    "acceptor": {
        "raw": "acceptor",
        "canonical": "acceptor_canonical",
        "smiles": "acceptor_smiles",
    },
}


def material_mapping_audit(df, role, fields):

    raw = fields["raw"]
    canonical = fields["canonical"]
    smiles = fields["smiles"]

    tmp = df[[raw, canonical, smiles]].copy()

    # Normalize whitespace only for audit purposes.
    # The raw dataframe is not modified.
    for col in [raw, canonical, smiles]:
        tmp[col] = (
            tmp[col]
            .astype("string")
            .str.strip()
        )

    summary = {
        "role": role,
        "records": len(tmp),
        "unique_raw_labels": tmp[raw].nunique(dropna=True),
        "unique_canonical_labels": tmp[canonical].nunique(dropna=True),
        "unique_smiles_strings": tmp[smiles].nunique(dropna=True),
    }

    # One raw label -> number of canonical names / SMILES
    raw_mapping = (
        tmp.groupby(raw, dropna=False)
        .agg(
            n_canonical=(canonical, "nunique"),
            n_smiles=(smiles, "nunique"),
            records=(raw, "size"),
        )
        .reset_index()
    )

    # One canonical label -> number of raw aliases / SMILES
    canonical_mapping = (
        tmp.groupby(canonical, dropna=False)
        .agg(
            n_raw_labels=(raw, "nunique"),
            n_smiles=(smiles, "nunique"),
            records=(canonical, "size"),
        )
        .reset_index()
    )

    # One SMILES -> number of canonical labels / raw labels
    smiles_mapping = (
        tmp.groupby(smiles, dropna=False)
        .agg(
            n_canonical_labels=(canonical, "nunique"),
            n_raw_labels=(raw, "nunique"),
            records=(smiles, "size"),
        )
        .reset_index()
    )

    return summary, raw_mapping, canonical_mapping, smiles_mapping


identity_results = {}
summary_rows = []

for role, fields in identity_fields.items():

    results = material_mapping_audit(
        strict_molecular,
        role,
        fields
    )

    identity_results[role] = results
    summary_rows.append(results[0])


identity_summary = pd.DataFrame(summary_rows)

display(identity_summary)



# ------------------------------------------------------------------
# Quantify potentially ambiguous mappings
# ------------------------------------------------------------------

ambiguity_rows = []

for role, (
    summary,
    raw_mapping,
    canonical_mapping,
    smiles_mapping
) in identity_results.items():

    ambiguity_rows.append({
        "role": role,

        "raw_labels_with_multiple_canonical_labels":
            int((raw_mapping["n_canonical"] > 1).sum()),

        "raw_labels_with_multiple_smiles":
            int((raw_mapping["n_smiles"] > 1).sum()),

        "canonical_labels_with_multiple_raw_aliases":
            int((canonical_mapping["n_raw_labels"] > 1).sum()),

        "canonical_labels_with_multiple_smiles":
            int((canonical_mapping["n_smiles"] > 1).sum()),

        "smiles_shared_by_multiple_canonical_labels":
            int((smiles_mapping["n_canonical_labels"] > 1).sum()),
    })


identity_ambiguity = pd.DataFrame(ambiguity_rows)

display(identity_ambiguity)

,role,records,unique_raw_labels,unique_canonical_labels,unique_smiles_strings
0,donor,21720,2178,1903,2001
1,acceptor,21720,1583,1365,1457


,role,raw_labels_with_multiple_canonical_labels,raw_labels_with_multiple_smiles,canonical_labels_with_multiple_raw_aliases,canonical_labels_with_multiple_smiles,smiles_shared_by_multiple_canonical_labels
0,donor,26,113,191,87,24
1,acceptor,13,91,131,81,18


### 2.2 Examination of ambiguous material mappings

Material mappings identified as one-to-many or many-to-one are inspected
before molecular graph canonicalization.

Ambiguity may arise from legitimate aliases, alternative polymer repeat-unit
representations, stereochemical or charge conventions, literature naming
differences, or incorrect/incomplete mappings.

These cases are not automatically corrected. Their structure is first
characterized so that later train/test grouping does not rely on an
over-simplified material identifier.

In [13]:
# ------------------------------------------------------------------
# Build compact tables of ambiguous mappings
# ------------------------------------------------------------------

def ambiguous_mapping_examples(df, raw, canonical, smiles):

    tmp = df[[raw, canonical, smiles]].copy()

    for col in [raw, canonical, smiles]:
        tmp[col] = tmp[col].astype("string").str.strip()

    # Canonical labels represented by multiple SMILES
    canon_counts = (
        tmp.groupby(canonical)[smiles]
        .nunique()
    )

    ambiguous_canon = canon_counts[
        canon_counts > 1
    ].sort_values(ascending=False)

    canon_examples = []

    for label in ambiguous_canon.head(20).index:

        subset = tmp[tmp[canonical] == label]

        canon_examples.append({
            "canonical_label": label,
            "records": len(subset),
            "n_raw_labels": subset[raw].nunique(),
            "n_smiles": subset[smiles].nunique(),
            "raw_labels":
                " | ".join(
                    subset[raw]
                    .dropna()
                    .unique()[:6]
                ),
        })

    # Identical SMILES represented by multiple canonical labels
    smiles_counts = (
        tmp.groupby(smiles)[canonical]
        .nunique()
    )

    shared_smiles = smiles_counts[
        smiles_counts > 1
    ].sort_values(ascending=False)

    smiles_examples = []

    for smi in shared_smiles.head(20).index:

        subset = tmp[tmp[smiles] == smi]

        smiles_examples.append({
            "records": len(subset),
            "n_canonical_labels":
                subset[canonical].nunique(),
            "canonical_labels":
                " | ".join(
                    subset[canonical]
                    .dropna()
                    .unique()[:8]
                ),
            "smiles_preview":
                str(smi)[:100],
        })

    return (
        pd.DataFrame(canon_examples),
        pd.DataFrame(smiles_examples)
    )


donor_canon_ambig, donor_smiles_shared = (
    ambiguous_mapping_examples(
        strict_molecular,
        "donor",
        "donor_canonical",
        "donor_smiles"
    )
)

acceptor_canon_ambig, acceptor_smiles_shared = (
    ambiguous_mapping_examples(
        strict_molecular,
        "acceptor",
        "acceptor_canonical",
        "acceptor_smiles"
    )
)


print("DONORS — canonical labels with multiple SMILES")
display(donor_canon_ambig)

print("\nDONORS — identical SMILES under multiple canonical labels")
display(donor_smiles_shared)

print("\nACCEPTORS — canonical labels with multiple SMILES")
display(acceptor_canon_ambig)

print("\nACCEPTORS — identical SMILES under multiple canonical labels")
display(acceptor_smiles_shared)

DONORS — canonical labels with multiple SMILES


,canonical_label,records,n_raw_labels,n_smiles,raw_labels
0,P1,311,2,12,P1 | PIDTDPP1
1,P3,162,2,6,P3 | P1
2,PTB7-Th,1721,13,4,PTB7-Th | PBDTT-TT-F | PTB7-TH | P1 | P3 | PTB...
3,PBDB-T,1639,7,4,PBDB-T | PBDBT | PB1 | PBDTBDD-T | PBDB-TT0 | PM5
4,P3HT,1836,6,3,P3HT | rr-P3HT | P3HT-end-H/Br | P3HT-end-OXD ...
5,P2,6,3,3,P3 | P2 | P(DKPP-TPTH)
6,PBN-S,5,1,3,PBN-S
7,PT1,5,1,3,PT1
8,PTB7,649,1,3,PTB7
9,PPDT2FBT,41,1,3,PPDT2FBT



DONORS — identical SMILES under multiple canonical labels


,records,n_canonical_labels,canonical_labels,smiles_preview
0,1573,5,P3HT | T7Bz-Bz2 | T7Bz-Dp2 | T7Bz-TSO2 | T7SBz...,CCCCCCc1ccsc1
1,21,5,PDBT-T1 | 2.1 | 2.2 | 2.8 | 2.11,CCCCCCCCc1ccc(-c2c3sc4cc(-c5ccc(-c6sc(-c7cccs7...
2,6,4,P3TEA | 2.13 | 2.14 | 3.1,CCCCCCCCCCC(CCCCCCCC)COC(=O)c1cc(-c2c(F)c(F)c(...
3,5286,3,PM6 | 2.15 | PM1,COC(=O)C(C)(C)c1cc(C)cc(-c2sc3c4sc5cc(/C=C6/C(...
4,3,2,P(BDT-TBTF) | PBDT-TBT-F,CCCCC(CC)COc1c2cc(-c3cc(/C=C/c4sc(-c5c(F)c(F)c...
5,8,2,PBDB-ST | PBDT-ST1,CCCCC(CC)CSc1ccc(-c2c3cc(-c4ccc(-c5sc(-c6cccs6...
6,1470,2,PBDB-T | 2.6,CCCCC(CC)Cc1ccc(-c2c3cc(-c4ccc(-c5sc(-c6cccs6)...
7,88,2,PBDB-T-2Cl | PBDB-T2Cl,CCCCC(CC)Cc1sc(-c2c3cc(-c4ccc(-c5sc(-c6cccs6)c...
8,84,2,J52 | PBZ,CCCCCCCCC(CCCCCC)Cn1nc2c(-c3cccs3)c(F)c(F)c(-c...
9,11,2,P5 | P7,CCCCCCCCC(CCCCCC)Cn1c2ccccc2c2ccc(-c3c4cc(-c5c...



ACCEPTORS — canonical labels with multiple SMILES


,canonical_label,records,n_raw_labels,n_smiles,raw_labels
0,NNFA,5,1,5,NNFA
1,Y5,17,3,4,Y5 | BTP | Y5-2BO
2,BTA3,24,2,3,BTA3 | BTA3-4F
3,IDSe-T-IC,3,1,3,IDSe-T-IC
4,IEICO,29,1,3,IEICO
5,L8-BO,1862,3,3,L8-BO | L8BO | L8-Bo
6,Y6,3411,4,3,Y6 | N3 | BTP-4F | BTP-4F-12
7,A1,9,1,2,A1
8,BP-4F,6,1,2,BP-4F
9,BP-PDI4,2,1,2,BP-PDI4



ACCEPTORS — identical SMILES under multiple canonical labels


,records,n_canonical_labels,canonical_labels,smiles_preview
0,3627,5,PC71BM | PC<sub>71</sub>BM | Zn2(ZnTCPP)-PC71B...,COC(=O)CCCC1(c2ccccc2)C23c4c5ccc6c7cc8c9c%10c(...
1,869,5,PC61BM | P61BM | (3BS)2-SiPc | PC[61]BM | PC<s...,COC(=O)CCCC1(c2ccccc2)C23c4c5c6c7c8c9c(c%10c%1...
2,175,3,Y6-BO | Y6BO | Y6(BO),CCCCCCCCCCCc1c(/C=C2\C(=O)c3cc(F)c(F)cc3C2=C(C...
3,20,2,L8-BO-X | L8BO-X,CCCCCCC(CCCC)CCC1=C(/C=C2\C(=O)C3=CC(F)=C(F)C=...
4,44,2,PNDI2HD-T | P(NDI2HD-T),CCCCCCCCC(CCCCCC)CN1C(=O)c2ccc3c4c(c(-c5cccs5)...
5,3,2,IFTIC | IFT-IC,CCCCCCCCC1(CCCCCCCC)c2cc(-c3ccc(/C=C4\C(=O)c5c...
6,113,2,Y7 | Y6-2Cl,CCCCCCCCCCCc1c(/C=C2\C(=O)c3cc(Cl)c(Cl)cc3C2=C...
7,226,2,Y6 | N3,CCCCCCCCCCCC1=C(/C=C2\C(=O)C3=CC(F)=C(F)C=C3C2...
8,4,2,PPDI-DTT | P(PDI-DTT),CCCCCCCCCCCCC(CCCCCCCCCC)CN1C(=O)c2ccc3c4ccc5c...
9,47,2,PY-T | PYT,CCCCCCCCCCCCC(CCCCCCCCCC)Cn1c2c3sc(/C=C4\C(=O)...


### 2.3 Molecular-graph canonicalization and structure-level identity

Raw SMILES strings are textual molecular representations and are not
necessarily unique: equivalent molecular graphs can be written using
different SMILES strings.

RDKit is therefore used to parse each donor and acceptor SMILES and generate
a canonical isomeric SMILES representation. This provides a stronger
structure-level identity key for later overlap and train/test leakage
analysis.

Canonicalization is used only to normalize the molecular graph represented
by each SMILES. It is not assumed to resolve all polymer-identity issues,
such as alternative repeat-unit conventions, end-group definitions, or
different representations of the same macromolecular material.

In [14]:
# ------------------------------------------------------------------
# RDKit availability and molecular-graph canonicalization
# ------------------------------------------------------------------

from rdkit import Chem, RDLogger

# Prevent long parser warnings from flooding the notebook.
# Invalid structures will still be counted explicitly below.
RDLogger.DisableLog("rdApp.error")


def canonicalize_smiles(smiles):
    """
    Parse a SMILES string with RDKit and return canonical isomeric SMILES.

    No fragment removal, charge neutralization, or structure correction is
    performed. We preserve the molecular graph supplied by the dataset.
    """
    if pd.isna(smiles):
        return pd.NA

    smiles = str(smiles).strip()

    if not smiles:
        return pd.NA

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return pd.NA

    return Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )


# Work on a copy; raw dataset remains untouched
molecular_audit = strict_molecular.copy()

molecular_audit["donor_graph_smiles"] = (
    molecular_audit["donor_smiles"]
    .apply(canonicalize_smiles)
)

molecular_audit["acceptor_graph_smiles"] = (
    molecular_audit["acceptor_smiles"]
    .apply(canonicalize_smiles)
)


# ------------------------------------------------------------------
# Parsing success
# ------------------------------------------------------------------

structure_parse_summary = pd.DataFrame({
    "role": ["donor", "acceptor"],

    "records": [
        len(molecular_audit),
        len(molecular_audit)
    ],

    "parsed_records": [
        molecular_audit["donor_graph_smiles"].notna().sum(),
        molecular_audit["acceptor_graph_smiles"].notna().sum()
    ],

    "failed_records": [
        molecular_audit["donor_graph_smiles"].isna().sum(),
        molecular_audit["acceptor_graph_smiles"].isna().sum()
    ],

    "unique_raw_smiles": [
        molecular_audit["donor_smiles"].nunique(),
        molecular_audit["acceptor_smiles"].nunique()
    ],

    "unique_canonical_graphs": [
        molecular_audit["donor_graph_smiles"].nunique(),
        molecular_audit["acceptor_graph_smiles"].nunique()
    ]
})

structure_parse_summary["parse_success_percent"] = (
    100
    * structure_parse_summary["parsed_records"]
    / structure_parse_summary["records"]
).round(3)

display(structure_parse_summary)



# ------------------------------------------------------------------
# How much textual SMILES redundancy disappears after canonicalization?
# ------------------------------------------------------------------

for role in ["donor", "acceptor"]:

    raw_col = f"{role}_smiles"
    graph_col = f"{role}_graph_smiles"

    raw_unique = molecular_audit[raw_col].nunique()
    graph_unique = molecular_audit[graph_col].nunique()

    print(
        f"{role.capitalize():8s}: "
        f"{raw_unique:,} raw SMILES -> "
        f"{graph_unique:,} canonicalized molecular graphs "
        f"(reduction = {raw_unique - graph_unique:,})"
    )

,role,records,parsed_records,failed_records,unique_raw_smiles,unique_canonical_graphs,parse_success_percent
0,donor,21720,21720,0,2001,1987,100.0
1,acceptor,21720,21720,0,1457,1436,100.0


Donor   : 2,001 raw SMILES -> 1,987 canonicalized molecular graphs (reduction = 14)
Acceptor: 1,457 raw SMILES -> 1,436 canonicalized molecular graphs (reduction = 21)


In [15]:
# ------------------------------------------------------------------
# Label ↔ canonicalized molecular-graph consistency
# ------------------------------------------------------------------

graph_identity_rows = []

for role in ["donor", "acceptor"]:

    canonical_label = f"{role}_canonical"
    graph_smiles = f"{role}_graph_smiles"

    tmp = molecular_audit[
        [canonical_label, graph_smiles]
    ].dropna().copy()

    # One database canonical label -> multiple molecular graphs
    label_to_graph = (
        tmp.groupby(canonical_label)[graph_smiles]
        .nunique()
    )

    # One molecular graph -> multiple database canonical labels
    graph_to_label = (
        tmp.groupby(graph_smiles)[canonical_label]
        .nunique()
    )

    graph_identity_rows.append({
        "role": role,

        "canonical_labels":
            tmp[canonical_label].nunique(),

        "canonicalized_graphs":
            tmp[graph_smiles].nunique(),

        "labels_mapping_to_multiple_graphs":
            int((label_to_graph > 1).sum()),

        "graphs_mapping_to_multiple_labels":
            int((graph_to_label > 1).sum()),
    })


graph_identity_summary = pd.DataFrame(graph_identity_rows)

display(graph_identity_summary)

,role,canonical_labels,canonicalized_graphs,labels_mapping_to_multiple_graphs,graphs_mapping_to_multiple_labels
0,donor,1903,1967,73,24
1,acceptor,1365,1409,64,19


### 2.4 Publication-scoped material identity

Generic material labels such as `P1`, `P2`, or `A1` may be reused across
independent publications for chemically unrelated compounds.

To distinguish cross-publication label reuse from within-publication identity
inconsistency, canonical material labels are evaluated both globally and
within source DOI.

If a canonical label maps to several molecular graphs globally but usually
maps to only one graph within each DOI, the ambiguity primarily reflects
publication-local naming rather than structural inconsistency within a
particular study.

In [16]:
# ------------------------------------------------------------------
# Does DOI context resolve canonical-label ambiguity?
# ------------------------------------------------------------------

doi_identity_rows = []

for role in ["donor", "acceptor"]:

    label_col = f"{role}_canonical"
    graph_col = f"{role}_graph_smiles"

    tmp = molecular_audit[
        ["doi_norm", label_col, graph_col]
    ].dropna().copy()

    # --------------------------------------------------------------
    # Global: one canonical label -> how many molecular graphs?
    # --------------------------------------------------------------

    global_mapping = (
        tmp.groupby(label_col)[graph_col]
        .nunique()
    )

    global_ambiguous_labels = int(
        (global_mapping > 1).sum()
    )

    # --------------------------------------------------------------
    # DOI-scoped:
    # one (DOI, canonical label) -> how many molecular graphs?
    # --------------------------------------------------------------

    doi_mapping = (
        tmp.groupby(
            ["doi_norm", label_col]
        )[graph_col]
        .nunique()
    )

    doi_ambiguous_groups = int(
        (doi_mapping > 1).sum()
    )

    total_doi_label_groups = len(doi_mapping)

    doi_identity_rows.append({
        "role": role,
        "global_canonical_labels":
            tmp[label_col].nunique(),

        "globally_ambiguous_labels":
            global_ambiguous_labels,

        "doi_label_groups":
            total_doi_label_groups,

        "doi_label_groups_with_multiple_graphs":
            doi_ambiguous_groups,

        "doi_scoped_ambiguity_percent":
            round(
                100 * doi_ambiguous_groups /
                total_doi_label_groups,
                3
            ),
    })


doi_identity_summary = pd.DataFrame(
    doi_identity_rows
)

display(doi_identity_summary)



# ------------------------------------------------------------------
# Inspect remaining within-publication conflicts
# ------------------------------------------------------------------

for role in ["donor", "acceptor"]:

    label_col = f"{role}_canonical"
    graph_col = f"{role}_graph_smiles"

    tmp = molecular_audit[
        [
            "doi_norm",
            role,
            label_col,
            graph_col
        ]
    ].dropna().copy()

    mapping = (
        tmp.groupby(
            ["doi_norm", label_col]
        )[graph_col]
        .nunique()
        .reset_index(name="n_graphs")
    )

    conflicts = mapping[
        mapping["n_graphs"] > 1
    ].sort_values(
        "n_graphs",
        ascending=False
    )

    print("\n" + "=" * 80)
    print(
        f"{role.upper()} — "
        "within-DOI labels mapping to multiple graphs"
    )
    print("=" * 80)

    print(
        f"Conflicting DOI-label groups: "
        f"{len(conflicts):,}"
    )

    display(
        conflicts.head(25)
    )

,role,global_canonical_labels,globally_ambiguous_labels,doi_label_groups,doi_label_groups_with_multiple_graphs,doi_scoped_ambiguity_percent
0,donor,1903,73,7791,52,0.667
1,acceptor,1365,64,8006,61,0.762



DONOR — within-DOI labels mapping to multiple graphs
Conflicting DOI-label groups: 52


,doi_norm,donor_canonical,n_graphs
21,10.1002/adfm.201504153,P3HT,2
63,10.1002/adfm.201808828,L810,2
405,10.1002/adma.201500577,PSEHTT,2
420,10.1002/adma.201503801,PSEHTT,2
469,10.1002/adma.201606396,PTzBI,2
477,10.1002/adma.201702291,PTZ1,2
1027,10.1002/aenm.201701125,PTPTI-T100,2
1041,10.1002/aenm.201701674,PTFB-P,2
1065,10.1002/aenm.201801214,PBDT-TT,2
1098,10.1002/aenm.201903298,PTQ10,2



ACCEPTOR — within-DOI labels mapping to multiple graphs
Conflicting DOI-label groups: 61


,doi_norm,acceptor_canonical,n_graphs
6716,10.1039/c8qm00238j,NNFA,5
6478,10.1039/c6ee00315j,IDSe-T-IC,3
48,10.1002/adfm.201704507,BTA3,2
47,10.1002/adfm.201704507,BTA2,2
46,10.1002/adfm.201704507,BTA1,2
516,10.1002/adma.201703080,IEICO-4Cl,2
521,10.1002/adma.201703973,ITIC,2
552,10.1002/adma.201707508,F-M,2
555,10.1002/adma.201800052,IDT6CN-M,2
556,10.1002/adma.201800052,IDT6CN-Th,2


## 3. Photovoltaic Target Integrity

### 3.1 Verification of documented quality-control criteria

The strict molecular benchmark was released after OPV-DB quality-control
filtering. Rather than inventing new physical thresholds, the documented
quality-tier rules supplied with the database are inspected first.

The subsequent checks independently verify that the strict molecular cohort
satisfies the authors' stated photovoltaic consistency requirements before
additional analysis is performed.

In [17]:
import json

# ------------------------------------------------------------------
# Inspect the OPV-DB quality-tier rules supplied with the release
# ------------------------------------------------------------------

with open(
    opvdb_files["quality_rules"],
    "r",
    encoding="utf-8"
) as f:
    quality_rules = json.load(f)

quality_rules

{'generated_for': 'opvdb_scidata_release_v1_20260629',
 'versioning': {'missing_values': 'Missing values are represented as empty cells in CSV exports.'},
 'tiers': {'full': 'Publication-facing single-junction OPV device records after missing-value cleanup and exclusion of physically non-single-cell, module, tandem/subcell, silicon-hybrid, perovskite/lead-halide hybrid, theoretical/computational/device-simulation, photodiode, Schottky, Voc >2.0 V, and PCE-recomputation error >50% rows.',
  'strict_performance_benchmark': 'Requires normalized DOI, reported donor label, reported acceptor label, PCE, Voc, Jsc, FF, internal source-scope checks, single-junction public-scope checks excluding module, tandem/subcell, silicon-hybrid, perovskite/lead-halide hybrid, theoretical/computational/device-simulation rows, physical-range checks including Voc <=2.0 V, exact duplicate core-record removal on both reported and canonical material labels with core metrics, and PCE recomputation relative error 

### 3.2 Independent verification of strict molecular benchmark integrity

The strict molecular benchmark is independently checked against the explicit
criteria documented in the OPV-DB quality-tier rules.

The verification focuses only on criteria that can be reproduced directly
from the public release:

- completeness of required provenance, material, structure, and photovoltaic
  fields;
- consistency of the supplied recomputed PCE with
  `Voc × Jsc × FF / 100`;
- compliance with the documented PCE relative-error threshold of 2%;
- compliance with the documented `Voc ≤ 2.0 V` requirement;
- and absence of exact duplicate rows.

This is a verification of the released benchmark, not a new filtering step.

In [18]:
# ------------------------------------------------------------------
# Required-field completeness
# ------------------------------------------------------------------

required_fields = [
    "doi_norm",
    "donor",
    "acceptor",
    "donor_smiles",
    "acceptor_smiles",
    "voc",
    "jsc",
    "ff",
    "pce",
    "pce_recomputed",
    "pce_relative_error_percent",
]

required_field_check = pd.DataFrame({
    "field": required_fields,
    "missing_records": [
        strict_molecular[field].isna().sum()
        for field in required_fields
    ],
})

required_field_check["complete"] = (
    required_field_check["missing_records"] == 0
)

display(required_field_check)



# ------------------------------------------------------------------
# Independently recompute PCE
# ------------------------------------------------------------------

pce_formula_check = (
    strict_molecular["voc"]
    * strict_molecular["jsc"]
    * strict_molecular["ff"]
    / 100
)

pce_formula_difference = (
    pce_formula_check
    - strict_molecular["pce_recomputed"]
).abs()


# ------------------------------------------------------------------
# Documented benchmark criteria
# ------------------------------------------------------------------

integrity_checks = pd.DataFrame({

    "check": [
        "Required fields complete",
        "PCE recomputation matches formula",
        "PCE relative error <= 2%",
        "Voc <= 2.0 V",
        "Exact full-row duplicates absent",
    ],

    "violating_records": [
        int(
            strict_molecular[
                required_fields
            ].isna().any(axis=1).sum()
        ),

        int(
            (pce_formula_difference > 1e-10).sum()
        ),

        int(
            (
                strict_molecular[
                    "pce_relative_error_percent"
                ] > 2
            ).sum()
        ),

        int(
            (strict_molecular["voc"] > 2.0).sum()
        ),

        int(
            strict_molecular.duplicated().sum()
        ),
    ]
})

integrity_checks["passed"] = (
    integrity_checks["violating_records"] == 0
)

display(integrity_checks)



# ------------------------------------------------------------------
# Numerical summary of PCE-consistency diagnostics
# ------------------------------------------------------------------

pce_consistency_summary = pd.DataFrame({
    "metric": [
        "Maximum |independent PCE recomputation - supplied recomputation|",
        "Median reported-vs-recomputed relative error (%)",
        "95th percentile relative error (%)",
        "Maximum relative error (%)",
    ],

    "value": [
        pce_formula_difference.max(),
        strict_molecular[
            "pce_relative_error_percent"
        ].median(),
        strict_molecular[
            "pce_relative_error_percent"
        ].quantile(0.95),
        strict_molecular[
            "pce_relative_error_percent"
        ].max(),
    ]
})

display(pce_consistency_summary)

,field,missing_records,complete
0,doi_norm,0,True
1,donor,0,True
2,acceptor,0,True
3,donor_smiles,0,True
4,acceptor_smiles,0,True
5,voc,0,True
6,jsc,0,True
7,ff,0,True
8,pce,0,True
9,pce_recomputed,0,True


,check,violating_records,passed
0,Required fields complete,0,True
1,PCE recomputation matches formula,0,True
2,PCE relative error <= 2%,0,True
3,Voc <= 2.0 V,0,True
4,Exact full-row duplicates absent,0,True


,metric,value
0,Maximum |independent PCE recomputation - suppl...,7.105427e-15
1,Median reported-vs-recomputed relative error (%),1.533060e-01
2,95th percentile relative error (%),1.380337e+00
3,Maximum relative error (%),2.000000e+00


### 3.3 Distribution and range of photovoltaic targets

After verifying the documented benchmark criteria, the distributions of the
four photovoltaic performance variables are examined without applying
additional filtering.

The purpose is to characterize:

- central tendency and spread,
- low- and high-performance tails,
- potential floor or ceiling effects,
- and the frequency of non-positive values.

Extreme observations are inspected rather than automatically removed because
unusual performance values may represent legitimate experimental devices.

In [20]:
# ------------------------------------------------------------------
# Photovoltaic target distribution audit
# ------------------------------------------------------------------

target_fields = ["voc", "jsc", "ff", "pce"]

target_summary = strict_molecular[target_fields].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
).T

display(target_summary)



# ------------------------------------------------------------------
# Non-positive and boundary-value checks
# ------------------------------------------------------------------

target_boundary_rows = []

for field in target_fields:

    s = strict_molecular[field]

    target_boundary_rows.append({
        "field": field,
        "minimum": s.min(),
        "maximum": s.max(),
        "zero_records": int((s == 0).sum()),
        "negative_records": int((s < 0).sum()),
        "positive_records": int((s > 0).sum()),
    })

target_boundary_summary = pd.DataFrame(
    target_boundary_rows
)

display(target_boundary_summary)



# ------------------------------------------------------------------
# Inspect the tails of the PCE distribution
# ------------------------------------------------------------------

low_pce_records = (
    strict_molecular[
        [
            "id",
            "doi_norm",
            "donor_canonical",
            "acceptor_canonical",
            "voc",
            "jsc",
            "ff",
            "pce",
        ]
    ]
    .sort_values("pce")
    .head(20)
)

high_pce_records = (
    strict_molecular[
        [
            "id",
            "doi_norm",
            "donor_canonical",
            "acceptor_canonical",
            "voc",
            "jsc",
            "ff",
            "pce",
        ]
    ]
    .sort_values("pce", ascending=False)
    .head(20)
)

print("LOWEST-PCE RECORDS")
display(low_pce_records)

print("\nHIGHEST-PCE RECORDS")
display(high_pce_records)

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
voc,21720.0,0.819287,0.129637,0.030000,0.4400,0.57,0.640,0.770,0.84,0.89,0.94,0.978,1.10,1.953
jsc,21720.0,17.162038,7.671949,0.000153,1.6419,4.30,6.469,10.750,17.15,24.93,26.58,27.160,28.20,44.320
ff,21720.0,62.989607,12.531833,16.000000,28.8357,38.00,44.449,55.300,65.60,73.00,77.00,78.570,80.69,87.670
pce,21720.0,9.690211,5.687498,0.000022,0.4000,1.23,2.160,4.643,9.26,15.20,17.54,18.360,19.53,21.830


,field,minimum,maximum,zero_records,negative_records,positive_records
0,voc,0.030000,1.953,0,0,21720
1,jsc,0.000153,44.320,0,0,21720
2,ff,16.000000,87.670,0,0,21720
3,pce,0.000022,21.830,0,0,21720


LOWEST-PCE RECORDS


,id,doi_norm,donor_canonical,acceptor_canonical,voc,jsc,ff,pce
12253,81675,10.1016/j.orgel.2020.106046,P3HT,PCBM,0.467000,0.000153,31.0000,0.000022
13418,86457,10.1007/s00339-025-08810-6,PM6,L8-BO,0.212000,0.066000,28.1000,0.004000
18697,108170,10.1016/j.solmat.2012.04.041,P3HT,PCDTNDI,0.140000,0.140000,25.4000,0.005000
13420,86460,10.1007/s00339-025-08810-6,PM6,BTP-eC9,0.306000,0.124000,28.9000,0.011000
20175,110603,10.1039/c2jm32514d,PTBDT-BTZ,PC71BM,0.920000,0.040000,35.5000,0.013000
15240,98690,10.1007/s11664-021-09020-5,P3HT,PCBM,0.126334,0.416349,25.0126,0.013200
8403,60511,10.1016/j.orgel.2023.106876,rubrene,NDI-C6,1.040000,0.025000,52.8000,0.014000
13419,86459,10.1007/s00339-025-08810-6,PM6,BTP-eC9,0.235000,0.234000,27.6000,0.015000
9635,65764,10.1039/d0tc03393f,NDT,PC61BM,0.740000,0.089000,24.7000,0.016000
13379,86241,10.1002/solr.202300322,PM6,L8-BO,0.080000,0.760000,32.8900,0.020000



HIGHEST-PCE RECORDS


,id,doi_norm,donor_canonical,acceptor_canonical,voc,jsc,ff,pce
6535,53569,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.540,31.87,44.44,21.83
6534,53568,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.540,31.85,44.50,21.81
6533,53567,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.540,31.84,44.55,21.79
6532,53566,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.540,31.84,44.58,21.79
6531,53565,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.530,31.83,44.59,21.79
6530,53564,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.530,31.84,44.62,21.78
12950,83963,10.1002/adfm.202208793,PM6,BTP-eC11,1.090,24.41,81.30,21.63
14303,94647,10.1039/d0ta08706h,PDTPDTBT,PC61BM,1.157,22.76,79.90,21.04
12949,83962,10.1002/adfm.202208793,PM6,BTP-eC11,1.074,24.25,80.74,21.02
3497,43927,10.1002/adma.202519230,D18,L8-BO,0.901,27.69,83.93,20.94


### 3.4 Provenance audit triggered by extreme-performance records

Inspection of the high-PCE tail identified records that satisfy the numerical
quality-control criteria but originate from a numerically simulated device
rather than an experimentally measured photovoltaic device.

For DOI `10.1007/s40243-025-00304-y`, six PBDB-T:ITIC records with reported
PCE values of approximately 21.78–21.83% correspond to SCAPS-1D simulated
devices described in the source publication.

This demonstrates that internal photovoltaic consistency checks alone cannot
establish experimental provenance.

Because the OPV-DB release documentation states that device-simulation and
theoretical/computational records were excluded, the presence of this source
motivates an additional publication-level provenance audit before final
modeling cohorts are defined.

No records are removed at this stage.

In [21]:
# ------------------------------------------------------------------
# Confirmed provenance exception identified during target-tail audit
# ------------------------------------------------------------------

confirmed_provenance_exceptions = pd.DataFrame([
    {
        "doi_norm": "10.1007/s40243-025-00304-y",
        "reason": "device simulation",
        "method": "SCAPS-1D numerical simulation",
        "verification_status": "manually verified from source publication",
        "action": "flag only; cohort exclusion not yet applied",
    }
])

exception_rows = strict_molecular[
    strict_molecular["doi_norm"].isin(
        confirmed_provenance_exceptions["doi_norm"]
    )
].copy()

print("Confirmed provenance-exception records:")
print(f"Records : {len(exception_rows):,}")
print(f"DOIs    : {exception_rows['doi_norm'].nunique():,}")

display(
    exception_rows[
        [
            "id",
            "doi_norm",
            "donor_canonical",
            "acceptor_canonical",
            "voc",
            "jsc",
            "ff",
            "pce",
            "device_structure",
            "etl",
            "htl",
        ]
    ]
)

Confirmed provenance-exception records:
Records : 6
DOIs    : 1


,id,doi_norm,donor_canonical,acceptor_canonical,voc,jsc,ff,pce,device_structure,etl,htl
6530,53564,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.53,31.84,44.62,21.78,FTO/ETL/PBDB-T/ITIC/Cu2O/Ag,TiO2,Cu2O
6531,53565,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.53,31.83,44.59,21.79,FTO/ETL/PBDB-T/ITIC/Cu2O/Ag,Tm(III)-doped TiO2 (0.2 mol%),Cu2O
6532,53566,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.54,31.84,44.58,21.79,FTO/ETL/PBDB-T/ITIC/Cu2O/Ag,Tm(III)-doped TiO2 (0.4 mol%),Cu2O
6533,53567,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.54,31.84,44.55,21.79,FTO/ETL/PBDB-T/ITIC/Cu2O/Ag,Tm(III)-doped TiO2 (0.6 mol%),Cu2O
6534,53568,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.54,31.85,44.50,21.81,FTO/ETL/PBDB-T/ITIC/Cu2O/Ag,Tm(III)-doped TiO2 (0.8 mol%),Cu2O
6535,53569,10.1007/s40243-025-00304-y,PBDB-T,ITIC,1.54,31.87,44.44,21.83,FTO/ETL/PBDB-T/ITIC/Cu2O/Ag,Tm(III)-doped TiO2 (1.0 mol%),Cu2O


### 3.5 Publication-level provenance screening

The discovery of a numerically simulated device within the strict molecular
benchmark motivates a systematic audit of publication provenance.

The audit is performed at the DOI level rather than the individual device-row
level because multiple device records can originate from the same publication.

For each source publication, the number of device records, material systems,
and observed PCE range are summarized. Publication metadata will subsequently
be used to identify candidate simulation, theoretical, or computational studies
for manual verification.

Automated keyword screening is used only for prioritization. A publication is
not excluded solely because computational terminology appears in its metadata,
since many experimental studies also contain DFT, simulation, or theoretical
analysis.

In [23]:
# ------------------------------------------------------------------
# DOI-level inventory for provenance screening
# ------------------------------------------------------------------

doi_inventory = (
    molecular_audit
    .groupby("doi_norm")
    .agg(
        records=("id", "size"),

        unique_donors=(
            "donor_graph_smiles",
            "nunique"
        ),

        unique_acceptors=(
            "acceptor_graph_smiles",
            "nunique"
        ),

        unique_donor_labels=(
            "donor_canonical",
            "nunique"
        ),

        unique_acceptor_labels=(
            "acceptor_canonical",
            "nunique"
        ),

        pce_min=("pce", "min"),
        pce_median=("pce", "median"),
        pce_max=("pce", "max"),

        voc_max=("voc", "max"),
        jsc_max=("jsc", "max"),
        ff_max=("ff", "max"),
    )
    .reset_index()
)


# ------------------------------------------------------------------
# Count distinct canonicalized donor–acceptor structure pairs per DOI
# ------------------------------------------------------------------

pair_audit = molecular_audit.copy()

pair_audit["graph_pair"] = list(
    zip(
        pair_audit["donor_graph_smiles"],
        pair_audit["acceptor_graph_smiles"]
    )
)

pair_counts_by_doi = (
    pair_audit
    .groupby("doi_norm")["graph_pair"]
    .nunique()
    .rename("unique_graph_pairs")
    .reset_index()
)


doi_inventory = doi_inventory.merge(
    pair_counts_by_doi,
    on="doi_norm",
    how="left"
)


print(f"Unique source DOIs : {len(doi_inventory):,}")
print(f"Device records     : {doi_inventory['records'].sum():,}")

display(
    doi_inventory
    .sort_values(
        ["pce_max", "records"],
        ascending=False
    )
    .head(30)
)

Unique source DOIs : 5,459
Device records     : 21,720


,doi_norm,records,unique_donors,unique_acceptors,unique_donor_labels,unique_acceptor_labels,pce_min,pce_median,pce_max,voc_max,jsc_max,ff_max,unique_graph_pairs
1761,10.1007/s40243-025-00304-y,6,1,1,1,1,21.78,21.790,21.83,1.5400,31.87,44.62,1
117,10.1002/adfm.202208793,7,1,1,1,1,5.44,17.460,21.63,1.0900,27.87,81.30,1
4815,10.1039/d0ta08706h,17,3,10,3,10,4.31,13.100,21.04,1.1570,23.15,79.90,10
601,10.1002/adma.202519230,4,1,1,1,1,19.70,20.145,20.94,0.9120,27.69,83.93,1
4139,10.1038/s41560-025-01862-1,12,2,4,2,4,17.50,19.250,20.90,0.9300,29.20,83.20,5
2272,10.1016/j.joule.2025.102135,12,1,1,1,1,13.52,18.450,20.87,0.9120,28.18,82.28,1
5431,10.3390/polym15040869,10,4,6,4,6,8.18,15.980,20.87,1.2090,28.37,78.50,6
5224,10.1039/d5ee02957k,3,1,1,1,1,18.53,19.430,20.81,0.9250,27.48,81.85,1
5228,10.1039/d5ee05708f,3,1,1,1,1,18.99,19.710,20.70,0.9100,27.72,82.06,1
589,10.1002/adma.202512197,13,1,1,1,1,16.60,20.030,20.64,0.9070,27.87,82.11,1


### 3.6 Publication-metadata enrichment

The DOI-level inventory is enriched with bibliographic metadata to support
publication-provenance screening.

Metadata retrieval is performed separately from the raw OPV-DB files and the
retrieved results are cached in the project `data/interim` directory. This
avoids repeated dependence on an external API and preserves the metadata
snapshot used for the analysis.

Publication metadata are used only to prioritize candidate records for manual
provenance verification. Keyword matches alone do not determine exclusion.

In [24]:
# ------------------------------------------------------------------
# Save DOI inventory before external metadata enrichment
# ------------------------------------------------------------------

doi_inventory_path = (
    INTERIM_DIR / "opvdb_strict_molecular_doi_inventory.csv"
)

doi_inventory.to_csv(
    doi_inventory_path,
    index=False
)

print("Saved DOI inventory:")
print(doi_inventory_path)
print(f"Rows: {len(doi_inventory):,}")

Saved DOI inventory:
<PROJECT_ROOT>\data\interim\opvdb_strict_molecular_doi_inventory.csv
Rows: 5,459


**Metadata-provider note.**  
The initial Semantic Scholar batch request returned HTTP 429 (rate limited)
during the 10-publication test. No metadata were collected from this request.

To avoid making the provenance workflow dependent on an already rate-limited
service, Crossref was adopted as the primary DOI-metadata source.

In [26]:
import requests
import time
import re
from urllib.parse import quote

# ------------------------------------------------------------------
# Crossref DOI metadata test
# ------------------------------------------------------------------

CONTACT_EMAIL = input(
    "Enter your email for the Crossref polite API pool: "
).strip()

CROSSREF_BASE = "https://api.crossref.org/works"

session = requests.Session()

session.headers.update({
    "User-Agent": (
        f"OPV-Paper3-Provenance-Audit/0.1 "
        f"(mailto:{CONTACT_EMAIL})"
    )
})


def crossref_get_doi(doi, max_retries=5):
    """
    Retrieve one Crossref DOI record with conservative retry/backoff.
    """

    encoded_doi = quote(str(doi).strip(), safe="")

    url = f"{CROSSREF_BASE}/{encoded_doi}"

    for attempt in range(max_retries):

        response = session.get(
            url,
            params={"mailto": CONTACT_EMAIL},
            timeout=30,
        )

        if response.status_code == 200:
            return response.json()["message"], 200

        if response.status_code == 404:
            return None, 404

        if response.status_code == 429:

            retry_after = response.headers.get("Retry-After")

            if retry_after:
                try:
                    wait_seconds = float(retry_after)
                except ValueError:
                    wait_seconds = 2 ** attempt
            else:
                wait_seconds = 2 ** attempt

            print(
                f"Rate limited. Waiting "
                f"{wait_seconds:.1f} s..."
            )

            time.sleep(wait_seconds)
            continue

        # Retry temporary server failures
        if response.status_code >= 500:
            time.sleep(2 ** attempt)
            continue

        return None, response.status_code

    return None, 429



    # ------------------------------------------------------------------
# Test on only 10 publications
# ------------------------------------------------------------------

known_simulation_doi = "10.1007/s40243-025-00304-y"

test_dois = [known_simulation_doi]

test_dois += (
    doi_inventory.loc[
        doi_inventory["doi_norm"] != known_simulation_doi,
        "doi_norm"
    ]
    .head(9)
    .tolist()
)


crossref_test_rows = []

for i, doi in enumerate(test_dois, start=1):

    item, status = crossref_get_doi(doi)

    if item is None:

        crossref_test_rows.append({
            "doi_norm": doi,
            "http_status": status,
            "matched": False,
            "title": None,
            "year": None,
            "journal": None,
            "type": None,
            "abstract_available": False,
        })

    else:

        title = item.get("title", [])
        title = title[0] if title else None

        journal = item.get("container-title", [])
        journal = journal[0] if journal else None

        # Prefer published-print/online date, otherwise issued
        date_info = (
            item.get("published-print")
            or item.get("published-online")
            or item.get("issued")
            or {}
        )

        date_parts = date_info.get("date-parts", [])
        year = (
            date_parts[0][0]
            if date_parts and date_parts[0]
            else None
        )

        crossref_test_rows.append({
            "doi_norm": doi,
            "http_status": status,
            "matched": True,
            "title": title,
            "year": year,
            "journal": journal,
            "type": item.get("type"),
            "abstract_available":
                bool(item.get("abstract")),
        })

    # Stay comfortably below Crossref request limits
    time.sleep(0.20)


crossref_test_df = pd.DataFrame(
    crossref_test_rows
)

display(crossref_test_df)

Enter your email for the Crossref polite API pool:  [redacted-email]


,doi_norm,http_status,matched,title,year,journal,type,abstract_available
0,10.1007/s40243-025-00304-y,200,True,An experimental and computational investigatio...,2025,Materials for Renewable and Sustainable Energy,journal-article,True
1,10.1002/adem.202001305,200,True,Silver Nanowires Digital Printing for Inverted...,2021,Advanced Engineering Materials,journal-article,True
2,10.1002/adem.202300595,200,True,Overcoming Moisture‐Induced Degradation in Org...,2023,Advanced Engineering Materials,journal-article,True
3,10.1002/adfm.201001807,200,True,A Simple and Effective Modification of PCBM fo...,2011,Advanced Functional Materials,journal-article,True
4,10.1002/adfm.201100708,200,True,"Alternating Copolymers of Cyclopenta[2,1‐b;3,4...",2011,Advanced Functional Materials,journal-article,True
5,10.1002/adfm.201102771,200,True,Highly Efficient and Thermally Stable Polymer ...,2012,Advanced Functional Materials,journal-article,True
6,10.1002/adfm.201102937,200,True,High‐Performance Inverted Polymer Solar Cells:...,2012,Advanced Functional Materials,journal-article,True
7,10.1002/adfm.201200729,200,True,Synthesis of a Modified PC<sub>70</sub>BM and ...,2012,Advanced Functional Materials,journal-article,True
8,10.1002/adfm.201203251,200,True,Cyclobutadiene–C<sub>60</sub> Adducts: N‐Type ...,2013,Advanced Functional Materials,journal-article,True
9,10.1002/adfm.201303219,200,True,Alkoxy‐Functionalized Thienyl‐Vinylene Polymer...,2014,Advanced Functional Materials,journal-article,True


### 3.7 Retrieval and caching of publication metadata

Crossref successfully resolved the pilot DOI sample and was therefore used as
the primary publication-metadata provider.

Metadata retrieval is performed conservatively using the Crossref polite API
pool. Results are cached incrementally so that interrupted retrieval can resume
without repeating completed requests.

The metadata snapshot is treated as an auxiliary provenance resource rather
than part of the original OPV-DB release.

In [27]:
from getpass import getpass

# Re-enter only if CONTACT_EMAIL is not already available
if "CONTACT_EMAIL" not in globals() or not CONTACT_EMAIL:
    CONTACT_EMAIL = getpass(
        "Crossref contact email (hidden): "
    ).strip()

session.headers.update({
    "User-Agent": (
        f"OPV-Paper3-Provenance-Audit/0.1 "
        f"(mailto:{CONTACT_EMAIL})"
    )
})

In [28]:
import json
import time
from pathlib import Path

# ------------------------------------------------------------------
# Crossref metadata cache
# ------------------------------------------------------------------

CROSSREF_CACHE = (
    INTERIM_DIR / "crossref_opvdb_metadata.jsonl"
)

CROSSREF_FAILURES = (
    INTERIM_DIR / "crossref_opvdb_failures.csv"
)


def extract_crossref_metadata(doi, item, status):
    """Convert Crossref response into the fields needed for provenance audit."""

    if item is None:
        return {
            "doi_norm": doi,
            "http_status": status,
            "matched": False,
            "title": None,
            "abstract": None,
            "year": None,
            "journal": None,
            "type": None,
            "publisher": None,
        }

    title = item.get("title", [])
    title = title[0] if title else None

    journal = item.get("container-title", [])
    journal = journal[0] if journal else None

    date_info = (
        item.get("published-print")
        or item.get("published-online")
        or item.get("issued")
        or {}
    )

    date_parts = date_info.get("date-parts", [])

    year = (
        date_parts[0][0]
        if date_parts and date_parts[0]
        else None
    )

    return {
        "doi_norm": doi,
        "http_status": status,
        "matched": True,
        "title": title,
        "abstract": item.get("abstract"),
        "year": year,
        "journal": journal,
        "type": item.get("type"),
        "publisher": item.get("publisher"),
    }



    # ------------------------------------------------------------------
# Load previously cached records, if any
# ------------------------------------------------------------------

cached_records = {}

if CROSSREF_CACHE.exists():

    with open(
        CROSSREF_CACHE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                record = json.loads(line)

                cached_records[
                    record["doi_norm"]
                ] = record


all_dois = doi_inventory["doi_norm"].tolist()

remaining_dois = [
    doi
    for doi in all_dois
    if doi not in cached_records
]


print(f"Total DOIs     : {len(all_dois):,}")
print(f"Already cached : {len(cached_records):,}")
print(f"Remaining      : {len(remaining_dois):,}")


# ------------------------------------------------------------------
# Retrieve remaining DOI metadata
# ------------------------------------------------------------------

failure_rows = []

start_time = time.time()

with open(
    CROSSREF_CACHE,
    "a",
    encoding="utf-8"
) as cache_file:

    for i, doi in enumerate(
        remaining_dois,
        start=1
    ):

        item, status = crossref_get_doi(
            doi,
            max_retries=5
        )

        record = extract_crossref_metadata(
            doi,
            item,
            status
        )

        # Write immediately so progress survives interruption
        cache_file.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

        cache_file.flush()

        if not record["matched"]:

            failure_rows.append({
                "doi_norm": doi,
                "http_status": status
            })

        # Progress report every 100 new requests
        if i % 100 == 0 or i == len(remaining_dois):

            elapsed_min = (
                time.time() - start_time
            ) / 60

            print(
                f"{i:,}/{len(remaining_dois):,} "
                f"new DOIs processed | "
                f"{elapsed_min:.1f} min elapsed"
            )

        # Conservative request spacing
        time.sleep(0.15)



        # ------------------------------------------------------------------
# Save failures for later inspection
# ------------------------------------------------------------------

pd.DataFrame(
    failure_rows
).to_csv(
    CROSSREF_FAILURES,
    index=False
)

print("\nRetrieval pass complete.")
print("Cache :", CROSSREF_CACHE)
print("Failures this pass :", len(failure_rows))

Total DOIs     : 5,459
Already cached : 0
Remaining      : 5,459
100/5,459 new DOIs processed | 0.7 min elapsed
200/5,459 new DOIs processed | 1.5 min elapsed
300/5,459 new DOIs processed | 2.2 min elapsed
400/5,459 new DOIs processed | 3.0 min elapsed
500/5,459 new DOIs processed | 3.7 min elapsed
600/5,459 new DOIs processed | 4.4 min elapsed
700/5,459 new DOIs processed | 5.1 min elapsed
800/5,459 new DOIs processed | 5.9 min elapsed
900/5,459 new DOIs processed | 6.8 min elapsed
1,000/5,459 new DOIs processed | 7.5 min elapsed
1,100/5,459 new DOIs processed | 8.2 min elapsed
1,200/5,459 new DOIs processed | 8.9 min elapsed
1,300/5,459 new DOIs processed | 9.6 min elapsed
1,400/5,459 new DOIs processed | 10.4 min elapsed
1,500/5,459 new DOIs processed | 11.1 min elapsed
1,600/5,459 new DOIs processed | 11.9 min elapsed
1,700/5,459 new DOIs processed | 12.6 min elapsed
1,800/5,459 new DOIs processed | 13.4 min elapsed
1,900/5,459 new DOIs processed | 14.2 min elapsed
2,000/5,459 new 

### 3.8 Crossref metadata completeness audit

The cached Crossref metadata are evaluated before provenance keyword
screening.

Coverage is quantified for DOI matching, title, abstract, publication year,
journal, and publication type. Failed DOI resolutions are inspected
separately.

This ensures that subsequent provenance screening is interpreted relative to
the metadata actually available rather than assuming uniform bibliographic
coverage.

In [29]:
# ------------------------------------------------------------------
# Load the complete cached Crossref metadata snapshot
# ------------------------------------------------------------------

crossref_records = []

with open(
    CROSSREF_CACHE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():
            crossref_records.append(
                json.loads(line)
            )


crossref_metadata = pd.DataFrame(
    crossref_records
)

# Deduplicate defensively by DOI in case a retrieval was ever resumed
crossref_metadata = (
    crossref_metadata
    .drop_duplicates(
        subset="doi_norm",
        keep="last"
    )
    .reset_index(drop=True)
)


print(f"Cached unique DOIs : {len(crossref_metadata):,}")

metadata_coverage = pd.DataFrame({
    "field": [
        "matched",
        "title",
        "abstract",
        "year",
        "journal",
        "type",
        "publisher",
    ],

    "available_records": [
        crossref_metadata["matched"].eq(True).sum(),
        crossref_metadata["title"].notna().sum(),
        crossref_metadata["abstract"].notna().sum(),
        crossref_metadata["year"].notna().sum(),
        crossref_metadata["journal"].notna().sum(),
        crossref_metadata["type"].notna().sum(),
        crossref_metadata["publisher"].notna().sum(),
    ]
})

metadata_coverage["percent_of_dois"] = (
    100
    * metadata_coverage["available_records"]
    / len(doi_inventory)
).round(2)

display(metadata_coverage)



# ------------------------------------------------------------------
# Inspect unresolved DOIs
# ------------------------------------------------------------------

failed_metadata = crossref_metadata[
    ~crossref_metadata["matched"]
].copy()

print(
    "Unresolved DOI records:",
    len(failed_metadata)
)

display(
    failed_metadata[
        [
            "doi_norm",
            "http_status"
        ]
    ]
)



# ------------------------------------------------------------------
# Merge publication metadata with the DOI-level OPV inventory
# ------------------------------------------------------------------

doi_metadata_audit = doi_inventory.merge(
    crossref_metadata,
    on="doi_norm",
    how="left"
)

print(
    "DOI inventory after metadata merge:",
    len(doi_metadata_audit)
)

print(
    "DOIs without any cached metadata row:",
    doi_metadata_audit["matched"].isna().sum()
)

display(
    doi_metadata_audit[
        [
            "doi_norm",
            "records",
            "pce_max",
            "year",
            "journal",
            "title",
            "abstract"
        ]
    ]
    .sort_values(
        "pce_max",
        ascending=False
    )
    .head(15)
)

Cached unique DOIs : 5,459


,field,available_records,percent_of_dois
0,matched,5456,99.95
1,title,5456,99.95
2,abstract,2849,52.19
3,year,5442,99.69
4,journal,5442,99.69
5,type,5456,99.95
6,publisher,5456,99.95


Unresolved DOI records: 3


,doi_norm,http_status
759,10.1002/aenm.201606574,404
3131,10.1020/jacs.7b13239,404
3136,10.1021/acenergylett.8b00627,404


DOI inventory after metadata merge: 5459
DOIs without any cached metadata row: 0


,doi_norm,records,pce_max,year,journal,title,abstract
1761,10.1007/s40243-025-00304-y,6,21.83,2025.0,Materials for Renewable and Sustainable Energy,An experimental and computational investigatio...,<jats:title>Abstract</jats:title>\n ...
117,10.1002/adfm.202208793,7,21.63,2022.0,Advanced Functional Materials,Versatile Hole Selective Molecules Containing ...,<jats:title>Abstract</jats:title>\n ...
4815,10.1039/d0ta08706h,17,21.04,2020.0,Journal of Materials Chemistry A,"The design of dithieno[3,2-\n ...",<p>\n The design of dithien...
601,10.1002/adma.202519230,4,20.94,2026.0,Advanced Materials,Achieving a Record Fill Factor of Approaching ...,<jats:title>ABSTRACT</jats:title>\n ...
4139,10.1038/s41560-025-01862-1,12,20.90,2025.0,Nature Energy,Two-step crystallization modulated through ace...,NaN
5431,10.3390/polym15040869,10,20.87,2023.0,Polymers,Device Modeling of Efficient PBDB-T:PZT-Based ...,"<jats:p>In this study, we present some design ..."
2272,10.1016/j.joule.2025.102135,12,20.87,2025.0,Joule,Elevating dielectric constant via additive eng...,NaN
5224,10.1039/d5ee02957k,3,20.81,2025.0,Energy &amp; Environmental Science,Balanced distribution of donors and acceptors ...,<jats:p>Volatile isomerization additives induc...
5228,10.1039/d5ee05708f,3,20.70,2026.0,Energy &amp; Environmental Science,Mitigating photon escape in thin-film photovol...,<jats:p>Subwavelength periodic pyramids with h...
583,10.1002/adma.202509806,11,20.64,2025.0,Advanced Materials,20.64% Efficient and Stable Binary Organic Sol...,<jats:title>Abstract</jats:title>\n ...


### 3.9 Conservative publication-provenance candidate screening

Crossref provides titles for nearly all source publications but abstracts for
only a subset. Provenance screening therefore combines universally available
titles with abstracts where available.

Bibliographic text is normalized only for screening purposes. Publications
are flagged using terms associated with device simulation, numerical modeling,
and theoretical prediction.

Keyword matches are treated as candidate flags rather than exclusion criteria.
Every candidate publication must be manually assessed before any device
records are removed from the experimental modeling cohort.

In [31]:
# ------------------------------------------------------------------
# Clean bibliographic text for provenance screening
# ------------------------------------------------------------------

from html import unescape

def clean_bibliographic_text(value):

    if pd.isna(value):
        return ""

    text = unescape(str(value))

    # Remove JATS/XML/HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace and case
    text = re.sub(r"\s+", " ", text).strip().lower()

    return text


doi_metadata_audit["title_clean"] = (
    doi_metadata_audit["title"]
    .apply(clean_bibliographic_text)
)

doi_metadata_audit["abstract_clean"] = (
    doi_metadata_audit["abstract"]
    .apply(clean_bibliographic_text)
)

doi_metadata_audit["screening_text"] = (
    doi_metadata_audit["title_clean"]
    + " "
    + doi_metadata_audit["abstract_clean"]
).str.strip()



# ------------------------------------------------------------------
# High-recall provenance keyword groups
# ------------------------------------------------------------------

provenance_patterns = {

    "scaps": (
        r"\bscaps(?:-1d)?\b"
    ),

    "device_simulation": (
        r"\bdevice simulation\b|"
        r"\bsimulated device\b|"
        r"\bsimulation of (?:an? )?(?:organic )?solar cell\b"
    ),

    "numerical": (
        r"\bnumerical simulation\b|"
        r"\bnumerical model(?:ing|ling)?\b|"
        r"\bnumerical investigation\b|"
        r"\bnumerical analysis\b"
    ),

    "computational": (
        r"\bcomputational investigation\b|"
        r"\bcomputational study\b|"
        r"\bcomputational model(?:ing|ling)?\b|"
        r"\bcomputational simulation\b"
    ),

    "theoretical": (
        r"\btheoretical investigation\b|"
        r"\btheoretical study\b|"
        r"\btheoretical model(?:ing|ling)?\b|"
        r"\btheoretical simulation\b"
    ),

    "predictive_only": (
        r"\bpredicted photovoltaic\b|"
        r"\bpredicted efficiency\b|"
        r"\btheoretically predicted\b"
    ),
}


for flag_name, pattern in provenance_patterns.items():

    doi_metadata_audit[f"flag_{flag_name}"] = (
        doi_metadata_audit["screening_text"]
        .str.contains(
            pattern,
            regex=True,
            na=False
        )
    )


flag_columns = [
    f"flag_{name}"
    for name in provenance_patterns
]

doi_metadata_audit["provenance_keyword_flag"] = (
    doi_metadata_audit[flag_columns]
    .any(axis=1)
)



# ------------------------------------------------------------------
# Candidate publication summary
# ------------------------------------------------------------------

keyword_summary = pd.DataFrame({
    "flag": list(provenance_patterns.keys()),

    "publications": [
        int(
            doi_metadata_audit[
                f"flag_{name}"
            ].sum()
        )
        for name in provenance_patterns
    ]
})

display(keyword_summary)


provenance_candidates = (
    doi_metadata_audit[
        doi_metadata_audit[
            "provenance_keyword_flag"
        ]
    ]
    .copy()
)


print(
    "Candidate publications:",
    f"{len(provenance_candidates):,}"
)

print(
    "Device records represented:",
    f"{provenance_candidates['records'].sum():,}"
)


display(
    provenance_candidates[
        [
            "doi_norm",
            "records",
            "pce_max",
            "year",
            "journal",
            "title",
        ]
        + flag_columns
    ]
    .sort_values(
        ["records", "pce_max"],
        ascending=False
    )
    .head(50)
)



# ------------------------------------------------------------------
# Positive-control check: known SCAPS publication
# ------------------------------------------------------------------

positive_control = provenance_candidates[
    provenance_candidates["doi_norm"]
    == "10.1007/s40243-025-00304-y"
]

print(
    "Known SCAPS paper successfully flagged:",
    len(positive_control) == 1
)

display(
    positive_control[
        [
            "doi_norm",
            "title",
        ]
        + flag_columns
    ]
)

,flag,publications
0,scaps,13
1,device_simulation,4
2,numerical,11
3,computational,3
4,theoretical,4
5,predictive_only,0


Candidate publications: 30
Device records represented: 174


,doi_norm,records,pce_max,year,journal,title,flag_scaps,flag_device_simulation,flag_numerical,flag_computational,flag_theoretical,flag_predictive_only
2877,10.1016/j.rio.2024.100748,29,12.260,2024.0,Results in Optics,Numerical investigation of graphene derivative...,False,False,True,False,False,False
2878,10.1016/j.rsurfi.2025.100474,14,11.220,2025.0,Results in Surfaces and Interfaces,Numerical investigation of PBDB-T:INTIC based ...,False,False,True,False,False,False
681,10.1002/advs.202202150,13,17.250,2022.0,Advanced Science,High‐Performance Semitransparent Organic Solar...,False,False,False,False,True,False
1739,10.1007/s11664-021-09020-5,11,1.163,2021.0,Journal of Electronic Materials,Optimization of Nanoparticle Organic Photovolt...,True,False,False,False,False,False
5431,10.3390/polym15040869,10,20.870,2023.0,Polymers,Device Modeling of Efficient PBDB-T:PZT-Based ...,True,True,False,False,False,False
120,10.1002/adfm.202209728,9,18.100,2023.0,Advanced Functional Materials,An n‐n Heterojunction Configuration for Effici...,False,False,True,False,False,False
1333,10.1002/pssa.202500317,8,19.350,2025.0,physica status solidi (a),Enhancing Organic Solar Cells through Dual Ele...,True,False,False,False,False,False
1970,10.1016/j.cjph.2023.12.028,8,18.380,2024.0,Chinese Journal of Physics,Enhancing the efficiency of PM6:Y6 bulk-hetero...,True,False,False,False,False,False
656,10.1002/adts.202400725,7,18.540,2025.0,Advanced Theory and Simulations,Validating the Novel Electron Transport Layer ...,False,True,False,False,False,False
1761,10.1007/s40243-025-00304-y,6,21.830,2025.0,Materials for Renewable and Sustainable Energy,An experimental and computational investigatio...,True,False,True,True,False,False


Known SCAPS paper successfully flagged: True


,doi_norm,title,flag_scaps,flag_device_simulation,flag_numerical,flag_computational,flag_theoretical,flag_predictive_only
1761,10.1007/s40243-025-00304-y,An experimental and computational investigatio...,True,False,True,True,False,False


### 3.10 Manual verification of provenance candidates

Automated screening identified a small set of candidate publications containing
terminology associated with device simulation, numerical modeling, or
computational analysis.

Because such terminology can also occur in otherwise experimental studies,
automated flags are not used as exclusion criteria.

Each candidate publication is therefore assigned a manual provenance status:

- `experimental` — the OPV device records correspond to experimentally
  fabricated/measured devices;
- `simulation_only` — the reported device-performance records arise from
  numerical/device simulation rather than experimental measurements;
- `mixed` — the publication contains both experimental and simulated device
  results and the provenance of the database rows requires row-level
  verification;
- `uncertain` — available evidence is insufficient for confident
  classification.

For every exclusion decision, a short evidence note is retained.

In [32]:
# ------------------------------------------------------------------
# Create manual provenance-review table
# ------------------------------------------------------------------

review_columns = [
    "doi_norm",
    "records",
    "pce_min",
    "pce_median",
    "pce_max",
    "year",
    "journal",
    "title",
]

provenance_review = (
    provenance_candidates[
        review_columns
    ]
    .copy()
    .sort_values(
        ["records", "pce_max"],
        ascending=False
    )
    .reset_index(drop=True)
)

# Fields to be completed during manual source verification
provenance_review["review_status"] = "unreviewed"
provenance_review["device_method"] = ""
provenance_review["evidence_note"] = ""
provenance_review["exclude_from_experimental_cohort"] = False


# Known positive-control paper already manually verified
known_doi = "10.1007/s40243-025-00304-y"

mask = provenance_review["doi_norm"].eq(known_doi)

provenance_review.loc[
    mask,
    "review_status"
] = "simulation_only"

provenance_review.loc[
    mask,
    "device_method"
] = "SCAPS-1D numerical simulation"

provenance_review.loc[
    mask,
    "evidence_note"
] = (
    "Source publication describes the high-efficiency PBDB-T:ITIC "
    "device values as results of SCAPS-1D modeling."
)

provenance_review.loc[
    mask,
    "exclude_from_experimental_cohort"
] = True


PROVENANCE_REVIEW_PATH = (
    INTERIM_DIR / "opvdb_provenance_manual_review.csv"
)

provenance_review.to_csv(
    PROVENANCE_REVIEW_PATH,
    index=False
)

print("Candidate publications :", len(provenance_review))
print(
    "Already verified       :",
    (provenance_review["review_status"] != "unreviewed").sum()
)
print(
    "Remaining for review   :",
    (provenance_review["review_status"] == "unreviewed").sum()
)

display(provenance_review)

Candidate publications : 30
Already verified       : 1
Remaining for review   : 29


,doi_norm,records,pce_min,pce_median,pce_max,year,journal,title,review_status,device_method,evidence_note,exclude_from_experimental_cohort
0,10.1016/j.rio.2024.100748,29,3.7800,7.8300,12.260,2024.0,Results in Optics,Numerical investigation of graphene derivative...,unreviewed,,,False
1,10.1016/j.rsurfi.2025.100474,14,5.0200,7.8950,11.220,2025.0,Results in Surfaces and Interfaces,Numerical investigation of PBDB-T:INTIC based ...,unreviewed,,,False
2,10.1002/advs.202202150,13,9.4100,13.2500,17.250,2022.0,Advanced Science,High‐Performance Semitransparent Organic Solar...,unreviewed,,,False
3,10.1007/s11664-021-09020-5,11,0.0132,0.4720,1.163,2021.0,Journal of Electronic Materials,Optimization of Nanoparticle Organic Photovolt...,unreviewed,,,False
4,10.3390/polym15040869,10,8.1800,15.9800,20.870,2023.0,Polymers,Device Modeling of Efficient PBDB-T:PZT-Based ...,unreviewed,,,False
5,10.1002/adfm.202209728,9,8.6000,14.5000,18.100,2023.0,Advanced Functional Materials,An n‐n Heterojunction Configuration for Effici...,unreviewed,,,False
6,10.1002/pssa.202500317,8,14.6800,16.3900,19.350,2025.0,physica status solidi (a),Enhancing Organic Solar Cells through Dual Ele...,unreviewed,,,False
7,10.1016/j.cjph.2023.12.028,8,6.1800,16.6300,18.380,2024.0,Chinese Journal of Physics,Enhancing the efficiency of PM6:Y6 bulk-hetero...,unreviewed,,,False
8,10.1002/adts.202400725,7,2.3300,10.1900,18.540,2025.0,Advanced Theory and Simulations,Validating the Novel Electron Transport Layer ...,unreviewed,,,False
9,10.1007/s40243-025-00304-y,6,21.7800,21.7900,21.830,2025.0,Materials for Renewable and Sustainable Energy,An experimental and computational investigatio...,simulation_only,SCAPS-1D numerical simulation,Source publication describes the high-efficien...,True


In [33]:
# ------------------------------------------------------------------
# High-confidence manual provenance classifications
# based on source-publication verification
# ------------------------------------------------------------------

verified_updates = {

    # ---------------- SIMULATION ONLY ----------------
    "10.1016/j.rio.2024.100748": {
        "review_status": "simulation_only",
        "device_method": "numerical device simulation",
        "evidence_note":
            "Paper explicitly models/simulates PBDB-T:NCBDT OSC configurations.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1016/j.rsurfi.2025.100474": {
        "review_status": "simulation_only",
        "device_method": "numerical device simulation",
        "evidence_note":
            "Paper explicitly numerically simulates PBDB-T:INTIC OSC configurations.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1007/s11664-021-09020-5": {
        "review_status": "simulation_only",
        "device_method": "SCAPS numerical simulation",
        "evidence_note":
            "SCAPS drift-diffusion simulation is used to optimize nanoparticle OPV performance.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1002/pssa.202500317": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D + ML",
        "evidence_note":
            "Reported device optimization and ML predictions are based on SCAPS-1D simulation.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1016/j.cjph.2023.12.028": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D numerical simulation",
        "evidence_note":
            "Paper explicitly reports simulated PM6:Y6 photovoltaic performance.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1002/pssa.202400654": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D + optimization",
        "evidence_note":
            "Study is based on SCAPS-1D simulation combined with optimization methods.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1007/s12648-023-03042-x": {
        "review_status": "simulation_only",
        "device_method": "numerical drift-diffusion modeling",
        "evidence_note":
            "Study is explicitly a numerical investigation of OSC performance.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1039/d5ra04889c": {
        "review_status": "simulation_only",
        "device_method": "DFT + SCAPS-1D",
        "evidence_note":
            "Novel donor and photovoltaic device are theoretically studied using DFT and SCAPS-1D.",
        "exclude_from_experimental_cohort": True,
    },

    "10.3390/polym14173610": {
        "review_status": "simulation_only",
        "device_method": "device simulation",
        "evidence_note":
            "OSC electron-transport-layer optimization is conducted through numerical device simulation.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1016/j.ijleo.2021.168457": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D",
        "evidence_note":
            "Paper explicitly performs simulation of PBDB-T/ITIC devices using SCAPS-1D.",
        "exclude_from_experimental_cohort": True,
    },

    "10.3390/polym15183674": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D",
        "evidence_note":
            "Paper states that the polymer solar cell is simulated using SCAPS-1D.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1016/j.chphi.2023.100407": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D theoretical analysis",
        "evidence_note":
            "Study is explicitly a theoretical SCAPS-1D analysis of PBDB-T/ITIC devices.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1016/j.solener.2020.09.068": {
        "review_status": "simulation_only",
        "device_method": "SCAPS numerical simulation",
        "evidence_note":
            "Paper explicitly models graded-BHJ OSCs using SCAPS simulation.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1039/d5na00721f": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D",
        "evidence_note":
            "Paper is explicitly a numerical SCAPS-1D investigation and optimization study.",
        "exclude_from_experimental_cohort": True,
    },

    "10.3390/polym15112578": {
        "review_status": "simulation_only",
        "device_method": "TCAD/device simulation",
        "evidence_note":
            "Proposed tandem architecture is studied through numerical device simulations.",
        "exclude_from_experimental_cohort": True,
    },

    "10.1007/s10825-021-01843-z": {
        "review_status": "simulation_only",
        "device_method": "numerical device simulation",
        "evidence_note":
            "Publication is a numerical investigation rather than a fabricated-device study.",
        "exclude_from_experimental_cohort": True,
    },

    # ---------------- EXPERIMENTAL ----------------
    "10.1002/advs.202202150": {
        "review_status": "experimental",
        "device_method": "fabricated and characterized OPV devices",
        "evidence_note":
            "Study experimentally fabricates semitransparent OSCs; theoretical simulation supports device design/analysis.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1002/aenm.201300031": {
        "review_status": "experimental",
        "device_method": "polymer synthesis + fabricated PSC devices",
        "evidence_note":
            "Polymers were synthesized and photovoltaic devices experimentally characterized; DFT is supporting analysis.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1039/c7py00292k": {
        "review_status": "experimental",
        "device_method": "polymer synthesis + fabricated PSC devices",
        "evidence_note":
            "Computational molecular design is followed by synthesis and experimental photovoltaic device measurement.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1073/pnas.1807535115": {
        "review_status": "experimental",
        "device_method": "fabricated photovoltaic devices",
        "evidence_note":
            "Paper contains explicit solar-cell fabrication and measured photovoltaic performance; calculations are supporting characterization.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1002/adfm.202402045": {
        "review_status": "experimental",
        "device_method": "fabricated polymer solar cells",
        "evidence_note":
            "Measured efficiency and stability of fabricated ternary polymer solar cells; computational analysis is supporting.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1002/aesr.202400114": {
        "review_status": "experimental",
        "device_method": "fabricated polymer solar cells",
        "evidence_note":
            "Source reports successfully fabricated P3HT:PCBM devices with nanoparticle-modified transport layers.",
        "exclude_from_experimental_cohort": False,
    },

    # ---------------- MIXED / ROW-LEVEL CHECK REQUIRED ----------------
    "10.3390/polym15040869": {
        "review_status": "mixed",
        "device_method": "experimental reference device + numerical device simulation",
        "evidence_note":
            "Paper starts from a previously fabricated PBDB-T:PZT cell but performs simulated device optimization; database rows require row-level provenance comparison.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1002/adts.202400725": {
        "review_status": "mixed",
        "device_method": "experimental literature reference + SCAPS-1D simulations",
        "evidence_note":
            "Paper validates simulations against a reported experimental D18:Y6 device, then numerically optimizes device structures; database rows require row-level verification.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1002/aenm.202000823": {
        "review_status": "mixed",
        "device_method": "fabricated tandem devices + forward simulation",
        "evidence_note":
            "Paper experimentally fabricates tandem OSCs but also reports simulated future performance; the single OPV-DB row requires row-level verification.",
        "exclude_from_experimental_cohort": False,
    },
}



for doi, update in verified_updates.items():

    mask = provenance_review["doi_norm"].eq(doi)

    for column, value in update.items():
        provenance_review.loc[mask, column] = value


provenance_review.to_csv(
    PROVENANCE_REVIEW_PATH,
    index=False
)


print(
    provenance_review["review_status"]
    .value_counts()
)

print(
    "\nRemaining unreviewed:",
    (provenance_review["review_status"] == "unreviewed").sum()
)

review_status
simulation_only    17
experimental        6
unreviewed          4
mixed               3
Name: count, dtype: int64

Remaining unreviewed: 4


In [34]:
final_unreviewed_updates = {

    "10.1002/adfm.202209728": {
        "review_status": "experimental",
        "device_method": "fabricated OPV devices + supporting numerical simulation",
        "evidence_note":
            "Experimental OPV performance is reported for devices using an "
            "n-n heterojunction ETL; numerical J-V simulation supports "
            "mechanistic interpretation.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1002/admi.202000577": {
        "review_status": "experimental",
        "device_method": "experimental BHJ devices + supporting numerical analysis",
        "evidence_note":
            "Study combines experimental and numerical analysis of vertical "
            "miscibility and reports experimentally optimized PM6:Y6-based devices.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1002/aenm.202202729": {
        "review_status": "experimental",
        "device_method": "fabricated all-polymer solar cells + supporting simulation",
        "evidence_note":
            "Study experimentally fabricates rigid and flexible all-polymer "
            "solar cells; numerical simulation is used to interpret FF enhancement.",
        "exclude_from_experimental_cohort": False,
    },

    "10.1016/j.optmat.2021.111588": {
        "review_status": "simulation_only",
        "device_method": "SCAPS-1D numerical simulation",
        "evidence_note":
            "Paper explicitly studies inverted BHJ organic solar cells "
            "numerically using SCAPS-1D.",
        "exclude_from_experimental_cohort": True,
    },
}


for doi, update in final_unreviewed_updates.items():

    mask = provenance_review["doi_norm"].eq(doi)

    for column, value in update.items():
        provenance_review.loc[mask, column] = value


provenance_review.to_csv(
    PROVENANCE_REVIEW_PATH,
    index=False
)


print(provenance_review["review_status"].value_counts())

print(
    "\nRemaining unreviewed:",
    (provenance_review["review_status"] == "unreviewed").sum()
)

review_status
simulation_only    18
experimental        9
mixed               3
Name: count, dtype: int64

Remaining unreviewed: 0


In [35]:
# ------------------------------------------------------------------
# Row-level inspection of mixed-provenance publications
# ------------------------------------------------------------------

mixed_dois = provenance_review.loc[
    provenance_review["review_status"].eq("mixed"),
    "doi_norm"
].tolist()

mixed_rows = (
    strict_molecular[
        strict_molecular["doi_norm"].isin(mixed_dois)
    ]
    [
        [
            "id",
            "doi_norm",
            "donor_canonical",
            "acceptor_canonical",
            "voc",
            "jsc",
            "ff",
            "pce",
            "pce_recomputed",
            "device_structure",
            "device_type",
            "etl",
            "htl",
            "active_layer_thickness",
        ]
    ]
    .sort_values(
        ["doi_norm", "pce"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

print("Mixed-provenance publications :", len(mixed_dois))
print("Rows requiring verification    :", len(mixed_rows))

display(mixed_rows)

Mixed-provenance publications : 3
Rows requiring verification    : 18


,id,doi_norm,donor_canonical,acceptor_canonical,voc,jsc,ff,pce,pce_recomputed,device_structure,device_type,etl,htl,active_layer_thickness
0,44508,10.1002/adts.202400725,P3HT,PCBM,0.550,7.08,58.80,2.33,2.289672,ITO/ZnO/P3HT:PCBM/MoO3/Ag,inverted,ZnO,MoO3,NaN
1,44509,10.1002/adts.202400725,P3HT,PCBM,0.580,8.93,52.80,2.74,2.734723,ITO/ZnO/P3HT:PCBM/PEDOT:PSS/Al,inverted,ZnO,PEDOT:PSS,NaN
2,44510,10.1002/adts.202400725,P3HT,PCBM,0.620,8.58,52.97,2.81,2.817792,ITO/ZnO/P3HT:PCBM/PEDOT:PSS/Al,inverted,ZnO,PEDOT:PSS,NaN
3,44505,10.1002/adts.202400725,PBDB-T,NCBDT,0.840,18.64,64.60,10.19,10.114810,ITO/PEDOT:PSS/PBDB-T:NCBDT/PDINO/Al,conventional,PDINO,PEDOT:PSS,NaN
4,44506,10.1002/adts.202400725,PBDB-T,NCBDT,0.850,18.74,64.54,10.33,10.280577,ITO/PEDOT:PSS/PBDB-T:NCBDT/PDINO/Al,conventional,PDINO,PEDOT:PSS,NaN
5,44511,10.1002/adts.202400725,D18,Y6,0.850,27.80,76.66,18.22,18.114758,ITO/PEDOT:PSS/D18:Y6/PDIN/Ag,conventional,PDIN,PEDOT:PSS,NaN
6,44512,10.1002/adts.202400725,D18,Y6,0.850,28.00,77.17,18.54,18.366460,ITO/PEDOT:PSS/D18:Y6/PDIN/Ag,conventional,PDIN,PEDOT:PSS,NaN
7,1783,10.1002/aenm.202000823,FTAZ,IT-M,0.950,16.30,64.00,9.80,9.910400,inverted,inverted,ZnO,MoOx,100.0
8,71588,10.3390/polym15040869,PTB7,PC70BM,0.731,16.43,68.05,8.18,8.173030,NaN,conventional,PFN-Br,PEDOT,NaN
9,71578,10.3390/polym15040869,PBDB-T,PZT-γ,0.912,23.90,68.50,14.90,14.930808,ITO/PEDOT:PSS/PBDB-T:PZT/PFN-Br/Ag,conventional,PFN-Br,PEDOT:PSS,100.0


### 3.11 Row-level resolution of mixed-provenance publications

For publication-held-out analysis, an experimental value is considered
suitable only when it represents an experimentally measured device reported
by the publication identified by `doi_norm`.

Experimental values reproduced in simulation papers from earlier literature
are not retained in the primary experimental cohort because the source DOI
does not correspond to the original experimental measurement.

This distinction prevents secondary literature tables from creating incorrect
publication provenance or duplicating measurements under a different DOI.

In [36]:
# ------------------------------------------------------------------
# Row-level provenance decisions for the three mixed publications
# ------------------------------------------------------------------

mixed_row_decisions = {

    # ==============================================================
    # Advanced Theory and Simulations 2024, 2400725
    # Simulation study. Some rows reproduce experimental literature
    # values, but none are primary measurements from this DOI.
    # ==============================================================

    44508: ("simulation", False,
            "SCAPS-simulated P3HT:PCBM device."),

    44509: ("secondary_experimental_reference", False,
            "Experimental P3HT:PCBM value reproduced from prior literature; "
            "not measured in this source publication."),

    44510: ("simulation", False,
            "SCAPS-simulated P3HT:PCBM device."),

    44505: ("secondary_experimental_reference", False,
            "Experimental PBDB-T:NCBDT value reproduced from prior literature."),

    44506: ("simulation", False,
            "Simulated PBDB-T:NCBDT comparison value."),

    44511: ("secondary_experimental_reference", False,
            "Experimental D18:Y6 value reproduced from prior literature."),

    44512: ("simulation", False,
            "SCAPS-simulated D18:Y6 validation device."),


    # ==============================================================
    # Polymers 2023, 15, 869
    # Device-modeling paper. Experimental rows are literature
    # comparators, not measurements performed in this publication.
    # ==============================================================

    71588: ("simulation", False,
            "PTB7:PC70BM simulated comparison device."),

    71578: ("secondary_experimental_reference", False,
            "14.90% measured PBDB-T:PZT reference device used to calibrate SCAPS."),

    71579: ("simulation", False,
            "14.91% calibrated SCAPS result corresponding to the reference device."),

    71580: ("secondary_experimental_reference", False,
            "PM6:Y6 experimental literature comparator."),

    71581: ("secondary_experimental_reference", False,
            "PBDB-T:PZT-gamma experimental literature comparator."),

    71583: ("secondary_experimental_reference", False,
            "PBDB-T:PN-Se experimental literature comparator."),

    71584: ("secondary_experimental_reference", False,
            "PM6:PY-IT experimental literature comparator."),

    71589: ("simulation", False,
            "PTB7:PC70BM simulated comparison device."),

    71587: ("secondary_experimental_reference", False,
            "D18:N3 experimental literature comparator."),

    71590: ("simulation", False,
            "20.87% PBDB-T:PZT SCAPS optimization from this work."),


    # ==============================================================
    # Advanced Energy Materials 2020, 2000823
    # Experimental paper. This exact FTAZ:IT-M single-junction
    # device is experimentally reported in Table 2.
    # ==============================================================

    1783: ("primary_experimental", True,
           "Experimentally fabricated FTAZ:IT-M single-junction device "
           "reported by this publication."),
}


mixed_row_review = mixed_rows.copy()

mixed_row_review["row_provenance"] = (
    mixed_row_review["id"]
    .map(lambda x: mixed_row_decisions[x][0])
)

mixed_row_review["keep_primary_experimental"] = (
    mixed_row_review["id"]
    .map(lambda x: mixed_row_decisions[x][1])
)

mixed_row_review["evidence_note"] = (
    mixed_row_review["id"]
    .map(lambda x: mixed_row_decisions[x][2])
)

display(
    mixed_row_review[
        [
            "id",
            "doi_norm",
            "donor_canonical",
            "acceptor_canonical",
            "pce",
            "row_provenance",
            "keep_primary_experimental",
            "evidence_note",
        ]
    ]
)

print("\nRow-level provenance:")
print(
    mixed_row_review["row_provenance"]
    .value_counts()
)

print(
    "\nRows retained from mixed publications:",
    mixed_row_review["keep_primary_experimental"].sum()
)

,id,doi_norm,donor_canonical,acceptor_canonical,pce,row_provenance,keep_primary_experimental,evidence_note
0,44508,10.1002/adts.202400725,P3HT,PCBM,2.33,simulation,False,SCAPS-simulated P3HT:PCBM device.
1,44509,10.1002/adts.202400725,P3HT,PCBM,2.74,secondary_experimental_reference,False,Experimental P3HT:PCBM value reproduced from p...
2,44510,10.1002/adts.202400725,P3HT,PCBM,2.81,simulation,False,SCAPS-simulated P3HT:PCBM device.
3,44505,10.1002/adts.202400725,PBDB-T,NCBDT,10.19,secondary_experimental_reference,False,Experimental PBDB-T:NCBDT value reproduced fro...
4,44506,10.1002/adts.202400725,PBDB-T,NCBDT,10.33,simulation,False,Simulated PBDB-T:NCBDT comparison value.
5,44511,10.1002/adts.202400725,D18,Y6,18.22,secondary_experimental_reference,False,Experimental D18:Y6 value reproduced from prio...
6,44512,10.1002/adts.202400725,D18,Y6,18.54,simulation,False,SCAPS-simulated D18:Y6 validation device.
7,1783,10.1002/aenm.202000823,FTAZ,IT-M,9.80,primary_experimental,True,Experimentally fabricated FTAZ:IT-M single-jun...
8,71588,10.3390/polym15040869,PTB7,PC70BM,8.18,simulation,False,PTB7:PC70BM simulated comparison device.
9,71578,10.3390/polym15040869,PBDB-T,PZT-γ,14.90,secondary_experimental_reference,False,14.90% measured PBDB-T:PZT reference device us...



Row-level provenance:
row_provenance
secondary_experimental_reference    9
simulation                          8
primary_experimental                1
Name: count, dtype: int64

Rows retained from mixed publications: 1


### 3.12 Construction of the provenance-clean experimental cohort

Publication-level and row-level provenance decisions are now combined to
construct a provisional experimental cohort.

Records are excluded when:

1. the source publication was manually verified as simulation-only; or
2. a mixed-provenance publication contains a simulated device value; or
3. a mixed-provenance publication reproduces an experimental value from an
   earlier paper rather than reporting a primary experimental measurement.

The original OPV-DB tables remain unchanged. All exclusions are retained in a
separate audit table with explicit reasons.

The cohort remains provisional until the small number of DOI records without
Crossref metadata have been assessed.

In [37]:
# ------------------------------------------------------------------
# Combine publication-level and row-level provenance decisions
# ------------------------------------------------------------------

provenance_audit = molecular_audit.copy()


# Publication-level review status
publication_status_map = (
    provenance_review
    .set_index("doi_norm")["review_status"]
    .to_dict()
)

provenance_audit["publication_review_status"] = (
    provenance_audit["doi_norm"]
    .map(publication_status_map)
    .fillna("not_flagged")
)


# Start by retaining every record
provenance_audit["provenance_keep"] = True
provenance_audit["provenance_exclusion_reason"] = pd.NA


# ------------------------------------------------------------------
# Exclude all records from publications verified as simulation-only
# ------------------------------------------------------------------

simulation_publication_mask = (
    provenance_audit["publication_review_status"]
    .eq("simulation_only")
)

provenance_audit.loc[
    simulation_publication_mask,
    "provenance_keep"
] = False

provenance_audit.loc[
    simulation_publication_mask,
    "provenance_exclusion_reason"
] = "simulation_only_publication"



# ------------------------------------------------------------------
# Apply row-level decisions for mixed publications
# ------------------------------------------------------------------

mixed_keep_map = (
    mixed_row_review
    .set_index("id")["keep_primary_experimental"]
    .to_dict()
)

mixed_provenance_map = (
    mixed_row_review
    .set_index("id")["row_provenance"]
    .to_dict()
)


mixed_ids = set(mixed_keep_map)

mixed_mask = provenance_audit["id"].isin(mixed_ids)

provenance_audit.loc[
    mixed_mask,
    "provenance_keep"
] = (
    provenance_audit.loc[mixed_mask, "id"]
    .map(mixed_keep_map)
    .astype(bool)
)


# Give excluded mixed rows their specific reason
for row_id, row_type in mixed_provenance_map.items():

    keep = mixed_keep_map[row_id]

    if not keep:

        if row_type == "simulation":
            reason = "simulation_row_in_mixed_publication"

        elif row_type == "secondary_experimental_reference":
            reason = "secondary_experimental_reference"

        else:
            reason = f"mixed_publication_{row_type}"

        provenance_audit.loc[
            provenance_audit["id"].eq(row_id),
            "provenance_exclusion_reason"
        ] = reason



        # ------------------------------------------------------------------
# Separate retained and excluded records
# ------------------------------------------------------------------

provenance_clean = (
    provenance_audit[
        provenance_audit["provenance_keep"]
    ]
    .copy()
    .reset_index(drop=True)
)

provenance_exclusions = (
    provenance_audit[
        ~provenance_audit["provenance_keep"]
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

provenance_summary = pd.DataFrame({

    "metric": [
        "Original strict molecular records",
        "Retained provenance-clean records",
        "Excluded records",
        "Original source DOIs",
        "Retained source DOIs",
        "Fully removed source DOIs",
        "PCE maximum before provenance cleaning",
        "PCE maximum after provenance cleaning",
    ],

    "value": [
        len(molecular_audit),
        len(provenance_clean),
        len(provenance_exclusions),

        molecular_audit["doi_norm"].nunique(),
        provenance_clean["doi_norm"].nunique(),

        (
            molecular_audit["doi_norm"].nunique()
            - provenance_clean["doi_norm"].nunique()
        ),

        molecular_audit["pce"].max(),
        provenance_clean["pce"].max(),
    ]
})

display(provenance_summary)



print("EXCLUSIONS BY REASON")

display(
    provenance_exclusions[
        "provenance_exclusion_reason"
    ]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="records")
)


print("\nEXCLUDED PUBLICATIONS")

excluded_doi_summary = (
    provenance_exclusions
    .groupby("doi_norm")
    .agg(
        excluded_records=("id", "size"),
        pce_min=("pce", "min"),
        pce_max=("pce", "max"),
    )
    .reset_index()
    .sort_values(
        "excluded_records",
        ascending=False
    )
)

display(excluded_doi_summary)



# ------------------------------------------------------------------
# Crossref-unresolved publications still present in the clean cohort
# ------------------------------------------------------------------

unresolved_crossref_dois = set(
    failed_metadata["doi_norm"]
)

unresolved_provenance_rows = (
    provenance_clean[
        provenance_clean["doi_norm"]
        .isin(unresolved_crossref_dois)
    ]
    [
        [
            "id",
            "doi_norm",
            "donor_canonical",
            "acceptor_canonical",
            "voc",
            "jsc",
            "ff",
            "pce",
        ]
    ]
    .sort_values(
        ["doi_norm", "pce"]
    )
)

print(
    "\nCrossref-unresolved DOIs still retained:",
    unresolved_provenance_rows["doi_norm"].nunique()
)

print(
    "Device records represented:",
    len(unresolved_provenance_rows)
)

display(unresolved_provenance_rows)

,metric,value
0,Original strict molecular records,21720.00
1,Retained provenance-clean records,21590.00
2,Excluded records,130.00
3,Original source DOIs,5459.00
4,Retained source DOIs,5439.00
5,Fully removed source DOIs,20.00
6,PCE maximum before provenance cleaning,21.83
7,PCE maximum after provenance cleaning,21.63


EXCLUSIONS BY REASON


,reason,records
0,simulation_only_publication,113
1,secondary_experimental_reference,9
2,simulation_row_in_mixed_publication,8



EXCLUDED PUBLICATIONS


,doi_norm,excluded_records,pce_min,pce_max
11,10.1016/j.rio.2024.100748,29,3.7800,12.260
12,10.1016/j.rsurfi.2025.100474,14,5.0200,11.220
4,10.1007/s11664-021-09020-5,11,0.0132,1.163
17,10.3390/polym15040869,10,8.1800,20.870
8,10.1016/j.cjph.2023.12.028,8,6.1800,18.380
2,10.1002/pssa.202500317,8,14.6800,19.350
0,10.1002/adts.202400725,7,2.3300,18.540
6,10.1007/s40243-025-00304-y,6,21.7800,21.830
10,10.1016/j.optmat.2021.111588,6,2.1400,5.460
1,10.1002/pssa.202400654,5,7.7200,11.560



Crossref-unresolved DOIs still retained: 3
Device records represented: 5


,id,doi_norm,donor_canonical,acceptor_canonical,voc,jsc,ff,pce
617,1059,10.1002/aenm.201606574,PTB7-Th,ATT-2,0.730,20.75,63.0,9.58
606,1044,10.1020/jacs.7b13239,PTB7-Th,DTPC-IC,0.863,8.53,42.4,3.12
605,1043,10.1020/jacs.7b13239,PTB7-Th,DTPC-DFIC,0.760,21.92,61.3,10.21
1007,1921,10.1021/acenergylett.8b00627,J52,IEICO-4F,0.704,20.48,56.9,8.20
1009,1923,10.1021/acenergylett.8b00627,J52,i-IEICO-4F,0.849,22.86,67.9,13.18


### 3.13 Resolution of Crossref-unmatched publications

Three source DOIs were not resolved through the automated Crossref retrieval.
These publications were therefore verified independently from publisher and
literature sources.

All three correspond to experimentally fabricated and characterized organic
photovoltaic devices. Their five associated OPV-DB records are therefore
retained.

No additional exclusions were made following this manual verification.

In [38]:
# ------------------------------------------------------------------
# Manual resolution of Crossref-unmatched publications
# ------------------------------------------------------------------

crossref_unresolved_review = pd.DataFrame([
    {
        "doi_norm": "10.1002/adma.201606574",
        "review_status": "experimental",
        "decision": "retain",
        "evidence_note":
            "Experimental PTB7-Th:ATT-2 organic solar cells; "
            "reported device PCE reaches 9.58%."
    },

    {
        "doi_norm": "10.1021/jacs.7b13239",
        "review_status": "experimental",
        "decision": "retain",
        "evidence_note":
            "DTPC-based acceptors synthesized and experimentally evaluated "
            "in PTB7-Th solar cells."
    },

    {
        "doi_norm": "10.1021/acsenergylett.8b00627",
        "review_status": "experimental",
        "decision": "retain",
        "evidence_note":
            "J52-based nonfullerene solar cells experimentally fabricated "
            "and characterized."
    },
])

UNRESOLVED_REVIEW_PATH = (
    INTERIM_DIR / "opvdb_crossref_unresolved_manual_review.csv"
)

crossref_unresolved_review.to_csv(
    UNRESOLVED_REVIEW_PATH,
    index=False
)

display(crossref_unresolved_review)

print(
    "\nAll Crossref-unresolved publications retained:",
    crossref_unresolved_review["decision"].eq("retain").all()
)



# ------------------------------------------------------------------
# Save provenance-audit outputs
# ------------------------------------------------------------------

provenance_clean.to_csv(
    INTERIM_DIR / "opvdb_provenance_clean_interim.csv",
    index=False
)

provenance_exclusions.to_csv(
    INTERIM_DIR / "opvdb_provenance_exclusions.csv",
    index=False
)

print(f"Retained records : {len(provenance_clean):,}")
print(f"Excluded records : {len(provenance_exclusions):,}")
print(f"Retained DOIs    : {provenance_clean['doi_norm'].nunique():,}")

,doi_norm,review_status,decision,evidence_note
0,10.1002/adma.201606574,experimental,retain,Experimental PTB7-Th:ATT-2 organic solar cells...
1,10.1021/jacs.7b13239,experimental,retain,DTPC-based acceptors synthesized and experimen...
2,10.1021/acsenergylett.8b00627,experimental,retain,J52-based nonfullerene solar cells experimenta...



All Crossref-unresolved publications retained: True
Retained records : 21,590
Excluded records : 130
Retained DOIs    : 5,439


## 4. Chemical-Space and Literature Concentration

### 4.1 Repetition of donors, acceptors, and donor–acceptor systems

A large number of device records does not necessarily imply a comparably large
number of independent material systems.

The provenance-clean cohort is therefore examined at the molecular-graph level
to quantify repetition of donors, acceptors, and donor–acceptor combinations.

This analysis establishes whether the dataset is dominated by a small number
of heavily studied OPV systems and characterizes the long tail of sparsely
represented chemistry.

In [41]:
# ------------------------------------------------------------------
# Frequency distributions
# ------------------------------------------------------------------

def safe_mode(series):
    """
    Return the most common non-missing label.
    If no label is available, return <NA>.
    """
    s = series.dropna()

    if len(s) == 0:
        return pd.NA

    modes = s.mode()

    if len(modes) == 0:
        return s.iloc[0]

    return modes.iloc[0]


donor_frequency = (
    concentration_audit
    .groupby(
        "donor_graph_smiles",
        dropna=False
    )
    .agg(
        records=("id", "size"),
        donor_label=("donor_canonical", safe_mode),
        dois=("doi_norm", "nunique"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)


acceptor_frequency = (
    concentration_audit
    .groupby(
        "acceptor_graph_smiles",
        dropna=False
    )
    .agg(
        records=("id", "size"),
        acceptor_label=("acceptor_canonical", safe_mode),
        dois=("doi_norm", "nunique"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)


pair_frequency = (
    concentration_audit
    .groupby(
        [
            "donor_graph_smiles",
            "acceptor_graph_smiles",
        ],
        dropna=False
    )
    .agg(
        records=("id", "size"),
        dois=("doi_norm", "nunique"),

        donor_label=(
            "donor_canonical",
            safe_mode
        ),

        acceptor_label=(
            "acceptor_canonical",
            safe_mode
        ),

        pce_min=("pce", "min"),
        pce_median=("pce", "median"),
        pce_max=("pce", "max"),
    )
    .reset_index()
    .sort_values(
        "records",
        ascending=False
    )
)


print("TOP 20 D:A SYSTEMS")

display(
    pair_frequency[
        [
            "donor_label",
            "acceptor_label",
            "records",
            "dois",
            "pce_min",
            "pce_median",
            "pce_max",
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------------
# Concentration and long-tail statistics
# ------------------------------------------------------------------

pair_counts = pair_frequency["records"]


concentration_summary = pd.DataFrame({

    "metric": [
        "Unique D:A graph pairs",
        "Pairs observed exactly once",
        "Pairs observed <= 2 times",
        "Pairs observed >= 10 times",
        "Pairs observed >= 100 times",
        "Records contributed by top 1 pair (%)",
        "Records contributed by top 5 pairs (%)",
        "Records contributed by top 10 pairs (%)",
        "Records contributed by top 20 pairs (%)",
    ],

    "value": [
        len(pair_frequency),

        int(
            (pair_counts == 1).sum()
        ),

        int(
            (pair_counts <= 2).sum()
        ),

        int(
            (pair_counts >= 10).sum()
        ),

        int(
            (pair_counts >= 100).sum()
        ),

        round(
            100 * pair_counts.iloc[:1].sum()
            / len(concentration_audit),
            2
        ),

        round(
            100 * pair_counts.iloc[:5].sum()
            / len(concentration_audit),
            2
        ),

        round(
            100 * pair_counts.iloc[:10].sum()
            / len(concentration_audit),
            2
        ),

        round(
            100 * pair_counts.iloc[:20].sum()
            / len(concentration_audit),
            2
        ),
    ]
})


display(concentration_summary)

TOP 20 D:A SYSTEMS


,donor_label,acceptor_label,records,dois,pce_min,pce_median,pce_max
4605,PM6,Y6,2559,795,0.060000,15.340,20.2800
4502,PM6,L8-BO,1220,408,0.004000,17.785,20.5200
4645,PM6,BTP-eC9,944,351,0.011000,17.200,20.4000
4181,P3HT,PCBM,782,199,0.000022,2.800,10.0200
514,PTB7-Th,PC71BM,642,176,0.060000,8.500,12.5000
1811,D18,L8-BO,501,202,4.800000,18.590,20.9400
283,PTB7,PC71BM,427,99,0.120000,6.960,16.3800
1175,PBDB-T,ITIC,409,134,0.120000,9.340,17.4037
4703,PM6,IT-4F,328,141,0.500000,12.600,14.6500
4298,P3HT,PC61BM,298,94,0.100000,3.000,15.6000


,metric,value
0,Unique D:A graph pairs,4853.00
1,Pairs observed exactly once,3008.00
2,Pairs observed <= 2 times,3832.00
3,Pairs observed >= 10 times,186.00
4,Pairs observed >= 100 times,17.00
5,Records contributed by top 1 pair (%),11.85
6,Records contributed by top 5 pairs (%),28.47
7,Records contributed by top 10 pairs (%),37.56
8,Records contributed by top 20 pairs (%),44.45


## 5. Validation Leakage and Generalization Structure

### 5.1 Chemical and publication overlap under conventional random splitting

The strong repetition and concentration of donor–acceptor systems raise the
possibility that conventional row-level random train/test splitting may place
closely related or identical chemical systems on both sides of the split.

To quantify this effect before model training, repeated 80:20 random splits
are generated.

For every split, test records are evaluated according to whether their:

- source DOI,
- donor molecular graph,
- acceptor molecular graph,
- and complete donor–acceptor graph pair

have already appeared in the training set.

This analysis measures structural and publication overlap independently of any
machine-learning algorithm.

In [42]:
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------
# Repeated random-split leakage audit
# ------------------------------------------------------------------

leakage_df = concentration_audit[
    [
        "id",
        "doi_norm",
        "donor_graph_smiles",
        "acceptor_graph_smiles",
        "graph_pair",
        "pce",
    ]
].copy()


def evaluate_split_overlap(train_df, test_df):

    train_dois = set(train_df["doi_norm"])
    train_donors = set(train_df["donor_graph_smiles"])
    train_acceptors = set(train_df["acceptor_graph_smiles"])
    train_pairs = set(train_df["graph_pair"])

    return {
        "test_records": len(test_df),

        "doi_seen_percent":
            100 * test_df["doi_norm"].isin(train_dois).mean(),

        "donor_seen_percent":
            100 * test_df["donor_graph_smiles"].isin(train_donors).mean(),

        "acceptor_seen_percent":
            100 * test_df["acceptor_graph_smiles"].isin(train_acceptors).mean(),

        "pair_seen_percent":
            100 * test_df["graph_pair"].isin(train_pairs).mean(),
    }


random_split_results = []

for seed in range(100):

    train_idx, test_idx = train_test_split(
        np.arange(len(leakage_df)),
        test_size=0.20,
        random_state=seed,
        shuffle=True,
    )

    train_part = leakage_df.iloc[train_idx]
    test_part = leakage_df.iloc[test_idx]

    result = evaluate_split_overlap(
        train_part,
        test_part
    )

    result["seed"] = seed

    random_split_results.append(result)


random_split_results = pd.DataFrame(
    random_split_results
)


# ------------------------------------------------------------------
# Summarize overlap across 100 independent random splits
# ------------------------------------------------------------------

overlap_metrics = [
    "doi_seen_percent",
    "donor_seen_percent",
    "acceptor_seen_percent",
    "pair_seen_percent",
]

random_overlap_summary = (
    random_split_results[
        overlap_metrics
    ]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .T
    .round(2)
)

display(random_overlap_summary)

,mean,std,min,max
doi_seen_percent,91.43,0.57,90.09,93.10
donor_seen_percent,94.93,0.32,94.26,95.65
acceptor_seen_percent,95.84,0.27,95.14,96.48
pair_seen_percent,84.24,0.55,82.65,85.50


In [43]:
# ------------------------------------------------------------------
# Chemical novelty categories for a representative random split
# ------------------------------------------------------------------

REPRESENTATIVE_SEED = 42

train_idx, test_idx = train_test_split(
    np.arange(len(leakage_df)),
    test_size=0.20,
    random_state=REPRESENTATIVE_SEED,
    shuffle=True,
)

train_random = leakage_df.iloc[train_idx].copy()
test_random = leakage_df.iloc[test_idx].copy()


train_donors = set(
    train_random["donor_graph_smiles"]
)

train_acceptors = set(
    train_random["acceptor_graph_smiles"]
)

train_pairs = set(
    train_random["graph_pair"]
)


def novelty_category(row):

    donor_seen = (
        row["donor_graph_smiles"]
        in train_donors
    )

    acceptor_seen = (
        row["acceptor_graph_smiles"]
        in train_acceptors
    )

    pair_seen = (
        row["graph_pair"]
        in train_pairs
    )

    if pair_seen:
        return "seen exact D:A pair"

    if donor_seen and acceptor_seen:
        return "new pairing of seen donor + seen acceptor"

    if donor_seen and not acceptor_seen:
        return "unseen acceptor"

    if not donor_seen and acceptor_seen:
        return "unseen donor"

    return "unseen donor + unseen acceptor"


test_random["novelty_category"] = (
    test_random.apply(
        novelty_category,
        axis=1
    )
)


novelty_summary = (
    test_random["novelty_category"]
    .value_counts()
    .rename_axis("novelty_category")
    .reset_index(name="test_records")
)

novelty_summary["percent_of_test"] = (
    100
    * novelty_summary["test_records"]
    / len(test_random)
).round(2)

display(novelty_summary)

,novelty_category,test_records,percent_of_test
0,seen exact D:A pair,3692,85.50
1,new pairing of seen donor + seen acceptor,260,6.02
2,unseen donor,186,4.31
3,unseen acceptor,178,4.12
4,unseen donor + unseen acceptor,2,0.05


In [44]:
# ------------------------------------------------------------------
# DOI overlap in the representative split
# ------------------------------------------------------------------

train_dois = set(train_random["doi_norm"])

test_random["publication_status"] = np.where(
    test_random["doi_norm"].isin(train_dois),
    "DOI already represented in training",
    "unseen DOI"
)

publication_overlap = (
    test_random["publication_status"]
    .value_counts()
    .rename_axis("publication_status")
    .reset_index(name="test_records")
)

publication_overlap["percent_of_test"] = (
    100
    * publication_overlap["test_records"]
    / len(test_random)
).round(2)

display(publication_overlap)

,publication_status,test_records,percent_of_test
0,DOI already represented in training,3938,91.2
1,unseen DOI,380,8.8


### 5.2 Feasibility of grouped generalization benchmarks

The random-split audit demonstrates extensive chemical and publication overlap
between training and test records.

Distinct grouped validation regimes are therefore evaluated before model
training:

1. publication-held-out;
2. donor–acceptor-pair-held-out;
3. donor-held-out;
4. acceptor-held-out.

These regimes represent different scientific deployment questions and are not
assumed to be equivalent.

Because group sizes are highly imbalanced, repeated grouped splits are first
examined for actual test-set size, chemical novelty, and target-distribution
stability before any model-performance comparison is attempted.

In [45]:
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------------
# Helper: evaluate one grouped split
# ------------------------------------------------------------------

def grouped_split_audit(df, groups, split_name, n_splits=100, test_size=0.20):

    rows = []

    splitter = GroupShuffleSplit(
        n_splits=n_splits,
        test_size=test_size,
        random_state=42
    )

    X_dummy = np.zeros(len(df))

    for split_id, (train_idx, test_idx) in enumerate(
        splitter.split(X_dummy, groups=groups)
    ):

        train = df.iloc[train_idx]
        test = df.iloc[test_idx]

        train_dois = set(train["doi_norm"])
        train_donors = set(train["donor_graph_smiles"])
        train_acceptors = set(train["acceptor_graph_smiles"])
        train_pairs = set(train["graph_pair"])

        rows.append({
            "split_type": split_name,
            "split_id": split_id,

            "train_records": len(train),
            "test_records": len(test),
            "test_percent":
                100 * len(test) / len(df),

            "test_unique_dois":
                test["doi_norm"].nunique(),

            "test_unique_donors":
                test["donor_graph_smiles"].nunique(),

            "test_unique_acceptors":
                test["acceptor_graph_smiles"].nunique(),

            "test_unique_pairs":
                test["graph_pair"].nunique(),

            "doi_seen_percent":
                100 * test["doi_norm"].isin(train_dois).mean(),

            "donor_seen_percent":
                100 * test["donor_graph_smiles"].isin(train_donors).mean(),

            "acceptor_seen_percent":
                100 * test["acceptor_graph_smiles"].isin(train_acceptors).mean(),

            "pair_seen_percent":
                100 * test["graph_pair"].isin(train_pairs).mean(),

            "train_pce_mean":
                train["pce"].mean(),

            "test_pce_mean":
                test["pce"].mean(),

            "train_pce_median":
                train["pce"].median(),

            "test_pce_median":
                test["pce"].median(),
        })

    return pd.DataFrame(rows)



    # ------------------------------------------------------------------
# Evaluate four grouped validation regimes
# ------------------------------------------------------------------

doi_split_audit = grouped_split_audit(
    leakage_df,
    groups=leakage_df["doi_norm"],
    split_name="DOI-held-out"
)

pair_split_audit = grouped_split_audit(
    leakage_df,
    groups=leakage_df["graph_pair"],
    split_name="D:A-pair-held-out"
)

donor_split_audit = grouped_split_audit(
    leakage_df,
    groups=leakage_df["donor_graph_smiles"],
    split_name="donor-held-out"
)

acceptor_split_audit = grouped_split_audit(
    leakage_df,
    groups=leakage_df["acceptor_graph_smiles"],
    split_name="acceptor-held-out"
)


grouped_audit_results = pd.concat(
    [
        doi_split_audit,
        pair_split_audit,
        donor_split_audit,
        acceptor_split_audit,
    ],
    ignore_index=True
)

In [46]:
summary_fields = [
    "test_percent",
    "doi_seen_percent",
    "donor_seen_percent",
    "acceptor_seen_percent",
    "pair_seen_percent",
]

grouped_split_summary = (
    grouped_audit_results
    .groupby("split_type")[summary_fields]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)

display(grouped_split_summary)



# ------------------------------------------------------------------
# PCE-distribution stability across grouped splits
# ------------------------------------------------------------------

grouped_audit_results["pce_mean_shift"] = (
    grouped_audit_results["test_pce_mean"]
    - grouped_audit_results["train_pce_mean"]
)

grouped_audit_results["pce_median_shift"] = (
    grouped_audit_results["test_pce_median"]
    - grouped_audit_results["train_pce_median"]
)

pce_shift_summary = (
    grouped_audit_results
    .groupby("split_type")[
        [
            "test_pce_mean",
            "pce_mean_shift",
            "pce_median_shift",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

display(pce_shift_summary)

test_percent                      doi_seen_percent        \
                          mean    std    min    max             mean   std   
split_type                                                                   
D:A-pair-held-out        20.15   5.81  10.23  35.51            53.10  6.04   
DOI-held-out             20.08   0.53  18.99  21.57             0.00  0.00   
acceptor-held-out        19.78  10.41   3.93  50.80            39.18  9.70   
donor-held-out           20.75  14.37   6.00  54.28            40.11  9.86   

                                donor_seen_percent                      \
                     min    max               mean   std    min    max   
split_type                                                               
D:A-pair-held-out  40.24  68.36              83.55  4.72  71.19  92.17   
DOI-held-out        0.00   0.00              84.97  1.27  81.72  87.82   
acceptor-held-out  21.06  65.49              86.00  8.85  64.28  97.75   
donor-held-out     18.14  58.98               0.00  0.00   0.00   0.00   

                  acceptor_seen_percent                     pair_seen_percent  \
                                   mean   std    min    max              mean   
split_type                                                                      
D:A-pair-held-out                 90.96  2.63  82.25  95.50              0.00   
DOI-held-out                      93.02  0.79  90.95  94.96             69.27   
acceptor-held-out                  0.00  0.00   0.00   0.00              0.00   
donor-held-out                    91.20  4.19  80.05  97.23              0.00   

                                       
                    std    min    max  
split_type                             
D:A-pair-held-out  0.00   0.00   0.00  
DOI-held-out       1.56  64.53  72.28  
acceptor-held-out  0.00   0.00   0.00  
donor-held-out     0.00   0.00   0.00

test_pce_mean                       pce_mean_shift         \
                           mean    std    min     max           mean    std   
split_type                                                                    
D:A-pair-held-out         9.233  1.550  6.640  12.447         -0.481  1.979   
DOI-held-out              9.697  0.191  9.322  10.173          0.010  0.240   
acceptor-held-out         9.198  2.356  5.077  14.377         -0.605  3.028   
donor-held-out            8.444  2.606  4.544  13.817         -1.018  3.761   

                                pce_median_shift                       
                     min    max             mean    std    min    max  
split_type                                                             
D:A-pair-held-out -3.789  4.228           -0.258  2.869 -5.255  6.880  
DOI-held-out      -0.460  0.610            0.004  0.370 -0.865  1.085  
acceptor-held-out -5.983  5.429           -0.297  4.328 -7.240  8.270  
donor-held-out    -6.093  7.416           -0.765  4.890 -6.960  9.390

### 5.3 Record-balanced grouped holdout construction

Conventional group-level random splitting produces highly variable test-set
sizes because donor, acceptor, and donor–acceptor groups differ greatly in
their number of associated device records.

To enable fairer comparison across validation regimes, grouped holdouts are
constructed using group membership and group size only.

For each random seed, groups are randomly ordered and accumulated until the
subset closest to the desired 20% of device records is obtained.

The photovoltaic target is not used to select or optimize the split. Therefore,
any resulting difference in PCE distribution between training and test data is
preserved as a genuine property of the held-out chemical domain.

In [47]:
# ------------------------------------------------------------------
# Record-balanced grouped splitting
# ------------------------------------------------------------------

def record_balanced_group_split(
    df,
    group_col,
    test_fraction=0.20,
    random_state=42
):
    """
    Select complete groups while targeting a desired fraction of records.

    Group sizes, but NOT the target variable, are used in split construction.
    """

    rng = np.random.default_rng(random_state)

    group_sizes = (
        df.groupby(group_col, dropna=False)
        .size()
        .rename("n_records")
        .reset_index()
    )

    # Randomize group order
    group_sizes = group_sizes.iloc[
        rng.permutation(len(group_sizes))
    ].reset_index(drop=True)

    target_n = test_fraction * len(df)

    cumulative = group_sizes["n_records"].cumsum()

    crossing_idx = int(
        np.searchsorted(
            cumulative.to_numpy(),
            target_n,
            side="left"
        )
    )

    # Candidate A: groups before crossing point
    candidate_a = group_sizes.iloc[:crossing_idx]

    # Candidate B: include crossing group
    candidate_b = group_sizes.iloc[:crossing_idx + 1]

    n_a = candidate_a["n_records"].sum()
    n_b = candidate_b["n_records"].sum()

    # Choose whichever is closer to the target number of rows
    if abs(n_a - target_n) <= abs(n_b - target_n):
        selected_groups = set(
            candidate_a[group_col]
        )
    else:
        selected_groups = set(
            candidate_b[group_col]
        )

    test_mask = df[group_col].isin(
        selected_groups
    )

    test_idx = np.flatnonzero(
        test_mask.to_numpy()
    )

    train_idx = np.flatnonzero(
        (~test_mask).to_numpy()
    )

    return train_idx, test_idx

In [49]:
# ------------------------------------------------------------------
# Audit balanced grouped splits
# ------------------------------------------------------------------

def audit_balanced_splits(
    df,
    group_col,
    split_name,
    n_splits=100
):
    """
    Generate repeated record-balanced grouped holdouts and
    measure chemical/publication overlap.
    """

    rows = []

    for seed in range(n_splits):

        train_idx, test_idx = record_balanced_group_split(
            df,
            group_col=group_col,
            test_fraction=0.20,
            random_state=seed
        )

        train = df.iloc[train_idx].copy()
        test = df.iloc[test_idx].copy()

        train_dois = set(train["doi_norm"])
        train_donors = set(train["donor_graph_smiles"])
        train_acceptors = set(train["acceptor_graph_smiles"])
        train_pairs = set(train["graph_pair"])

        rows.append({
            "split_type": split_name,
            "seed": seed,

            "train_records": len(train),
            "test_records": len(test),

            "test_percent":
                100 * len(test) / len(df),

            "doi_seen_percent":
                100 * test["doi_norm"].isin(train_dois).mean(),

            "donor_seen_percent":
                100 * test["donor_graph_smiles"].isin(train_donors).mean(),

            "acceptor_seen_percent":
                100 * test["acceptor_graph_smiles"].isin(train_acceptors).mean(),

            "pair_seen_percent":
                100 * test["graph_pair"].isin(train_pairs).mean(),

            "train_pce_mean":
                train["pce"].mean(),

            "test_pce_mean":
                test["pce"].mean(),

            "train_pce_median":
                train["pce"].median(),

            "test_pce_median":
                test["pce"].median(),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------------
# Create the four balanced split audits
# ------------------------------------------------------------------

balanced_doi = audit_balanced_splits(
    leakage_df,
    "doi_norm",
    "DOI-held-out"
)

balanced_pair = audit_balanced_splits(
    leakage_df,
    "graph_pair",
    "D:A-pair-held-out"
)

balanced_donor = audit_balanced_splits(
    leakage_df,
    "donor_graph_smiles",
    "donor-held-out"
)

balanced_acceptor = audit_balanced_splits(
    leakage_df,
    "acceptor_graph_smiles",
    "acceptor-held-out"
)


# ------------------------------------------------------------------
# Combine results
# ------------------------------------------------------------------

balanced_results = pd.concat(
    [
        balanced_doi,
        balanced_pair,
        balanced_donor,
        balanced_acceptor,
    ],
    ignore_index=True
)


print("Balanced split audits created successfully.")
print("DOI     :", balanced_doi.shape)
print("Pair    :", balanced_pair.shape)
print("Donor   :", balanced_donor.shape)
print("Acceptor:", balanced_acceptor.shape)
print("Combined:", balanced_results.shape)

Balanced split audits created successfully.
DOI     : (100, 13)
Pair    : (100, 13)
Donor   : (100, 13)
Acceptor: (100, 13)
Combined: (400, 13)


In [51]:
# ------------------------------------------------------------------
# Summarize balanced grouped validation regimes
# ------------------------------------------------------------------

balanced_results["pce_mean_shift"] = (
    balanced_results["test_pce_mean"]
    - balanced_results["train_pce_mean"]
)

balanced_results["pce_median_shift"] = (
    balanced_results["test_pce_median"]
    - balanced_results["train_pce_median"]
)


balanced_split_summary = (
    balanced_results
    .groupby("split_type")[
        [
            "test_percent",
            "doi_seen_percent",
            "donor_seen_percent",
            "acceptor_seen_percent",
            "pair_seen_percent",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)

display(balanced_split_summary)


balanced_pce_shift_summary = (
    balanced_results
    .groupby("split_type")[
        [
            "test_pce_mean",
            "pce_mean_shift",
            "pce_median_shift",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

display(balanced_pce_shift_summary)

test_percent                     doi_seen_percent        \
                          mean   std    min    max             mean   std   
split_type                                                                  
D:A-pair-held-out        19.99  1.41  14.56  25.31            51.92  4.54   
DOI-held-out             20.00  0.01  19.97  20.05             0.00  0.00   
acceptor-held-out        20.15  3.18  11.73  27.59            37.10  7.39   
donor-held-out           18.20  6.07   4.68  35.69            40.74  5.96   

                                donor_seen_percent                       \
                     min    max               mean    std    min    max   
split_type                                                                
D:A-pair-held-out  41.66  62.85              84.32   5.47  71.24  95.23   
DOI-held-out        0.00   0.00              85.01   1.14  82.24  87.38   
acceptor-held-out  18.53  51.20              85.27  11.14  60.98  97.71   
donor-held-out     25.34  55.69               0.00   0.00   0.00   0.00   

                  acceptor_seen_percent                     pair_seen_percent  \
                                   mean   std    min    max              mean   
split_type                                                                      
D:A-pair-held-out                 91.38  3.12  84.43  97.60              0.00   
DOI-held-out                      92.95  0.71  91.18  94.51             69.32   
acceptor-held-out                  0.00  0.00   0.00   0.00              0.00   
donor-held-out                    90.72  4.03  78.14  98.39              0.00   

                                       
                    std    min    max  
split_type                             
D:A-pair-held-out  0.00   0.00   0.00  
DOI-held-out       1.51  65.89  72.72  
acceptor-held-out  0.00   0.00   0.00  
donor-held-out     0.00   0.00   0.00

test_pce_mean                       pce_mean_shift         \
                           mean    std    min     max           mean    std   
split_type                                                                    
D:A-pair-held-out         9.630  1.677  5.938  13.310         -0.063  2.097   
DOI-held-out              9.656  0.159  9.316  10.048         -0.042  0.198   
acceptor-held-out         9.902  2.846  5.496  15.475          0.241  3.569   
donor-held-out            7.827  2.218  4.371  14.521         -2.217  2.898   

                                pce_median_shift                       
                     min    max             mean    std    min    max  
split_type                                                             
D:A-pair-held-out -4.391  4.675            0.201  3.126 -5.560  7.280  
DOI-held-out      -0.467  0.448           -0.050  0.302 -0.675  0.685  
acceptor-held-out -5.456  6.582            0.961  4.812 -6.315  8.950  
donor-held-out    -6.033  7.147           -2.588  3.676 -6.950  9.030

### 5.4 Balanced grouped cross-validation

Repeated record-balanced holdouts remained unstable for highly imbalanced
chemical groups, particularly donor identities.

To avoid systematically favoring group combinations that happen to produce a
20% test fraction, the grouped benchmarks are instead formulated as
five-fold cross-validation problems.

Complete groups are assigned to one of five folds using group size only. Large
groups are allocated first to the currently smallest fold, producing
approximately balanced numbers of device records while ensuring that every
group and every device record appears in a held-out test fold exactly once.

The photovoltaic target is not used in fold construction.

In [52]:
# ------------------------------------------------------------------
# Balanced grouped K-fold assignment
# ------------------------------------------------------------------

def balanced_group_folds(
    df,
    group_col,
    n_splits=5,
    random_state=42
):
    """
    Assign complete groups to folds while balancing record counts.

    Groups are ordered from largest to smallest. Random numbers are used
    only to break ties between groups of equal size.
    """

    rng = np.random.default_rng(random_state)

    group_sizes = (
        df.groupby(group_col, dropna=False)
        .size()
        .rename("n_records")
        .reset_index()
    )

    # Random tie-breaker; group size remains the primary criterion
    group_sizes["tie_break"] = rng.random(
        len(group_sizes)
    )

    group_sizes = (
        group_sizes
        .sort_values(
            ["n_records", "tie_break"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

    fold_loads = np.zeros(
        n_splits,
        dtype=int
    )

    fold_groups = [
        []
        for _ in range(n_splits)
    ]

    for _, row in group_sizes.iterrows():

        # Assign next group to currently smallest fold
        smallest_load = fold_loads.min()

        candidate_folds = np.flatnonzero(
            fold_loads == smallest_load
        )

        chosen_fold = rng.choice(
            candidate_folds
        )

        group_value = row[group_col]
        group_n = int(row["n_records"])

        fold_groups[chosen_fold].append(
            group_value
        )

        fold_loads[chosen_fold] += group_n


    # Create group -> fold mapping
    group_to_fold = {}

    for fold_id, groups in enumerate(
        fold_groups
    ):
        for group in groups:
            group_to_fold[group] = fold_id


    row_fold = df[group_col].map(
        group_to_fold
    ).astype(int)

    fold_summary = pd.DataFrame({
        "fold": np.arange(n_splits),
        "test_records": fold_loads,
    })

    fold_summary["test_percent"] = (
        100
        * fold_summary["test_records"]
        / len(df)
    )

    return row_fold, fold_summary

In [53]:
# ------------------------------------------------------------------
# Build five-fold grouped benchmarks
# ------------------------------------------------------------------

cv_assignments = {}
cv_size_summaries = {}

group_definitions = {
    "DOI-held-out":
        "doi_norm",

    "D:A-pair-held-out":
        "graph_pair",

    "donor-held-out":
        "donor_graph_smiles",

    "acceptor-held-out":
        "acceptor_graph_smiles",
}


for split_name, group_col in group_definitions.items():

    fold_assignment, fold_sizes = (
        balanced_group_folds(
            leakage_df,
            group_col=group_col,
            n_splits=5,
            random_state=42
        )
    )

    cv_assignments[
        split_name
    ] = fold_assignment

    fold_sizes[
        "split_type"
    ] = split_name

    cv_size_summaries[
        split_name
    ] = fold_sizes


cv_fold_sizes = pd.concat(
    cv_size_summaries.values(),
    ignore_index=True
)

display(
    cv_fold_sizes[
        [
            "split_type",
            "fold",
            "test_records",
            "test_percent",
        ]
    ]
)

,split_type,fold,test_records,test_percent
0,DOI-held-out,0,4318,20.000000
1,DOI-held-out,1,4318,20.000000
2,DOI-held-out,2,4318,20.000000
3,DOI-held-out,3,4318,20.000000
4,DOI-held-out,4,4318,20.000000
5,D:A-pair-held-out,0,4318,20.000000
6,D:A-pair-held-out,1,4318,20.000000
7,D:A-pair-held-out,2,4318,20.000000
8,D:A-pair-held-out,3,4318,20.000000
9,D:A-pair-held-out,4,4318,20.000000


In [54]:
# ------------------------------------------------------------------
# Audit chemical/publication overlap in every CV fold
# ------------------------------------------------------------------

cv_audit_rows = []

for split_name, fold_assignment in cv_assignments.items():

    for fold in range(5):

        test_mask = (
            fold_assignment == fold
        )

        train = leakage_df.loc[
            ~test_mask
        ]

        test = leakage_df.loc[
            test_mask
        ]

        train_dois = set(
            train["doi_norm"]
        )

        train_donors = set(
            train["donor_graph_smiles"]
        )

        train_acceptors = set(
            train["acceptor_graph_smiles"]
        )

        train_pairs = set(
            train["graph_pair"]
        )

        cv_audit_rows.append({
            "split_type": split_name,
            "fold": fold,

            "test_records":
                len(test),

            "test_percent":
                100 * len(test)
                / len(leakage_df),

            "doi_seen_percent":
                100 * test["doi_norm"]
                .isin(train_dois).mean(),

            "donor_seen_percent":
                100 * test[
                    "donor_graph_smiles"
                ]
                .isin(train_donors).mean(),

            "acceptor_seen_percent":
                100 * test[
                    "acceptor_graph_smiles"
                ]
                .isin(train_acceptors).mean(),

            "pair_seen_percent":
                100 * test["graph_pair"]
                .isin(train_pairs).mean(),

            "train_pce_mean":
                train["pce"].mean(),

            "test_pce_mean":
                test["pce"].mean(),

            "train_pce_median":
                train["pce"].median(),

            "test_pce_median":
                test["pce"].median(),
        })


cv_audit = pd.DataFrame(
    cv_audit_rows
)

cv_audit["pce_mean_shift"] = (
    cv_audit["test_pce_mean"]
    - cv_audit["train_pce_mean"]
)

cv_audit["pce_median_shift"] = (
    cv_audit["test_pce_median"]
    - cv_audit["train_pce_median"]
)



cv_overlap_summary = (
    cv_audit
    .groupby("split_type")[
        [
            "test_percent",
            "doi_seen_percent",
            "donor_seen_percent",
            "acceptor_seen_percent",
            "pair_seen_percent",
        ]
    ]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .round(2)
)

display(cv_overlap_summary)


cv_pce_summary = (
    cv_audit
    .groupby("split_type")[
        [
            "test_pce_mean",
            "pce_mean_shift",
            "pce_median_shift",
        ]
    ]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .round(3)
)

display(cv_pce_summary)

test_percent                     doi_seen_percent        \
                          mean   std    min    max             mean   std   
split_type                                                                  
D:A-pair-held-out         20.0  0.00  20.00  20.00            52.21  3.13   
DOI-held-out              20.0  0.00  20.00  20.00             0.00  0.00   
acceptor-held-out         20.0  0.00  20.00  20.00            37.58  5.78   
donor-held-out            20.0  6.44  17.11  31.53            37.23  7.47   

                                donor_seen_percent                       \
                     min    max               mean    std    min    max   
split_type                                                                
D:A-pair-held-out  46.90  54.35              84.63   0.80  83.81  85.71   
DOI-held-out        0.00   0.00              84.80   2.48  82.05  88.81   
acceptor-held-out  27.95  43.54              85.35  11.29  66.26  95.37   
donor-held-out     24.33  42.64               0.00   0.00   0.00   0.00   

                  acceptor_seen_percent                     pair_seen_percent  \
                                   mean   std    min    max              mean   
split_type                                                                      
D:A-pair-held-out                 91.41  0.98  90.27  92.68              0.00   
DOI-held-out                      93.02  1.27  91.48  94.37             69.03   
acceptor-held-out                  0.00  0.00   0.00   0.00              0.00   
donor-held-out                    91.19  5.07  84.17  96.32              0.00   

                                       
                    std    min    max  
split_type                             
D:A-pair-held-out  0.00   0.00   0.00  
DOI-held-out       2.12  65.77  71.49  
acceptor-held-out  0.00   0.00   0.00  
donor-held-out     0.00   0.00   0.00

test_pce_mean                       pce_mean_shift         \
                           mean    std    min     max           mean    std   
split_type                                                                    
D:A-pair-held-out         9.690  1.839  6.618  11.399         -0.000  2.299   
DOI-held-out              9.690  0.116  9.511   9.832          0.000  0.144   
acceptor-held-out         9.690  2.997  6.019  13.015          0.000  3.746   
donor-held-out            8.841  3.732  4.781  14.730         -0.768  5.015   

                                pce_median_shift                       
                     min    max             mean    std    min    max  
split_type                                                             
D:A-pair-held-out -3.839  2.136            0.209  3.427 -5.070  4.355  
DOI-held-out      -0.223  0.179            0.006  0.410 -0.680  0.375  
acceptor-held-out -4.588  4.156            0.902  5.179 -5.130  6.490  
donor-held-out    -5.922  7.362           -0.840  5.972 -6.975  9.110

In [55]:
for split_name, assignment in cv_assignments.items():

    print(
        split_name,
        "folds =", sorted(assignment.unique()),
        "| missing assignments =", assignment.isna().sum(),
        "| records =", len(assignment)
    )

DOI-held-out folds = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | missing assignments = 0 | records = 21590
D:A-pair-held-out folds = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | missing assignments = 0 | records = 21590
donor-held-out folds = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | missing assignments = 0 | records = 21590
acceptor-held-out folds = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | missing assignments = 0 | records = 21590


### 5.5 Common-fold feasibility under extreme donor imbalance

Five-fold grouped cross-validation produces well-balanced publication,
donor–acceptor-pair, and acceptor holdouts, but donor holdout cannot achieve
20% test folds because the largest donor group alone represents more than
30% of all device records.

A three-fold grouped design is therefore evaluated as a potential common
validation framework. A common number of folds would keep the approximate
training fraction comparable across generalization regimes while preserving
complete group separation.

This is evaluated before final benchmark selection rather than imposed in
advance.

In [56]:
# ------------------------------------------------------------------
# Test a common balanced 3-fold design for all grouped regimes
# ------------------------------------------------------------------

cv3_assignments = {}
cv3_size_summaries = {}

for split_name, group_col in group_definitions.items():

    assignment, fold_sizes = balanced_group_folds(
        leakage_df,
        group_col=group_col,
        n_splits=3,
        random_state=42
    )

    cv3_assignments[split_name] = assignment

    fold_sizes["split_type"] = split_name

    cv3_size_summaries[split_name] = fold_sizes


cv3_fold_sizes = pd.concat(
    cv3_size_summaries.values(),
    ignore_index=True
)

display(
    cv3_fold_sizes[
        [
            "split_type",
            "fold",
            "test_records",
            "test_percent",
        ]
    ]
)



# ------------------------------------------------------------------
# Audit the common 3-fold grouped benchmarks
# ------------------------------------------------------------------

cv3_audit_rows = []

for split_name, fold_assignment in cv3_assignments.items():

    for fold in range(3):

        test_mask = fold_assignment.eq(fold)

        train = leakage_df.loc[~test_mask]
        test = leakage_df.loc[test_mask]

        train_dois = set(train["doi_norm"])
        train_donors = set(train["donor_graph_smiles"])
        train_acceptors = set(train["acceptor_graph_smiles"])
        train_pairs = set(train["graph_pair"])

        cv3_audit_rows.append({
            "split_type": split_name,
            "fold": fold,

            "train_records": len(train),
            "test_records": len(test),

            "test_percent":
                100 * len(test) / len(leakage_df),

            "doi_seen_percent":
                100 * test["doi_norm"].isin(train_dois).mean(),

            "donor_seen_percent":
                100 * test["donor_graph_smiles"]
                .isin(train_donors).mean(),

            "acceptor_seen_percent":
                100 * test["acceptor_graph_smiles"]
                .isin(train_acceptors).mean(),

            "pair_seen_percent":
                100 * test["graph_pair"]
                .isin(train_pairs).mean(),

            "train_pce_mean":
                train["pce"].mean(),

            "test_pce_mean":
                test["pce"].mean(),

            "train_pce_median":
                train["pce"].median(),

            "test_pce_median":
                test["pce"].median(),
        })


cv3_audit = pd.DataFrame(cv3_audit_rows)

cv3_audit["pce_mean_shift"] = (
    cv3_audit["test_pce_mean"]
    - cv3_audit["train_pce_mean"]
)

cv3_audit["pce_median_shift"] = (
    cv3_audit["test_pce_median"]
    - cv3_audit["train_pce_median"]
)



cv3_overlap_summary = (
    cv3_audit
    .groupby("split_type")[
        [
            "test_percent",
            "doi_seen_percent",
            "donor_seen_percent",
            "acceptor_seen_percent",
            "pair_seen_percent",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)

display(cv3_overlap_summary)


cv3_pce_summary = (
    cv3_audit
    .groupby("split_type")[
        [
            "test_pce_mean",
            "pce_mean_shift",
            "pce_median_shift",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

display(cv3_pce_summary)

,split_type,fold,test_records,test_percent
0,DOI-held-out,0,7197,33.334877
1,DOI-held-out,1,7196,33.330245
2,DOI-held-out,2,7197,33.334877
3,D:A-pair-held-out,0,7196,33.330245
4,D:A-pair-held-out,1,7197,33.334877
5,D:A-pair-held-out,2,7197,33.334877
6,donor-held-out,0,7197,33.334877
7,donor-held-out,1,7197,33.334877
8,donor-held-out,2,7196,33.330245
9,acceptor-held-out,0,7197,33.334877


test_percent                    doi_seen_percent        \
                          mean  std    min    max             mean   std   
split_type                                                                 
D:A-pair-held-out        33.33  0.0  33.33  33.33            49.30  1.00   
DOI-held-out             33.33  0.0  33.33  33.33             0.00  0.00   
acceptor-held-out        33.33  0.0  33.33  33.33            34.01  3.33   
donor-held-out           33.33  0.0  33.33  33.33            31.64  3.93   

                                donor_seen_percent                      \
                     min    max               mean   std    min    max   
split_type                                                               
D:A-pair-held-out  48.60  50.44              84.15  1.00  83.08  85.05   
DOI-held-out        0.00   0.00              84.06  0.70  83.53  84.85   
acceptor-held-out  30.56  37.21              84.92  9.62  74.19  92.80   
donor-held-out     27.23  34.78               0.00  0.00   0.00   0.00   

                  acceptor_seen_percent                     pair_seen_percent  \
                                   mean   std    min    max              mean   
split_type                                                                      
D:A-pair-held-out                 90.60  0.74  89.86  91.34              0.00   
DOI-held-out                      92.38  0.42  91.93  92.76             68.02   
acceptor-held-out                  0.00  0.00   0.00   0.00              0.00   
donor-held-out                    91.43  4.03  87.85  95.79              0.00   

                                       
                    std    min    max  
split_type                             
D:A-pair-held-out  0.00   0.00   0.00  
DOI-held-out       0.87  67.27  68.97  
acceptor-held-out  0.00   0.00   0.00  
donor-held-out     0.00   0.00   0.00

test_pce_mean                       pce_mean_shift         \
                           mean    std    min     max           mean    std   
split_type                                                                    
D:A-pair-held-out         9.690  0.183  9.542   9.895            0.0  0.275   
DOI-held-out              9.690  0.029  9.672   9.723            0.0  0.043   
acceptor-held-out         9.689  1.916  7.641  11.438           -0.0  2.874   
donor-held-out            9.689  4.106  6.318  14.262           -0.0  6.158   

                                pce_median_shift                      
                     min    max             mean    std   min    max  
split_type                                                            
D:A-pair-held-out -0.221  0.308            0.107  1.166 -0.91  1.380  
DOI-held-out      -0.026  0.050            0.002  0.098 -0.10  0.095  
acceptor-held-out -3.073  2.623            0.557  4.384 -4.28  4.270  
donor-held-out    -5.056  6.858           -0.207  8.111 -6.59  8.920

In [57]:
# ------------------------------------------------------------------
# Save frozen grouped-CV assignments
# ------------------------------------------------------------------

split_assignments = leakage_df[
    [
        "id",
        "doi_norm",
        "donor_graph_smiles",
        "acceptor_graph_smiles",
        "graph_pair",
    ]
].copy()

for split_name, assignment in cv3_assignments.items():

    safe_name = (
        split_name
        .lower()
        .replace(":", "")
        .replace("-", "_")
        .replace(" ", "_")
    )

    split_assignments[
        f"fold_{safe_name}"
    ] = assignment.to_numpy()


SPLIT_ASSIGNMENT_PATH = (
    INTERIM_DIR / "opvdb_grouped_cv_assignments.csv"
)

split_assignments.to_csv(
    SPLIT_ASSIGNMENT_PATH,
    index=False
)

print("Saved:")
print(SPLIT_ASSIGNMENT_PATH)

print(
    "Records:",
    len(split_assignments)
)

Saved:
<PROJECT_ROOT>\data\interim\opvdb_grouped_cv_assignments.csv
Records: 21590


## 6. Processing-Information Coverage and Selection Bias

### 6.1 Does processing-data availability define a biased chemical subset?

Processing variables are incompletely reported across the OPV literature.

Restricting analysis to records with processing information may therefore
change not only sample size but also the chemical and performance distribution
of the modeling cohort.

For each major processing variable, records with and without reported values
are compared in terms of:

- sample size,
- source-publication coverage,
- donor and acceptor diversity,
- donor–acceptor pair diversity,
- PCE distribution,
- and representation of heavily studied material systems.

This analysis determines whether complete-case processing cohorts can be
treated as representative subsets of the broader OPV database.

In [59]:
# ------------------------------------------------------------------
# Processing-variable availability audit
# ------------------------------------------------------------------

# Work on a copy and add the already-defined molecular D:A identity
processing_audit = provenance_clean.copy()

processing_audit["graph_pair"] = list(
    zip(
        processing_audit["donor_graph_smiles"],
        processing_audit["acceptor_graph_smiles"]
    )
)


processing_fields = [
    "d_a_ratio",
    "solvent",
    "additive",
    "additive_ratio",
    "active_layer_thickness",
    "annealing_temp",
]


processing_selection_rows = []

for field in processing_fields:

    available = processing_audit[
        processing_audit[field].notna()
    ].copy()

    missing = processing_audit[
        processing_audit[field].isna()
    ].copy()

    processing_selection_rows.append({

        "field": field,

        "available_records":
            len(available),

        "available_percent":
            100 * len(available) / len(processing_audit),

        "available_dois":
            available["doi_norm"].nunique(),

        "available_donors":
            available["donor_graph_smiles"].nunique(),

        "available_acceptors":
            available["acceptor_graph_smiles"].nunique(),

        "available_pairs":
            available["graph_pair"].nunique(),

        "available_pce_mean":
            available["pce"].mean(),

        "missing_pce_mean":
            missing["pce"].mean(),

        "available_pce_median":
            available["pce"].median(),

        "missing_pce_median":
            missing["pce"].median(),
    })


processing_selection_summary = pd.DataFrame(
    processing_selection_rows
)

processing_selection_summary["mean_pce_difference"] = (
    processing_selection_summary["available_pce_mean"]
    - processing_selection_summary["missing_pce_mean"]
)

processing_selection_summary["median_pce_difference"] = (
    processing_selection_summary["available_pce_median"]
    - processing_selection_summary["missing_pce_median"]
)

display(
    processing_selection_summary.round(3)
)


processing_cohorts = {

    "ratio_only":
        ["d_a_ratio"],

    "ratio_solvent":
        ["d_a_ratio", "solvent"],

    "ratio_solvent_thickness":
        [
            "d_a_ratio",
            "solvent",
            "active_layer_thickness",
        ],

    "ratio_solvent_annealing":
        [
            "d_a_ratio",
            "solvent",
            "annealing_temp",
        ],

    "ratio_solvent_thickness_annealing":
        [
            "d_a_ratio",
            "solvent",
            "active_layer_thickness",
            "annealing_temp",
        ],

    "all_six_processing_fields":
        processing_fields,
}


processing_cohort_rows = []

for cohort_name, required_fields in processing_cohorts.items():

    cohort = processing_audit.dropna(
        subset=required_fields
    ).copy()

    processing_cohort_rows.append({

        "cohort": cohort_name,

        "required_fields":
            len(required_fields),

        "records":
            len(cohort),

        "percent_of_full":
            100 * len(cohort) / len(processing_audit),

        "dois":
            cohort["doi_norm"].nunique(),

        "donor_graphs":
            cohort["donor_graph_smiles"].nunique(),

        "acceptor_graphs":
            cohort["acceptor_graph_smiles"].nunique(),

        "graph_pairs":
            cohort["graph_pair"].nunique(),

        "pce_mean":
            cohort["pce"].mean(),

        "pce_median":
            cohort["pce"].median(),
    })


processing_cohort_summary = pd.DataFrame(
    processing_cohort_rows
)


total_donors = processing_audit[
    "donor_graph_smiles"
].nunique()

total_acceptors = processing_audit[
    "acceptor_graph_smiles"
].nunique()

total_pairs = processing_audit[
    "graph_pair"
].nunique()


processing_cohort_summary[
    "donor_space_retained_percent"
] = (
    100
    * processing_cohort_summary["donor_graphs"]
    / total_donors
)

processing_cohort_summary[
    "acceptor_space_retained_percent"
] = (
    100
    * processing_cohort_summary["acceptor_graphs"]
    / total_acceptors
)

processing_cohort_summary[
    "pair_space_retained_percent"
] = (
    100
    * processing_cohort_summary["graph_pairs"]
    / total_pairs
)


display(
    processing_cohort_summary[
        [
            "cohort",
            "records",
            "percent_of_full",
            "donor_space_retained_percent",
            "acceptor_space_retained_percent",
            "pair_space_retained_percent",
            "pce_mean",
            "pce_median",
        ]
    ].round(2)
)

,field,available_records,available_percent,available_dois,available_donors,available_acceptors,available_pairs,available_pce_mean,missing_pce_mean,available_pce_median,missing_pce_median,mean_pce_difference,median_pce_difference
0,d_a_ratio,11183,51.797,3145,1660,1028,3474,7.610,11.925,6.700,12.880,-4.315,-6.18
1,solvent,11948,55.340,3168,1556,949,3255,8.228,11.501,7.290,12.040,-3.273,-4.75
2,additive,7576,35.090,2186,928,640,1933,10.105,9.465,9.510,9.100,0.640,0.41
3,additive_ratio,5050,23.390,1520,740,434,1385,9.486,9.752,8.815,9.435,-0.266,-0.62
4,active_layer_thickness,6531,30.250,1697,941,487,1796,7.816,10.502,6.980,10.500,-2.686,-3.52
5,annealing_temp,5390,24.965,1634,594,586,1439,9.226,9.844,9.070,9.320,-0.618,-0.25


,cohort,records,percent_of_full,donor_space_retained_percent,acceptor_space_retained_percent,pair_space_retained_percent,pce_mean,pce_median
0,ratio_only,11183,51.80,83.59,71.59,71.58,7.61,6.70
1,ratio_solvent,9129,42.28,71.85,55.64,57.80,7.24,6.20
2,ratio_solvent_thickness,4383,20.30,39.98,24.30,28.46,6.88,5.86
3,ratio_solvent_annealing,3814,17.67,24.97,30.50,22.95,8.59,8.21
4,ratio_solvent_thickness_annealing,1738,8.05,13.70,12.67,10.74,8.52,8.00
5,all_six_processing_fields,711,3.29,5.79,4.87,3.98,10.79,11.06


### 6.2 Chemical enrichment induced by processing-data availability

Differences in sample size and PCE distribution demonstrate that processing
information is not missing at random.

To determine whether processing availability also changes the chemical
composition of the dataset, donor–acceptor pair frequencies in selected
processing cohorts are compared with their frequencies in the full
provenance-clean benchmark.

Enrichment is interpreted descriptively and is not used as a statistical
exclusion criterion.

In [60]:
# ------------------------------------------------------------------
# Chemical enrichment in processing-available cohorts
# ------------------------------------------------------------------

full_pair_counts = (
    processing_audit["graph_pair"]
    .value_counts()
)

full_pair_fraction = (
    full_pair_counts / len(processing_audit)
)


def pair_enrichment_for_field(field, top_n=15):

    cohort = processing_audit[
        processing_audit[field].notna()
    ].copy()

    cohort_counts = (
        cohort["graph_pair"]
        .value_counts()
    )

    cohort_fraction = (
        cohort_counts / len(cohort)
    )

    common_pairs = cohort_fraction.index.intersection(
        full_pair_fraction.index
    )

    enrichment = pd.DataFrame({
        "graph_pair": common_pairs,
        "cohort_fraction": cohort_fraction.loc[common_pairs].values,
        "full_fraction": full_pair_fraction.loc[common_pairs].values,
        "cohort_records": cohort_counts.loc[common_pairs].values,
    })

    enrichment["enrichment_ratio"] = (
        enrichment["cohort_fraction"]
        / enrichment["full_fraction"]
    )

    # Attach readable labels
    label_lookup = (
        processing_audit
        .groupby("graph_pair")
        .agg(
            donor_label=("donor_canonical", safe_mode),
            acceptor_label=("acceptor_canonical", safe_mode),
        )
    )

    enrichment = enrichment.merge(
        label_lookup,
        left_on="graph_pair",
        right_index=True,
        how="left"
    )

    return (
        enrichment[
            enrichment["cohort_records"] >= 10
        ]
        .sort_values(
            "enrichment_ratio",
            ascending=False
        )
        .head(top_n)
    )


for field in [
    "d_a_ratio",
    "solvent",
    "active_layer_thickness",
    "annealing_temp",
]:

    print("\n" + "=" * 80)
    print(field.upper())
    print("=" * 80)

    display(
        pair_enrichment_for_field(field)
        [
            [
                "donor_label",
                "acceptor_label",
                "cohort_records",
                "cohort_fraction",
                "full_fraction",
                "enrichment_ratio",
            ]
        ]
        .round(3)
    )


D_A_RATIO


,donor_label,acceptor_label,cohort_records,cohort_fraction,full_fraction,enrichment_ratio
29,AnE-PVstat,PCBM,27,0.002,0.001,1.931
37,PBDT-TPD-8,PNDI-T-5,22,0.002,0.001,1.931
94,PM6,IEICO-4F,10,0.001,0.000,1.931
67,PBDTTT-EFT,PC70BM,13,0.001,0.001,1.931
68,P(BDTT-PDBT),PC70BM,13,0.001,0.001,1.931
73,PTB7-Th,PNDI2OD-T2,12,0.001,0.001,1.931
75,PTB7-Th,"3,9-bis(2-methylene-(3-(1,1-dicyanomethylene)-...",12,0.001,0.001,1.931
77,PBDTTPD-DDT,PC71BM,12,0.001,0.001,1.931
74,RTh-BSe-ThR,PC61BM,12,0.001,0.001,1.931
97,P3HT,PDPP2TzT,10,0.001,0.000,1.931



SOLVENT


,donor_label,acceptor_label,cohort_records,cohort_fraction,full_fraction,enrichment_ratio
67,PBDTBT,PC70BM,15,0.001,0.001,1.807
62,TQ1,PC71BM,15,0.001,0.001,1.807
58,PBTTbT,PC70BM,16,0.001,0.001,1.807
108,ZR1-C3,L8-BO,10,0.001,0.000,1.807
43,PBDT-TPD-8,PNDI-T-5,22,0.002,0.001,1.807
33,DRCN5T,PC71BM,28,0.002,0.001,1.807
26,PBDTTT-C-T,PC70BM,37,0.003,0.002,1.807
87,DT-PDPP2T-TT,PCBM,12,0.001,0.001,1.807
88,PTB7-Th,"3,9-bis(2-methylene-(3-(1,1-dicyanomethylene)-...",12,0.001,0.001,1.807
90,PBDTTPD-DDT,PC71BM,12,0.001,0.001,1.807



ACTIVE_LAYER_THICKNESS


,donor_label,acceptor_label,cohort_records,cohort_fraction,full_fraction,enrichment_ratio
67,PM6,IEICO-4F,10,0.002,0.000,3.306
60,J52,IEICO,10,0.002,0.000,3.306
49,DT-PDPP2T-TT,PCBM,12,0.002,0.001,3.306
24,AnE-PVstat,PCBM,27,0.004,0.001,3.306
65,PM2,Y6-BO,10,0.002,0.000,3.306
37,PBTTbT,PC70BM,16,0.002,0.001,3.306
50,PBDTTPD-DDT,PC71BM,12,0.002,0.001,3.306
58,PTP8,P(NDI2HD-T),11,0.002,0.001,3.306
40,C1,PCBM,14,0.002,0.001,3.306
43,PBDD-ff4T,PC71BM,14,0.002,0.001,3.306



ANNEALING_TEMP


,donor_label,acceptor_label,cohort_records,cohort_fraction,full_fraction,enrichment_ratio
58,PM6,IEICO-4F,10,0.002,0.000,4.006
45,RTh-BSe-ThR,PC61BM,12,0.002,0.001,4.006
55,PM2,Y6-BO,10,0.002,0.000,4.006
20,P3HT,ZY-4Cl,22,0.004,0.001,3.389
38,P3HT,PC70BM,14,0.003,0.001,3.299
44,P3HT,SF(DPPB)4,12,0.002,0.001,3.204
27,TPD-3F,IT-4F,18,0.003,0.001,3.135
33,PBDT-TPD-8,PNDI-T-5,17,0.003,0.001,3.095
29,PM6,ITIC-4F,18,0.003,0.001,3.004
39,PCE10-2F,Y6,13,0.002,0.001,2.893


### 6.3 Processing-audit conclusion

Processing-variable availability is strongly associated with both chemical
composition and photovoltaic-performance distribution.

Pair-level enrichment analysis further shows that some donor–acceptor systems
have near-complete reporting of particular processing variables even when the
corresponding field is sparse across the database as a whole. Consequently,
processing-data availability is clustered within specific regions of OPV
chemical space rather than randomly distributed.

Complete-case restriction therefore produces a chemically selective modeling
domain. In particular, requiring all six examined processing fields retains
only 3.29% of device records and approximately 3.98% of donor–acceptor pair
space.

OPV-DB is therefore retained primarily as the large-scale molecular
structure–performance and generalization benchmark. Dedicated
structure–processing–performance analyses will use the processing-focused
Wen–Zhang–Ma database rather than treating the small OPV-DB complete-case
subset as representative of the broader literature.

## 7. Cross-Dataset Identity and Overlap

### 7.1 OPV-DB versus Wen–Zhang–Ma global processing benchmark

The Wen–Zhang–Ma global processing dataset is evaluated for chemical overlap
with the provenance-clean OPV-DB benchmark before it is used for
cross-dataset or transfer-learning experiments.

Overlap is assessed progressively using:

1. reported material labels,
2. raw molecular SMILES,
3. RDKit-canonicalized molecular graphs,
4. complete donor–acceptor molecular-graph pairs.

This hierarchy is necessary because identical materials may be represented by
different names or different valid SMILES strings across independently curated
databases.

In [61]:
# ------------------------------------------------------------------
# Locate Wen/Ma global processing dataset
# ------------------------------------------------------------------

wen_global_candidates = list(
    WEN_DIR.rglob("global-all-nine-parameters.csv")
)

print("Matches found:", len(wen_global_candidates))

for path in wen_global_candidates:
    print(path)

Matches found: 1
<PROJECT_ROOT>\data\raw\opv-multi-tier-ml-database\opv-multi-tier-ml-database\03-global-model\global-all-nine-parameters.csv


### 7.2 Loading and structure normalization of the Wen–Zhang–Ma global dataset

The global nine-parameter Wen–Zhang–Ma dataset is loaded from the untouched
raw-data directory.

Donor and acceptor SMILES are processed using the same RDKit canonicalization
procedure applied to OPV-DB. This avoids introducing database-specific
structure-normalization rules when assessing cross-dataset chemical overlap.

No records are removed at this stage. SMILES that cannot be parsed are
explicitly identified before overlap calculations are performed.

In [63]:
# ------------------------------------------------------------------
# Load Wen/Ma global processing dataset
# ------------------------------------------------------------------

WEN_GLOBAL_PATH = wen_global_candidates[0]

wen_global = pd.read_csv(
    WEN_GLOBAL_PATH,
    low_memory=False
)

print("Wen/Ma global dataset loaded")
print(f"Records : {wen_global.shape[0]:,}")
print(f"Columns : {wen_global.shape[1]:,}")

print("\nFirst 20 column names:")
print(wen_global.columns[:20].tolist())



# ------------------------------------------------------------------
# Verify required molecular-identity fields
# ------------------------------------------------------------------

wen_required_identity_fields = [
    "Name_Donor",
    "Smiles_Donor",
    "Name_Acceptor",
    "Smiles_Acceptor",
    "PCE (%)",
]

wen_identity_check = pd.DataFrame({
    "field": wen_required_identity_fields,
    "exists": [
        field in wen_global.columns
        for field in wen_required_identity_fields
    ],
    "missing_records": [
        wen_global[field].isna().sum()
        if field in wen_global.columns
        else np.nan
        for field in wen_required_identity_fields
    ],
})

display(wen_identity_check)



# ------------------------------------------------------------------
# Canonicalize Wen/Ma donor and acceptor molecular structures
# using the SAME function previously applied to OPV-DB
# ------------------------------------------------------------------

wen_audit = wen_global.copy()

wen_audit["donor_graph_smiles"] = (
    wen_audit["Smiles_Donor"]
    .apply(canonicalize_smiles)
)

wen_audit["acceptor_graph_smiles"] = (
    wen_audit["Smiles_Acceptor"]
    .apply(canonicalize_smiles)
)


wen_structure_summary = pd.DataFrame({

    "role": [
        "donor",
        "acceptor",
    ],

    "records": [
        len(wen_audit),
        len(wen_audit),
    ],

    "unique_names": [
        wen_audit["Name_Donor"].nunique(),
        wen_audit["Name_Acceptor"].nunique(),
    ],

    "unique_raw_smiles": [
        wen_audit["Smiles_Donor"].nunique(),
        wen_audit["Smiles_Acceptor"].nunique(),
    ],

    "parsed_records": [
        wen_audit["donor_graph_smiles"].notna().sum(),
        wen_audit["acceptor_graph_smiles"].notna().sum(),
    ],

    "failed_records": [
        wen_audit["donor_graph_smiles"].isna().sum(),
        wen_audit["acceptor_graph_smiles"].isna().sum(),
    ],

    "unique_canonical_graphs": [
        wen_audit["donor_graph_smiles"].nunique(),
        wen_audit["acceptor_graph_smiles"].nunique(),
    ],
})

wen_structure_summary["parse_success_percent"] = (
    100
    * wen_structure_summary["parsed_records"]
    / wen_structure_summary["records"]
).round(2)

display(wen_structure_summary)



# ------------------------------------------------------------------
# Inspect unparseable structures, if present
# ------------------------------------------------------------------

failed_wen_donors = (
    wen_audit.loc[
        wen_audit["donor_graph_smiles"].isna(),
        ["Name_Donor", "Smiles_Donor"]
    ]
    .drop_duplicates()
)

failed_wen_acceptors = (
    wen_audit.loc[
        wen_audit["acceptor_graph_smiles"].isna(),
        ["Name_Acceptor", "Smiles_Acceptor"]
    ]
    .drop_duplicates()
)


print("Unique failed donor structures:",
      len(failed_wen_donors))

print("Unique failed acceptor structures:",
      len(failed_wen_acceptors))


if len(failed_wen_donors) > 0:
    print("\nFAILED DONORS")
    display(failed_wen_donors)


if len(failed_wen_acceptors) > 0:
    print("\nFAILED ACCEPTORS")
    display(failed_wen_acceptors)

Wen/Ma global dataset loaded
Records : 1,028
Columns : 2,089

First 20 column names:
['Name_Donor', 'Smiles_Donor', 'Name_Acceptor', 'Smiles_Acceptor', 'PCE (%)', 'D_A_Weight_Ratio', 'Blend_Concentration (mg/ml)', 'Solvent_DipoleMoment (Debye)', 'Solvent_EnergyGap (eV)', 'Solvent_Polarizability (a.u.)', 'Additive_MeltingPoint (℃)', 'Additive_BoilingPoint  (℃)', 'Additive_Density (g/cm3)', 'Additive_MolecularWeight', 'Additive_DipoleMoment  (Debye)', 'Additive_EnergyGap (eV)', 'Additive_Polarizability (a.u.)', 'Additive_Volume_Ratio (vol%)', 'Spin_Coating_Rate (rpm)', 'Annealing_Temperature(℃)']


,field,exists,missing_records
0,Name_Donor,True,0
1,Smiles_Donor,True,0
2,Name_Acceptor,True,0
3,Smiles_Acceptor,True,0
4,PCE (%),True,0


,role,records,unique_names,unique_raw_smiles,parsed_records,failed_records,unique_canonical_graphs,parse_success_percent
0,donor,1028,60,59,1028,0,59,100.00
1,acceptor,1028,181,181,757,271,160,73.64


Unique failed donor structures: 0
Unique failed acceptor structures: 21

FAILED ACCEPTORS


,Name_Acceptor,Smiles_Acceptor
93,E-SubPc-PDI,c1c(ccc2c1c1n3c2[n]c2n4c([n]c5n(c([n]1)c1c5cc(...
104,S-SubPc-PDI,c1c(ccc2c1c1n3c2[n]c2n4c([n]c5n(c([n]1)c1c5cc(...
168,o-TEH,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1c1sc(cc...
169,m-TEH,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C\c1sc3c(c1c1scc(c...
191,N3,C1=C(C(=C[C@@H]2[C@@H]1C(=C(C#N)C#N)/C(=C/c1sc...
202,Y6,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...
230,BT-LIC,C\1(=C\c2sc3c(c2CCCCCCCCCCC)sc2c3n(c3c2c2c(c4c...
235,BT-BO-L4F,C1(=C(c2c(C1=O)cc1c(c2)cc(c(c1)F)F)[C](C#N)C#N...
295,BTP-T-2Cl,c12c(c3c(c4c1nsn4)c1c(n3C[C@@H](CCCCCC)CCCC)c3...
296,BTP-T-3Cl,c12c(c3c(c4c1nsn4)c1c(n3C[C@@H](CCCCCC)CCCC)c3...


### 7.3 Audit and recovery of unparseable Wen–Zhang–Ma acceptor structures

Approximately one quarter of records in the Wen–Zhang–Ma global dataset
contain acceptor SMILES that cannot be parsed directly by RDKit.

Because the failures involve only a small number of unique acceptor identities
and include important modern non-fullerene acceptors, these records are not
discarded.

Instead, failed acceptor identities are first quantified and compared against
the independently curated OPV-DB material identities. Structure recovery is
attempted only when a Wen–Zhang–Ma acceptor can be linked unambiguously to a
corresponding OPV-DB material representation.

The original Wen–Zhang–Ma SMILES are preserved unchanged.

In [64]:
# ------------------------------------------------------------------
# Quantify the impact of failed Wen/Ma acceptor structures
# ------------------------------------------------------------------

failed_acceptor_rows = wen_audit[
    wen_audit["acceptor_graph_smiles"].isna()
].copy()

failed_acceptor_summary = (
    failed_acceptor_rows
    .groupby(
        ["Name_Acceptor", "Smiles_Acceptor"],
        dropna=False
    )
    .agg(
        records=("Name_Acceptor", "size"),
        pce_mean=("PCE (%)", "mean"),
        pce_median=("PCE (%)", "median"),
        pce_min=("PCE (%)", "min"),
        pce_max=("PCE (%)", "max"),
    )
    .reset_index()
    .sort_values(
        "records",
        ascending=False
    )
)

print(
    "Failed acceptor records:",
    f"{len(failed_acceptor_rows):,}",
    f"({100 * len(failed_acceptor_rows) / len(wen_audit):.2f}% of Wen/Ma global)"
)

print(
    "Unique failed acceptor names:",
    failed_acceptor_rows["Name_Acceptor"].nunique()
)

display(
    failed_acceptor_summary[
        [
            "Name_Acceptor",
            "records",
            "pce_mean",
            "pce_median",
            "pce_min",
            "pce_max",
        ]
    ]
)



# ------------------------------------------------------------------
# Compare failed Wen/Ma acceptor names with OPV-DB acceptor labels
# ------------------------------------------------------------------

def normalize_material_label(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    # Normalize only superficial typography/spacing.
    # Do not perform chemical alias substitution here.
    value = value.replace("–", "-").replace("—", "-")
    value = re.sub(r"\s+", "", value)

    return value


# OPV-DB acceptor identity table
opv_acceptor_identity = (
    provenance_clean[
        [
            "acceptor",
            "acceptor_canonical",
            "acceptor_graph_smiles",
        ]
    ]
    .dropna(
        subset=["acceptor_graph_smiles"]
    )
    .drop_duplicates()
    .copy()
)

opv_acceptor_identity["raw_norm"] = (
    opv_acceptor_identity["acceptor"]
    .apply(normalize_material_label)
)

opv_acceptor_identity["canonical_norm"] = (
    opv_acceptor_identity["acceptor_canonical"]
    .apply(normalize_material_label)
)


# Failed Wen/Ma identities
wen_failed_identity = (
    failed_acceptor_rows[
        ["Name_Acceptor", "Smiles_Acceptor"]
    ]
    .drop_duplicates()
    .copy()
)

wen_failed_identity["name_norm"] = (
    wen_failed_identity["Name_Acceptor"]
    .apply(normalize_material_label)
)



# ------------------------------------------------------------------
# Determine whether each failed Wen/Ma name has a unique OPV-DB graph
# ------------------------------------------------------------------

recovery_rows = []

for _, row in wen_failed_identity.iterrows():

    name = row["Name_Acceptor"]
    name_norm = row["name_norm"]

    candidates = opv_acceptor_identity[
        (
            opv_acceptor_identity["raw_norm"].eq(name_norm)
        )
        |
        (
            opv_acceptor_identity["canonical_norm"].eq(name_norm)
        )
    ].copy()

    unique_graphs = (
        candidates["acceptor_graph_smiles"]
        .dropna()
        .unique()
    )

    if len(unique_graphs) == 0:
        status = "no_exact_name_match"

    elif len(unique_graphs) == 1:
        status = "unique_opvdb_graph"

    else:
        status = "ambiguous_multiple_graphs"

    recovery_rows.append({
        "wen_acceptor": name,
        "opvdb_matching_rows": len(candidates),
        "opvdb_unique_graphs": len(unique_graphs),
        "recovery_status": status,
        "candidate_opvdb_labels":
            " | ".join(
                candidates["acceptor_canonical"]
                .dropna()
                .astype(str)
                .unique()[:8]
            ),
    })


wen_failed_recovery = pd.DataFrame(
    recovery_rows
)

display(wen_failed_recovery)

print("\nRecovery-status counts:")
display(
    wen_failed_recovery[
        "recovery_status"
    ]
    .value_counts()
    .rename_axis("recovery_status")
    .reset_index(name="acceptors")
)

Failed acceptor records: 271 (26.36% of Wen/Ma global)
Unique failed acceptor names: 21


,Name_Acceptor,records,pce_mean,pce_median,pce_min,pce_max
18,Y6,139,12.082734,12.960,0.23,17.62
11,L8-BO,28,10.300714,10.885,4.24,17.68
12,MQ6,14,14.791429,14.610,13.12,16.39
13,N3,13,13.353846,13.830,7.76,17.61
7,C7BTP-BO-2Cl-2F,12,16.175000,16.000,15.00,18.00
19,m-TEH,12,17.750000,17.675,17.18,18.51
16,S-SubPc-PDI,11,3.963636,4.060,2.93,4.53
10,E-SubPc-PDI,11,1.488182,1.650,0.81,1.78
17,Se46,8,18.178750,18.225,17.66,18.46
1,BT-LIC,5,12.052000,12.150,10.40,13.20


,wen_acceptor,opvdb_matching_rows,opvdb_unique_graphs,recovery_status,candidate_opvdb_labels
0,E-SubPc-PDI,0,0,no_exact_name_match,
1,S-SubPc-PDI,1,1,unique_opvdb_graph,S-SubPc-PDI
2,o-TEH,0,0,no_exact_name_match,
3,m-TEH,0,0,no_exact_name_match,
4,N3,2,1,unique_opvdb_graph,Y6 | N3
5,Y6,5,3,ambiguous_multiple_graphs,Y6
6,BT-LIC,1,1,unique_opvdb_graph,BT-LIC
7,BT-BO-L4F,2,1,unique_opvdb_graph,BT-BO-L4F
8,BTP-T-2Cl,0,0,no_exact_name_match,
9,BTP-T-3Cl,0,0,no_exact_name_match,



Recovery-status counts:


,recovery_status,acceptors
0,no_exact_name_match,14
1,unique_opvdb_graph,5
2,ambiguous_multiple_graphs,2


### 7.4 Reference-table-assisted identity resolution

Exact material-name matching against OPV-DB device records resolves only a
minority of the unparseable Wen–Zhang–Ma acceptor identities.

The OPV-DB material-reference table is therefore examined as an additional
identity resource because it contains material labels, aliases, and mapped
molecular structures.

Reference-table matches are accepted only when the combined name/alias evidence
maps a Wen–Zhang–Ma acceptor to a single canonicalized molecular graph.
Ambiguous mappings are retained as unresolved rather than resolved by
arbitrary selection.

In [65]:
# ------------------------------------------------------------------
# Inspect OPV-DB material-reference schema
# ------------------------------------------------------------------

print("Materials-reference dimensions:")
print(materials_ref.shape)

print("\nColumns:")
print(materials_ref.columns.tolist())

display(materials_ref.head(10))

Materials-reference dimensions:
(4548, 7)

Columns:
['name', 'smiles', 'material_type', 'chemical_class', 'aliases', 'homo', 'lumo']


,name,smiles,material_type,chemical_class,aliases,homo,lumo
0,PBFSF,CCCCCCC(CCCC)Cc1ccc(-c2c3cc(-c4c(F)cc(-c5cc6c(...,donor,NaN,[],-5.27,-3.72
1,N2200,CCCCCCCCCCC(CCCCCCCC)CN1C(=O)c2ccc3c4c(c(-c5cc...,acceptor,NaN,[],-5.71,-4.02
2,PBBSB,CCCCCCC(CCCC)Cc1ccc(-c2c3cc(-c4ccc(-c5cc6c(s5)...,donor,NaN,[],-5.19,-3.63
3,PTB7-Th,CCCCC(CC)COC(=O)c1sc2csc(-c3cc4c(-c5ccc(CC(CC)...,donor,polymer,"[""PCE10"", ""PBDTTT-EFT""]",-5.21,-3.60
4,IOTIC-2F,CCCCCCc1ccc(C2(c3ccc(CCCCCC)cc3)c3cc4c(cc3-c3s...,acceptor,NaN,[],-5.34,-4.06
5,3TT-CIC,CCCCCCc1ccc(C2(c3ccc(CCCCCC)cc3)c3c(sc4cc(/C=C...,acceptor,NaN,[],-5.24,-3.95
6,6TBA,CCCCCCc1ccc(C2(c3ccc(CCCCCC)cc3)c3c(sc4cc(C=C5...,acceptor,NaN,[],-5.24,-3.78
7,ITOTIC-2F,CCCCCCc1ccc(C2(c3ccc(CCCCCC)cc3)c3cc4c(cc3-c3s...,acceptor,NaN,[],-5.22,-4.11
8,O-IDTBCN,CCCCCCCCC1(CCCCCCCC)c2cc3c(cc2-c2sc(-c4ccc(C=C...,acceptor,NaN,[],-5.80,-3.80
9,3TT-OCIC,CCCCCCCCc1c(/C=C2\C(=O)c3cc(Cl)c(Cl)cc3C2=C(C#...,acceptor,NaN,[],-5.22,-3.91


### 7.5 Exact name-and-alias resolution using the OPV-DB material reference

The OPV-DB material-reference table provides an additional alias layer beyond
the device tables.

Aliases are parsed conservatively and normalized only for superficial
typographic differences. Failed Wen–Zhang–Ma acceptor identities are then
matched against both reference-table material names and aliases.

A structure is considered recoverable only when all matching reference entries
collapse to a single RDKit-canonicalized molecular graph.

In [66]:
# ------------------------------------------------------------------
# Inspect alias encoding
# ------------------------------------------------------------------

print("aliases dtype:", materials_ref["aliases"].dtype)

print("\nExample non-empty alias entries:")

display(
    materials_ref.loc[
        materials_ref["aliases"].notna()
        & materials_ref["aliases"].astype(str).ne("[]"),
        ["name", "material_type", "aliases"]
    ].head(30)
)

aliases dtype: str

Example non-empty alias entries:


,name,material_type,aliases
3,PTB7-Th,donor,"[""PCE10"", ""PBDTTT-EFT""]"
29,Y6,acceptor,"[""BTP-4F"", ""BTP-4F-12""]"
63,PBDB-TF,donor,"[""PBDB-T-2F"", ""PBDB-TF""]"
102,PBDB-T-2Cl,donor,"[""PBDB-T-2Cl""]"
103,IT-4F,acceptor,"[""ITIC-4F""]"
134,D18,donor,"[""D18-Cl""]"
244,L8-BO,acceptor,"[""BTP-eC9"", ""eC9""]"
397,eC9,acceptor,"[""BTP-eC9"", ""eC9""]"
445,PBDB-T2Cl,donor,"[""PBDB-T-2Cl""]"
453,PBDB-T2F,donor,"[""PBDB-T-2F"", ""PBDB-TF""]"


In [67]:
# ------------------------------------------------------------------
# Parse OPV-DB material-reference aliases
# ------------------------------------------------------------------

import ast


def parse_aliases(value):
    """
    Safely parse the Python-list-style alias field.
    Returns an empty list when no aliases are available.
    """
    if pd.isna(value):
        return []

    value = str(value).strip()

    if value in {"", "[]"}:
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except (ValueError, SyntaxError):
        pass

    return []


reference_acceptors = (
    materials_ref[
        materials_ref["material_type"]
        .astype(str)
        .str.lower()
        .eq("acceptor")
    ]
    .copy()
)

reference_acceptors["parsed_aliases"] = (
    reference_acceptors["aliases"]
    .apply(parse_aliases)
)

reference_acceptors["reference_graph_smiles"] = (
    reference_acceptors["smiles"]
    .apply(canonicalize_smiles)
)


print("Reference acceptor entries:",
      len(reference_acceptors))

print(
    "Reference SMILES successfully parsed:",
    reference_acceptors[
        "reference_graph_smiles"
    ].notna().sum()
)

print(
    "Reference SMILES failed:",
    reference_acceptors[
        "reference_graph_smiles"
    ].isna().sum()
)



# ------------------------------------------------------------------
# Expand each reference material into name + alias identity terms
# ------------------------------------------------------------------

reference_identity_rows = []

for _, row in reference_acceptors.iterrows():

    identity_terms = [row["name"]]

    identity_terms.extend(
        row["parsed_aliases"]
    )

    for term in identity_terms:

        reference_identity_rows.append({
            "reference_name":
                row["name"],

            "identity_term":
                term,

            "identity_norm":
                normalize_material_label(term),

            "reference_smiles":
                row["smiles"],

            "reference_graph_smiles":
                row["reference_graph_smiles"],
        })


reference_identity = pd.DataFrame(
    reference_identity_rows
).drop_duplicates()


print(
    "Expanded name/alias identities:",
    len(reference_identity)
)

print(
    "Unique normalized identity terms:",
    reference_identity[
        "identity_norm"
    ].nunique()
)



# ------------------------------------------------------------------
# Match failed Wen/Ma acceptors against OPV-DB names + aliases
# ------------------------------------------------------------------

reference_recovery_rows = []

for _, row in wen_failed_identity.iterrows():

    wen_name = row["Name_Acceptor"]
    wen_norm = row["name_norm"]

    matches = reference_identity[
        reference_identity["identity_norm"]
        .eq(wen_norm)
    ].copy()

    valid_graphs = (
        matches["reference_graph_smiles"]
        .dropna()
        .unique()
    )

    if len(matches) == 0:

        status = "no_reference_match"

    elif len(valid_graphs) == 0:

        status = "matched_but_reference_smiles_unparseable"

    elif len(valid_graphs) == 1:

        status = "unique_reference_graph"

    else:

        status = "ambiguous_multiple_reference_graphs"

    reference_recovery_rows.append({

        "wen_acceptor":
            wen_name,

        "reference_matches":
            len(matches),

        "unique_reference_names":
            matches["reference_name"]
            .nunique(),

        "unique_valid_graphs":
            len(valid_graphs),

        "recovery_status":
            status,

        "matched_reference_names":
            " | ".join(
                matches["reference_name"]
                .dropna()
                .astype(str)
                .unique()[:10]
            ),

        "matched_identity_terms":
            " | ".join(
                matches["identity_term"]
                .dropna()
                .astype(str)
                .unique()[:10]
            ),
    })


reference_recovery = pd.DataFrame(
    reference_recovery_rows
)

display(reference_recovery)


print("\nRecovery status counts:")

display(
    reference_recovery[
        "recovery_status"
    ]
    .value_counts()
    .rename_axis("recovery_status")
    .reset_index(name="acceptors")
)



# ------------------------------------------------------------------
# Translate identity-resolution status to affected Wen/Ma records
# ------------------------------------------------------------------

failed_name_record_counts = (
    failed_acceptor_rows[
        "Name_Acceptor"
    ]
    .value_counts()
)

reference_recovery["wen_records"] = (
    reference_recovery[
        "wen_acceptor"
    ]
    .map(failed_name_record_counts)
)


record_recovery_summary = (
    reference_recovery
    .groupby("recovery_status")[
        "wen_records"
    ]
    .sum()
    .rename("records")
    .reset_index()
)

record_recovery_summary[
    "percent_of_failed_records"
] = (
    100
    * record_recovery_summary["records"]
    / len(failed_acceptor_rows)
).round(2)

display(record_recovery_summary)


Reference acceptor entries: 1820
Reference SMILES successfully parsed: 1819
Reference SMILES failed: 1
Expanded name/alias identities: 1849
Unique normalized identity terms: 1718


,wen_acceptor,reference_matches,unique_reference_names,unique_valid_graphs,recovery_status,matched_reference_names,matched_identity_terms
0,E-SubPc-PDI,0,0,0,no_reference_match,,
1,S-SubPc-PDI,1,1,1,unique_reference_graph,S-SubPc-PDI,S-SubPc-PDI
2,o-TEH,0,0,0,no_reference_match,,
3,m-TEH,0,0,0,no_reference_match,,
4,N3,1,1,1,unique_reference_graph,N3,N3
5,Y6,1,1,1,unique_reference_graph,Y6,Y6
6,BT-LIC,1,1,1,unique_reference_graph,BT-LIC,BT-LIC
7,BT-BO-L4F,1,1,1,unique_reference_graph,BT-BO-L4F,BT-BO-L4F
8,BTP-T-2Cl,0,0,0,no_reference_match,,
9,BTP-T-3Cl,0,0,0,no_reference_match,,



Recovery status counts:


,recovery_status,acceptors
0,no_reference_match,14
1,unique_reference_graph,6
2,ambiguous_multiple_reference_graphs,1


,recovery_status,records,percent_of_failed_records
0,ambiguous_multiple_reference_graphs,28,10.33
1,no_reference_match,70,25.83
2,unique_reference_graph,173,63.84


### 7.6 Conservative structure recovery and cross-dataset comparability

Reference-table matching uniquely resolves the molecular graph for a substantial
fraction of Wen–Zhang–Ma acceptor records whose original SMILES could not be
parsed.

Only unambiguous reference mappings are accepted. Ambiguous mappings, including
L8-BO representations associated with more than one OPV-DB molecular graph,
are left unresolved rather than assigned arbitrarily.

The resulting structurally comparable cohort is used to quantify chemical
overlap between Wen–Zhang–Ma and OPV-DB. Unresolved records are retained
separately so that overlap conclusions can be expressed with conservative
bounds rather than assuming that unresolved chemistry is either novel or
already represented.

In [68]:
# ------------------------------------------------------------------
# Build unique reference-graph recovery map
# ------------------------------------------------------------------

unique_reference_graph_map = {}

for _, row in wen_failed_identity.iterrows():

    wen_name = row["Name_Acceptor"]
    wen_norm = row["name_norm"]

    matches = reference_identity[
        reference_identity["identity_norm"].eq(wen_norm)
    ]

    graphs = (
        matches["reference_graph_smiles"]
        .dropna()
        .unique()
    )

    if len(graphs) == 1:
        unique_reference_graph_map[wen_name] = graphs[0]


print(
    "Failed acceptor identities uniquely recovered:",
    len(unique_reference_graph_map)
)



# ------------------------------------------------------------------
# Add conservative resolved acceptor identity
# ------------------------------------------------------------------

wen_overlap = wen_audit.copy()

wen_overlap["acceptor_graph_resolved"] = (
    wen_overlap["acceptor_graph_smiles"]
    .copy()
)

recovery_mask = (
    wen_overlap["acceptor_graph_resolved"].isna()
    & wen_overlap["Name_Acceptor"].isin(
        unique_reference_graph_map
    )
)

wen_overlap.loc[
    recovery_mask,
    "acceptor_graph_resolved"
] = (
    wen_overlap.loc[
        recovery_mask,
        "Name_Acceptor"
    ]
    .map(unique_reference_graph_map)
)


# Resolution source
wen_overlap["acceptor_resolution_source"] = np.select(
    [
        wen_overlap["acceptor_graph_smiles"].notna(),
        recovery_mask,
    ],
    [
        "original_wen_smiles",
        "unique_opvdb_reference_match",
    ],
    default="unresolved"
)


wen_overlap["pair_structurally_resolved"] = (
    wen_overlap["donor_graph_smiles"].notna()
    & wen_overlap["acceptor_graph_resolved"].notna()
)


resolution_summary = (
    wen_overlap["acceptor_resolution_source"]
    .value_counts()
    .rename_axis("resolution_source")
    .reset_index(name="records")
)

resolution_summary["percent"] = (
    100
    * resolution_summary["records"]
    / len(wen_overlap)
).round(2)

display(resolution_summary)

print(
    "\nStructurally resolved D:A records:",
    f"{wen_overlap['pair_structurally_resolved'].sum():,}",
    "/",
    f"{len(wen_overlap):,}"
)

Failed acceptor identities uniquely recovered: 6


,resolution_source,records,percent
0,original_wen_smiles,757,73.64
1,unique_opvdb_reference_match,173,16.83
2,unresolved,98,9.53



Structurally resolved D:A records: 930 / 1,028


### 7.7 Molecular overlap between OPV-DB and Wen–Zhang–Ma

The structurally resolved Wen–Zhang–Ma records are compared against the full
provenance-clean OPV-DB benchmark using RDKit-canonicalized molecular graphs.

Overlap is evaluated independently for donor identity, acceptor identity, and
complete donor–acceptor pairs.

For complete pairs, unresolved Wen–Zhang–Ma records are not assumed to be
either overlapping or novel. Instead, lower and upper overlap bounds are
reported by treating all unresolved records as non-overlapping or overlapping,
respectively.

In [69]:
# ------------------------------------------------------------------
# OPV-DB molecular identity sets
# ------------------------------------------------------------------

opv_donor_graphs = set(
    provenance_clean[
        "donor_graph_smiles"
    ].dropna()
)

opv_acceptor_graphs = set(
    provenance_clean[
        "acceptor_graph_smiles"
    ].dropna()
)

opv_graph_pairs = set(
    zip(
        provenance_clean["donor_graph_smiles"],
        provenance_clean["acceptor_graph_smiles"]
    )
)


# ------------------------------------------------------------------
# Construct Wen/Ma resolved D:A pairs
# ------------------------------------------------------------------

wen_overlap["resolved_graph_pair"] = list(
    zip(
        wen_overlap["donor_graph_smiles"],
        wen_overlap["acceptor_graph_resolved"]
    )
)


wen_overlap["donor_seen_in_opvdb"] = (
    wen_overlap["donor_graph_smiles"]
    .isin(opv_donor_graphs)
)

wen_overlap["acceptor_seen_in_opvdb"] = (
    wen_overlap["acceptor_graph_resolved"]
    .isin(opv_acceptor_graphs)
    & wen_overlap["acceptor_graph_resolved"].notna()
)

wen_overlap["pair_seen_in_opvdb"] = (
    wen_overlap["resolved_graph_pair"]
    .isin(opv_graph_pairs)
    & wen_overlap["pair_structurally_resolved"]
)


# ------------------------------------------------------------------
# Row-level overlap among structurally resolved Wen/Ma records
# ------------------------------------------------------------------

wen_resolved = wen_overlap[
    wen_overlap["pair_structurally_resolved"]
].copy()


cross_dataset_overlap = pd.DataFrame({

    "identity_level": [
        "donor graph",
        "acceptor graph",
        "exact D:A graph pair",
    ],

    "overlapping_records": [
        int(
            wen_resolved[
                "donor_seen_in_opvdb"
            ].sum()
        ),

        int(
            wen_resolved[
                "acceptor_seen_in_opvdb"
            ].sum()
        ),

        int(
            wen_resolved[
                "pair_seen_in_opvdb"
            ].sum()
        ),
    ],

    "resolved_records": [
        len(wen_resolved),
        len(wen_resolved),
        len(wen_resolved),
    ]
})


cross_dataset_overlap[
    "overlap_percent_of_resolved"
] = (
    100
    * cross_dataset_overlap["overlapping_records"]
    / cross_dataset_overlap["resolved_records"]
).round(2)

display(cross_dataset_overlap)



# ------------------------------------------------------------------
# Unique-identity overlap
# ------------------------------------------------------------------

wen_unique_donors = set(
    wen_resolved["donor_graph_smiles"]
)

wen_unique_acceptors = set(
    wen_resolved["acceptor_graph_resolved"]
)

wen_unique_pairs = set(
    wen_resolved["resolved_graph_pair"]
)


unique_overlap_summary = pd.DataFrame({

    "identity_level": [
        "donor graphs",
        "acceptor graphs",
        "D:A graph pairs",
    ],

    "unique_wen_identities": [
        len(wen_unique_donors),
        len(wen_unique_acceptors),
        len(wen_unique_pairs),
    ],

    "also_present_in_opvdb": [
        len(
            wen_unique_donors
            & opv_donor_graphs
        ),

        len(
            wen_unique_acceptors
            & opv_acceptor_graphs
        ),

        len(
            wen_unique_pairs
            & opv_graph_pairs
        ),
    ]
})


unique_overlap_summary[
    "overlap_percent"
] = (
    100
    * unique_overlap_summary[
        "also_present_in_opvdb"
    ]
    / unique_overlap_summary[
        "unique_wen_identities"
    ]
).round(2)

display(unique_overlap_summary)



# ------------------------------------------------------------------
# Cross-dataset chemical novelty categories
# ------------------------------------------------------------------

def cross_dataset_novelty(row):

    if not row["pair_structurally_resolved"]:
        return "unresolved structure"

    donor_seen = row["donor_seen_in_opvdb"]
    acceptor_seen = row["acceptor_seen_in_opvdb"]
    pair_seen = row["pair_seen_in_opvdb"]

    if pair_seen:
        return "exact D:A pair already in OPV-DB"

    if donor_seen and acceptor_seen:
        return "new pairing of OPV-DB donor + acceptor"

    if donor_seen and not acceptor_seen:
        return "acceptor unseen in OPV-DB"

    if not donor_seen and acceptor_seen:
        return "donor unseen in OPV-DB"

    return "both donor and acceptor unseen in OPV-DB"


wen_overlap["opvdb_novelty_category"] = (
    wen_overlap.apply(
        cross_dataset_novelty,
        axis=1
    )
)


wen_novelty_summary = (
    wen_overlap[
        "opvdb_novelty_category"
    ]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="records")
)

wen_novelty_summary["percent_of_wen_global"] = (
    100
    * wen_novelty_summary["records"]
    / len(wen_overlap)
).round(2)

display(wen_novelty_summary)

,identity_level,overlapping_records,resolved_records,overlap_percent_of_resolved
0,donor graph,23,930,2.47
1,acceptor graph,266,930,28.60
2,exact D:A graph pair,0,930,0.00


,identity_level,unique_wen_identities,also_present_in_opvdb,overlap_percent
0,donor graphs,57,2,3.51
1,acceptor graphs,166,23,13.86
2,D:A graph pairs,225,0,0.00


,category,records,percent_of_wen_global
0,both donor and acceptor unseen in OPV-DB,641,62.35
1,donor unseen in OPV-DB,266,25.88
2,unresolved structure,98,9.53
3,acceptor unseen in OPV-DB,23,2.24


### 7.8 Name-level overlap versus molecular-graph overlap

Exact molecular-graph comparison indicates very limited donor overlap between
OPV-DB and the Wen–Zhang–Ma processing database.

For conjugated polymers, however, equivalent material identities can be
represented by different repeat-unit, termination, oligomer, or side-chain
SMILES conventions. Exact graph matching may therefore underestimate shared
polymer chemistry.

Material names are consequently used as a diagnostic identity layer—not as
the final definition of chemical equivalence—to determine whether nominally
identical donor or acceptor materials are represented by different molecular
graphs across the two databases.

A high frequency of same-name/different-graph cases would indicate
representation-domain shift rather than true chemical novelty.

In [70]:
# ------------------------------------------------------------------
# Build OPV-DB name/alias -> molecular-graph identity lexicon
# ------------------------------------------------------------------

def build_opv_identity_lexicon(role):

    if role == "donor":
        raw_col = "donor"
        canonical_col = "donor_canonical"
        graph_col = "donor_graph_smiles"

    elif role == "acceptor":
        raw_col = "acceptor"
        canonical_col = "acceptor_canonical"
        graph_col = "acceptor_graph_smiles"

    else:
        raise ValueError("role must be 'donor' or 'acceptor'")


    identity_rows = []

    # --------------------------------------------------------------
    # Device-table names
    # --------------------------------------------------------------

    device_identity = provenance_clean[
        [raw_col, canonical_col, graph_col]
    ].copy()

    for _, row in device_identity.iterrows():

        graph = row[graph_col]

        if pd.isna(graph):
            continue

        for source, value in [
            ("device_raw", row[raw_col]),
            ("device_canonical", row[canonical_col]),
        ]:

            norm = normalize_material_label(value)

            if pd.isna(norm):
                continue

            identity_rows.append({
                "identity_norm": norm,
                "identity_label": value,
                "graph_smiles": graph,
                "identity_source": source,
            })


    # --------------------------------------------------------------
    # Material-reference names and aliases
    # --------------------------------------------------------------

    ref = materials_ref[
        materials_ref["material_type"]
        .astype(str)
        .str.lower()
        .eq(role)
    ].copy()

    ref["parsed_aliases"] = (
        ref["aliases"]
        .apply(parse_aliases)
    )

    ref["graph_smiles"] = (
        ref["smiles"]
        .apply(canonicalize_smiles)
    )

    for _, row in ref.iterrows():

        graph = row["graph_smiles"]

        if pd.isna(graph):
            continue

        terms = [row["name"]] + row["parsed_aliases"]

        for term in terms:

            norm = normalize_material_label(term)

            if pd.isna(norm):
                continue

            identity_rows.append({
                "identity_norm": norm,
                "identity_label": term,
                "graph_smiles": graph,
                "identity_source": "material_reference",
            })


    return (
        pd.DataFrame(identity_rows)
        .drop_duplicates()
        .reset_index(drop=True)
    )


opv_donor_lexicon = build_opv_identity_lexicon(
    "donor"
)

opv_acceptor_lexicon = build_opv_identity_lexicon(
    "acceptor"
)


print(
    "Donor lexicon normalized names:",
    opv_donor_lexicon["identity_norm"].nunique()
)

print(
    "Acceptor lexicon normalized names:",
    opv_acceptor_lexicon["identity_norm"].nunique()
)

Donor lexicon normalized names: 2736
Acceptor lexicon normalized names: 1869


In [72]:
# ------------------------------------------------------------------
# Create Wen/Ma name-versus-graph audit tables
# ------------------------------------------------------------------

wen_donor_name_audit = diagnose_name_graph_overlap(
    wen_overlap,
    "Name_Donor",
    "donor_graph_smiles",
    opv_donor_lexicon,
    "donor"
)

wen_acceptor_name_audit = diagnose_name_graph_overlap(
    wen_overlap,
    "Name_Acceptor",
    "acceptor_graph_resolved",
    opv_acceptor_lexicon,
    "acceptor"
)


print("Donor audit rows   :", len(wen_donor_name_audit))
print("Acceptor audit rows:", len(wen_acceptor_name_audit))


# ------------------------------------------------------------------
# Combine donor and acceptor audits
# ------------------------------------------------------------------

name_graph_audit = pd.concat(
    [
        wen_donor_name_audit,
        wen_acceptor_name_audit,
    ],
    ignore_index=True
)


name_graph_summary = (
    name_graph_audit
    .groupby(
        ["role", "status"]
    )
    .size()
    .rename("unique_wen_materials")
    .reset_index()
)

display(name_graph_summary)

Donor audit rows   : 60
Acceptor audit rows: 181


,role,status,unique_wen_materials
0,acceptor,name_found_wen_graph_unresolved,1
1,acceptor,name_not_found_in_opvdb,109
2,acceptor,same_name_different_graph,43
3,acceptor,same_name_exact_graph_match,24
4,acceptor,same_name_opvdb_graph_ambiguous,4
5,donor,name_not_found_in_opvdb,31
6,donor,same_name_different_graph,18
7,donor,same_name_exact_graph_match,2
8,donor,same_name_opvdb_graph_ambiguous,9


In [73]:
# ------------------------------------------------------------------
# Translate name/graph status to Wen/Ma device-record counts
# ------------------------------------------------------------------

donor_status_map = (
    wen_donor_name_audit
    .set_index("wen_name")["status"]
    .to_dict()
)

acceptor_status_map = (
    wen_acceptor_name_audit
    .set_index("wen_name")["status"]
    .to_dict()
)


wen_overlap["donor_name_graph_status"] = (
    wen_overlap["Name_Donor"]
    .map(donor_status_map)
)

wen_overlap["acceptor_name_graph_status"] = (
    wen_overlap["Name_Acceptor"]
    .map(acceptor_status_map)
)


for role, column in [
    ("DONOR", "donor_name_graph_status"),
    ("ACCEPTOR", "acceptor_name_graph_status"),
]:

    print("\n" + "=" * 80)
    print(role)
    print("=" * 80)

    row_summary = (
        wen_overlap[column]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="records")
    )

    row_summary["percent_of_wen"] = (
        100
        * row_summary["records"]
        / len(wen_overlap)
    ).round(2)

    display(row_summary)


    print("DONORS — same name but different molecular graph")

display(
    wen_donor_name_audit[
        wen_donor_name_audit["status"]
        .eq("same_name_different_graph")
    ]
    .sort_values("wen_name")
)


print("\nACCEPTORS — same name but different molecular graph")

display(
    wen_acceptor_name_audit[
        wen_acceptor_name_audit["status"]
        .eq("same_name_different_graph")
    ]
    .sort_values("wen_name")
)


DONOR


,status,records,percent_of_wen
0,same_name_opvdb_graph_ambiguous,654,63.62
1,name_not_found_in_opvdb,244,23.74
2,same_name_different_graph,107,10.41
3,same_name_exact_graph_match,23,2.24


DONORS — same name but different molecular graph

ACCEPTOR


,status,records,percent_of_wen
0,name_not_found_in_opvdb,517,50.29
1,same_name_exact_graph_match,275,26.75
2,same_name_different_graph,202,19.65
3,name_found_wen_graph_unresolved,28,2.72
4,same_name_opvdb_graph_ambiguous,6,0.58


DONORS — same name but different molecular graph


,role,wen_name,wen_graph_resolved,opv_name_matches,opv_candidate_graphs,status,opv_matching_labels
42,donor,D18,True,3,1,same_name_different_graph,D18
0,donor,J61,True,3,1,same_name_different_graph,J61
18,donor,L1,True,3,1,same_name_different_graph,L1
19,donor,L2,True,3,1,same_name_different_graph,L2
44,donor,MPhS-C2,True,3,1,same_name_different_graph,MPhS-C2
7,donor,PBDT-Cl,True,2,1,same_name_different_graph,PBDT-Cl
23,donor,PBDT-TTz,True,5,1,same_name_different_graph,PBDT-TTz | PBdT-TTz
41,donor,PBDTTT-E-T,True,3,1,same_name_different_graph,PBDTTT-E-T
16,donor,PCE-10,True,2,1,same_name_different_graph,PCE-10
27,donor,POTz1,True,2,1,same_name_different_graph,POTz1



ACCEPTORS — same name but different molecular graph


,role,wen_name,wen_graph_resolved,opv_name_matches,opv_candidate_graphs,status,opv_matching_labels
73,acceptor,AOT3,True,1,1,same_name_different_graph,AOT3
106,acceptor,BTIC-4F,True,3,1,same_name_different_graph,BTIC-4F
179,acceptor,BTP-BO-4Cl,True,3,1,same_name_different_graph,BTP-BO-4Cl
11,acceptor,CNDTBT-IDTT-FINCN,True,3,1,same_name_different_graph,CNDTBT-IDTT-FINCN
113,acceptor,DCNBT-IDT,True,3,1,same_name_different_graph,DCNBT-IDT
136,acceptor,DCNBT-TPIC,True,3,1,same_name_different_graph,DCNBT-TPIC
33,acceptor,EH-IDTBR,True,4,1,same_name_different_graph,EH-IDTBR | eh-IDTBR
121,acceptor,F6IC,True,3,1,same_name_different_graph,F6IC
7,acceptor,F8-DPPTCN,True,3,1,same_name_different_graph,F8-DPPTCN
10,acceptor,FDTBT-IDTT-FINCN,True,3,1,same_name_different_graph,FDTBT-IDTT-FINCN


In [4]:
# ------------------------------------------------------------------
# Recover project paths correctly after kernel restart
# ------------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path
import ast

CURRENT_DIR = Path.cwd()

# Notebook is inside /notebooks, so project root is its parent
if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"

print("Current directory :", CURRENT_DIR)
print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)

print("\nData directory exists:", DATA_DIR.exists())
print("Interim exists       :", INTERIM_DIR.exists())

Current directory : <PROJECT_ROOT>\notebooks
Project directory : <PROJECT_ROOT>
Data directory    : <PROJECT_ROOT>\data

Data directory exists: True
Interim exists       : True


In [5]:
# ------------------------------------------------------------------
# Reload the saved core datasets
# ------------------------------------------------------------------

provenance_clean = pd.read_csv(
    INTERIM_DIR / "opvdb_provenance_clean_interim.csv"
)

materials_ref = pd.read_csv(
    RAW_DIR
    / "opvdb"
    / "data"
    / "materials_reference.csv"
)

wen_global_files = list(
    RAW_DIR.rglob("global-all-nine-parameters.csv")
)

print("Provenance-clean OPV-DB:", provenance_clean.shape)
print("Materials reference     :", materials_ref.shape)
print("Wen/Ma global files     :", len(wen_global_files))

for p in wen_global_files:
    print(p)

if len(wen_global_files) != 1:
    raise RuntimeError(
        f"Expected 1 Wen/Ma global file, found {len(wen_global_files)}"
    )

wen_global = pd.read_csv(wen_global_files[0])

print("Wen/Ma global dataset   :", wen_global.shape)

Provenance-clean OPV-DB: (21590, 42)
Materials reference     : (4548, 7)
Wen/Ma global files     : 1
<PROJECT_ROOT>\data\raw\opv-multi-tier-ml-database\opv-multi-tier-ml-database\03-global-model\global-all-nine-parameters.csv
Wen/Ma global dataset   : (1028, 2089)


In [7]:
# ------------------------------------------------------------------
# Recovery Cell 2
# Inspect which derived OPV-DB variables survived in saved CSV
# ------------------------------------------------------------------

important_opv_columns = [
    "doi_norm",
    "donor",
    "donor_canonical",
    "donor_smiles",
    "donor_graph_smiles",
    "acceptor",
    "acceptor_canonical",
    "acceptor_smiles",
    "acceptor_graph_smiles",
    "pce",
]

print("OPV-DB saved-column check:")
print("-" * 60)

for col in important_opv_columns:
    print(
        f"{col:25s}",
        "FOUND" if col in provenance_clean.columns else "MISSING"
    )


print("\nAll 42 saved columns:")
print(provenance_clean.columns.tolist())


print("\nWen/Ma identity columns:")

wen_identity_columns = [
    "Name_Donor",
    "Smiles_Donor",
    "Name_Acceptor",
    "Smiles_Acceptor",
    "PCE (%)",
]

for col in wen_identity_columns:
    print(
        f"{col:25s}",
        "FOUND" if col in wen_global.columns else "MISSING"
    )

OPV-DB saved-column check:
------------------------------------------------------------
doi_norm                  FOUND
donor                     FOUND
donor_canonical           FOUND
donor_smiles              FOUND
donor_graph_smiles        FOUND
acceptor                  FOUND
acceptor_canonical        FOUND
acceptor_smiles           FOUND
acceptor_graph_smiles     FOUND
pce                       FOUND

All 42 saved columns:
['id', 'doi', 'doi_norm', 'donor', 'acceptor', 'donor_canonical', 'acceptor_canonical', 'donor_smiles', 'acceptor_smiles', 'voc', 'jsc', 'ff', 'pce', 'pce_recomputed', 'pce_relative_error_percent', 'pce_avg', 'pce_best', 'd_a_ratio', 'additive', 'additive_canonical', 'additive_ratio', 'device_structure', 'device_type', 'etl', 'etl_canonical', 'htl', 'htl_canonical', 'active_layer_thickness', 'solvent', 'solvent_canonical', 'annealing_temp', 'homo_d', 'lumo_d', 'eg_d', 'homo_a', 'lumo_a', 'eg_a', 'donor_graph_smiles', 'acceptor_graph_smiles', 'publication_review_s

In [8]:
# ------------------------------------------------------------------
# Recovery Cell 3
# Restore chemistry helper functions and Wen/Ma molecular graphs
# ------------------------------------------------------------------

import re
import unicodedata
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.error")


def canonicalize_smiles(smiles):
    if pd.isna(smiles):
        return pd.NA

    smiles = str(smiles).strip()

    if not smiles:
        return pd.NA

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return pd.NA

    return Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )


def normalize_material_label(value):
    """
    Conservative normalization for matching material labels.
    Does not attempt chemical interpretation.
    """
    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize(
        "NFKC",
        str(value)
    ).strip()

    # Normalize dash variants
    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    # Case-insensitive identity comparison
    value = value.lower()

    # Ignore whitespace differences
    value = re.sub(r"\s+", "", value)

    return value if value else pd.NA


def parse_aliases(value):
    if pd.isna(value):
        return []

    value = str(value).strip()

    if value in {"", "[]"}:
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except (ValueError, SyntaxError):
        pass

    return []


# ------------------------------------------------------------------
# Reconstruct Wen/Ma molecular audit
# ------------------------------------------------------------------

wen_audit = wen_global.copy()

wen_audit["donor_graph_smiles"] = (
    wen_audit["Smiles_Donor"]
    .apply(canonicalize_smiles)
)

wen_audit["acceptor_graph_smiles"] = (
    wen_audit["Smiles_Acceptor"]
    .apply(canonicalize_smiles)
)


print("Wen/Ma records:", len(wen_audit))

print(
    "\nDonor SMILES parsed:",
    wen_audit["donor_graph_smiles"].notna().sum()
)

print(
    "Donor SMILES failed:",
    wen_audit["donor_graph_smiles"].isna().sum()
)

print(
    "\nAcceptor SMILES parsed:",
    wen_audit["acceptor_graph_smiles"].notna().sum()
)

print(
    "Acceptor SMILES failed:",
    wen_audit["acceptor_graph_smiles"].isna().sum()
)

print(
    "\nUnique donor names:",
    wen_audit["Name_Donor"].nunique()
)

print(
    "Unique acceptor names:",
    wen_audit["Name_Acceptor"].nunique()
)

print(
    "\nUnique failed acceptor names:",
    wen_audit.loc[
        wen_audit["acceptor_graph_smiles"].isna(),
        "Name_Acceptor"
    ].nunique()
)

Wen/Ma records: 1028

Donor SMILES parsed: 1028
Donor SMILES failed: 0

Acceptor SMILES parsed: 757
Acceptor SMILES failed: 271

Unique donor names: 60
Unique acceptor names: 181

Unique failed acceptor names: 21


In [9]:
# ------------------------------------------------------------------
# Recovery Cell 4
# Rebuild OPV-DB reference identity mapping and recover Wen acceptors
# ------------------------------------------------------------------

# Reference acceptors only
reference_acceptors = (
    materials_ref[
        materials_ref["material_type"]
        .astype(str)
        .str.lower()
        .eq("acceptor")
    ]
    .copy()
)

reference_acceptors["parsed_aliases"] = (
    reference_acceptors["aliases"]
    .apply(parse_aliases)
)

reference_acceptors["reference_graph_smiles"] = (
    reference_acceptors["smiles"]
    .apply(canonicalize_smiles)
)


# ------------------------------------------------------------------
# Expand reference name + aliases
# ------------------------------------------------------------------

reference_identity_rows = []

for _, row in reference_acceptors.iterrows():

    terms = [row["name"]] + row["parsed_aliases"]

    for term in terms:

        norm = normalize_material_label(term)

        if pd.isna(norm):
            continue

        reference_identity_rows.append({
            "reference_name": row["name"],
            "identity_term": term,
            "identity_norm": norm,
            "reference_smiles": row["smiles"],
            "reference_graph_smiles":
                row["reference_graph_smiles"],
        })


reference_identity = (
    pd.DataFrame(reference_identity_rows)
    .drop_duplicates()
    .reset_index(drop=True)
)


# ------------------------------------------------------------------
# Unique failed Wen/Ma acceptor identities
# ------------------------------------------------------------------

wen_failed_identity = (
    wen_audit.loc[
        wen_audit["acceptor_graph_smiles"].isna(),
        ["Name_Acceptor"]
    ]
    .drop_duplicates()
    .copy()
)

wen_failed_identity["name_norm"] = (
    wen_failed_identity["Name_Acceptor"]
    .apply(normalize_material_label)
)


# ------------------------------------------------------------------
# Accept recovery only when one unique reference graph exists
# ------------------------------------------------------------------

unique_reference_graph_map = {}

for _, row in wen_failed_identity.iterrows():

    matches = reference_identity[
        reference_identity["identity_norm"]
        .eq(row["name_norm"])
    ]

    graphs = (
        matches["reference_graph_smiles"]
        .dropna()
        .unique()
    )

    if len(graphs) == 1:
        unique_reference_graph_map[
            row["Name_Acceptor"]
        ] = graphs[0]


# ------------------------------------------------------------------
# Construct recovered Wen/Ma overlap table
# ------------------------------------------------------------------

wen_overlap = wen_audit.copy()

wen_overlap["acceptor_graph_resolved"] = (
    wen_overlap["acceptor_graph_smiles"]
    .copy()
)

recovery_mask = (
    wen_overlap["acceptor_graph_resolved"].isna()
    & wen_overlap["Name_Acceptor"].isin(
        unique_reference_graph_map
    )
)

wen_overlap.loc[
    recovery_mask,
    "acceptor_graph_resolved"
] = (
    wen_overlap.loc[
        recovery_mask,
        "Name_Acceptor"
    ]
    .map(unique_reference_graph_map)
)


wen_overlap["acceptor_resolution_source"] = np.select(
    [
        wen_overlap["acceptor_graph_smiles"].notna(),
        recovery_mask,
    ],
    [
        "original_wen_smiles",
        "unique_opvdb_reference_match",
    ],
    default="unresolved"
)


wen_overlap["pair_structurally_resolved"] = (
    wen_overlap["donor_graph_smiles"].notna()
    & wen_overlap["acceptor_graph_resolved"].notna()
)


# ------------------------------------------------------------------
# Verify restored state
# ------------------------------------------------------------------

resolution_summary = (
    wen_overlap["acceptor_resolution_source"]
    .value_counts()
    .rename_axis("resolution_source")
    .reset_index(name="records")
)

resolution_summary["percent"] = (
    100
    * resolution_summary["records"]
    / len(wen_overlap)
).round(2)

display(resolution_summary)

print(
    "\nFailed acceptor identities uniquely recovered:",
    len(unique_reference_graph_map)
)

print(
    "Structurally resolved D:A records:",
    int(wen_overlap["pair_structurally_resolved"].sum()),
    "/",
    len(wen_overlap)
)

print(
    "Unresolved records:",
    int(
        (~wen_overlap["pair_structurally_resolved"])
        .sum()
    )
)

,resolution_source,records,percent
0,original_wen_smiles,757,73.64
1,unique_opvdb_reference_match,173,16.83
2,unresolved,98,9.53



Failed acceptor identities uniquely recovered: 6
Structurally resolved D:A records: 930 / 1028
Unresolved records: 98


In [10]:
# ------------------------------------------------------------------
# Recovery Cell 5
# Rebuild OPV identity lexicons and Wen/Ma name-graph audits
# ------------------------------------------------------------------

def build_opv_identity_lexicon(role):

    if role == "donor":
        raw_col = "donor"
        canonical_col = "donor_canonical"
        graph_col = "donor_graph_smiles"

    elif role == "acceptor":
        raw_col = "acceptor"
        canonical_col = "acceptor_canonical"
        graph_col = "acceptor_graph_smiles"

    else:
        raise ValueError("role must be 'donor' or 'acceptor'")

    identity_rows = []

    # Device-table names
    for _, row in provenance_clean[
        [raw_col, canonical_col, graph_col]
    ].iterrows():

        graph = row[graph_col]

        if pd.isna(graph):
            continue

        for source, value in [
            ("device_raw", row[raw_col]),
            ("device_canonical", row[canonical_col]),
        ]:

            norm = normalize_material_label(value)

            if pd.isna(norm):
                continue

            identity_rows.append({
                "identity_norm": norm,
                "identity_label": value,
                "graph_smiles": graph,
                "identity_source": source,
            })

    # Materials-reference names + aliases
    ref = materials_ref[
        materials_ref["material_type"]
        .astype(str)
        .str.lower()
        .eq(role)
    ].copy()

    ref["parsed_aliases"] = ref["aliases"].apply(parse_aliases)
    ref["graph_smiles"] = ref["smiles"].apply(canonicalize_smiles)

    for _, row in ref.iterrows():

        graph = row["graph_smiles"]

        if pd.isna(graph):
            continue

        terms = [row["name"]] + row["parsed_aliases"]

        for term in terms:

            norm = normalize_material_label(term)

            if pd.isna(norm):
                continue

            identity_rows.append({
                "identity_norm": norm,
                "identity_label": term,
                "graph_smiles": graph,
                "identity_source": "material_reference",
            })

    return (
        pd.DataFrame(identity_rows)
        .drop_duplicates()
        .reset_index(drop=True)
    )


opv_donor_lexicon = build_opv_identity_lexicon("donor")
opv_acceptor_lexicon = build_opv_identity_lexicon("acceptor")


def diagnose_name_graph_overlap(
    wen_df,
    wen_name_col,
    wen_graph_col,
    opv_lexicon,
    role
):

    rows = []

    unique_wen = (
        wen_df[[wen_name_col, wen_graph_col]]
        .drop_duplicates()
        .copy()
    )

    for _, row in unique_wen.iterrows():

        wen_name = row[wen_name_col]
        wen_graph = row[wen_graph_col]

        name_norm = normalize_material_label(wen_name)

        matches = opv_lexicon[
            opv_lexicon["identity_norm"].eq(name_norm)
        ]

        candidate_graphs = (
            matches["graph_smiles"]
            .dropna()
            .unique()
        )

        if len(matches) == 0:
            status = "name_not_found_in_opvdb"

        elif pd.isna(wen_graph):
            status = "name_found_wen_graph_unresolved"

        elif wen_graph in candidate_graphs:
            status = "same_name_exact_graph_match"

        elif len(candidate_graphs) == 1:
            status = "same_name_different_graph"

        else:
            status = "same_name_opvdb_graph_ambiguous"

        rows.append({
            "role": role,
            "wen_name": wen_name,
            "wen_graph_resolved": pd.notna(wen_graph),
            "opv_name_matches": len(matches),
            "opv_candidate_graphs": len(candidate_graphs),
            "status": status,
            "opv_matching_labels": " | ".join(
                matches["identity_label"]
                .dropna()
                .astype(str)
                .unique()[:10]
            ),
        })

    return pd.DataFrame(rows)


wen_donor_name_audit = diagnose_name_graph_overlap(
    wen_overlap,
    "Name_Donor",
    "donor_graph_smiles",
    opv_donor_lexicon,
    "donor"
)

wen_acceptor_name_audit = diagnose_name_graph_overlap(
    wen_overlap,
    "Name_Acceptor",
    "acceptor_graph_resolved",
    opv_acceptor_lexicon,
    "acceptor"
)


name_graph_audit = pd.concat(
    [
        wen_donor_name_audit,
        wen_acceptor_name_audit
    ],
    ignore_index=True
)


name_graph_summary = (
    name_graph_audit
    .groupby(["role", "status"])
    .size()
    .rename("unique_wen_materials")
    .reset_index()
)

display(name_graph_summary)

,role,status,unique_wen_materials
0,acceptor,name_found_wen_graph_unresolved,1
1,acceptor,name_not_found_in_opvdb,109
2,acceptor,same_name_different_graph,43
3,acceptor,same_name_exact_graph_match,24
4,acceptor,same_name_opvdb_graph_ambiguous,4
5,donor,name_not_found_in_opvdb,31
6,donor,same_name_different_graph,18
7,donor,same_name_exact_graph_match,2
8,donor,same_name_opvdb_graph_ambiguous,9


In [11]:
# ------------------------------------------------------------------
# Save current Wen/Ma cross-dataset audit state
# ------------------------------------------------------------------

wen_overlap.to_csv(
    INTERIM_DIR / "wen_global_cross_dataset_audit_interim.csv",
    index=False
)

wen_donor_name_audit.to_csv(
    INTERIM_DIR / "wen_donor_name_graph_audit.csv",
    index=False
)

wen_acceptor_name_audit.to_csv(
    INTERIM_DIR / "wen_acceptor_name_graph_audit.csv",
    index=False
)

print("Checkpoint saved.")
print("Wen/Ma overlap table:", wen_overlap.shape)

Checkpoint saved.
Wen/Ma overlap table: (1028, 2094)


### 7.9 Nominal material-system overlap across datasets

Exact molecular-graph comparison revealed substantial representation
disagreement between OPV-DB and Wen–Zhang–Ma, particularly for donor polymers.

A complementary nominal-identity analysis is therefore performed using
normalized material names. Name-level identity is treated as diagnostic
evidence rather than definitive proof of molecular equivalence.

The analysis first quantifies whether Wen–Zhang–Ma donor and acceptor names
occur anywhere in OPV-DB. It then evaluates whether complete named
donor–acceptor systems are also represented in OPV-DB.

Comparing nominal-system overlap with exact graph-pair overlap allows true
chemical-domain differences to be distinguished from differences in molecular
representation conventions.

In [12]:
# ------------------------------------------------------------------
# 7.9A
# Row-level nominal donor and acceptor overlap
# ------------------------------------------------------------------

donor_status_map = (
    wen_donor_name_audit
    .set_index("wen_name")["status"]
    .to_dict()
)

acceptor_status_map = (
    wen_acceptor_name_audit
    .set_index("wen_name")["status"]
    .to_dict()
)


wen_overlap["donor_name_graph_status"] = (
    wen_overlap["Name_Donor"]
    .map(donor_status_map)
)

wen_overlap["acceptor_name_graph_status"] = (
    wen_overlap["Name_Acceptor"]
    .map(acceptor_status_map)
)


# A material name is considered nominally represented if its
# normalized name was found somewhere in the OPV-DB identity lexicon.
wen_overlap["donor_name_seen_in_opvdb"] = (
    wen_overlap["donor_name_graph_status"]
    .notna()
    & ~wen_overlap["donor_name_graph_status"]
        .eq("name_not_found_in_opvdb")
)

wen_overlap["acceptor_name_seen_in_opvdb"] = (
    wen_overlap["acceptor_name_graph_status"]
    .notna()
    & ~wen_overlap["acceptor_name_graph_status"]
        .eq("name_not_found_in_opvdb")
)


both_names_seen = (
    wen_overlap["donor_name_seen_in_opvdb"]
    & wen_overlap["acceptor_name_seen_in_opvdb"]
)


name_overlap_summary = pd.DataFrame({
    "identity_level": [
        "donor name",
        "acceptor name",
        "both donor and acceptor names",
    ],

    "overlapping_records": [
        int(wen_overlap["donor_name_seen_in_opvdb"].sum()),
        int(wen_overlap["acceptor_name_seen_in_opvdb"].sum()),
        int(both_names_seen.sum()),
    ],
})


name_overlap_summary["total_wen_records"] = len(wen_overlap)

name_overlap_summary["percent_of_wen"] = (
    100
    * name_overlap_summary["overlapping_records"]
    / name_overlap_summary["total_wen_records"]
).round(2)


display(name_overlap_summary)

,identity_level,overlapping_records,total_wen_records,percent_of_wen
0,donor name,784,1028,76.26
1,acceptor name,511,1028,49.71
2,both donor and acceptor names,340,1028,33.07


#### 7.9.1 Exact nominal donor–acceptor pair overlap

Individual donor and acceptor overlap does not establish that the same
photovoltaic system occurs in both datasets. A donor and acceptor may each
be represented in OPV-DB while their specific combination is absent.

Normalized donor–acceptor name pairs are therefore compared directly between
the provenance-clean OPV-DB cohort and Wen–Zhang–Ma.

This name-based comparison remains a diagnostic identity layer rather than a
claim of exact structural equivalence.

In [13]:
# ------------------------------------------------------------------
# 7.9B
# Exact nominal donor-acceptor pair overlap
# ------------------------------------------------------------------

# Construct all nominal pair variants directly represented
# in the provenance-clean OPV-DB device table.

opv_named_pairs = set()

for _, row in provenance_clean[
    [
        "donor",
        "donor_canonical",
        "acceptor",
        "acceptor_canonical",
    ]
].iterrows():

    donor_terms = {
        normalize_material_label(row["donor"]),
        normalize_material_label(row["donor_canonical"]),
    }

    acceptor_terms = {
        normalize_material_label(row["acceptor"]),
        normalize_material_label(row["acceptor_canonical"]),
    }

    donor_terms = {
        x for x in donor_terms
        if pd.notna(x)
    }

    acceptor_terms = {
        x for x in acceptor_terms
        if pd.notna(x)
    }

    for donor_name in donor_terms:
        for acceptor_name in acceptor_terms:

            opv_named_pairs.add(
                (donor_name, acceptor_name)
            )


# Normalize Wen/Ma names
wen_overlap["donor_name_norm"] = (
    wen_overlap["Name_Donor"]
    .apply(normalize_material_label)
)

wen_overlap["acceptor_name_norm"] = (
    wen_overlap["Name_Acceptor"]
    .apply(normalize_material_label)
)


wen_overlap["nominal_pair"] = list(
    zip(
        wen_overlap["donor_name_norm"],
        wen_overlap["acceptor_name_norm"]
    )
)


wen_overlap["named_pair_seen_in_opvdb"] = (
    wen_overlap["nominal_pair"]
    .isin(opv_named_pairs)
)


print(
    "Unique OPV-DB nominal pair representations:",
    f"{len(opv_named_pairs):,}"
)

print(
    "\nWen/Ma records with exact named D:A pair in OPV-DB:",
    int(wen_overlap["named_pair_seen_in_opvdb"].sum()),
    "/",
    len(wen_overlap)
)

print(
    "Percentage:",
    round(
        100
        * wen_overlap["named_pair_seen_in_opvdb"].mean(),
        2
    ),
    "%"
)


# ------------------------------------------------------------------
# Nominal cross-dataset system categories
# ------------------------------------------------------------------

def classify_nominal_system(row):

    if row["named_pair_seen_in_opvdb"]:
        return "same named D:A pair in OPV-DB"

    donor_seen = row["donor_name_seen_in_opvdb"]
    acceptor_seen = row["acceptor_name_seen_in_opvdb"]

    if donor_seen and acceptor_seen:
        return "both material names seen, pairing unseen"

    if donor_seen:
        return "donor name seen only"

    if acceptor_seen:
        return "acceptor name seen only"

    return "neither material name seen"


wen_overlap["nominal_system_category"] = (
    wen_overlap.apply(
        classify_nominal_system,
        axis=1
    )
)


nominal_novelty_summary = (
    wen_overlap["nominal_system_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="records")
)

nominal_novelty_summary["percent_of_wen"] = (
    100
    * nominal_novelty_summary["records"]
    / len(wen_overlap)
).round(2)


display(nominal_novelty_summary)



# ------------------------------------------------------------------
# Which shared nominal pairs account for the overlap?
# ------------------------------------------------------------------

shared_named_pairs = (
    wen_overlap[
        wen_overlap["named_pair_seen_in_opvdb"]
    ]
    .groupby(
        ["Name_Donor", "Name_Acceptor"]
    )
    .size()
    .rename("wen_records")
    .reset_index()
    .sort_values(
        "wen_records",
        ascending=False
    )
)

print(
    "Unique Wen/Ma named D:A pairs also present in OPV-DB:",
    len(shared_named_pairs)
)

display(
    shared_named_pairs.head(30)
)

Unique OPV-DB nominal pair representations: 5,535

Wen/Ma records with exact named D:A pair in OPV-DB: 288 / 1028
Percentage: 28.02 %


,category,records,percent_of_wen
0,donor name seen only,444,43.19
1,same named D:A pair in OPV-DB,288,28.02
2,acceptor name seen only,171,16.63
3,neither material name seen,73,7.10
4,"both material names seen, pairing unseen",52,5.06


Unique Wen/Ma named D:A pairs also present in OPV-DB: 69


,Name_Donor,Name_Acceptor,wen_records
64,Qx-12F,Y6,19
9,L2,Y6,18
39,PM6,PTIC-4Cl,13
16,PBDB-T,CNDTBT-IDTT-FINCN,12
17,PBDB-T,FDTBT-IDTT-FINCN,12
36,PM6,BTIC-4F,11
46,PM6,TPDC-4F,11
26,PBDB-T,S-SubPc-PDI,11
61,PTB7-Th,SSBR,10
7,J71,ZITI-4F,10


#### 7.9.2 Structural representation within shared nominal systems

A substantial subset of Wen–Zhang–Ma records shares the same normalized
donor–acceptor names with OPV-DB despite zero overlap under exact
canonical-graph pair identity.

To determine the source of this discrepancy, shared nominal systems are
examined at the component-graph level.

For each Wen–Zhang–Ma record whose named donor–acceptor pair occurs in
OPV-DB, donor and acceptor molecular graphs are independently tested for
presence in the OPV-DB structural vocabulary. This distinguishes
representation disagreement in the donor, acceptor, or both components from
cases where both component graphs are individually represented but not as the
same graph pair.

In [14]:
# ------------------------------------------------------------------
# 7.9C
# Structural compatibility inside shared nominal D:A systems
# ------------------------------------------------------------------

opv_donor_graphs = set(
    provenance_clean[
        "donor_graph_smiles"
    ].dropna()
)

opv_acceptor_graphs = set(
    provenance_clean[
        "acceptor_graph_smiles"
    ].dropna()
)

opv_graph_pairs = set(
    zip(
        provenance_clean["donor_graph_smiles"],
        provenance_clean["acceptor_graph_smiles"]
    )
)


wen_overlap["donor_graph_seen_in_opvdb"] = (
    wen_overlap["donor_graph_smiles"]
    .isin(opv_donor_graphs)
)

wen_overlap["acceptor_graph_seen_in_opvdb"] = (
    wen_overlap["acceptor_graph_resolved"]
    .notna()
    & wen_overlap["acceptor_graph_resolved"]
        .isin(opv_acceptor_graphs)
)


wen_overlap["resolved_graph_pair"] = list(
    zip(
        wen_overlap["donor_graph_smiles"],
        wen_overlap["acceptor_graph_resolved"]
    )
)

wen_overlap["exact_graph_pair_seen_in_opvdb"] = (
    wen_overlap["pair_structurally_resolved"]
    & wen_overlap["resolved_graph_pair"]
        .isin(opv_graph_pairs)
)


shared_nominal = wen_overlap[
    wen_overlap["named_pair_seen_in_opvdb"]
].copy()

print(
    "Shared nominal-system records:",
    len(shared_nominal)
)

print(
    "Exact graph-pair matches among them:",
    int(
        shared_nominal[
            "exact_graph_pair_seen_in_opvdb"
        ].sum()
    )
)


# ------------------------------------------------------------------
# Source of graph disagreement
# ------------------------------------------------------------------

def representation_status(row):

    if not row["pair_structurally_resolved"]:
        return "acceptor structure unresolved"

    donor_graph_seen = (
        row["donor_graph_seen_in_opvdb"]
    )

    acceptor_graph_seen = (
        row["acceptor_graph_seen_in_opvdb"]
    )

    pair_seen = (
        row["exact_graph_pair_seen_in_opvdb"]
    )

    if pair_seen:
        return "exact graph pair also represented"

    if donor_graph_seen and acceptor_graph_seen:
        return "both component graphs seen, exact pair absent"

    if donor_graph_seen and not acceptor_graph_seen:
        return "acceptor graph representation differs"

    if not donor_graph_seen and acceptor_graph_seen:
        return "donor graph representation differs"

    return "both graph representations differ"


shared_nominal[
    "representation_status"
] = shared_nominal.apply(
    representation_status,
    axis=1
)


representation_summary = (
    shared_nominal[
        "representation_status"
    ]
    .value_counts()
    .rename_axis("representation_status")
    .reset_index(name="records")
)

representation_summary[
    "percent_of_shared_nominal"
] = (
    100
    * representation_summary["records"]
    / len(shared_nominal)
).round(2)

display(representation_summary)



# ------------------------------------------------------------------
# Which shared named systems dominate the representation disagreement?
# ------------------------------------------------------------------

shared_pair_representation = (
    shared_nominal
    .groupby(
        [
            "Name_Donor",
            "Name_Acceptor",
            "representation_status",
        ]
    )
    .size()
    .rename("records")
    .reset_index()
    .sort_values(
        "records",
        ascending=False
    )
)

display(
    shared_pair_representation.head(40)
)

Shared nominal-system records: 288
Exact graph-pair matches among them: 0


,representation_status,records,percent_of_shared_nominal
0,both graph representations differ,145,50.35
1,donor graph representation differs,136,47.22
2,acceptor graph representation differs,7,2.43


,Name_Donor,Name_Acceptor,representation_status,records
64,Qx-12F,Y6,donor graph representation differs,19
9,L2,Y6,donor graph representation differs,18
39,PM6,PTIC-4Cl,both graph representations differ,13
16,PBDB-T,CNDTBT-IDTT-FINCN,both graph representations differ,12
17,PBDB-T,FDTBT-IDTT-FINCN,both graph representations differ,12
36,PM6,BTIC-4F,both graph representations differ,11
46,PM6,TPDC-4F,both graph representations differ,11
26,PBDB-T,S-SubPc-PDI,donor graph representation differs,11
61,PTB7-Th,SSBR,both graph representations differ,10
7,J71,ZITI-4F,both graph representations differ,10


#### 7.9.3 Structural similarity within nominally identical materials

Exact graph equality is highly sensitive to molecular representation choices,
particularly for conjugated polymers represented using different repeat units,
terminations, oligomer lengths, or attachment conventions.

To determine whether same-named materials with non-identical graph
representations nevertheless retain substantial local structural similarity,
Morgan fingerprints are generated consistently from both datasets.

For each Wen–Zhang–Ma molecular graph, similarity is calculated against all
OPV-DB molecular graphs carrying the same normalized material identity. The
maximum Tanimoto similarity is used as a diagnostic measure.

Similarity is not treated as proof of chemical identity; it is used only to
characterize the magnitude of cross-dataset representation disagreement.

In [15]:
# ------------------------------------------------------------------
# 7.9D
# Same-name molecular similarity across datasets
# ------------------------------------------------------------------

from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)


def smiles_to_morgan(smiles):

    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None

    return morgan_generator.GetFingerprint(mol)


def max_same_name_similarity(
    wen_name,
    wen_graph,
    opv_lexicon
):

    if pd.isna(wen_graph):
        return np.nan

    name_norm = normalize_material_label(
        wen_name
    )

    candidates = (
        opv_lexicon.loc[
            opv_lexicon["identity_norm"]
            .eq(name_norm),
            "graph_smiles"
        ]
        .dropna()
        .unique()
    )

    if len(candidates) == 0:
        return np.nan

    wen_fp = smiles_to_morgan(
        wen_graph
    )

    if wen_fp is None:
        return np.nan

    similarities = []

    for candidate in candidates:

        candidate_fp = smiles_to_morgan(
            candidate
        )

        if candidate_fp is None:
            continue

        similarities.append(
            DataStructs.TanimotoSimilarity(
                wen_fp,
                candidate_fp
            )
        )

    if not similarities:
        return np.nan

    return max(similarities)


    # ------------------------------------------------------------------
# Donor similarity
# ------------------------------------------------------------------

wen_donor_similarity = (
    wen_overlap[
        [
            "Name_Donor",
            "donor_graph_smiles"
        ]
    ]
    .drop_duplicates()
    .copy()
)

wen_donor_similarity[
    "max_same_name_tanimoto"
] = wen_donor_similarity.apply(
    lambda row:
        max_same_name_similarity(
            row["Name_Donor"],
            row["donor_graph_smiles"],
            opv_donor_lexicon
        ),
    axis=1
)


# ------------------------------------------------------------------
# Acceptor similarity
# ------------------------------------------------------------------

wen_acceptor_similarity = (
    wen_overlap[
        [
            "Name_Acceptor",
            "acceptor_graph_resolved"
        ]
    ]
    .drop_duplicates()
    .copy()
)

wen_acceptor_similarity[
    "max_same_name_tanimoto"
] = (
    wen_acceptor_similarity.apply(
        lambda row:
            max_same_name_similarity(
                row["Name_Acceptor"],
                row["acceptor_graph_resolved"],
                opv_acceptor_lexicon
            ),
        axis=1
    )
)



# ------------------------------------------------------------------
# Same-name similarity summary
# ------------------------------------------------------------------

similarity_summary = pd.DataFrame({

    "role": [
        "donor",
        "acceptor"
    ],

    "materials_with_same_name_comparison": [
        wen_donor_similarity[
            "max_same_name_tanimoto"
        ].notna().sum(),

        wen_acceptor_similarity[
            "max_same_name_tanimoto"
        ].notna().sum(),
    ],

    "median_max_tanimoto": [
        wen_donor_similarity[
            "max_same_name_tanimoto"
        ].median(),

        wen_acceptor_similarity[
            "max_same_name_tanimoto"
        ].median(),
    ],

    "mean_max_tanimoto": [
        wen_donor_similarity[
            "max_same_name_tanimoto"
        ].mean(),

        wen_acceptor_similarity[
            "max_same_name_tanimoto"
        ].mean(),
    ],

    "min_max_tanimoto": [
        wen_donor_similarity[
            "max_same_name_tanimoto"
        ].min(),

        wen_acceptor_similarity[
            "max_same_name_tanimoto"
        ].min(),
    ],
})

display(
    similarity_summary.round(3)
)



donor_similarity_diagnostic = (
    wen_donor_name_audit[
        wen_donor_name_audit["status"]
        .isin([
            "same_name_different_graph",
            "same_name_opvdb_graph_ambiguous",
        ])
    ]
    .merge(
        wen_donor_similarity,
        left_on="wen_name",
        right_on="Name_Donor",
        how="left"
    )
)


display(
    donor_similarity_diagnostic[
        [
            "wen_name",
            "status",
            "opv_candidate_graphs",
            "max_same_name_tanimoto",
        ]
    ]
    .sort_values(
        "max_same_name_tanimoto",
        ascending=False
    )
)



acceptor_similarity_diagnostic = (
    wen_acceptor_name_audit[
        wen_acceptor_name_audit["status"]
        .isin([
            "same_name_different_graph",
            "same_name_opvdb_graph_ambiguous",
        ])
    ]
    .merge(
        wen_acceptor_similarity,
        left_on="wen_name",
        right_on="Name_Acceptor",
        how="left"
    )
)


display(
    acceptor_similarity_diagnostic[
        [
            "wen_name",
            "status",
            "opv_candidate_graphs",
            "max_same_name_tanimoto",
        ]
    ]
    .sort_values(
        "max_same_name_tanimoto",
        ascending=False
    )
)

,role,materials_with_same_name_comparison,median_max_tanimoto,mean_max_tanimoto,min_max_tanimoto
0,donor,29,1.000,0.832,0.147
1,acceptor,71,0.958,0.852,0.241


,wen_name,status,opv_candidate_graphs,max_same_name_tanimoto
0,J61,same_name_different_graph,1,1.000000
1,J71,same_name_opvdb_graph_ambiguous,2,1.000000
2,PBDB-T,same_name_opvdb_graph_ambiguous,5,1.000000
3,PTB7,same_name_opvdb_graph_ambiguous,3,1.000000
6,PM6,same_name_opvdb_graph_ambiguous,4,1.000000
8,PBDB-T-2Cl,same_name_opvdb_graph_ambiguous,2,1.000000
7,PBDB-T-2F,same_name_opvdb_graph_ambiguous,4,1.000000
9,PCE-10,same_name_different_graph,1,1.000000
20,SM1-S,same_name_different_graph,1,1.000000
22,PBDTTT-E-T,same_name_different_graph,1,1.000000


,wen_name,status,opv_candidate_graphs,max_same_name_tanimoto
1,TTz1,same_name_different_graph,1,1.000000
2,TTz2,same_name_different_graph,1,1.000000
9,EH-IDTBR,same_name_different_graph,1,1.000000
14,SSBRC,same_name_different_graph,1,1.000000
13,SSBR,same_name_different_graph,1,1.000000
41,TPIC,same_name_different_graph,1,1.000000
29,PY-IT,same_name_different_graph,1,1.000000
45,PBDB-T,same_name_different_graph,1,1.000000
35,PNDIBSF,same_name_different_graph,1,1.000000
34,PNDIBS,same_name_different_graph,1,1.000000


### 7.10 Cross-dataset identity audit: conclusions

Cross-dataset comparison revealed a pronounced distinction between nominal
material identity and exact molecular-graph identity.

Among the 1,028 Wen–Zhang–Ma global records, 784 (76.26%) contain a donor
whose normalized name occurs in OPV-DB, 511 (49.71%) contain an acceptor name
represented in OPV-DB, and 340 (33.07%) contain both individually represented
material names.

More importantly, 288 records (28.02%), spanning 69 distinct donor–acceptor
systems, have the same normalized donor–acceptor pair in provenance-clean
OPV-DB. Nevertheless, none of these records matches OPV-DB under exact
canonical molecular-graph pair identity.

Within the 288 shared nominal systems, donor graph representation differs in
281 records (97.57%), while acceptor graph representation differs in 152
records (52.78%). Same-name Morgan-fingerprint comparisons further show high
best-match similarity for many materials despite non-identical canonical
graphs: the median maximum Tanimoto similarity is 1.000 for donors and 0.958
for acceptors.

However, substantial low-similarity exceptions remain. Consequently, material
names are not treated as definitive structural identities, and fingerprint
similarity is not interpreted as proof of molecular equivalence.

Together, these results demonstrate substantial nominal chemical overlap
between the datasets alongside strong molecular-representation domain shift,
particularly for conjugated-polymer donors. Subsequent cross-dataset learning
experiments must therefore distinguish chemical-domain transfer from
representation-domain transfer rather than assuming directly interchangeable
molecular encodings.

In [16]:
# ------------------------------------------------------------------
# Save completed cross-dataset identity audit
# ------------------------------------------------------------------

wen_overlap.to_csv(
    INTERIM_DIR / "wen_global_cross_dataset_audit_interim.csv",
    index=False
)

name_overlap_summary.to_csv(
    INTERIM_DIR / "cross_dataset_name_overlap_summary.csv",
    index=False
)

nominal_novelty_summary.to_csv(
    INTERIM_DIR / "cross_dataset_nominal_system_summary.csv",
    index=False
)

representation_summary.to_csv(
    INTERIM_DIR / "shared_system_representation_summary.csv",
    index=False
)

wen_donor_similarity.to_csv(
    INTERIM_DIR / "wen_donor_same_name_similarity.csv",
    index=False
)

wen_acceptor_similarity.to_csv(
    INTERIM_DIR / "wen_acceptor_same_name_similarity.csv",
    index=False
)

print("Section 7 cross-dataset identity audit saved.")

Section 7 cross-dataset identity audit saved.


## 8. Wen–Zhang–Ma internal data-quality audit

The Wen–Zhang–Ma global dataset is now evaluated independently of OPV-DB.

The audit examines whether records are duplicated, whether identical predictor
descriptions are associated with conflicting photovoltaic efficiencies, how
processing variables are encoded, and whether descriptor and fingerprint
blocks are internally consistent.

These checks are performed before model construction so that repeated
measurements, inconsistent targets, or representation artifacts are not
mistaken for predictive signal.

In [17]:
# ------------------------------------------------------------------
# 8.1
# Wen/Ma global-table schema reconnaissance
# ------------------------------------------------------------------

print("Dataset dimensions:")
print(wen_global.shape)

print("\nFirst 40 columns:")
for i, col in enumerate(wen_global.columns[:40]):
    print(f"{i:4d}  {col}")

print("\nLast 20 columns:")
for i, col in enumerate(
    wen_global.columns[-20:],
    start=len(wen_global.columns) - 20
):
    print(f"{i:4d}  {col}")

print("\nColumns containing common performance terms:")

performance_terms = [
    "pce",
    "voc",
    "jsc",
    "ff",
    "efficiency",
]

performance_columns = [
    col
    for col in wen_global.columns
    if any(
        term in str(col).lower()
        for term in performance_terms
    )
]

print(performance_columns)



# ------------------------------------------------------------------
# 8.2
# Exact full-row duplicates
# ------------------------------------------------------------------

exact_duplicate_mask = (
    wen_global.duplicated(
        keep=False
    )
)

exact_duplicate_rows = (
    wen_global[
        exact_duplicate_mask
    ]
    .copy()
)

print(
    "Rows participating in exact full-row duplicates:",
    exact_duplicate_mask.sum()
)

print(
    "Unique exact duplicate groups:",
    len(
        wen_global[
            exact_duplicate_mask
        ].drop_duplicates()
    )
)

print(
    "Completely unique rows:",
    (~wen_global.duplicated()).sum()
)



# ------------------------------------------------------------------
# 8.3
# Identical predictor rows with potentially different PCE
# ------------------------------------------------------------------

TARGET_COL = "PCE (%)"

assert TARGET_COL in wen_global.columns

predictor_columns = [
    col
    for col in wen_global.columns
    if col != TARGET_COL
]


predictor_duplicate_mask = (
    wen_global.duplicated(
        subset=predictor_columns,
        keep=False
    )
)

predictor_duplicate_rows = (
    wen_global.loc[
        predictor_duplicate_mask
    ]
    .copy()
)


print(
    "Rows sharing an identical predictor vector:",
    predictor_duplicate_mask.sum()
)



# ------------------------------------------------------------------
# Group repeated predictor vectors and quantify target disagreement
# ------------------------------------------------------------------

predictor_groups = (
    predictor_duplicate_rows
    .groupby(
        predictor_columns,
        dropna=False
    )[TARGET_COL]
    .agg(
        n_records="size",
        n_unique_pce="nunique",
        pce_min="min",
        pce_max="max",
        pce_mean="mean",
        pce_std="std",
    )
    .reset_index()
)

predictor_groups[
    "pce_range"
] = (
    predictor_groups["pce_max"]
    - predictor_groups["pce_min"]
)


print(
    "Repeated predictor groups:",
    len(predictor_groups)
)

print(
    "Groups with identical PCE:",
    (
        predictor_groups[
            "n_unique_pce"
        ] == 1
    ).sum()
)

print(
    "Groups with conflicting PCE:",
    (
        predictor_groups[
            "n_unique_pce"
        ] > 1
    ).sum()
)


conflicting_predictor_groups = (
    predictor_groups[
        predictor_groups[
            "n_unique_pce"
        ] > 1
    ]
    .sort_values(
        "pce_range",
        ascending=False
    )
    .copy()
)


print(
    "\nMaximum PCE disagreement:",
    conflicting_predictor_groups[
        "pce_range"
    ].max()
)



# ------------------------------------------------------------------
# Compact view of conflicting duplicate predictor groups
# ------------------------------------------------------------------

compact_columns = [
    col for col in [
        "Name_Donor",
        "Name_Acceptor",
        "Smiles_Donor",
        "Smiles_Acceptor",
        "D_A_Weight_Ratio",
        "Blend_Concentration",
        "Solvent",
        "Additive",
        "Additive_Volume_Ratio",
        "Spin_Coating_Speed",
        "Active_Layer_Thickness",
        "Annealing_Temperature",
        "Annealing_Time",
        "n_records",
        "n_unique_pce",
        "pce_min",
        "pce_max",
        "pce_range",
    ]
    if col in conflicting_predictor_groups.columns
]

display(
    conflicting_predictor_groups[
        compact_columns
    ].head(30)
)

Dataset dimensions:
(1028, 2089)

First 40 columns:
   0  Name_Donor
   1  Smiles_Donor
   2  Name_Acceptor
   3  Smiles_Acceptor
   4  PCE (%)
   5  D_A_Weight_Ratio
   6  Blend_Concentration (mg/ml)
   7  Solvent_DipoleMoment (Debye)
   8  Solvent_EnergyGap (eV)
   9  Solvent_Polarizability (a.u.)
  10  Additive_MeltingPoint (℃)
  11  Additive_BoilingPoint  (℃)
  12  Additive_Density (g/cm3)
  13  Additive_MolecularWeight
  14  Additive_DipoleMoment  (Debye)
  15  Additive_EnergyGap (eV)
  16  Additive_Polarizability (a.u.)
  17  Additive_Volume_Ratio (vol%)
  18  Spin_Coating_Rate (rpm)
  19  Annealing_Temperature(℃)
  20  Annealing_Time (min)
  21  Active_Layer_Thickness (nm)
  22  E_DH-1 (eV)
  23  E_DH (eV)
  24  E_DL (eV)
  25  E_DL+1 (eV)
  26  E_AH-1 (eV)
  27  E_AH (eV)
  28  E_AL (eV)
  29  E_AL+1 (eV)
  30  delta_E_DL_DH (eV)
  31  delta_E_AL_AH (eV)
  32  delta_E_DL_AL (eV)
  33  delta_E_DH_AH (eV)
  34  delta_E_AL-DH (eV)
  35  delta_E_DH_DH-1 (eV)
  36  delta_E_DL+1_DL (

<TEMP_KERNEL_PATH>\2438855076.py:138: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()
<TEMP_KERNEL_PATH>\2438855076.py:141: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  predictor_groups[


Repeated predictor groups: 28
Groups with identical PCE: 18
Groups with conflicting PCE: 10

Maximum PCE disagreement: 4.41


,Name_Donor,Name_Acceptor,Smiles_Donor,Smiles_Acceptor,D_A_Weight_Ratio,n_records,n_unique_pce,pce_min,pce_max,pce_range
4,P130,Y6,c1(c2c3c(c(c(c2F)F)c2sc(cc2)c2c(c4c(s2)c2c(c5c...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,0.83,2,2,10.87,15.28,4.41
5,P131,Y6,c1(c2c3c(c(cc2)c2sc(cc2)c2c(c4c(s2)c2c(c5c4c(c...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,0.83,2,2,7.60,11.13,3.53
13,PM6,ML-2FM,s1c2c(cc1)c(c1c(c2c2sc(c(c2)F)C[C@H](CCCC)CC)c...,c1c(cc(c2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1)n(c1c3...,1.00,2,2,13.32,15.33,2.01
0,C-2F,N3,c12c(c(c3c(c1c1cc(c(c(c1)F)C[C@H](CCCC)CC)F)cc...,C1=C(C(=C[C@@H]2[C@@H]1C(=C(C#N)C#N)/C(=C/c1sc...,1.60,2,2,13.21,14.31,1.10
2,L2,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,1.50,2,2,14.90,15.80,0.90
7,P3HT,TTDTC-4F,c1scc(c1)CCCCCC,C1(=C(C(=O)c2c1cc(c(c2)F)F)[CH]c1sc2c(c1)[C@](...,1.25,2,2,4.34,4.81,0.47
3,L2,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,1.50,2,2,14.02,14.30,0.28
11,PM6,C7BTP-BO-2Cl-2F,s1c2c(cc1)c(c1c(c2c2sc(c(c2)F)C[C@H](CCCC)CC)c...,c12c(c3c(c4c1nsn4)c1c(n3C[C@@H](CCCCCC)CCCC)c3...,0.83,2,2,16.20,16.40,0.20
6,P3HT,TTDTC-0F,c1scc(c1)CCCCCC,C1(=C(C#N)C#N)c2c(C(=O)/C/1=C\c1sc3c(c1)[C@](c...,1.25,2,2,3.77,3.95,0.18
17,PM6,PTIC-4Cl,s1c2c(cc1)c(c1c(c2c2sc(c(c2)F)C[C@H](CCCC)CC)c...,c1(cc2c(cc1Cl)C(=O)C(=C2[C](C#N)C#N)/C=C\1/[C]...,0.83,2,2,14.81,14.82,0.01


### 8.4 Repeated predictor vectors and target inconsistency

Identical feature vectors occur multiple times in the Wen–Zhang–Ma global
dataset. Some repeated vectors have identical PCE values, whereas others are
associated with different efficiencies.

Because the complete recorded predictor representation is identical within
these groups, target disagreement reflects either unrecorded experimental
variation, source-level differences, or curation inconsistency.

The repeated groups are therefore characterized before deciding whether exact
duplicates should be removed, conflicting observations retained as grouped
replicates, or replicate targets aggregated for modeling.

In [18]:
# ------------------------------------------------------------------
# 8.4A
# Size and severity of repeated predictor groups
# ------------------------------------------------------------------

print("Repeated predictor group-size distribution:")

display(
    predictor_groups["n_records"]
    .value_counts()
    .sort_index()
    .rename_axis("records_per_group")
    .reset_index(name="groups")
)


conflict_ranges = (
    conflicting_predictor_groups["pce_range"]
)

print("\nConflicting groups:", len(conflicting_predictor_groups))
print(
    "Rows involved in conflicting groups:",
    int(conflicting_predictor_groups["n_records"].sum())
)

print(
    "\nMedian PCE disagreement:",
    round(conflict_ranges.median(), 3)
)

print(
    "Mean PCE disagreement:",
    round(conflict_ranges.mean(), 3)
)

print(
    "Maximum PCE disagreement:",
    round(conflict_ranges.max(), 3)
)


for threshold in [0.1, 0.5, 1.0, 2.0]:
    print(
        f"Groups with PCE range > {threshold}:",
        int((conflict_ranges > threshold).sum())
    )



    # ------------------------------------------------------------------
# 8.4B
# Assign a stable identifier to identical predictor vectors
# ------------------------------------------------------------------

wen_duplicate_audit = wen_global.copy()

wen_duplicate_audit["predictor_hash"] = (
    pd.util.hash_pandas_object(
        wen_duplicate_audit[predictor_columns],
        index=False
    )
)


hash_summary = (
    wen_duplicate_audit
    .groupby("predictor_hash")[TARGET_COL]
    .agg(
        n_records="size",
        n_unique_pce="nunique",
        pce_min="min",
        pce_max="max",
    )
    .reset_index()
)

hash_summary["pce_range"] = (
    hash_summary["pce_max"]
    - hash_summary["pce_min"]
)


conflicting_hashes = set(
    hash_summary.loc[
        hash_summary["n_unique_pce"] > 1,
        "predictor_hash"
    ]
)


conflicting_rows = (
    wen_duplicate_audit[
        wen_duplicate_audit["predictor_hash"]
        .isin(conflicting_hashes)
    ]
    .copy()
)

print(
    "Conflicting original rows:",
    len(conflicting_rows)
)

print(
    "Conflicting predictor groups:",
    conflicting_rows["predictor_hash"].nunique()
)



# ------------------------------------------------------------------
# 8.4C
# Compact conflicting-row inspection
# ------------------------------------------------------------------

processing_columns = [
    "D_A_Weight_Ratio",
    "Blend_Concentration (mg/ml)",
    "Solvent_DipoleMoment (Debye)",
    "Solvent_EnergyGap (eV)",
    "Solvent_Polarizability (a.u.)",
    "Additive_MeltingPoint (℃)",
    "Additive_BoilingPoint  (℃)",
    "Additive_Density (g/cm3)",
    "Additive_MolecularWeight",
    "Additive_DipoleMoment  (Debye)",
    "Additive_EnergyGap (eV)",
    "Additive_Polarizability (a.u.)",
    "Additive_Volume_Ratio (vol%)",
    "Spin_Coating_Rate (rpm)",
    "Annealing_Temperature(℃)",
    "Annealing_Time (min)",
    "Active_Layer_Thickness (nm)",
]

display_columns = [
    "predictor_hash",
    "Name_Donor",
    "Name_Acceptor",
    *[
        col
        for col in processing_columns
        if col in conflicting_rows.columns
    ],
    TARGET_COL,
]


display(
    conflicting_rows[
        display_columns
    ]
    .sort_values(
        ["predictor_hash", TARGET_COL]
    )
)



# ------------------------------------------------------------------
# 8.4D
# Effective sample counts
# ------------------------------------------------------------------

n_original = len(wen_global)

n_after_exact_dedup = len(
    wen_global.drop_duplicates()
)

n_unique_predictor_vectors = len(
    wen_global.drop_duplicates(
        subset=predictor_columns
    )
)


effective_sample_summary = pd.DataFrame({
    "dataset_definition": [
        "original rows",
        "after exact full-row deduplication",
        "unique predictor vectors",
    ],
    "records": [
        n_original,
        n_after_exact_dedup,
        n_unique_predictor_vectors,
    ],
})

effective_sample_summary["percent_of_original"] = (
    100
    * effective_sample_summary["records"]
    / n_original
).round(2)

display(effective_sample_summary)

Repeated predictor group-size distribution:


,records_per_group,groups
0,2,28



Conflicting groups: 10
Rows involved in conflicting groups: 20

Median PCE disagreement: 0.685
Mean PCE disagreement: 1.309
Maximum PCE disagreement: 4.41
Groups with PCE range > 0.1: 9
Groups with PCE range > 0.5: 5
Groups with PCE range > 1.0: 4
Groups with PCE range > 2.0: 3
Conflicting original rows: 20
Conflicting predictor groups: 10


,predictor_hash,Name_Donor,Name_Acceptor,D_A_Weight_Ratio,Blend_Concentration (mg/ml),Solvent_DipoleMoment (Debye),Solvent_EnergyGap (eV),Solvent_Polarizability (a.u.),Additive_MeltingPoint (℃),Additive_BoilingPoint (℃),...,Additive_MolecularWeight,Additive_DipoleMoment (Debye),Additive_EnergyGap (eV),Additive_Polarizability (a.u.),Additive_Volume_Ratio (vol%),Spin_Coating_Rate (rpm),Annealing_Temperature(℃),Annealing_Time (min),Active_Layer_Thickness (nm),PCE (%)
194,220727013179546949,C-2F,N3,1.60,18.0,1.2948,10.388224,39.062667,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,1500,120,10,120.0,13.21
198,220727013179546949,C-2F,N3,1.60,18.0,1.2948,10.388224,39.062667,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,1500,120,10,120.0,14.31
205,5584278388726656165,P130,Y6,0.83,16.0,1.2948,10.388224,39.062667,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,2000,0,0,95.0,10.87
202,5584278388726656165,P130,Y6,0.83,16.0,1.2948,10.388224,39.062667,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,2000,0,0,95.0,15.28
221,7173200220340131552,P3HT,TTDTC-4F,1.25,22.0,1.9304,8.740846,65.217333,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,2000,0,0,100.0,4.34
222,7173200220340131552,P3HT,TTDTC-4F,1.25,22.0,1.9304,8.740846,65.217333,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,2000,0,0,100.0,4.81
189,9864860838704130922,PM6,ML-2FM,1.00,16.0,1.2948,10.388224,39.062667,-20.0,260.27,...,162.6156,1.833775,6.857817,108.221667,0.5,3500,90,8,110.0,13.32
184,9864860838704130922,PM6,ML-2FM,1.00,16.0,1.2948,10.388224,39.062667,-20.0,260.27,...,162.6156,1.833775,6.857817,108.221667,0.5,3500,90,8,110.0,15.33
213,13500842269689763437,P3HT,TTDTC-0F,1.25,22.0,1.9304,8.740846,65.217333,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,2000,0,0,100.0,3.77
214,13500842269689763437,P3HT,TTDTC-0F,1.25,22.0,1.9304,8.740846,65.217333,0.0,0.00,...,0.0000,0.000000,0.000000,0.000000,0.0,2000,0,0,100.0,3.95


,dataset_definition,records,percent_of_original
0,original rows,1028,100.00
1,after exact full-row deduplication,1010,98.25
2,unique predictor vectors,1000,97.28


### 8.5 Duplicate-resolution policy

The global Wen–Zhang–Ma table contains 18 pairs of exact full-row duplicates
and 10 additional pairs with identical recorded predictor vectors but
different PCE values.

Exact duplicates are removed because retaining them would give duplicated
observations additional statistical weight without adding information.

For the primary processing benchmark, records sharing an identical complete
predictor vector are represented as a single condition-level observation.
When repeated predictor vectors have different PCE values, the mean PCE is
used as the condition-level target while the replicate count, target range,
and target standard deviation are retained as metadata.

This produces one observation per unique recorded experimental condition.

A secondary sensitivity cohort retains the conflicting observations
individually after removal of exact duplicates. Identical predictor vectors
are assigned to the same cross-validation fold in this sensitivity analysis
to prevent identical feature representations from appearing simultaneously
in training and test sets.

Target disagreement among identical recorded predictors is interpreted as
evidence of experimental or curation variability not captured by the supplied
feature representation rather than as model error.

In [19]:
# ------------------------------------------------------------------
# 8.5A
# Exact-deduplicated sensitivity cohort
# ------------------------------------------------------------------

wen_exact_dedup = (
    wen_global
    .drop_duplicates()
    .copy()
    .reset_index(drop=True)
)

# Recompute predictor hashes after exact deduplication
wen_exact_dedup["predictor_hash"] = (
    pd.util.hash_pandas_object(
        wen_exact_dedup[predictor_columns],
        index=False
    )
)

print(
    "Exact-deduplicated sensitivity cohort:",
    wen_exact_dedup.shape
)

print(
    "Unique predictor vectors:",
    wen_exact_dedup["predictor_hash"].nunique()
)

Exact-deduplicated sensitivity cohort: (1010, 2090)
Unique predictor vectors: 1000


In [20]:
# ------------------------------------------------------------------
# 8.5B
# One row per unique recorded predictor condition
# ------------------------------------------------------------------

condition_target_summary = (
    wen_exact_dedup
    .groupby("predictor_hash")[TARGET_COL]
    .agg(
        pce_condition_mean="mean",
        pce_condition_std="std",
        pce_condition_min="min",
        pce_condition_max="max",
        n_condition_records="size",
    )
    .reset_index()
)

condition_target_summary["pce_condition_range"] = (
    condition_target_summary["pce_condition_max"]
    - condition_target_summary["pce_condition_min"]
)

condition_target_summary[
    "pce_condition_std"
] = (
    condition_target_summary[
        "pce_condition_std"
    ].fillna(0.0)
)


# Keep one copy of each predictor vector
wen_condition_level = (
    wen_exact_dedup
    .drop_duplicates(
        subset=["predictor_hash"],
        keep="first"
    )
    .drop(columns=[TARGET_COL])
    .merge(
        condition_target_summary,
        on="predictor_hash",
        how="left",
        validate="one_to_one"
    )
    .rename(
        columns={
            "pce_condition_mean": TARGET_COL
        }
    )
    .reset_index(drop=True)
)


print(
    "Condition-level main cohort:",
    wen_condition_level.shape
)

print(
    "Unique predictor hashes:",
    wen_condition_level[
        "predictor_hash"
    ].nunique()
)

print(
    "Conditions with >1 observed PCE:",
    (
        wen_condition_level[
            "n_condition_records"
        ] > 1
    ).sum()
)

Condition-level main cohort: (1000, 2095)
Unique predictor hashes: 1000
Conditions with >1 observed PCE: 10


In [21]:
# ------------------------------------------------------------------
# 8.5C
# Target-distribution sensitivity to duplicate policy
# ------------------------------------------------------------------

duplicate_policy_target_summary = pd.DataFrame({
    "cohort": [
        "raw global",
        "exact-deduplicated",
        "condition-level",
    ],

    "n": [
        len(wen_global),
        len(wen_exact_dedup),
        len(wen_condition_level),
    ],

    "pce_mean": [
        wen_global[TARGET_COL].mean(),
        wen_exact_dedup[TARGET_COL].mean(),
        wen_condition_level[TARGET_COL].mean(),
    ],

    "pce_median": [
        wen_global[TARGET_COL].median(),
        wen_exact_dedup[TARGET_COL].median(),
        wen_condition_level[TARGET_COL].median(),
    ],

    "pce_std": [
        wen_global[TARGET_COL].std(),
        wen_exact_dedup[TARGET_COL].std(),
        wen_condition_level[TARGET_COL].std(),
    ],

    "pce_min": [
        wen_global[TARGET_COL].min(),
        wen_exact_dedup[TARGET_COL].min(),
        wen_condition_level[TARGET_COL].min(),
    ],

    "pce_max": [
        wen_global[TARGET_COL].max(),
        wen_exact_dedup[TARGET_COL].max(),
        wen_condition_level[TARGET_COL].max(),
    ],
})

display(
    duplicate_policy_target_summary.round(3)
)



# ------------------------------------------------------------------
# 8.5D
# Conditions affected by conflicting targets
# ------------------------------------------------------------------

condition_conflicts = (
    wen_condition_level[
        wen_condition_level[
            "n_condition_records"
        ] > 1
    ][
        [
            "Name_Donor",
            "Name_Acceptor",
            "D_A_Weight_Ratio",
            TARGET_COL,
            "pce_condition_min",
            "pce_condition_max",
            "pce_condition_range",
            "pce_condition_std",
            "n_condition_records",
        ]
    ]
    .sort_values(
        "pce_condition_range",
        ascending=False
    )
)

display(condition_conflicts)




# ------------------------------------------------------------------
# Save duplicate-resolved Wen/Ma cohorts
# ------------------------------------------------------------------

wen_exact_dedup.to_csv(
    INTERIM_DIR
    / "wen_global_exact_deduplicated.csv",
    index=False
)

wen_condition_level.to_csv(
    INTERIM_DIR
    / "wen_global_condition_level.csv",
    index=False
)

condition_conflicts.to_csv(
    INTERIM_DIR
    / "wen_global_conflicting_conditions.csv",
    index=False
)

print("Saved.")
print(
    "Sensitivity cohort:",
    len(wen_exact_dedup)
)
print(
    "Main condition-level cohort:",
    len(wen_condition_level)
)

,cohort,n,pce_mean,pce_median,pce_std,pce_min,pce_max
0,raw global,1028,10.027,10.855,4.530,0.1,19.06
1,exact-deduplicated,1010,9.999,10.795,4.553,0.1,19.06
2,condition-level,1000,9.980,10.740,4.550,0.1,19.06


,Name_Donor,Name_Acceptor,D_A_Weight_Ratio,PCE (%),pce_condition_min,pce_condition_max,pce_condition_range,pce_condition_std,n_condition_records
200,P130,Y6,0.83,13.075,10.87,15.28,4.41,3.118341,2
204,P131,Y6,0.83,9.365,7.60,11.13,3.53,2.496087,2
184,PM6,ML-2FM,1.00,14.325,13.32,15.33,2.01,1.421285,2
193,C-2F,N3,1.60,13.760,13.21,14.31,1.10,0.777817,2
450,L2,Y6,1.50,15.350,14.90,15.80,0.90,0.636396,2
216,P3HT,TTDTC-4F,1.25,4.575,4.34,4.81,0.47,0.332340,2
445,L2,Y6,1.50,14.160,14.02,14.30,0.28,0.197990,2
854,PM6,C7BTP-BO-2Cl-2F,0.83,16.300,16.20,16.40,0.20,0.141421,2
209,P3HT,TTDTC-0F,1.25,3.860,3.77,3.95,0.18,0.127279,2
387,PM6,PTIC-4Cl,0.83,14.815,14.81,14.82,0.01,0.007071,2


Saved.
Sensitivity cohort: 1010
Main condition-level cohort: 1000


### 8.6 Processing-variable semantics, ranges, and zero encoding

The condition-level Wen–Zhang–Ma cohort contains nine fabrication dimensions,
but these dimensions are not represented uniformly.

Donor–acceptor ratio, blend concentration, additive loading, spin-coating rate,
annealing temperature, annealing time, and active-layer thickness are supplied
as scalar variables. Solvent and additive identity are represented through
physicochemical descriptor blocks.

Before modeling, each processing dimension is examined for missing values,
zero-valued observations, numerical range, and internal consistency.

Particular attention is given to zero encoding because zero may represent
absence of an additive or thermal-annealing step rather than a physically
measured value. Such values must therefore not be interpreted automatically
as ordinary continuous measurements.

In [22]:
# ------------------------------------------------------------------
# 8.6A
# Direct processing-variable audit
# ------------------------------------------------------------------

direct_processing_columns = [
    "D_A_Weight_Ratio",
    "Blend_Concentration (mg/ml)",
    "Additive_Volume_Ratio (vol%)",
    "Spin_Coating_Rate (rpm)",
    "Annealing_Temperature(℃)",
    "Annealing_Time (min)",
    "Active_Layer_Thickness (nm)",
]


processing_summary_rows = []

for col in direct_processing_columns:

    s = pd.to_numeric(
        wen_condition_level[col],
        errors="coerce"
    )

    processing_summary_rows.append({

        "variable": col,

        "n": len(s),

        "missing": int(s.isna().sum()),

        "missing_percent":
            round(100 * s.isna().mean(), 2),

        "zero_count":
            int((s == 0).sum()),

        "zero_percent":
            round(100 * (s == 0).mean(), 2),

        "negative_count":
            int((s < 0).sum()),

        "min":
            s.min(),

        "q01":
            s.quantile(0.01),

        "q05":
            s.quantile(0.05),

        "median":
            s.median(),

        "q95":
            s.quantile(0.95),

        "q99":
            s.quantile(0.99),

        "max":
            s.max(),

        "unique_values":
            s.nunique(),
    })


processing_scalar_summary = pd.DataFrame(
    processing_summary_rows
)

display(
    processing_scalar_summary.round(3)
)

,variable,n,missing,missing_percent,zero_count,zero_percent,negative_count,min,q01,q05,median,q95,q99,max,unique_values
0,D_A_Weight_Ratio,1000,0,0.0,0,0.0,0,0.25,0.33,0.50,1.0,1.6,2.002,2.5,32
1,Blend_Concentration (mg/ml),1000,0,0.0,0,0.0,0,5.00,5.00,9.95,16.0,23.0,30.000,48.0,25
2,Additive_Volume_Ratio (vol%),1000,0,0.0,532,53.2,0,0.00,0.00,0.00,0.0,1.0,12.000,18.0,22
3,Spin_Coating_Rate (rpm),1000,0,0.0,0,0.0,0,800.00,800.00,1500.00,2500.0,4000.0,5000.000,5000.0,20
4,Annealing_Temperature(℃),1000,0,0.0,333,33.3,0,0.00,0.00,0.00,90.0,160.0,200.000,200.0,22
5,Annealing_Time (min),1000,0,0.0,339,33.9,0,0.00,0.00,0.00,5.0,15.0,20.000,20.0,11
6,Active_Layer_Thickness (nm),1000,0,0.0,0,0.0,0,26.50,45.94,80.00,100.0,150.0,270.300,510.0,60


In [23]:
# ------------------------------------------------------------------
# 8.6B
# Zero-state relationships
# ------------------------------------------------------------------

zero_state_summary = pd.DataFrame({

    "condition": [
        "additive volume = 0",
        "annealing temperature = 0",
        "annealing time = 0",
        "both annealing temperature and time = 0",
        "temperature > 0 but time = 0",
        "temperature = 0 but time > 0",
    ],

    "records": [
        int(
            (
                wen_condition_level[
                    "Additive_Volume_Ratio (vol%)"
                ] == 0
            ).sum()
        ),

        int(
            (
                wen_condition_level[
                    "Annealing_Temperature(℃)"
                ] == 0
            ).sum()
        ),

        int(
            (
                wen_condition_level[
                    "Annealing_Time (min)"
                ] == 0
            ).sum()
        ),

        int(
            (
                (
                    wen_condition_level[
                        "Annealing_Temperature(℃)"
                    ] == 0
                )
                &
                (
                    wen_condition_level[
                        "Annealing_Time (min)"
                    ] == 0
                )
            ).sum()
        ),

        int(
            (
                (
                    wen_condition_level[
                        "Annealing_Temperature(℃)"
                    ] > 0
                )
                &
                (
                    wen_condition_level[
                        "Annealing_Time (min)"
                    ] == 0
                )
            ).sum()
        ),

        int(
            (
                (
                    wen_condition_level[
                        "Annealing_Temperature(℃)"
                    ] == 0
                )
                &
                (
                    wen_condition_level[
                        "Annealing_Time (min)"
                    ] > 0
                )
            ).sum()
        ),
    ],
})


zero_state_summary["percent"] = (
    100
    * zero_state_summary["records"]
    / len(wen_condition_level)
).round(2)

display(zero_state_summary)

,condition,records,percent
0,additive volume = 0,532,53.2
1,annealing temperature = 0,333,33.3
2,annealing time = 0,339,33.9
3,both annealing temperature and time = 0,333,33.3
4,temperature > 0 but time = 0,6,0.6
5,temperature = 0 but time > 0,0,0.0


In [24]:
# ------------------------------------------------------------------
# 8.6C
# Additive-presence encoding consistency
# ------------------------------------------------------------------

additive_descriptor_columns = [
    "Additive_MeltingPoint (℃)",
    "Additive_BoilingPoint  (℃)",
    "Additive_Density (g/cm3)",
    "Additive_MolecularWeight",
    "Additive_DipoleMoment  (Debye)",
    "Additive_EnergyGap (eV)",
    "Additive_Polarizability (a.u.)",
]


additive_desc = (
    wen_condition_level[
        additive_descriptor_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


wen_condition_level[
    "additive_descriptors_all_zero"
] = (
    additive_desc.fillna(0)
    .eq(0)
    .all(axis=1)
)


wen_condition_level[
    "additive_loading_zero"
] = (
    wen_condition_level[
        "Additive_Volume_Ratio (vol%)"
    ] == 0
)


additive_encoding_crosstab = pd.crosstab(
    wen_condition_level[
        "additive_loading_zero"
    ],
    wen_condition_level[
        "additive_descriptors_all_zero"
    ],
    rownames=["additive_loading_zero"],
    colnames=["descriptor_block_all_zero"],
    margins=True
)

display(additive_encoding_crosstab)



# ------------------------------------------------------------------
# Additive encoding exceptions
# ------------------------------------------------------------------

additive_encoding_exceptions = (
    wen_condition_level[
        wen_condition_level[
            "additive_loading_zero"
        ]
        !=
        wen_condition_level[
            "additive_descriptors_all_zero"
        ]
    ][
        [
            "Name_Donor",
            "Name_Acceptor",
            "Additive_Volume_Ratio (vol%)",
            *additive_descriptor_columns,
            TARGET_COL,
        ]
    ]
    .copy()
)


print(
    "Additive zero-encoding exceptions:",
    len(additive_encoding_exceptions)
)

display(
    additive_encoding_exceptions.head(30)
)

descriptor_block_all_zero,False,True,All
additive_loading_zero,,,
False,468,0,468
True,12,520,532
All,480,520,1000


Additive zero-encoding exceptions: 12


,Name_Donor,Name_Acceptor,Additive_Volume_Ratio (vol%),Additive_MeltingPoint (℃),Additive_BoilingPoint (℃),Additive_Density (g/cm3),Additive_MolecularWeight,Additive_DipoleMoment (Debye),Additive_EnergyGap (eV),Additive_Polarizability (a.u.),PCE (%)
115,PBDB-T,T2-SePDI2,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,4.10
116,PBDB-T,T3B-SePDI3,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,5.82
117,PBDB-T,T4B-SePDI4,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,5.10
901,PTzBI-dF,NTIC-4F,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,11.49
902,PTzBI-dF,NTIC-4Cl,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,11.75
951,P3-2023,L8-BO,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,10.70
988,PDBD-2FBT,Y6-HU,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,11.86
989,PDBD-2FBT,Y6-HU,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,12.63
990,PDBD-2FBT,Y6-HU,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,10.05
991,PDBD-2FBT,Y6-HU,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,11.87


### 8.7 Ambiguous zero-valued processing states

Zero-valued processing variables are not interpreted automatically as
ordinary continuous measurements.

The thermal-annealing variables show a dominant `(temperature = 0,
time = 0)` state consistent with absence of thermal annealing, but a small
number of records contain a positive annealing temperature with zero
annealing time.

Similarly, most zero additive-loading records contain an all-zero additive
descriptor block, consistent with absence of an additive. A small subset,
however, contains nonzero additive descriptors despite zero recorded loading.

These exceptional states are examined separately before categorical process
indicators are defined.

In [25]:
# ------------------------------------------------------------------
# 8.7A
# Positive annealing temperature with zero annealing time
# ------------------------------------------------------------------

annealing_zero_time_exceptions = (
    wen_condition_level[
        (
            wen_condition_level[
                "Annealing_Temperature(℃)"
            ] > 0
        )
        &
        (
            wen_condition_level[
                "Annealing_Time (min)"
            ] == 0
        )
    ][
        [
            "Name_Donor",
            "Name_Acceptor",
            "D_A_Weight_Ratio",
            "Blend_Concentration (mg/ml)",
            "Additive_Volume_Ratio (vol%)",
            "Spin_Coating_Rate (rpm)",
            "Annealing_Temperature(℃)",
            "Annealing_Time (min)",
            "Active_Layer_Thickness (nm)",
            TARGET_COL,
        ]
    ]
    .copy()
)

print(
    "Positive-temperature / zero-time records:",
    len(annealing_zero_time_exceptions)
)

display(annealing_zero_time_exceptions)

Positive-temperature / zero-time records: 6


,Name_Donor,Name_Acceptor,D_A_Weight_Ratio,Blend_Concentration (mg/ml),Additive_Volume_Ratio (vol%),Spin_Coating_Rate (rpm),Annealing_Temperature(℃),Annealing_Time (min),Active_Layer_Thickness (nm),PCE (%)
572,PFBDT-8ttTPD,Y6,1.00,16.0,0.0,2000,90,0,110.0,15.05
573,PClBDT-8ttTPD,Y6,1.00,16.0,0.0,2000,90,0,110.0,10.02
730,PBDTTT-E-T,DCNBT-TPC,1.43,13.5,2.0,3000,100,0,100.0,9.34
731,PBDTTT-E-T,DCNBT-TPIC,1.43,13.5,2.0,3000,120,0,100.0,10.22
856,PM6,C7BTP-BO-2Cl-2F,0.83,15.4,0.5,2800,80,0,110.0,15.90
857,PM6,C7BTP-BO-2Cl-2F,0.83,15.4,0.5,2800,100,0,110.0,15.40


In [26]:
# ------------------------------------------------------------------
# 8.7B
# Zero additive loading with nonzero additive descriptors
# ------------------------------------------------------------------

additive_zero_descriptor_exceptions = (
    wen_condition_level[
        (
            wen_condition_level[
                "Additive_Volume_Ratio (vol%)"
            ] == 0
        )
        &
        (
            ~wen_condition_level[
                "additive_descriptors_all_zero"
            ]
        )
    ]
    .copy()
)


print(
    "Zero-loading but nonzero-descriptor records:",
    len(additive_zero_descriptor_exceptions)
)

print(
    "Unique donor–acceptor systems:",
    additive_zero_descriptor_exceptions[
        ["Name_Donor", "Name_Acceptor"]
    ]
    .drop_duplicates()
    .shape[0]
)


display(
    additive_zero_descriptor_exceptions[
        [
            "Name_Donor",
            "Name_Acceptor",
            "D_A_Weight_Ratio",
            "Additive_Volume_Ratio (vol%)",
            *additive_descriptor_columns,
            "Annealing_Temperature(℃)",
            "Annealing_Time (min)",
            TARGET_COL,
        ]
    ]
)

Zero-loading but nonzero-descriptor records: 12
Unique donor–acceptor systems: 7


,Name_Donor,Name_Acceptor,D_A_Weight_Ratio,Additive_Volume_Ratio (vol%),Additive_MeltingPoint (℃),Additive_BoilingPoint (℃),Additive_Density (g/cm3),Additive_MolecularWeight,Additive_DipoleMoment (Debye),Additive_EnergyGap (eV),Additive_Polarizability (a.u.),Annealing_Temperature(℃),Annealing_Time (min),PCE (%)
115,PBDB-T,T2-SePDI2,1.00,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,0,0,4.10
116,PBDB-T,T3B-SePDI3,1.00,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,0,0,5.82
117,PBDB-T,T4B-SePDI4,1.00,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,0,0,5.10
901,PTzBI-dF,NTIC-4F,1.00,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,110,10,11.49
902,PTzBI-dF,NTIC-4Cl,1.00,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,110,10,11.75
951,P3-2023,L8-BO,0.83,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,70,3,10.70
988,PDBD-2FBT,Y6-HU,0.67,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,0,0,11.86
989,PDBD-2FBT,Y6-HU,0.67,0.0,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,150,5,12.63
990,PDBD-2FBT,Y6-HU,0.67,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,0,0,10.05
991,PDBD-2FBT,Y6-HU,0.67,0.0,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,150,5,11.87


In [27]:
# ------------------------------------------------------------------
# 8.7C
# Distinct additive identities represented among exceptions
# ------------------------------------------------------------------

exception_additive_signatures = (
    additive_zero_descriptor_exceptions[
        additive_descriptor_columns
    ]
    .drop_duplicates()
)

print(
    "Distinct additive descriptor signatures:",
    len(exception_additive_signatures)
)

display(exception_additive_signatures)

Distinct additive descriptor signatures: 3


,Additive_MeltingPoint (℃),Additive_BoilingPoint (℃),Additive_Density (g/cm3),Additive_MolecularWeight,Additive_DipoleMoment (Debye),Additive_EnergyGap (eV),Additive_Polarizability (a.u.)
115,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000
901,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667
992,28.0,257.90,1.0748,170.2100,1.408600,8.659484,114.134333


### 8.8 Contextual zero encoding in processing series

A small number of records contain processing descriptors that are inconsistent
with a simple interpretation of zero as process absence.

Zero additive-loading records with nonzero additive descriptors may represent
zero-loading control conditions embedded within additive-specific experimental
series. Likewise, positive annealing temperatures paired with zero annealing
time may reflect special or incompletely encoded annealing-series conditions.

These records are compared with neighboring conditions sharing the same
material system and processing descriptors to determine whether they form
systematic experimental series rather than isolated data errors.

In [28]:
# ------------------------------------------------------------------
# 8.8A
# Are zero-loading/nonzero-descriptor records members of
# additive-loading experimental series?
# ------------------------------------------------------------------

additive_series_rows = []

for idx, row in additive_zero_descriptor_exceptions.iterrows():

    same_system = (
        wen_condition_level["Name_Donor"].eq(row["Name_Donor"])
        &
        wen_condition_level["Name_Acceptor"].eq(row["Name_Acceptor"])
    )

    same_signature = pd.Series(
        True,
        index=wen_condition_level.index
    )

    for col in additive_descriptor_columns:
        same_signature &= (
            wen_condition_level[col]
            .eq(row[col])
        )

    matches = (
        wen_condition_level[
            same_system & same_signature
        ]
        .copy()
    )

    matches["zero_control_index"] = idx

    additive_series_rows.append(matches)


additive_series_context = pd.concat(
    additive_series_rows,
    ignore_index=False
).drop_duplicates()


additive_series_display_cols = [
    "Name_Donor",
    "Name_Acceptor",
    "D_A_Weight_Ratio",
    "Additive_Volume_Ratio (vol%)",
    "Spin_Coating_Rate (rpm)",
    "Annealing_Temperature(℃)",
    "Annealing_Time (min)",
    "Active_Layer_Thickness (nm)",
    TARGET_COL,
]


display(
    additive_series_context[
        additive_series_display_cols
    ]
    .sort_values(
        [
            "Name_Donor",
            "Name_Acceptor",
            "Additive_Volume_Ratio (vol%)",
        ]
    )
)



# ------------------------------------------------------------------
# Does each zero-loading exception have positive-loading companions?
# ------------------------------------------------------------------

additive_zero_series_summary = []

for idx, row in additive_zero_descriptor_exceptions.iterrows():

    mask = (
        wen_condition_level["Name_Donor"].eq(row["Name_Donor"])
        &
        wen_condition_level["Name_Acceptor"].eq(row["Name_Acceptor"])
    )

    for col in additive_descriptor_columns:
        mask &= wen_condition_level[col].eq(row[col])

    series = wen_condition_level[mask]

    positive_loadings = sorted(
        series.loc[
            series["Additive_Volume_Ratio (vol%)"] > 0,
            "Additive_Volume_Ratio (vol%)"
        ]
        .unique()
        .tolist()
    )

    additive_zero_series_summary.append({
        "Name_Donor": row["Name_Donor"],
        "Name_Acceptor": row["Name_Acceptor"],
        "series_records": len(series),
        "has_positive_loading_companion":
            len(positive_loadings) > 0,
        "positive_loadings": positive_loadings,
    })


additive_zero_series_summary = pd.DataFrame(
    additive_zero_series_summary
)

display(additive_zero_series_summary)

,Name_Donor,Name_Acceptor,D_A_Weight_Ratio,Additive_Volume_Ratio (vol%),Spin_Coating_Rate (rpm),Annealing_Temperature(℃),Annealing_Time (min),Active_Layer_Thickness (nm),PCE (%)
951,P3-2023,L8-BO,0.83,0.0,2600,70,3,100.0,10.70
943,P3-2023,L8-BO,1.00,0.3,2600,90,5,100.0,10.59
944,P3-2023,L8-BO,0.83,0.3,2600,90,5,100.0,11.18
945,P3-2023,L8-BO,0.67,0.3,2600,90,5,100.0,10.61
946,P3-2023,L8-BO,0.83,0.3,2600,0,0,100.0,10.85
947,P3-2023,L8-BO,0.83,0.3,2600,90,3,100.0,11.27
948,P3-2023,L8-BO,0.83,0.3,2600,90,10,100.0,10.92
949,P3-2023,L8-BO,0.83,0.3,2600,70,3,100.0,11.77
950,P3-2023,L8-BO,0.83,0.3,2600,110,3,100.0,11.02
952,P3-2023,L8-BO,0.83,0.5,2600,70,3,100.0,12.56


,Name_Donor,Name_Acceptor,series_records,has_positive_loading_companion,positive_loadings
0,PBDB-T,T2-SePDI2,1,False,[]
1,PBDB-T,T3B-SePDI3,1,False,[]
2,PBDB-T,T4B-SePDI4,1,False,[]
3,PTzBI-dF,NTIC-4F,1,False,[]
4,PTzBI-dF,NTIC-4Cl,1,False,[]
5,P3-2023,L8-BO,10,True,"[0.3, 0.5]"
6,PDBD-2FBT,Y6-HU,2,False,[]
7,PDBD-2FBT,Y6-HU,2,False,[]
8,PDBD-2FBT,Y6-HU,2,False,[]
9,PDBD-2FBT,Y6-HU,2,False,[]


In [29]:
# ------------------------------------------------------------------
# 8.8B
# Context surrounding positive-temperature / zero-time rows
# ------------------------------------------------------------------

annealing_context_rows = []

for idx, row in annealing_zero_time_exceptions.iterrows():

    same_system = (
        wen_condition_level["Name_Donor"].eq(row["Name_Donor"])
        &
        wen_condition_level["Name_Acceptor"].eq(row["Name_Acceptor"])
    )

    context = (
        wen_condition_level[
            same_system
        ][
            [
                "Name_Donor",
                "Name_Acceptor",
                "D_A_Weight_Ratio",
                "Blend_Concentration (mg/ml)",
                "Additive_Volume_Ratio (vol%)",
                "Spin_Coating_Rate (rpm)",
                "Annealing_Temperature(℃)",
                "Annealing_Time (min)",
                "Active_Layer_Thickness (nm)",
                TARGET_COL,
            ]
        ]
        .copy()
    )

    context["exception_index"] = idx

    annealing_context_rows.append(context)


annealing_series_context = (
    pd.concat(
        annealing_context_rows,
        ignore_index=False
    )
    .drop_duplicates()
    .sort_values(
        [
            "Name_Donor",
            "Name_Acceptor",
            "Annealing_Temperature(℃)",
            "Annealing_Time (min)",
        ]
    )
)

display(annealing_series_context)



# ------------------------------------------------------------------
# Compact annealing-series summary
# ------------------------------------------------------------------

annealing_series_summary = (
    annealing_series_context
    .groupby(
        ["Name_Donor", "Name_Acceptor"]
    )
    .agg(
        records=("PCE (%)", "size"),
        temperatures=(
            "Annealing_Temperature(℃)",
            lambda x: sorted(set(x))
        ),
        times=(
            "Annealing_Time (min)",
            lambda x: sorted(set(x))
        ),
        pce_min=("PCE (%)", "min"),
        pce_max=("PCE (%)", "max"),
    )
    .reset_index()
)

display(annealing_series_summary)

,Name_Donor,Name_Acceptor,D_A_Weight_Ratio,Blend_Concentration (mg/ml),Additive_Volume_Ratio (vol%),Spin_Coating_Rate (rpm),Annealing_Temperature(℃),Annealing_Time (min),Active_Layer_Thickness (nm),PCE (%),exception_index
730,PBDTTT-E-T,DCNBT-TPC,1.43,13.5,2.00,3000,100,0,100.0,9.34,730
731,PBDTTT-E-T,DCNBT-TPIC,1.43,13.5,2.00,3000,120,0,100.0,10.22,731
573,PClBDT-8ttTPD,Y6,1.00,16.0,0.00,2000,90,0,110.0,10.02,573
572,PFBDT-8ttTPD,Y6,1.00,16.0,0.00,2000,90,0,110.0,15.05,572
852,PM6,C7BTP-BO-2Cl-2F,0.83,15.4,0.00,2800,0,0,110.0,15.00,856
853,PM6,C7BTP-BO-2Cl-2F,0.83,15.4,0.50,2800,0,0,110.0,15.60,856
854,PM6,C7BTP-BO-2Cl-2F,0.83,15.4,0.50,2800,0,0,110.0,16.30,856
855,PM6,C7BTP-BO-2Cl-2F,0.83,15.4,0.50,2800,0,0,110.0,15.20,856
858,PM6,C7BTP-BO-2Cl-2F,0.83,15.4,0.10,2800,0,0,110.0,16.10,856
859,PM6,C7BTP-BO-2Cl-2F,1.00,15.4,0.25,2800,0,0,110.0,17.40,856


,Name_Donor,Name_Acceptor,records,temperatures,times,pce_min,pce_max
0,PBDTTT-E-T,DCNBT-TPC,1,[100],[0],9.34,9.34
1,PBDTTT-E-T,DCNBT-TPIC,1,[120],[0],10.22,10.22
2,PClBDT-8ttTPD,Y6,1,[90],[0],10.02,10.02
3,PFBDT-8ttTPD,Y6,1,[90],[0],15.05,15.05
4,PM6,C7BTP-BO-2Cl-2F,22,"[0, 80, 100]",[0],15.00,18.00


In [30]:
# ------------------------------------------------------------------
# Provisional semantic process states
# ------------------------------------------------------------------

wen_condition_level["annealing_state"] = np.select(
    [
        (
            wen_condition_level["Annealing_Temperature(℃)"].eq(0)
            &
            wen_condition_level["Annealing_Time (min)"].eq(0)
        ),

        (
            wen_condition_level["Annealing_Temperature(℃)"].gt(0)
            &
            wen_condition_level["Annealing_Time (min)"].gt(0)
        ),

        (
            wen_condition_level["Annealing_Temperature(℃)"].gt(0)
            &
            wen_condition_level["Annealing_Time (min)"].eq(0)
        ),
    ],
    [
        "no_thermal_annealing",
        "explicit_thermal_annealing",
        "positive_temperature_zero_time",
    ],
    default="other"
)


wen_condition_level["additive_state"] = np.select(
    [
        (
            wen_condition_level["Additive_Volume_Ratio (vol%)"].eq(0)
            &
            wen_condition_level["additive_descriptors_all_zero"]
        ),

        (
            wen_condition_level["Additive_Volume_Ratio (vol%)"].eq(0)
            &
            ~wen_condition_level["additive_descriptors_all_zero"]
        ),

        wen_condition_level[
            "Additive_Volume_Ratio (vol%)"
        ].gt(0),
    ],
    [
        "no_additive",
        "zero_loading_with_additive_identity",
        "positive_additive_loading",
    ],
    default="other"
)


print("Annealing states:")
display(
    wen_condition_level[
        "annealing_state"
    ].value_counts()
)

print("\nAdditive states:")
display(
    wen_condition_level[
        "additive_state"
    ].value_counts()
)

Annealing states:


annealing_state
explicit_thermal_annealing        661
no_thermal_annealing              333
positive_temperature_zero_time      6
Name: count, dtype: int64


Additive states:


additive_state
no_additive                            520
positive_additive_loading              468
zero_loading_with_additive_identity     12
Name: count, dtype: int64

### 8.9 Physically consistent processing representation

The raw global table contains a small number of context-dependent zero
encodings.

For additive processing, zero loading is interpreted as physical absence of
the additive. Consequently, additive physicochemical descriptors are set to
zero whenever the recorded additive volume ratio is zero. The original
contextual representation is retained separately for sensitivity analysis.

Thermal annealing is represented differently. Records with both zero
temperature and zero time are treated as no thermal annealing. Records with
positive temperature and positive time represent explicitly specified thermal
annealing. Positive-temperature/zero-time records are retained as annealed
conditions with unresolved duration rather than being interpreted as
zero-minute annealing.

These transformations are applied only to a derived modeling representation;
the original curated values remain unchanged.

In [31]:
# ------------------------------------------------------------------
# 8.9A
# Construct physically interpreted processing representation
# ------------------------------------------------------------------

wen_physical = wen_condition_level.copy()


# --------------------------------------------------------------
# Additive representation
# --------------------------------------------------------------

zero_additive_mask = (
    wen_physical[
        "Additive_Volume_Ratio (vol%)"
    ].eq(0)
)

# Physical absence: descriptors should not describe material
# that is present at zero concentration.
wen_physical.loc[
    zero_additive_mask,
    additive_descriptor_columns
] = 0.0


wen_physical["Additive_Present"] = (
    wen_physical[
        "Additive_Volume_Ratio (vol%)"
    ].gt(0)
).astype(int)


# Keep an audit-only flag describing the original contextual encoding.
wen_physical[
    "Additive_Context_At_Zero"
] = (
    zero_additive_mask
    &
    ~wen_condition_level[
        "additive_descriptors_all_zero"
    ]
).astype(int)


# --------------------------------------------------------------
# Thermal annealing representation
# --------------------------------------------------------------

temperature = wen_physical[
    "Annealing_Temperature(℃)"
]

time = wen_physical[
    "Annealing_Time (min)"
]


no_anneal = (
    temperature.eq(0)
    & time.eq(0)
)

duration_known = (
    temperature.gt(0)
    & time.gt(0)
)

duration_unresolved = (
    temperature.gt(0)
    & time.eq(0)
)


wen_physical["Thermal_Annealing_Present"] = (
    ~no_anneal
).astype(int)

wen_physical[
    "Annealing_Duration_Unresolved"
] = (
    duration_unresolved
).astype(int)


# Physical/modeling duration:
# 0 = genuinely no thermal annealing
# positive value = reported duration
# NaN = annealing occurred but duration is unresolved
wen_physical[
    "Annealing_Time_Physical"
] = time.astype(float)

wen_physical.loc[
    duration_unresolved,
    "Annealing_Time_Physical"
] = np.nan


print("Physical-processing cohort:", wen_physical.shape)

print(
    "\nAdditive descriptors zeroed:",
    int(zero_additive_mask.sum())
)

print(
    "Contextual zero-additive cases corrected:",
    int(
        wen_physical[
            "Additive_Context_At_Zero"
        ].sum()
    )
)

print(
    "\nNo thermal annealing:",
    int(no_anneal.sum())
)

print(
    "Annealing with known duration:",
    int(duration_known.sum())
)

print(
    "Annealing with unresolved duration:",
    int(duration_unresolved.sum())
)

Physical-processing cohort: (1000, 2104)

Additive descriptors zeroed: 532
Contextual zero-additive cases corrected: 12

No thermal annealing: 333
Annealing with known duration: 661
Annealing with unresolved duration: 6


In [32]:
# ------------------------------------------------------------------
# 8.9B
# Define the physical feature representation
# ------------------------------------------------------------------

physical_feature_columns = [
    col
    for col in predictor_columns
    if col != "Annealing_Time (min)"
]

physical_feature_columns += [
    "Annealing_Time_Physical",
    "Additive_Present",
    "Thermal_Annealing_Present",
    "Annealing_Duration_Unresolved",
]


print(
    "Number of physical feature columns:",
    len(physical_feature_columns)
)

print(
    "Duplicate feature names:",
    len(physical_feature_columns)
    - len(set(physical_feature_columns))
)

Number of physical feature columns: 2091
Duplicate feature names: 0


In [33]:
# ------------------------------------------------------------------
# 8.9C
# Re-audit repeated conditions after semantic correction
# ------------------------------------------------------------------

wen_physical["physical_predictor_hash"] = (
    pd.util.hash_pandas_object(
        wen_physical[
            physical_feature_columns
        ],
        index=False
    )
)


physical_group_summary = (
    wen_physical
    .groupby(
        "physical_predictor_hash"
    )[TARGET_COL]
    .agg(
        n_records="size",
        n_unique_pce="nunique",
        pce_min="min",
        pce_max="max",
        pce_mean="mean",
        pce_std="std",
    )
    .reset_index()
)

physical_group_summary["pce_range"] = (
    physical_group_summary["pce_max"]
    - physical_group_summary["pce_min"]
)


repeated_physical_groups = (
    physical_group_summary[
        physical_group_summary[
            "n_records"
        ] > 1
    ]
)

conflicting_physical_groups = (
    repeated_physical_groups[
        repeated_physical_groups[
            "n_unique_pce"
        ] > 1
    ]
)


print(
    "Original condition-level records:",
    len(wen_physical)
)

print(
    "Unique physical predictor vectors:",
    wen_physical[
        "physical_predictor_hash"
    ].nunique()
)

print(
    "Repeated physical-condition groups:",
    len(repeated_physical_groups)
)

print(
    "Rows involved:",
    int(
        repeated_physical_groups[
            "n_records"
        ].sum()
    )
)

print(
    "Physical groups with conflicting PCE:",
    len(conflicting_physical_groups)
)

if len(conflicting_physical_groups):
    print(
        "Maximum PCE range:",
        conflicting_physical_groups[
            "pce_range"
        ].max()
    )

Original condition-level records: 1000
Unique physical predictor vectors: 994
Repeated physical-condition groups: 2
Rows involved: 8
Physical groups with conflicting PCE: 2
Maximum PCE range: 2.41


In [34]:
# ------------------------------------------------------------------
# 8.9D
# Inspect physical conditions that collapse after correction
# ------------------------------------------------------------------

repeated_physical_hashes = set(
    repeated_physical_groups[
        "physical_predictor_hash"
    ]
)

physical_repeat_rows = (
    wen_physical[
        wen_physical[
            "physical_predictor_hash"
        ].isin(
            repeated_physical_hashes
        )
    ]
    .copy()
)


display(
    physical_repeat_rows[
        [
            "physical_predictor_hash",
            "Name_Donor",
            "Name_Acceptor",
            "D_A_Weight_Ratio",
            "Additive_Volume_Ratio (vol%)",
            "Additive_Context_At_Zero",
            "Annealing_Temperature(℃)",
            "Annealing_Time (min)",
            "Annealing_Time_Physical",
            TARGET_COL,
        ]
    ]
    .sort_values(
        [
            "physical_predictor_hash",
            TARGET_COL
        ]
    )
)

,physical_predictor_hash,Name_Donor,Name_Acceptor,D_A_Weight_Ratio,Additive_Volume_Ratio (vol%),Additive_Context_At_Zero,Annealing_Temperature(℃),Annealing_Time (min),Annealing_Time_Physical,PCE (%)
991,12240771959050453599,PDBD-2FBT,Y6-HU,0.67,0.0,1,150,5,5.0,11.87
989,12240771959050453599,PDBD-2FBT,Y6-HU,0.67,0.0,1,150,5,5.0,12.63
993,12240771959050453599,PDBD-2FBT,Y6-HU,0.67,0.0,1,150,5,5.0,13.94
977,12240771959050453599,PDBD-2FBT,Y6-HU,0.67,0.0,0,150,5,5.0,14.14
990,15999201343441973721,PDBD-2FBT,Y6-HU,0.67,0.0,1,0,0,0.0,10.05
978,15999201343441973721,PDBD-2FBT,Y6-HU,0.67,0.0,0,0,0,0.0,10.62
988,15999201343441973721,PDBD-2FBT,Y6-HU,0.67,0.0,1,0,0,0.0,11.86
992,15999201343441973721,PDBD-2FBT,Y6-HU,0.67,0.0,1,0,0,0.0,12.46


### 8.10 Final physical-condition cohort

Semantic correction of additive zero states reduces the 1,000 nominally
unique recorded predictor vectors to 994 unique physical predictor vectors.

The newly collapsed conditions occur because additive descriptors associated
with zero additive loading distinguish experimental-series context without
representing material physically present in the active layer.

The final primary Wen–Zhang–Ma benchmark is therefore reconstructed directly
from the exact-deduplicated observation table. Physically equivalent predictor
vectors are aggregated to a single condition-level target, while replicate
count, PCE dispersion, and PCE range are retained as uncertainty metadata.

The original 1,000-condition contextual representation is preserved for
sensitivity analysis.

In [35]:
# ------------------------------------------------------------------
# 8.10A
# Apply physical processing semantics to exact-deduplicated records
# ------------------------------------------------------------------

wen_exact_physical = wen_exact_dedup.copy()


# --------------------------------------------------------------
# Additive: zero loading means physical absence
# --------------------------------------------------------------

zero_additive_exact = (
    wen_exact_physical[
        "Additive_Volume_Ratio (vol%)"
    ].eq(0)
)

wen_exact_physical.loc[
    zero_additive_exact,
    additive_descriptor_columns
] = 0.0


wen_exact_physical["Additive_Present"] = (
    wen_exact_physical[
        "Additive_Volume_Ratio (vol%)"
    ].gt(0)
).astype(int)


# --------------------------------------------------------------
# Thermal annealing
# --------------------------------------------------------------

temp_exact = wen_exact_physical[
    "Annealing_Temperature(℃)"
]

time_exact = wen_exact_physical[
    "Annealing_Time (min)"
]

no_anneal_exact = (
    temp_exact.eq(0)
    & time_exact.eq(0)
)

unresolved_duration_exact = (
    temp_exact.gt(0)
    & time_exact.eq(0)
)


wen_exact_physical[
    "Thermal_Annealing_Present"
] = (
    ~no_anneal_exact
).astype(int)


wen_exact_physical[
    "Annealing_Duration_Unresolved"
] = (
    unresolved_duration_exact
).astype(int)


wen_exact_physical[
    "Annealing_Time_Physical"
] = time_exact.astype(float)

wen_exact_physical.loc[
    unresolved_duration_exact,
    "Annealing_Time_Physical"
] = np.nan



# ------------------------------------------------------------------
# 8.10B
# Physical predictor-vector identity
# ------------------------------------------------------------------

final_physical_feature_columns = [
    col
    for col in predictor_columns
    if col != "Annealing_Time (min)"
]

final_physical_feature_columns += [
    "Annealing_Time_Physical",
    "Additive_Present",
    "Thermal_Annealing_Present",
    "Annealing_Duration_Unresolved",
]


assert (
    len(final_physical_feature_columns)
    ==
    len(set(final_physical_feature_columns))
)


wen_exact_physical[
    "physical_predictor_hash"
] = (
    pd.util.hash_pandas_object(
        wen_exact_physical[
            final_physical_feature_columns
        ],
        index=False
    )
)


print(
    "Exact-deduplicated observations:",
    len(wen_exact_physical)
)

print(
    "Unique physical predictor vectors:",
    wen_exact_physical[
        "physical_predictor_hash"
    ].nunique()
)

Exact-deduplicated observations: 1010
Unique physical predictor vectors: 994


In [36]:
# ------------------------------------------------------------------
# 8.10C
# Aggregate observations belonging to the same physical condition
# ------------------------------------------------------------------

physical_target_summary = (
    wen_exact_physical
    .groupby(
        "physical_predictor_hash"
    )[TARGET_COL]
    .agg(
        pce_physical_mean="mean",
        pce_physical_std="std",
        pce_physical_min="min",
        pce_physical_max="max",
        n_physical_records="size",
    )
    .reset_index()
)


physical_target_summary[
    "pce_physical_range"
] = (
    physical_target_summary[
        "pce_physical_max"
    ]
    -
    physical_target_summary[
        "pce_physical_min"
    ]
)


physical_target_summary[
    "pce_physical_std"
] = (
    physical_target_summary[
        "pce_physical_std"
    ].fillna(0.0)
)


physical_replicate_groups = (
    physical_target_summary[
        physical_target_summary[
            "n_physical_records"
        ] > 1
    ]
    .copy()
)


print(
    "Physical conditions represented by >1 observation:",
    len(physical_replicate_groups)
)

print(
    "Underlying observations in those conditions:",
    int(
        physical_replicate_groups[
            "n_physical_records"
        ].sum()
    )
)

print(
    "Maximum replicate PCE range:",
    round(
        physical_replicate_groups[
            "pce_physical_range"
        ].max(),
        3
    )
)

Physical conditions represented by >1 observation: 12
Underlying observations in those conditions: 28
Maximum replicate PCE range: 4.41


In [37]:
# ------------------------------------------------------------------
# 8.10D
# Final primary Wen/Ma physical-condition cohort
# ------------------------------------------------------------------

wen_physical_condition_level = (
    wen_exact_physical
    .drop_duplicates(
        subset=["physical_predictor_hash"],
        keep="first"
    )
    .drop(
        columns=[
            TARGET_COL,
            "predictor_hash",
        ],
        errors="ignore"
    )
    .merge(
        physical_target_summary,
        on="physical_predictor_hash",
        how="left",
        validate="one_to_one"
    )
    .rename(
        columns={
            "pce_physical_mean": TARGET_COL
        }
    )
    .reset_index(drop=True)
)


print(
    "Final physical-condition cohort:",
    wen_physical_condition_level.shape
)

print(
    "Unique physical hashes:",
    wen_physical_condition_level[
        "physical_predictor_hash"
    ].nunique()
)

print(
    "Conditions with replicate observations:",
    (
        wen_physical_condition_level[
            "n_physical_records"
        ] > 1
    ).sum()
)

Final physical-condition cohort: (994, 2099)
Unique physical hashes: 994
Conditions with replicate observations: 12


In [38]:
# ------------------------------------------------------------------
# 8.10E
# Final target-distribution check
# ------------------------------------------------------------------

final_cohort_comparison = pd.DataFrame({

    "cohort": [
        "raw global",
        "recorded condition-level",
        "physical condition-level",
    ],

    "n": [
        len(wen_global),
        len(wen_condition_level),
        len(wen_physical_condition_level),
    ],

    "pce_mean": [
        wen_global[TARGET_COL].mean(),
        wen_condition_level[TARGET_COL].mean(),
        wen_physical_condition_level[TARGET_COL].mean(),
    ],

    "pce_median": [
        wen_global[TARGET_COL].median(),
        wen_condition_level[TARGET_COL].median(),
        wen_physical_condition_level[TARGET_COL].median(),
    ],

    "pce_std": [
        wen_global[TARGET_COL].std(),
        wen_condition_level[TARGET_COL].std(),
        wen_physical_condition_level[TARGET_COL].std(),
    ],

    "pce_min": [
        wen_global[TARGET_COL].min(),
        wen_condition_level[TARGET_COL].min(),
        wen_physical_condition_level[TARGET_COL].min(),
    ],

    "pce_max": [
        wen_global[TARGET_COL].max(),
        wen_condition_level[TARGET_COL].max(),
        wen_physical_condition_level[TARGET_COL].max(),
    ],
})


display(
    final_cohort_comparison.round(3)
)

,cohort,n,pce_mean,pce_median,pce_std,pce_min,pce_max
0,raw global,1028,10.027,10.855,4.530,0.1,19.06
1,recorded condition-level,1000,9.980,10.740,4.550,0.1,19.06
2,physical condition-level,994,9.966,10.710,4.559,0.1,19.06


In [39]:
# ------------------------------------------------------------------
# Save final Wen/Ma physical benchmark
# ------------------------------------------------------------------

wen_physical_condition_level.to_csv(
    INTERIM_DIR
    / "wen_global_physical_condition_level.csv",
    index=False
)

physical_target_summary.to_csv(
    INTERIM_DIR
    / "wen_global_physical_replicate_summary.csv",
    index=False
)

print(
    "Saved final physical cohort:",
    len(wen_physical_condition_level)
)

Saved final physical cohort: 994


### 8.11 Solvent and additive descriptor identity audit

Solvent and additive identity in the global Wen–Zhang–Ma table is represented
through physicochemical descriptor vectors rather than explicit categorical
material names.

Before these descriptors are used as model inputs, their internal structure is
examined to determine how many distinct descriptor signatures occur, whether
zero-valued signatures are present, whether distinct processing conditions
collapse onto identical descriptor representations, and how concentrated the
descriptor space is.

This audit distinguishes representation of processing-media identity from
the numerical loading variables themselves.

In [40]:
# ------------------------------------------------------------------
# 8.11A
# Solvent descriptor signatures
# ------------------------------------------------------------------

solvent_descriptor_columns = [
    "Solvent_DipoleMoment (Debye)",
    "Solvent_EnergyGap (eV)",
    "Solvent_Polarizability (a.u.)",
]


solvent_desc = (
    wen_physical_condition_level[
        solvent_descriptor_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


print("Solvent descriptor missing values:")
display(
    solvent_desc.isna()
    .sum()
    .rename("missing")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)


wen_physical_condition_level[
    "solvent_descriptor_all_zero"
] = (
    solvent_desc.fillna(0)
    .eq(0)
    .all(axis=1)
)


solvent_signature_counts = (
    wen_physical_condition_level
    .groupby(
        solvent_descriptor_columns,
        dropna=False
    )
    .size()
    .rename("records")
    .reset_index()
    .sort_values(
        "records",
        ascending=False
    )
)


print(
    "\nDistinct solvent descriptor signatures:",
    len(solvent_signature_counts)
)

print(
    "All-zero solvent signatures:",
    int(
        wen_physical_condition_level[
            "solvent_descriptor_all_zero"
        ].sum()
    )
)

display(
    solvent_signature_counts.head(30)
)

Solvent descriptor missing values:


,descriptor,missing
0,Solvent_DipoleMoment (Debye),0
1,Solvent_EnergyGap (eV),0
2,Solvent_Polarizability (a.u.),0



Distinct solvent descriptor signatures: 7
All-zero solvent signatures: 0


,Solvent_DipoleMoment (Debye),Solvent_EnergyGap (eV),Solvent_Polarizability (a.u.),records
2,1.294800,10.388224,39.062667,593
5,1.930400,8.740846,65.217333,330
6,2.792000,8.537577,76.008667,66
0,0.142000,8.592272,235.839333,2
1,0.541500,8.819759,77.751667,1
4,1.825934,11.942811,43.441333,1
3,1.419090,8.213217,235.839333,1


In [41]:
# ------------------------------------------------------------------
# Solvent representation concentration
# ------------------------------------------------------------------

solvent_signature_counts[
    "percent_of_dataset"
] = (
    100
    * solvent_signature_counts["records"]
    / len(wen_physical_condition_level)
).round(2)


solvent_signature_counts[
    "cumulative_percent"
] = (
    100
    * solvent_signature_counts["records"]
      .cumsum()
    / len(wen_physical_condition_level)
).round(2)


display(
    solvent_signature_counts.head(20)
)


for top_n in [1, 3, 5, 10]:

    pct = (
        100
        * solvent_signature_counts[
            "records"
        ].head(top_n).sum()
        / len(wen_physical_condition_level)
    )

    print(
        f"Top {top_n} solvent signatures:",
        round(pct, 2),
        "%"
    )

,Solvent_DipoleMoment (Debye),Solvent_EnergyGap (eV),Solvent_Polarizability (a.u.),records,percent_of_dataset,cumulative_percent
2,1.294800,10.388224,39.062667,593,59.66,59.66
5,1.930400,8.740846,65.217333,330,33.20,92.86
6,2.792000,8.537577,76.008667,66,6.64,99.50
0,0.142000,8.592272,235.839333,2,0.20,99.70
1,0.541500,8.819759,77.751667,1,0.10,99.80
4,1.825934,11.942811,43.441333,1,0.10,99.90
3,1.419090,8.213217,235.839333,1,0.10,100.00


Top 1 solvent signatures: 59.66 %
Top 3 solvent signatures: 99.5 %
Top 5 solvent signatures: 99.8 %
Top 10 solvent signatures: 100.0 %


In [42]:
# ------------------------------------------------------------------
# 8.11C
# Additive descriptor signatures after physical correction
# ------------------------------------------------------------------

additive_desc_physical = (
    wen_physical_condition_level[
        additive_descriptor_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


wen_physical_condition_level[
    "additive_descriptor_all_zero_physical"
] = (
    additive_desc_physical.fillna(0)
    .eq(0)
    .all(axis=1)
)


additive_signature_counts = (
    wen_physical_condition_level
    .groupby(
        additive_descriptor_columns,
        dropna=False
    )
    .size()
    .rename("records")
    .reset_index()
    .sort_values(
        "records",
        ascending=False
    )
)


print(
    "Distinct additive descriptor signatures:",
    len(additive_signature_counts)
)

print(
    "All-zero additive descriptor records:",
    int(
        wen_physical_condition_level[
            "additive_descriptor_all_zero_physical"
        ].sum()
    )
)

print(
    "Positive-loading records:",
    int(
        (
            wen_physical_condition_level[
                "Additive_Volume_Ratio (vol%)"
            ] > 0
        ).sum()
    )
)

display(
    additive_signature_counts.head(30)
)

Distinct additive descriptor signatures: 12
All-zero additive descriptor records: 526
Positive-loading records: 468


,Additive_MeltingPoint (℃),Additive_BoilingPoint (℃),Additive_Density (g/cm3),Additive_MolecularWeight,Additive_DipoleMoment (Debye),Additive_EnergyGap (eV),Additive_Polarizability (a.u.),records
3,0.0,0.00,0.0000,0.0000,0.000000,0.000000,0.000000,526
2,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,267
7,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,124
8,28.0,257.90,1.0748,170.2100,1.408600,8.659484,114.134333,37
10,91.0,256.00,2.0210,270.3500,0.127900,8.285055,103.408667,12
6,6.0,182.00,3.3250,267.8360,1.359300,7.199592,40.018667,11
9,45.0,324.00,1.0800,204.2700,0.147035,6.697814,160.724333,6
0,-37.3,153.80,0.9950,108.1380,1.402117,9.023572,69.529667,3
4,0.9,269.50,0.9700,178.3590,0.000000,9.775695,119.478667,3
1,-24.0,202.00,1.0280,99.1310,3.769535,10.219513,58.860667,2


In [44]:
# ------------------------------------------------------------------
# 8.11D
# Unique processing-media descriptor states
# ------------------------------------------------------------------

media_summary = pd.DataFrame({
    "representation": [
        "solvent descriptor signature",
        "additive descriptor signature",
        "positive-loading additive signatures",
    ],

    "unique_signatures": [
        len(solvent_signature_counts),

        len(additive_signature_counts),

        wen_physical_condition_level.loc[
            wen_physical_condition_level[
                "Additive_Volume_Ratio (vol%)"
            ] > 0,
            additive_descriptor_columns
        ]
        .drop_duplicates()
        .shape[0],
    ],
})

display(media_summary)



# ------------------------------------------------------------------
# Additive signatures versus loading diversity
# ------------------------------------------------------------------

positive_additive = (
    wen_physical_condition_level[
        wen_physical_condition_level[
            "Additive_Volume_Ratio (vol%)"
        ] > 0
    ]
    .copy()
)


additive_loading_diversity = (
    positive_additive
    .groupby(
        additive_descriptor_columns,
        dropna=False
    )
    .agg(
        records=(
            "Additive_Volume_Ratio (vol%)",
            "size"
        ),

        unique_loadings=(
            "Additive_Volume_Ratio (vol%)",
            "nunique"
        ),

        min_loading=(
            "Additive_Volume_Ratio (vol%)",
            "min"
        ),

        max_loading=(
            "Additive_Volume_Ratio (vol%)",
            "max"
        ),
    )
    .reset_index()
    .sort_values(
        ["records", "unique_loadings"],
        ascending=False
    )
)


display(
    additive_loading_diversity.head(30)
)

,representation,unique_signatures
0,solvent descriptor signature,7
1,additive descriptor signature,12
2,positive-loading additive signatures,11


,Additive_MeltingPoint (℃),Additive_BoilingPoint (℃),Additive_Density (g/cm3),Additive_MolecularWeight,Additive_DipoleMoment (Debye),Additive_EnergyGap (eV),Additive_Polarizability (a.u.),records,unique_loadings,min_loading,max_loading
2,-20.0,260.27,1.1940,162.6156,1.833775,6.857817,108.221667,267,16,0.20,5.0
6,16.0,168.00,1.8400,366.0400,0.000000,8.251041,121.152000,124,14,0.10,3.0
7,28.0,257.90,1.0748,170.2100,1.408600,8.659484,114.134333,37,7,0.10,2.0
9,91.0,256.00,2.0210,270.3500,0.127900,8.285055,103.408667,12,2,12.00,18.0
5,6.0,182.00,3.3250,267.8360,1.359300,7.199592,40.018667,11,3,0.10,0.5
8,45.0,324.00,1.0800,204.2700,0.147035,6.697814,160.724333,6,3,0.25,1.0
3,0.9,269.50,0.9700,178.3590,0.000000,9.775695,119.478667,3,2,0.50,1.0
0,-37.3,153.80,0.9950,108.1380,1.402117,9.023572,69.529667,3,1,1.00,1.0
1,-24.0,202.00,1.0280,99.1310,3.769535,10.219513,58.860667,2,1,0.50,0.5
4,4.0,298.00,1.0430,198.2600,0.826900,8.609959,138.526000,2,1,1.00,1.0


In [45]:
# ------------------------------------------------------------------
# 8.11E
# Descriptor variability
# ------------------------------------------------------------------

media_descriptor_columns = (
    solvent_descriptor_columns
    + additive_descriptor_columns
)


descriptor_variability = []

for col in media_descriptor_columns:

    s = pd.to_numeric(
        wen_physical_condition_level[col],
        errors="coerce"
    )

    descriptor_variability.append({
        "descriptor": col,
        "unique_values": s.nunique(dropna=True),
        "missing": s.isna().sum(),
        "min": s.min(),
        "median": s.median(),
        "max": s.max(),
        "std": s.std(),
    })


descriptor_variability = pd.DataFrame(
    descriptor_variability
)

display(
    descriptor_variability.round(4)
)

,descriptor,unique_values,missing,min,median,max,std
0,Solvent_DipoleMoment (Debye),7,0,0.1420,1.2948,2.7920,0.4380
1,Solvent_EnergyGap (eV),7,0,8.2132,10.3882,11.9428,0.8299
2,Solvent_Polarizability (a.u.),6,0,39.0627,39.0627,235.8393,17.2572
3,Additive_MeltingPoint (℃),12,0,-37.3000,0.0000,131.0000,17.4150
4,Additive_BoilingPoint (℃),12,0,0.0000,0.0000,324.0000,120.4562
5,Additive_Density (g/cm3),12,0,0.0000,0.0000,3.3250,0.7709
6,Additive_MolecularWeight,12,0,0.0000,0.0000,366.0400,127.1562
7,Additive_DipoleMoment (Debye),9,0,0.0000,0.0000,3.7695,0.8383
8,Additive_EnergyGap (eV),12,0,0.0000,0.0000,10.2195,3.7672
9,Additive_Polarizability (a.u.),12,0,0.0000,0.0000,160.7243,56.2117


### 8.12 Molecular-fingerprint integrity and resolution

The Wen–Zhang–Ma global dataset contains two 1,024-bit molecular fingerprint
blocks representing donor and acceptor structure.

Before these fingerprints are used as model inputs, their numerical integrity
and effective structural resolution are audited.

The analysis verifies block boundaries, binary encoding, missingness,
constant and near-constant bits, fingerprint sparsity, and the relationship
between fingerprint identity and nominal molecular identity.

Particular attention is given to fingerprint collisions: distinct named or
SMILES-encoded materials that share an identical fingerprint vector. Such
collisions would establish that fingerprint identity cannot be treated as
equivalent to exact molecular identity.

In [46]:
# ------------------------------------------------------------------
# 8.12A
# Locate fingerprint columns and inspect block boundaries
# ------------------------------------------------------------------

fp_columns = [
    col for col in wen_physical_condition_level.columns
    if re.fullmatch(r"FP\d+(?:\.1)?", str(col))
]

donor_fp_columns = [
    f"FP{i}"
    for i in range(1, 1025)
    if f"FP{i}" in wen_physical_condition_level.columns
]

acceptor_fp_columns = [
    f"FP{i}.1"
    for i in range(1, 1025)
    if f"FP{i}.1" in wen_physical_condition_level.columns
]


print("Total fingerprint columns:", len(fp_columns))
print("First FP block:", len(donor_fp_columns))
print("Second FP block:", len(acceptor_fp_columns))

print("\nFirst 5 first-block columns:")
print(donor_fp_columns[:5])

print("\nLast 5 first-block columns:")
print(donor_fp_columns[-5:])

print("\nFirst 5 second-block columns:")
print(acceptor_fp_columns[:5])

print("\nLast 5 second-block columns:")
print(acceptor_fp_columns[-5:])

Total fingerprint columns: 2048
First FP block: 1024
Second FP block: 1024

First 5 first-block columns:
['FP1', 'FP2', 'FP3', 'FP4', 'FP5']

Last 5 first-block columns:
['FP1020', 'FP1021', 'FP1022', 'FP1023', 'FP1024']

First 5 second-block columns:
['FP1.1', 'FP2.1', 'FP3.1', 'FP4.1', 'FP5.1']

Last 5 second-block columns:
['FP1020.1', 'FP1021.1', 'FP1022.1', 'FP1023.1', 'FP1024.1']


In [47]:
# ------------------------------------------------------------------
# Fingerprint positions in table
# ------------------------------------------------------------------

first_fp_index = min(
    wen_physical_condition_level.columns.get_loc(col)
    for col in fp_columns
)

second_block_index = (
    wen_physical_condition_level.columns.get_loc("FP1.1")
)

print("First fingerprint starts at column:", first_fp_index)
print("Second fingerprint starts at column:", second_block_index)

print("\nColumns immediately before first fingerprint:")
print(
    wen_physical_condition_level.columns[
        max(0, first_fp_index - 10):
        first_fp_index
    ].tolist()
)

print("\nColumns around transition between FP blocks:")
print(
    wen_physical_condition_level.columns[
        second_block_index - 5:
        second_block_index + 5
    ].tolist()
)

First fingerprint starts at column: 40
Second fingerprint starts at column: 1064

Columns immediately before first fingerprint:
['delta_E_AL_AH (eV)', 'delta_E_DL_AL (eV)', 'delta_E_DH_AH (eV)', 'delta_E_AL-DH (eV)', 'delta_E_DH_DH-1 (eV)', 'delta_E_DL+1_DL (eV)', 'delta_E_AH_AH-1 (eV)', 'delta_E_AL+1_AL (eV)', 'Donor_DipoleMoment (Debye)', 'Acceptor_DipoleMoment (Debye)']

Columns around transition between FP blocks:
['FP1020', 'FP1021', 'FP1022', 'FP1023', 'FP1024', 'FP1.1', 'FP2.1', 'FP3.1', 'FP4.1', 'FP5.1']


In [48]:
# ------------------------------------------------------------------
# 8.12B
# Fingerprint numerical integrity
# ------------------------------------------------------------------

def fingerprint_integrity(df, columns, block_name):

    block = df[columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    values = pd.unique(
        block.to_numpy().ravel()
    )

    values_nonmissing = sorted(
        x for x in values
        if pd.notna(x)
    )

    return {
        "block": block_name,
        "rows": len(block),
        "bits": len(columns),
        "missing_cells": int(block.isna().sum().sum()),
        "non_binary_cells": int(
            (~block.isin([0, 1]) & block.notna())
            .sum()
            .sum()
        ),
        "observed_values": values_nonmissing[:20],
    }


fingerprint_integrity_summary = pd.DataFrame([
    fingerprint_integrity(
        wen_physical_condition_level,
        donor_fp_columns,
        "first FP block"
    ),

    fingerprint_integrity(
        wen_physical_condition_level,
        acceptor_fp_columns,
        "second FP block"
    ),
])

display(fingerprint_integrity_summary)

,block,rows,bits,missing_cells,non_binary_cells,observed_values
0,first FP block,994,1024,0,0,"[0, 1]"
1,second FP block,994,1024,0,0,"[0, 1]"


In [49]:
# ------------------------------------------------------------------
# 8.12C
# Fingerprint bit prevalence
# ------------------------------------------------------------------

def fingerprint_bit_summary(df, columns, block_name):

    block = (
        df[columns]
        .apply(pd.to_numeric, errors="coerce")
    )

    prevalence = block.mean(axis=0)

    summary = pd.DataFrame({
        "bit": columns,
        "prevalence": prevalence.values,
    })

    summary["block"] = block_name

    return summary


donor_fp_bit_summary = fingerprint_bit_summary(
    wen_physical_condition_level,
    donor_fp_columns,
    "first FP block"
)

acceptor_fp_bit_summary = fingerprint_bit_summary(
    wen_physical_condition_level,
    acceptor_fp_columns,
    "second FP block"
)


fp_bit_summary = pd.concat(
    [
        donor_fp_bit_summary,
        acceptor_fp_bit_summary
    ],
    ignore_index=True
)


fp_block_variability = (
    fp_bit_summary
    .groupby("block")
    .agg(
        bits=("bit", "size"),
        always_zero=("prevalence", lambda x: int((x == 0).sum())),
        always_one=("prevalence", lambda x: int((x == 1).sum())),
        prevalence_lt_1pct=("prevalence", lambda x: int((x < 0.01).sum())),
        prevalence_lt_5pct=("prevalence", lambda x: int((x < 0.05).sum())),
        prevalence_gt_95pct=("prevalence", lambda x: int((x > 0.95).sum())),
        median_prevalence=("prevalence", "median"),
        mean_prevalence=("prevalence", "mean"),
    )
    .reset_index()
)

display(
    fp_block_variability.round(4)
)

,block,bits,always_zero,always_one,prevalence_lt_1pct,prevalence_lt_5pct,prevalence_gt_95pct,median_prevalence,mean_prevalence
0,first FP block,1024,0,39,1,16,300,0.7656,0.6207
1,second FP block,1024,0,20,0,2,135,0.6922,0.6611


In [50]:
# ------------------------------------------------------------------
# 8.12D
# Number of active fingerprint bits per observation
# ------------------------------------------------------------------

donor_fp_matrix = (
    wen_physical_condition_level[
        donor_fp_columns
    ]
    .astype(int)
)

acceptor_fp_matrix = (
    wen_physical_condition_level[
        acceptor_fp_columns
    ]
    .astype(int)
)


donor_active_bits = donor_fp_matrix.sum(axis=1)
acceptor_active_bits = acceptor_fp_matrix.sum(axis=1)


fp_sparsity_summary = pd.DataFrame({
    "block": [
        "first FP block",
        "second FP block"
    ],

    "mean_active_bits": [
        donor_active_bits.mean(),
        acceptor_active_bits.mean(),
    ],

    "median_active_bits": [
        donor_active_bits.median(),
        acceptor_active_bits.median(),
    ],

    "min_active_bits": [
        donor_active_bits.min(),
        acceptor_active_bits.min(),
    ],

    "max_active_bits": [
        donor_active_bits.max(),
        acceptor_active_bits.max(),
    ],
})

display(
    fp_sparsity_summary.round(2)
)

,block,mean_active_bits,median_active_bits,min_active_bits,max_active_bits
0,first FP block,635.57,634.0,133,808
1,second FP block,676.98,675.0,181,943


In [51]:
# ------------------------------------------------------------------
# 8.12E
# Unique molecular fingerprint vectors
# ------------------------------------------------------------------

donor_fp_hash = pd.util.hash_pandas_object(
    donor_fp_matrix,
    index=False
)

acceptor_fp_hash = pd.util.hash_pandas_object(
    acceptor_fp_matrix,
    index=False
)


fp_identity_summary = pd.DataFrame({
    "identity_layer": [
        "donor names",
        "donor SMILES",
        "donor fingerprint vectors",
        "acceptor names",
        "acceptor SMILES",
        "acceptor fingerprint vectors",
    ],

    "unique_count": [
        wen_physical_condition_level[
            "Name_Donor"
        ].nunique(),

        wen_physical_condition_level[
            "Smiles_Donor"
        ].nunique(),

        donor_fp_hash.nunique(),

        wen_physical_condition_level[
            "Name_Acceptor"
        ].nunique(),

        wen_physical_condition_level[
            "Smiles_Acceptor"
        ].nunique(),

        acceptor_fp_hash.nunique(),
    ],
})

display(fp_identity_summary)

,identity_layer,unique_count
0,donor names,60
1,donor SMILES,59
2,donor fingerprint vectors,56
3,acceptor names,181
4,acceptor SMILES,181
5,acceptor fingerprint vectors,175


In [52]:
# ------------------------------------------------------------------
# 8.12F
# Do different material names share identical fingerprint vectors?
# ------------------------------------------------------------------

fp_identity_audit = wen_physical_condition_level[
    [
        "Name_Donor",
        "Smiles_Donor",
        "Name_Acceptor",
        "Smiles_Acceptor",
    ]
].copy()

fp_identity_audit["donor_fp_hash"] = donor_fp_hash.values
fp_identity_audit["acceptor_fp_hash"] = acceptor_fp_hash.values


donor_fp_collisions = (
    fp_identity_audit
    .groupby("donor_fp_hash")
    .agg(
        records=("Name_Donor", "size"),
        unique_names=("Name_Donor", "nunique"),
        unique_smiles=("Smiles_Donor", "nunique"),
        names=("Name_Donor", lambda x: " | ".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

donor_fp_collisions = donor_fp_collisions[
    (
        donor_fp_collisions["unique_names"] > 1
    )
    |
    (
        donor_fp_collisions["unique_smiles"] > 1
    )
].sort_values(
    ["unique_names", "unique_smiles", "records"],
    ascending=False
)


acceptor_fp_collisions = (
    fp_identity_audit
    .groupby("acceptor_fp_hash")
    .agg(
        records=("Name_Acceptor", "size"),
        unique_names=("Name_Acceptor", "nunique"),
        unique_smiles=("Smiles_Acceptor", "nunique"),
        names=("Name_Acceptor", lambda x: " | ".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

acceptor_fp_collisions = acceptor_fp_collisions[
    (
        acceptor_fp_collisions["unique_names"] > 1
    )
    |
    (
        acceptor_fp_collisions["unique_smiles"] > 1
    )
].sort_values(
    ["unique_names", "unique_smiles", "records"],
    ascending=False
)


print(
    "Donor fingerprint collision groups:",
    len(donor_fp_collisions)
)

print(
    "Acceptor fingerprint collision groups:",
    len(acceptor_fp_collisions)
)


print("\nDonor collisions:")
display(
    donor_fp_collisions.head(30)
)

print("\nAcceptor collisions:")
display(
    acceptor_fp_collisions.head(30)
)

Donor fingerprint collision groups: 4
Acceptor fingerprint collision groups: 5

Donor collisions:


,donor_fp_hash,records,unique_names,unique_smiles,names
38,15061697634968986668,268,2,2,PBDB-T-2F | PM6
22,6522670753636113825,20,2,2,PBDT-T12BT | PBDT-TEhBT
46,16684571256274381032,11,2,2,PBDT-Cl | PClBDT-8ttTPD
52,17435187375811762129,21,2,1,L1 | L2



Acceptor collisions:


,acceptor_fp_hash,records,unique_names,unique_smiles,names
20,2438120008006758194,13,3,3,C5BTP-BO-2Cl-2F | C7BTP-BO-2Cl-2F | C9BTP-BO-2...
147,15639734997745665733,162,2,2,L8-BO | Y6
69,7068929206331922141,14,2,2,DPCT10-4F | DPCT8-4F
57,5976514415849695160,10,2,2,TT-Naph1 | TT-Naph2
134,14301433736181656624,9,2,2,M12 | M13


### 8.13 Fingerprint determinism and collision semantics

Fingerprint collisions do not necessarily indicate corrupted data because
finite bit-vector representations can map distinct molecular structures onto
the same fingerprint.

A more fundamental integrity requirement is determinism: a given supplied
molecular structure should always be represented by the same fingerprint
vector throughout the dataset.

The donor and acceptor fingerprint blocks are therefore tested for
SMILES-to-fingerprint consistency. Name-level multiplicity is examined
separately because material labels may represent aliases, publication-local
names, or multiple molecular representations.

In [53]:
# ------------------------------------------------------------------
# 8.13A
# Attach fingerprint hashes to physical-condition cohort
# ------------------------------------------------------------------

wen_fp_audit = (
    wen_physical_condition_level.copy()
)

wen_fp_audit["donor_fp_hash"] = (
    pd.util.hash_pandas_object(
        wen_fp_audit[donor_fp_columns].astype(int),
        index=False
    ).values
)

wen_fp_audit["acceptor_fp_hash"] = (
    pd.util.hash_pandas_object(
        wen_fp_audit[acceptor_fp_columns].astype(int),
        index=False
    ).values
)

print("Fingerprint-audit records:", len(wen_fp_audit))



# ------------------------------------------------------------------
# 8.13B
# Does each supplied SMILES map to exactly one fingerprint?
# ------------------------------------------------------------------

donor_smiles_fp = (
    wen_fp_audit
    .groupby("Smiles_Donor")
    .agg(
        records=("donor_fp_hash", "size"),
        unique_names=("Name_Donor", "nunique"),
        unique_fp_vectors=("donor_fp_hash", "nunique"),
    )
    .reset_index()
)

acceptor_smiles_fp = (
    wen_fp_audit
    .groupby("Smiles_Acceptor")
    .agg(
        records=("acceptor_fp_hash", "size"),
        unique_names=("Name_Acceptor", "nunique"),
        unique_fp_vectors=("acceptor_fp_hash", "nunique"),
    )
    .reset_index()
)


donor_non_deterministic = donor_smiles_fp[
    donor_smiles_fp["unique_fp_vectors"] > 1
]

acceptor_non_deterministic = acceptor_smiles_fp[
    acceptor_smiles_fp["unique_fp_vectors"] > 1
]


print(
    "Donor SMILES mapping to >1 fingerprint:",
    len(donor_non_deterministic)
)

print(
    "Acceptor SMILES mapping to >1 fingerprint:",
    len(acceptor_non_deterministic)
)

Fingerprint-audit records: 994
Donor SMILES mapping to >1 fingerprint: 0
Acceptor SMILES mapping to >1 fingerprint: 0


In [55]:
# ------------------------------------------------------------------
# 8.13C
# Material-name representation multiplicity
# ------------------------------------------------------------------

donor_name_fp = (
    wen_fp_audit
    .groupby("Name_Donor")
    .agg(
        records=("donor_fp_hash", "size"),
        unique_smiles=("Smiles_Donor", "nunique"),
        unique_fp_vectors=("donor_fp_hash", "nunique"),
    )
    .reset_index()
)

acceptor_name_fp = (
    wen_fp_audit
    .groupby("Name_Acceptor")
    .agg(
        records=("acceptor_fp_hash", "size"),
        unique_smiles=("Smiles_Acceptor", "nunique"),
        unique_fp_vectors=("acceptor_fp_hash", "nunique"),
    )
    .reset_index()
)


print("Donor names with multiple supplied SMILES:")
display(
    donor_name_fp[
        donor_name_fp["unique_smiles"] > 1
    ]
    .sort_values(
        ["unique_smiles", "records"],
        ascending=False
    )
)


print("\nAcceptor names with multiple supplied SMILES:")
display(
    acceptor_name_fp[
        acceptor_name_fp["unique_smiles"] > 1
    ]
    .sort_values(
        ["unique_smiles", "records"],
        ascending=False
    )
)



# ------------------------------------------------------------------
# Same name + same supplied structure but multiple fingerprints
# ------------------------------------------------------------------

donor_name_inconsistency = donor_name_fp[
    (donor_name_fp["unique_smiles"] == 1)
    &
    (donor_name_fp["unique_fp_vectors"] > 1)
]

acceptor_name_inconsistency = acceptor_name_fp[
    (acceptor_name_fp["unique_smiles"] == 1)
    &
    (acceptor_name_fp["unique_fp_vectors"] > 1)
]


print(
    "Donor name/structure fingerprint inconsistencies:",
    len(donor_name_inconsistency)
)

print(
    "Acceptor name/structure fingerprint inconsistencies:",
    len(acceptor_name_inconsistency)
)

Donor names with multiple supplied SMILES:


,Name_Donor,records,unique_smiles,unique_fp_vectors



Acceptor names with multiple supplied SMILES:


,Name_Acceptor,records,unique_smiles,unique_fp_vectors


Donor name/structure fingerprint inconsistencies: 0
Acceptor name/structure fingerprint inconsistencies: 0


### 8.14 Molecular-descriptor integrity and redundancy

In addition to molecular fingerprints, the Wen–Zhang–Ma global dataset
contains a compact block of donor, acceptor, and donor–acceptor electronic
descriptors.

These variables are audited separately from the fingerprints to verify
numerical completeness, finite values, effective variability, deterministic
mapping to molecular identity, and possible redundancy.

The purpose is to distinguish informative physicochemical descriptors from
constant, duplicated, or identity-inconsistent variables before the final
model matrix is frozen.

In [56]:
# ------------------------------------------------------------------
# 8.14A
# Locate non-fingerprint molecular descriptor block
# ------------------------------------------------------------------

first_fp_position = (
    wen_physical_condition_level
    .columns
    .get_loc("FP1")
)

thickness_position = (
    wen_physical_condition_level
    .columns
    .get_loc("Active_Layer_Thickness (nm)")
)

molecular_descriptor_columns = (
    wen_physical_condition_level
    .columns[
        thickness_position + 1:
        first_fp_position
    ]
    .tolist()
)


print(
    "Molecular descriptor columns:",
    len(molecular_descriptor_columns)
)

for i, col in enumerate(
    molecular_descriptor_columns
):
    print(f"{i:2d}  {col}")

Molecular descriptor columns: 19
 0  E_DH-1 (eV)
 1  E_DH (eV)
 2  E_DL (eV)
 3  E_DL+1 (eV)
 4  E_AH-1 (eV)
 5  E_AH (eV)
 6  E_AL (eV)
 7  E_AL+1 (eV)
 8  delta_E_DL_DH (eV)
 9  delta_E_AL_AH (eV)
10  delta_E_DL_AL (eV)
11  delta_E_DH_AH (eV)
12  delta_E_AL-DH (eV)
13  delta_E_DH_DH-1 (eV)
14  delta_E_DL+1_DL (eV)
15  delta_E_AH_AH-1 (eV)
16  delta_E_AL+1_AL (eV)
17  Donor_DipoleMoment (Debye)
18  Acceptor_DipoleMoment (Debye)


In [57]:
# ------------------------------------------------------------------
# 8.14B
# Numerical integrity
# ------------------------------------------------------------------

molecular_desc = (
    wen_physical_condition_level[
        molecular_descriptor_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


descriptor_integrity_rows = []

for col in molecular_descriptor_columns:

    s = molecular_desc[col]

    finite_mask = np.isfinite(
        s.dropna()
    )

    descriptor_integrity_rows.append({
        "descriptor": col,

        "missing":
            int(s.isna().sum()),

        "infinite":
            int(
                (~finite_mask).sum()
            ),

        "unique_values":
            int(s.nunique()),

        "min":
            s.min(),

        "median":
            s.median(),

        "max":
            s.max(),

        "mean":
            s.mean(),

        "std":
            s.std(),
    })


molecular_descriptor_integrity = (
    pd.DataFrame(
        descriptor_integrity_rows
    )
)


display(
    molecular_descriptor_integrity
    .round(4)
)



constant_molecular_descriptors = (
    molecular_descriptor_integrity[
        molecular_descriptor_integrity[
            "unique_values"
        ] <= 1
    ]
)

print(
    "Constant molecular descriptors:",
    len(constant_molecular_descriptors)
)

display(
    constant_molecular_descriptors
)

,descriptor,missing,infinite,unique_values,min,median,max,mean,std
0,E_DH-1 (eV),0,0,60,-7.9068,-6.8497,-6.3212,-6.8482,0.2201
1,E_DH (eV),0,0,59,-7.5713,-6.3569,-5.9185,-6.3800,0.2106
2,E_DL (eV),0,0,60,-2.5565,-1.5693,0.9644,-1.5227,0.4401
3,E_DL+1 (eV),0,0,60,-2.4697,-1.3018,2.2251,-1.1349,0.6151
4,E_AH-1 (eV),0,0,177,-8.2173,-7.2722,-6.3471,-7.2163,0.2991
5,E_AH (eV),0,0,175,-7.5757,-6.5286,-5.8768,-6.5376,0.2018
6,E_AL (eV),0,0,171,-3.2387,-2.8158,-0.7274,-2.7312,0.2781
7,E_AL+1 (eV),0,0,173,-2.9339,-2.5818,0.1755,-2.4734,0.3876
8,delta_E_DL_DH (eV),0,0,60,3.6602,4.8872,8.5357,4.8573,0.6104
9,delta_E_AL_AH (eV),0,0,171,2.6381,3.6871,5.2714,3.8064,0.3208


Constant molecular descriptors: 0


,descriptor,missing,infinite,unique_values,min,median,max,mean,std


In [58]:
# ------------------------------------------------------------------
# 8.14C
# Structural pair -> descriptor-vector determinism
# ------------------------------------------------------------------

descriptor_hash = (
    pd.util.hash_pandas_object(
        molecular_desc,
        index=False
    )
)


descriptor_identity_audit = (
    wen_physical_condition_level[
        [
            "Name_Donor",
            "Smiles_Donor",
            "Name_Acceptor",
            "Smiles_Acceptor",
        ]
    ]
    .copy()
)

descriptor_identity_audit[
    "descriptor_hash"
] = descriptor_hash.values


pair_descriptor_determinism = (
    descriptor_identity_audit
    .groupby(
        [
            "Smiles_Donor",
            "Smiles_Acceptor",
        ],
        dropna=False
    )
    .agg(
        records=(
            "descriptor_hash",
            "size"
        ),

        unique_descriptor_vectors=(
            "descriptor_hash",
            "nunique"
        ),
    )
    .reset_index()
)


pair_descriptor_inconsistencies = (
    pair_descriptor_determinism[
        pair_descriptor_determinism[
            "unique_descriptor_vectors"
        ] > 1
    ]
)


print(
    "Unique supplied D:A structure pairs:",
    len(pair_descriptor_determinism)
)

print(
    "D:A structural pairs mapping to >1 descriptor vector:",
    len(pair_descriptor_inconsistencies)
)

Unique supplied D:A structure pairs: 246
D:A structural pairs mapping to >1 descriptor vector: 1


In [59]:
# ------------------------------------------------------------------
# 8.14D
# Descriptor-vector resolution
# ------------------------------------------------------------------

descriptor_resolution_summary = pd.DataFrame({

    "identity_layer": [
        "donor names",
        "acceptor names",
        "named D:A pairs",
        "supplied-SMILES D:A pairs",
        "molecular descriptor vectors",
    ],

    "unique_count": [
        wen_physical_condition_level[
            "Name_Donor"
        ].nunique(),

        wen_physical_condition_level[
            "Name_Acceptor"
        ].nunique(),

        wen_physical_condition_level[
            ["Name_Donor", "Name_Acceptor"]
        ].drop_duplicates().shape[0],

        wen_physical_condition_level[
            ["Smiles_Donor", "Smiles_Acceptor"]
        ].drop_duplicates().shape[0],

        descriptor_hash.nunique(),
    ],
})

display(
    descriptor_resolution_summary
)

,identity_layer,unique_count
0,donor names,60
1,acceptor names,181
2,named D:A pairs,247
3,supplied-SMILES D:A pairs,246
4,molecular descriptor vectors,247


In [60]:
# ------------------------------------------------------------------
# 8.14E
# Exact duplicate descriptor columns
# ------------------------------------------------------------------

duplicate_descriptor_pairs = []

for i, col_a in enumerate(
    molecular_descriptor_columns
):

    for col_b in (
        molecular_descriptor_columns[
            i + 1:
        ]
    ):

        if molecular_desc[col_a].equals(
            molecular_desc[col_b]
        ):

            duplicate_descriptor_pairs.append(
                (col_a, col_b)
            )


print(
    "Exactly duplicated molecular-descriptor column pairs:",
    len(duplicate_descriptor_pairs)
)

for pair in duplicate_descriptor_pairs:
    print(pair)

Exactly duplicated molecular-descriptor column pairs: 0


In [61]:
# ------------------------------------------------------------------
# 8.14F
# Highly correlated descriptor pairs
# ------------------------------------------------------------------

descriptor_corr = (
    molecular_desc.corr()
)

high_corr_pairs = []

for i, col_a in enumerate(
    molecular_descriptor_columns
):

    for col_b in (
        molecular_descriptor_columns[
            i + 1:
        ]
    ):

        r = descriptor_corr.loc[
            col_a,
            col_b
        ]

        if pd.notna(r) and abs(r) >= 0.95:

            high_corr_pairs.append({
                "descriptor_1": col_a,
                "descriptor_2": col_b,
                "pearson_r": r,
            })


high_corr_pairs = (
    pd.DataFrame(
        high_corr_pairs
    )
)

if len(high_corr_pairs):

    high_corr_pairs = (
        high_corr_pairs
        .assign(
            abs_r=lambda x:
                x["pearson_r"].abs()
        )
        .sort_values(
            "abs_r",
            ascending=False
        )
        .drop(columns="abs_r")
    )


print(
    "Descriptor pairs with |r| >= 0.95:",
    len(high_corr_pairs)
)

display(
    high_corr_pairs
)

Descriptor pairs with |r| >= 0.95: 1


,descriptor_1,descriptor_2,pearson_r
0,E_DL (eV),delta_E_DL_DH (eV),0.971483


### 8.15 Investigation of the single structure–descriptor inconsistency

The molecular-descriptor audit identified one supplied donor–acceptor SMILES
pair associated with more than one electronic-descriptor vector.

Because molecular descriptors should normally be deterministic with respect to
the supplied molecular representation, this case is inspected directly before
the descriptor block is accepted for modeling.

Differences in material naming, individual descriptor values, and record
context are examined to determine whether the inconsistency reflects an alias,
rounding or calculation difference, or a genuine curation inconsistency.

In [62]:
# ------------------------------------------------------------------
# 8.15A
# Identify D:A structural pair with multiple descriptor vectors
# ------------------------------------------------------------------

problem_structure_pairs = (
    pair_descriptor_inconsistencies[
        [
            "Smiles_Donor",
            "Smiles_Acceptor",
            "records",
            "unique_descriptor_vectors",
        ]
    ]
    .copy()
)

display(problem_structure_pairs)

,Smiles_Donor,Smiles_Acceptor,records,unique_descriptor_vectors
2,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,21,2


In [63]:
# ------------------------------------------------------------------
# 8.15B
# Retrieve records associated with inconsistent descriptor pair
# ------------------------------------------------------------------

problem_donor_smiles = (
    problem_structure_pairs.iloc[0][
        "Smiles_Donor"
    ]
)

problem_acceptor_smiles = (
    problem_structure_pairs.iloc[0][
        "Smiles_Acceptor"
    ]
)


problem_descriptor_rows = (
    wen_physical_condition_level[
        wen_physical_condition_level[
            "Smiles_Donor"
        ].eq(problem_donor_smiles)
        &
        wen_physical_condition_level[
            "Smiles_Acceptor"
        ].eq(problem_acceptor_smiles)
    ][
        [
            "Name_Donor",
            "Name_Acceptor",
            "Smiles_Donor",
            "Smiles_Acceptor",
            *molecular_descriptor_columns,
            TARGET_COL,
        ]
    ]
    .copy()
)


print(
    "Records belonging to problematic structural pair:",
    len(problem_descriptor_rows)
)

print(
    "Donor names:",
    problem_descriptor_rows[
        "Name_Donor"
    ].unique()
)

print(
    "Acceptor names:",
    problem_descriptor_rows[
        "Name_Acceptor"
    ].unique()
)

display(problem_descriptor_rows)

Records belonging to problematic structural pair: 21
Donor names: <StringArray>
['L1', 'L2']
Length: 2, dtype: str
Acceptor names: <StringArray>
['Y6']
Length: 1, dtype: str


,Name_Donor,Name_Acceptor,Smiles_Donor,Smiles_Acceptor,E_DH-1 (eV),E_DH (eV),E_DL (eV),E_DL+1 (eV),E_AH-1 (eV),E_AH (eV),...,delta_E_DL_AL (eV),delta_E_DH_AH (eV),delta_E_AL-DH (eV),delta_E_DH_DH-1 (eV),delta_E_DL+1_DL (eV),delta_E_AH_AH-1 (eV),delta_E_AL+1_AL (eV),Donor_DipoleMoment (Debye),Acceptor_DipoleMoment (Debye),PCE (%)
437,L1,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.465701,-6.068686,-1.912417,-1.850919,-7.402045,-6.53645,...,0.963011,0.467764,3.193258,0.397014,0.061498,0.865595,0.25225,5.6300,2.3626,13.50
438,L1,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.465701,-6.068686,-1.912417,-1.850919,-7.402045,-6.53645,...,0.963011,0.467764,3.193258,0.397014,0.061498,0.865595,0.25225,5.6300,2.3626,13.10
439,L1,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.465701,-6.068686,-1.912417,-1.850919,-7.402045,-6.53645,...,0.963011,0.467764,3.193258,0.397014,0.061498,0.865595,0.25225,5.6300,2.3626,12.70
440,L1,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.465701,-6.068686,-1.912417,-1.850919,-7.402045,-6.53645,...,0.963011,0.467764,3.193258,0.397014,0.061498,0.865595,0.25225,5.6300,2.3626,12.90
441,L1,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.465701,-6.068686,-1.912417,-1.850919,-7.402045,-6.53645,...,0.963011,0.467764,3.193258,0.397014,0.061498,0.865595,0.25225,5.6300,2.3626,13.20
442,L1,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.465701,-6.068686,-1.912417,-1.850919,-7.402045,-6.53645,...,0.963011,0.467764,3.193258,0.397014,0.061498,0.865595,0.25225,5.6300,2.3626,13.80
443,L2,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.452095,-6.056169,-1.913778,-1.868063,-7.402045,-6.53645,...,0.961651,0.480281,3.180741,0.395926,0.045715,0.865595,0.25225,4.9389,2.3626,13.53
444,L2,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.452095,-6.056169,-1.913778,-1.868063,-7.402045,-6.53645,...,0.961651,0.480281,3.180741,0.395926,0.045715,0.865595,0.25225,4.9389,2.3626,13.49
445,L2,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.452095,-6.056169,-1.913778,-1.868063,-7.402045,-6.53645,...,0.961651,0.480281,3.180741,0.395926,0.045715,0.865595,0.25225,4.9389,2.3626,14.16
446,L2,Y6,N1(C(=S)S/C(=C/c2sc(cc2)c2c(cc(s2)c2sc(cc2CCCC...,c1c(c(cc2c1C(=C(C#N)C#N)/C(=C/c1sc3c(c1CCCCCCC...,-6.452095,-6.056169,-1.913778,-1.868063,-7.402045,-6.53645,...,0.961651,0.480281,3.180741,0.395926,0.045715,0.865595,0.25225,4.9389,2.3626,13.64


In [64]:
# ------------------------------------------------------------------
# 8.15C
# Which molecular descriptors differ?
# ------------------------------------------------------------------

differing_descriptors = []

for col in molecular_descriptor_columns:

    if (
        problem_descriptor_rows[col]
        .nunique(dropna=False)
        > 1
    ):
        differing_descriptors.append(col)


print(
    "Number of differing descriptors:",
    len(differing_descriptors)
)

print("\nDiffering descriptors:")

for col in differing_descriptors:
    print(col)



    # ------------------------------------------------------------------
# 8.15D
# Compact view of descriptor inconsistency
# ------------------------------------------------------------------

display(
    problem_descriptor_rows[
        [
            "Name_Donor",
            "Name_Acceptor",
            *differing_descriptors,
            TARGET_COL,
        ]
    ]
    .drop_duplicates()
)

Number of differing descriptors: 11

Differing descriptors:
E_DH-1 (eV)
E_DH (eV)
E_DL (eV)
E_DL+1 (eV)
delta_E_DL_DH (eV)
delta_E_DL_AL (eV)
delta_E_DH_AH (eV)
delta_E_AL-DH (eV)
delta_E_DH_DH-1 (eV)
delta_E_DL+1_DL (eV)
Donor_DipoleMoment (Debye)


,Name_Donor,Name_Acceptor,E_DH-1 (eV),E_DH (eV),E_DL (eV),E_DL+1 (eV),delta_E_DL_DH (eV),delta_E_DL_AL (eV),delta_E_DH_AH (eV),delta_E_AL-DH (eV),delta_E_DH_DH-1 (eV),delta_E_DL+1_DL (eV),Donor_DipoleMoment (Debye),PCE (%)
437,L1,Y6,-6.465701,-6.068686,-1.912417,-1.850919,4.156269,0.963011,0.467764,3.193258,0.397014,0.061498,5.6300,13.50
438,L1,Y6,-6.465701,-6.068686,-1.912417,-1.850919,4.156269,0.963011,0.467764,3.193258,0.397014,0.061498,5.6300,13.10
439,L1,Y6,-6.465701,-6.068686,-1.912417,-1.850919,4.156269,0.963011,0.467764,3.193258,0.397014,0.061498,5.6300,12.70
440,L1,Y6,-6.465701,-6.068686,-1.912417,-1.850919,4.156269,0.963011,0.467764,3.193258,0.397014,0.061498,5.6300,12.90
441,L1,Y6,-6.465701,-6.068686,-1.912417,-1.850919,4.156269,0.963011,0.467764,3.193258,0.397014,0.061498,5.6300,13.20
442,L1,Y6,-6.465701,-6.068686,-1.912417,-1.850919,4.156269,0.963011,0.467764,3.193258,0.397014,0.061498,5.6300,13.80
443,L2,Y6,-6.452095,-6.056169,-1.913778,-1.868063,4.142391,0.961651,0.480281,3.180741,0.395926,0.045715,4.9389,13.53
444,L2,Y6,-6.452095,-6.056169,-1.913778,-1.868063,4.142391,0.961651,0.480281,3.180741,0.395926,0.045715,4.9389,13.49
445,L2,Y6,-6.452095,-6.056169,-1.913778,-1.868063,4.142391,0.961651,0.480281,3.180741,0.395926,0.045715,4.9389,14.16
446,L2,Y6,-6.452095,-6.056169,-1.913778,-1.868063,4.142391,0.961651,0.480281,3.180741,0.395926,0.045715,4.9389,13.64


In [65]:
# ------------------------------------------------------------------
# 8.15E
# Magnitude of descriptor disagreement
# ------------------------------------------------------------------

descriptor_difference_summary = []

for col in differing_descriptors:

    values = (
        problem_descriptor_rows[col]
        .dropna()
        .unique()
    )

    descriptor_difference_summary.append({
        "descriptor": col,
        "unique_values": len(values),
        "min": min(values),
        "max": max(values),
        "absolute_range":
            max(values) - min(values),
    })


descriptor_difference_summary = pd.DataFrame(
    descriptor_difference_summary
)

display(
    descriptor_difference_summary.round(6)
)

,descriptor,unique_values,min,max,absolute_range
0,E_DH-1 (eV),2,-6.465701,-6.452095,0.013606
1,E_DH (eV),2,-6.068686,-6.056169,0.012517
2,E_DL (eV),2,-1.913778,-1.912417,0.001361
3,E_DL+1 (eV),2,-1.868063,-1.850919,0.017143
4,delta_E_DL_DH (eV),2,4.142391,4.156269,0.013878
5,delta_E_DL_AL (eV),2,0.961651,0.963011,0.001361
6,delta_E_DH_AH (eV),2,0.467764,0.480281,0.012517
7,delta_E_AL-DH (eV),2,3.180741,3.193258,0.012517
8,delta_E_DH_DH-1 (eV),2,0.395926,0.397014,0.001088
9,delta_E_DL+1_DL (eV),2,0.045715,0.061498,0.015783


In [66]:
# ------------------------------------------------------------------
# 8.15F
# Is the molecular-descriptor representation deterministic
# at the nominal D:A-pair level?
# ------------------------------------------------------------------

named_pair_descriptor_determinism = (
    descriptor_identity_audit
    .groupby(
        ["Name_Donor", "Name_Acceptor"]
    )
    .agg(
        records=("descriptor_hash", "size"),
        unique_descriptor_vectors=(
            "descriptor_hash",
            "nunique"
        ),
    )
    .reset_index()
)


named_pair_descriptor_inconsistencies = (
    named_pair_descriptor_determinism[
        named_pair_descriptor_determinism[
            "unique_descriptor_vectors"
        ] > 1
    ]
)


print(
    "Named D:A pairs:",
    len(named_pair_descriptor_determinism)
)

print(
    "Named D:A pairs mapping to >1 descriptor vector:",
    len(named_pair_descriptor_inconsistencies)
)

display(
    named_pair_descriptor_inconsistencies
)

Named D:A pairs: 247
Named D:A pairs mapping to >1 descriptor vector: 0


,Name_Donor,Name_Acceptor,records,unique_descriptor_vectors


### 8.15 Structure–descriptor audit conclusion

The single apparent structure–descriptor inconsistency is confined to the
L1:Y6 and L2:Y6 systems.

L1 and L2 are represented by the same supplied donor SMILES and the same
supplied molecular fingerprint, but they possess distinct and internally
consistent donor electronic descriptors. Eleven molecular descriptors differ,
primarily donor-derived quantities and donor-dependent energy differences.

The molecular-descriptor representation therefore preserves a nominal material
distinction that is lost in the supplied SMILES and fingerprint representation.

Consequently, supplied SMILES equality and fingerprint equality are not used
as sufficient evidence for material equivalence within the Wen–Zhang–Ma
dataset. The curated electronic descriptors are retained without modification.

## 9.0 Relationship Among Wen–Zhang–Ma Modeling Tiers

The Wen–Zhang–Ma package contains single-parameter, stage-combined, and global
processing datasets.

These tables must not be assumed to represent independent experimental
collections. They may instead constitute partially overlapping views of the
same literature-derived device records with different requirements for
processing-parameter completeness.

Before any multi-tier or cross-tier learning experiment is considered, the
tables are inventoried and their molecular-system, target, and record-level
overlap is quantified.

In [68]:
# ------------------------------------------------------------------
# 9.1
# Inventory all Wen/Ma CSV datasets with encoding detection
# ------------------------------------------------------------------

wen_package_root = wen_global_files[0].parents[2]

wen_csv_files = sorted(
    wen_package_root.rglob("*.csv")
)

print("CSV files found:", len(wen_csv_files))
print()


def read_wen_csv(path):
    """
    Read Wen/Ma CSV while preserving text.
    Try a small set of likely encodings rather than silently
    dropping undecodable characters.
    """

    candidate_encodings = [
        "utf-8",
        "utf-8-sig",
        "gb18030",
        "cp1252",
    ]

    last_error = None

    for encoding in candidate_encodings:

        try:
            df = pd.read_csv(
                path,
                encoding=encoding
            )

            return df, encoding

        except UnicodeDecodeError as exc:
            last_error = exc

    raise UnicodeDecodeError(
        last_error.encoding,
        last_error.object,
        last_error.start,
        last_error.end,
        (
            f"Could not decode {path.name} "
            f"with tested encodings."
        )
    )


tier_inventory_rows = []

# Cache loaded tables so we do not repeatedly read them later
wen_tier_tables = {}


for path in wen_csv_files:

    df, encoding_used = read_wen_csv(path)

    relative_path = str(
        path.relative_to(wen_package_root)
    )

    wen_tier_tables[relative_path] = df

    tier_inventory_rows.append({
        "file": path.name,

        "relative_path":
            relative_path,

        "encoding":
            encoding_used,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "has_donor_name":
            "Name_Donor" in df.columns,

        "has_acceptor_name":
            "Name_Acceptor" in df.columns,

        "has_donor_smiles":
            "Smiles_Donor" in df.columns,

        "has_acceptor_smiles":
            "Smiles_Acceptor" in df.columns,

        "has_pce":
            "PCE (%)" in df.columns,
    })


wen_tier_inventory = pd.DataFrame(
    tier_inventory_rows
)


display(
    wen_tier_inventory[
        [
            "relative_path",
            "encoding",
            "rows",
            "columns",
            "has_pce",
        ]
    ]
)

CSV files found: 13



,relative_path,encoding,rows,columns,has_pce
0,opv-multi-tier-ml-database\01-single-parameter...,utf-8,1767,2073,False
1,opv-multi-tier-ml-database\01-single-parameter...,utf-8,128,2075,True
2,opv-multi-tier-ml-database\01-single-parameter...,utf-8,407,2079,True
3,opv-multi-tier-ml-database\01-single-parameter...,utf-8,1132,2073,True
4,opv-multi-tier-ml-database\01-single-parameter...,utf-8,95,2073,True
5,opv-multi-tier-ml-database\01-single-parameter...,utf-8,213,2073,True
6,opv-multi-tier-ml-database\01-single-parameter...,utf-8,199,2073,True
7,opv-multi-tier-ml-database\01-single-parameter...,utf-8,1600,2073,True
8,opv-multi-tier-ml-database\01-single-parameter...,utf-8,138,2073,True
9,opv-multi-tier-ml-database\02-stage-combined-m...,gb18030,2087,2074,True


In [69]:
# ------------------------------------------------------------------
# 9.1B
# Inspect exact filenames and any table lacking standard PCE label
# ------------------------------------------------------------------

pd.set_option("display.max_colwidth", None)

display(
    wen_tier_inventory[
        [
            "relative_path",
            "encoding",
            "rows",
            "columns",
            "has_pce",
        ]
    ]
)


print("\n" + "=" * 90)
print("TABLES WITHOUT STANDARD 'PCE (%)' COLUMN")
print("=" * 90)

no_standard_pce = wen_tier_inventory[
    ~wen_tier_inventory["has_pce"]
]

for _, row in no_standard_pce.iterrows():

    relative_path = row["relative_path"]

    df = wen_tier_tables[
        relative_path
    ]

    print("\nFile:")
    print(relative_path)

    print("\nColumns containing PCE / efficiency-like terms:")

    candidate_target_columns = [
        col
        for col in df.columns
        if (
            "pce" in str(col).lower()
            or "efficien" in str(col).lower()
            or "power conversion" in str(col).lower()
        )
    ]

    print(candidate_target_columns)

    print("\nFirst 30 columns:")
    print(df.columns[:30].tolist())

,relative_path,encoding,rows,columns,has_pce
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,utf-8,1767,2073,False
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,utf-8,128,2075,True
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,utf-8,407,2079,True
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,utf-8,1132,2073,True
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,utf-8,95,2073,True
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,utf-8,213,2073,True
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,utf-8,199,2073,True
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,utf-8,1600,2073,True
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,utf-8,138,2073,True
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,gb18030,2087,2074,True



TABLES WITHOUT STANDARD 'PCE (%)' COLUMN

File:
opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv

Columns containing PCE / efficiency-like terms:
['PCE (%）']

First 30 columns:
['Name_Donor', 'Smiles_Donor', 'Name_Acceptor', 'Smiles_Acceptor', 'PCE (%）', 'D_A_Weight_Ratio', 'E_DH-1 (eV)', 'E_DH (eV)', 'E_DL (eV)', 'E_DL+1 (eV)', 'E_AH-1 (eV)', 'E_AH (eV)', 'E_AL (eV)', 'E_AL+1 (eV)', 'delta_E_DL_DH (eV)', 'delta_E_AL_AH (eV)', 'delta_E_DL_AL (eV)', 'delta_E_DH_AH (eV)', 'delta_E_AL-DH (eV)', 'delta_E_DH_DH-1 (eV)', 'delta_E_DL+1_DL (eV)', 'delta_E_AH_AH-1 (eV)', 'delta_E_AL+1_AL (eV)', 'Donor_DipoleMoment (Debye)', 'Acceptor_DipoleMoment (Debye)', 'FP1', 'FP2', 'FP3', 'FP4', 'FP5']


### 9.1.1 Cross-tier schema normalization

The thirteen Wen–Zhang–Ma tables are not completely uniform in text encoding
or column-header representation.

Two stage-combined CSV files require GB18030 decoding, whereas the remaining
files are UTF-8 encoded. In addition, several headers contain Unicode
compatibility characters. Most notably, the D:A-weight-ratio table represents
the PCE target as `PCE (%）`, using a fullwidth right parenthesis (`U+FF09`).
Other processing-variable headers use compatibility forms such as the Unicode
Celsius symbol (`℃`).

Unicode NFKC normalization is therefore applied to all column headers before
cross-tier comparison. Header normalization is verified not to introduce
duplicate column names, and all thirteen tables subsequently contain the
standard `PCE (%)` target field.

No raw source files or data values are modified.

In [72]:
# ------------------------------------------------------------------
# 9.1C
# Normalize leading/trailing whitespace in tier-table headers
# ------------------------------------------------------------------

wen_tier_tables_clean = {}

header_normalization_rows = []


for relative_path, df in wen_tier_tables.items():

    clean_df = df.copy()

    original_columns = clean_df.columns.tolist()

    normalized_columns = [
        str(col).strip()
        for col in original_columns
    ]

    # Check whether stripping whitespace would collapse
    # originally distinct column names.
    duplicate_after_strip = (
        len(normalized_columns)
        - len(set(normalized_columns))
    )

    if duplicate_after_strip > 0:
        raise ValueError(
            f"Header collision after whitespace normalization: "
            f"{relative_path}"
        )

    changed_headers = [
        (old, new)
        for old, new in zip(
            original_columns,
            normalized_columns
        )
        if old != new
    ]

    clean_df.columns = normalized_columns

    wen_tier_tables_clean[
        relative_path
    ] = clean_df

    header_normalization_rows.append({
        "relative_path": relative_path,
        "changed_headers": len(changed_headers),
        "has_standard_pce":
            "PCE (%)" in clean_df.columns,
    })


header_normalization_summary = pd.DataFrame(
    header_normalization_rows
)

display(header_normalization_summary)

# ------------------------------------------------------------------
# Show tables whose headers required normalization
# ------------------------------------------------------------------

display(
    header_normalization_summary[
        header_normalization_summary[
            "changed_headers"
        ] > 0
    ]
)

,relative_path,changed_headers,has_standard_pce
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,0,False
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,0,True
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,0,True
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,0,True
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,0,True
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,0,True
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,0,True
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,0,True
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,0,True
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,0,True


,relative_path,changed_headers,has_standard_pce


In [73]:
# ------------------------------------------------------------------
# 9.1D
# Diagnose invisible characters in anomalous PCE header
# ------------------------------------------------------------------

problem_path = next(
    path
    for path in wen_tier_tables
    if path.endswith(
        "single-parameter-01-d-a-weight-ratio.csv"
    )
)

problem_df = wen_tier_tables[problem_path]


pce_like_columns = [
    col
    for col in problem_df.columns
    if "pce" in str(col).lower()
]


print("PCE-like columns:")
print(pce_like_columns)


for col in pce_like_columns:

    print("\nRaw repr:")
    print(repr(col))

    print("\nCharacter-by-character:")

    for i, char in enumerate(str(col)):

        print(
            i,
            repr(char),
            f"U+{ord(char):04X}",
            unicodedata.name(
                char,
                "UNKNOWN"
            )
        )

PCE-like columns:
['PCE (%）']

Raw repr:
'PCE (%）'

Character-by-character:
0 'P' U+0050 LATIN CAPITAL LETTER P
1 'C' U+0043 LATIN CAPITAL LETTER C
2 'E' U+0045 LATIN CAPITAL LETTER E
3 ' ' U+0020 SPACE
4 '(' U+0028 LEFT PARENTHESIS
5 '%' U+0025 PERCENT SIGN
6 '）' U+FF09 FULLWIDTH RIGHT PARENTHESIS


In [74]:
# ------------------------------------------------------------------
# 9.1E
# Robust Unicode-aware header normalization
# ------------------------------------------------------------------

def normalize_column_header(col):

    text = unicodedata.normalize(
        "NFKC",
        str(col)
    )

    # Remove invisible Unicode format characters if present
    text = "".join(
        char
        for char in text
        if unicodedata.category(char) != "Cf"
    )

    text = text.strip()

    return text


wen_tier_tables_clean = {}

header_normalization_rows = []


for relative_path, df in wen_tier_tables.items():

    clean_df = df.copy()

    original_columns = clean_df.columns.tolist()

    normalized_columns = [
        normalize_column_header(col)
        for col in original_columns
    ]

    # Safety check for accidental column collisions
    if len(normalized_columns) != len(set(normalized_columns)):

        duplicate_names = (
            pd.Series(normalized_columns)
            .loc[
                lambda x:
                    x.duplicated(keep=False)
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"Header collision after normalization "
            f"in {relative_path}: {duplicate_names}"
        )


    changed_headers = [
        (old, new)
        for old, new in zip(
            original_columns,
            normalized_columns
        )
        if old != new
    ]

    clean_df.columns = normalized_columns

    wen_tier_tables_clean[
        relative_path
    ] = clean_df


    header_normalization_rows.append({

        "relative_path":
            relative_path,

        "changed_headers":
            len(changed_headers),

        "has_standard_pce":
            "PCE (%)" in clean_df.columns,

        "changes":
            changed_headers,
    })


header_normalization_summary = pd.DataFrame(
    header_normalization_rows
)


display(
    header_normalization_summary[
        [
            "relative_path",
            "changed_headers",
            "has_standard_pce",
        ]
    ]
)

,relative_path,changed_headers,has_standard_pce
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1,True
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,0,True
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,2,True
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,0,True
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,0,True
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,0,True
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,0,True
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1,True
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,0,True
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,1,True


In [75]:
# ------------------------------------------------------------------
# 9.1F
# Final cross-tier target-schema verification
# ------------------------------------------------------------------

print(
    "Tables with standard PCE header:",
    int(
        header_normalization_summary[
            "has_standard_pce"
        ].sum()
    ),
    "/",
    len(header_normalization_summary)
)


print("\nHeaders changed during normalization:")

display(
    header_normalization_summary[
        header_normalization_summary[
            "changed_headers"
        ] > 0
    ][
        [
            "relative_path",
            "changed_headers",
            "changes",
        ]
    ]
)

Tables with standard PCE header: 13 / 13

Headers changed during normalization:


,relative_path,changed_headers,changes
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1,"[(PCE (%）, PCE (%))]"
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,2,"[(Additive_MeltingPoint (℃), Additive_MeltingPoint (°C)), (Additive_BoilingPoint (℃), Additive_BoilingPoint (°C))]"
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1,"[(Annealing_Temperature (℃), Annealing_Temperature (°C))]"
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,1,"[(Annealing_Temperature (℃), Annealing_Temperature (°C))]"
10,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-solution-preparation.csv,2,"[(Additive_MeltingPoint (℃), Additive_MeltingPoint (°C)), (Additive_BoilingPoint (℃), Additive_BoilingPoint (°C))]"
12,opv-multi-tier-ml-database\03-global-model\global-all-nine-parameters.csv,3,"[(Additive_MeltingPoint (℃), Additive_MeltingPoint (°C)), (Additive_BoilingPoint (℃), Additive_BoilingPoint (°C)), (Annealing_Temperature(℃), Annealing_Temperature(°C))]"


In [87]:
# ------------------------------------------------------------------
# 9.2
# Chemical-system coverage by modeling tier
# ------------------------------------------------------------------

tier_chemistry_rows = []


for relative_path, df in wen_tier_tables_clean.items():

    required = {
        "Name_Donor",
        "Name_Acceptor",
        "PCE (%)"
    }

    if not required.issubset(df.columns):
        raise ValueError(
            f"Required identity/target fields missing from: "
            f"{relative_path}"
        )

    pce_numeric = pd.to_numeric(
        df["PCE (%)"],
        errors="coerce"
    )

    tier_chemistry_rows.append({

        "file":
            relative_path,

        "records":
            len(df),

        "donor_names":
            df["Name_Donor"].nunique(),

        "acceptor_names":
            df["Name_Acceptor"].nunique(),

        "named_DA_pairs":
            df[
                ["Name_Donor", "Name_Acceptor"]
            ]
            .drop_duplicates()
            .shape[0],

        "unique_PCE":
            pce_numeric.nunique(),

        "pce_mean":
            pce_numeric.mean(),

        "pce_median":
            pce_numeric.median(),

        "pce_min":
            pce_numeric.min(),

        "pce_max":
            pce_numeric.max(),
    })


wen_tier_chemistry = pd.DataFrame(
    tier_chemistry_rows
)

display(
    wen_tier_chemistry.round(3)
)



# ------------------------------------------------------------------
# 9.3
# Nominal D:A + PCE overlap with global dataset
# ------------------------------------------------------------------

global_path = next(
    path
    for path in wen_tier_tables_clean
    if path.endswith(
        "global-all-nine-parameters.csv"
    )
)

global_clean = (
    wen_tier_tables_clean[
        global_path
    ]
)


global_identity_keys = set(
    zip(
        global_clean[
            "Name_Donor"
        ].map(
            normalize_material_label
        ),

        global_clean[
            "Name_Acceptor"
        ].map(
            normalize_material_label
        ),

        pd.to_numeric(
            global_clean["PCE (%)"],
            errors="coerce"
        ).round(6),
    )
)


tier_global_overlap_rows = []


for relative_path, df in wen_tier_tables_clean.items():

    pce_numeric = pd.to_numeric(
        df["PCE (%)"],
        errors="coerce"
    )

    keys = list(
        zip(
            df[
                "Name_Donor"
            ].map(
                normalize_material_label
            ),

            df[
                "Name_Acceptor"
            ].map(
                normalize_material_label
            ),

            pce_numeric.round(6),
        )
    )

    overlap = sum(
        key in global_identity_keys
        for key in keys
    )

    tier_global_overlap_rows.append({

        "file":
            relative_path,

        "records":
            len(df),

        "records_matching_global_DA_PCE":
            overlap,

        "percent_matching_global":
            100 * overlap / len(df),
    })


tier_global_overlap = (
    pd.DataFrame(
        tier_global_overlap_rows
    )
)

display(
    tier_global_overlap.round(2)
)

,file,records,donor_names,acceptor_names,named_DA_pairs,unique_PCE,pce_mean,pce_median,pce_min,pce_max
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1767,161,392,518,1075,8.701,8.800,0.02,18.400
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,128,24,50,54,124,6.581,5.765,0.08,18.320
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,407,40,108,126,343,8.319,7.800,0.28,18.620
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,1132,97,276,346,775,8.925,9.005,0.15,19.060
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,95,16,24,30,92,11.021,11.700,0.70,16.608
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,213,18,45,50,202,8.969,9.870,0.71,18.800
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,199,32,34,51,168,11.077,11.800,1.06,17.500
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1600,139,325,438,1002,9.354,9.570,0.02,18.600
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,138,26,26,39,122,10.795,11.115,0.85,18.040
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,2087,303,779,1138,1150,9.719,10.170,0.01,19.060


,file,records,records_matching_global_DA_PCE,percent_matching_global
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1767,269,15.22
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,128,11,8.59
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,407,55,13.51
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,1132,172,15.19
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,95,6,6.32
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,213,46,21.60
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,199,62,31.16
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1600,292,18.25
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,138,30,21.74
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,2087,460,22.04


### 9.4 Shared-column record overlap across modeling tiers

Donor–acceptor identity and PCE matching provides evidence that the modeling
tiers are drawn from overlapping experimental pools, but it is not sufficient
to establish reuse of the same individual records.

Each non-global table is therefore compared with the global table using all
columns shared between the two tables.

Two complementary overlap definitions are evaluated:

1. exact shared-record overlap, including the PCE target; and
2. shared-predictor overlap, excluding PCE.

The second comparison identifies conditions with identical available predictor
representations but potentially different reported PCE values.

Because different modeling tiers contain different subsets of processing
variables, overlap is interpreted relative to the information shared by each
tier and the global table rather than as absolute experimental identity.

In [86]:
# ------------------------------------------------------------------
# 9.4 A
# Dtype-robust pairwise normalization
# ------------------------------------------------------------------

def normalize_shared_frames(tier_df, global_df, columns):
    """
    Jointly normalize two tables for value-based comparison.

    Numerical interpretation is decided using values from BOTH tables,
    preventing integer/float/object dtype differences from changing
    otherwise equivalent row signatures.
    """

    tier_out = pd.DataFrame(index=tier_df.index)
    global_out = pd.DataFrame(index=global_df.index)

    for col in columns:

        tier_s = tier_df[col]
        global_s = global_df[col]

        combined = pd.concat(
            [tier_s, global_s],
            ignore_index=True
        )

        combined_numeric = pd.to_numeric(
            combined,
            errors="coerce"
        )

        original_nonmissing = combined.notna().sum()
        numeric_nonmissing = combined_numeric.notna().sum()

        # ----------------------------------------------------------
        # Treat as numeric only if every non-missing value from
        # both tables is numerically interpretable.
        # ----------------------------------------------------------
        if (
            original_nonmissing > 0
            and numeric_nonmissing == original_nonmissing
        ):

            tier_num = pd.to_numeric(
                tier_s,
                errors="coerce"
            ).astype(float).round(10)

            global_num = pd.to_numeric(
                global_s,
                errors="coerce"
            ).astype(float).round(10)

            tier_out[col] = tier_num
            global_out[col] = global_num

        else:

            def normalize_text_series(s):

                s = s.astype("string")

                return s.map(
                    lambda x:
                        unicodedata.normalize(
                            "NFKC",
                            str(x)
                        ).strip()
                        if pd.notna(x)
                        else pd.NA
                )

            tier_out[col] = normalize_text_series(
                tier_s
            )

            global_out[col] = normalize_text_series(
                global_s
            )

    return tier_out, global_out



    # ------------------------------------------------------------------
# 9.4  B
# Stable value-based row signatures
# ------------------------------------------------------------------

def canonical_row_signatures(df):

    canonical = pd.DataFrame(
        index=df.index
    )

    for col in df.columns:

        s = df[col]

        if pd.api.types.is_numeric_dtype(s):

            canonical[col] = s.map(
                lambda x:
                    "<NA>"
                    if pd.isna(x)
                    else f"{float(x):.10g}"
            )

        else:

            canonical[col] = s.map(
                lambda x:
                    "<NA>"
                    if pd.isna(x)
                    else str(x)
            )

    return pd.util.hash_pandas_object(
        canonical,
        index=False
    )



# ------------------------------------------------------------------
# 9.4C — OPTIMIZED
# Fast dtype-robust shared-column overlap
# ------------------------------------------------------------------

def fast_normalize_shared_pair(tier_df, global_df, columns):
    """
    Fast cross-table normalization.

    Numeric columns present as numeric in both tables are converted
    simultaneously to float64 and rounded.

    Remaining columns are treated as text and Unicode-normalized.
    """

    numeric_cols = [
        col for col in columns
        if (
            pd.api.types.is_numeric_dtype(tier_df[col])
            and
            pd.api.types.is_numeric_dtype(global_df[col])
        )
    ]

    text_cols = [
        col for col in columns
        if col not in numeric_cols
    ]


    # --------------------------------------------------------------
    # Numeric block -- vectorized
    # --------------------------------------------------------------

    tier_numeric = (
        tier_df[numeric_cols]
        .astype("float64")
        .round(10)
    )

    global_numeric = (
        global_df[numeric_cols]
        .astype("float64")
        .round(10)
    )


    # --------------------------------------------------------------
    # Text / identity block -- normally only a few columns
    # --------------------------------------------------------------

    tier_text = pd.DataFrame(
        index=tier_df.index
    )

    global_text = pd.DataFrame(
        index=global_df.index
    )

    for col in text_cols:

        tier_text[col] = (
            tier_df[col]
            .astype("string")
            .str.normalize("NFKC")
            .str.strip()
        )

        global_text[col] = (
            global_df[col]
            .astype("string")
            .str.normalize("NFKC")
            .str.strip()
        )


    # Preserve original shared-column order
    tier_norm = pd.concat(
        [tier_numeric, tier_text],
        axis=1
    )[columns]

    global_norm = pd.concat(
        [global_numeric, global_text],
        axis=1
    )[columns]


    return tier_norm, global_norm

In [88]:
# ------------------------------------------------------------------
# Run optimized comparison across all non-global tiers
# ------------------------------------------------------------------

corrected_overlap_rows = []


for tier_number, (relative_path, tier_df) in enumerate(
    wen_tier_tables_clean.items(),
    start=1
):

    if relative_path == global_path:
        continue


    print(
        f"Processing {tier_number}: "
        f"{Path(relative_path).name}"
    )


    shared_columns = [
        col
        for col in global_clean.columns
        if col in tier_df.columns
    ]


    # --------------------------------------------------------------
    # Normalize ONCE
    # --------------------------------------------------------------

    tier_norm, global_norm = (
        fast_normalize_shared_pair(
            tier_df,
            global_clean,
            shared_columns
        )
    )


    # --------------------------------------------------------------
    # Full shared-record signatures
    # --------------------------------------------------------------

    tier_full_hash = (
        pd.util.hash_pandas_object(
            tier_norm,
            index=False
        )
    )

    global_full_hash = (
        pd.util.hash_pandas_object(
            global_norm,
            index=False
        )
    )

    global_full_set = set(
        global_full_hash.to_numpy()
    )

    full_match = tier_full_hash.isin(
        global_full_set
    )


    # --------------------------------------------------------------
    # Predictor-only signatures
    # Reuse already-normalized data
    # --------------------------------------------------------------

    predictor_columns_shared = [
        col
        for col in shared_columns
        if col != "PCE (%)"
    ]


    tier_predictor_hash = (
        pd.util.hash_pandas_object(
            tier_norm[
                predictor_columns_shared
            ],
            index=False
        )
    )

    global_predictor_hash = (
        pd.util.hash_pandas_object(
            global_norm[
                predictor_columns_shared
            ],
            index=False
        )
    )

    global_predictor_set = set(
        global_predictor_hash.to_numpy()
    )

    predictor_match = (
        tier_predictor_hash.isin(
            global_predictor_set
        )
    )


    corrected_overlap_rows.append({

        "file":
            relative_path,

        "records":
            len(tier_df),

        "shared_columns":
            len(shared_columns),

        "shared_predictor_columns":
            len(predictor_columns_shared),

        "exact_shared_record_matches":
            int(full_match.sum()),

        "exact_shared_record_percent":
            100 * full_match.mean(),

        "shared_predictor_matches":
            int(predictor_match.sum()),

        "shared_predictor_percent":
            100 * predictor_match.mean(),

        "predictor_match_but_target_diff":
            int(
                (
                    predictor_match
                    & ~full_match
                ).sum()
            ),
    })


tier_shared_overlap_corrected = (
    pd.DataFrame(
        corrected_overlap_rows
    )
)


display(
    tier_shared_overlap_corrected.round(2)
)

Processing 1: single-parameter-01-d-a-weight-ratio.csv
Processing 2: single-parameter-02-solvent-type.csv
Processing 3: single-parameter-03-additive-type.csv
Processing 4: single-parameter-04-additive-volume-ratio.csv
Processing 5: single-parameter-05-blend-concentration.csv
Processing 6: single-parameter-06-spin-coating-speed.csv
Processing 7: single-parameter-07-active-layer-thickness.csv
Processing 8: single-parameter-08-annealing-temperature.csv
Processing 9: single-parameter-09-annealing-time.csv
Processing 10: stage-combined-post-processing.csv
Processing 11: stage-combined-solution-preparation.csv
Processing 12: stage-combined-spin-coating-film-formation.csv


,file,records,shared_columns,shared_predictor_columns,exact_shared_record_matches,exact_shared_record_percent,shared_predictor_matches,shared_predictor_percent,predictor_match_but_target_diff
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1767,2073,2072,269,15.22,278,15.73,9
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,128,2075,2074,11,8.59,12,9.38,1
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,407,2078,2077,45,11.06,49,12.04,4
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,1132,2073,2072,172,15.19,174,15.37,2
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,95,2073,2072,6,6.32,8,8.42,2
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,213,2073,2072,45,21.13,45,21.13,0
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,199,2073,2072,62,31.16,64,32.16,2
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1600,2072,2071,292,18.25,308,19.25,16
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,138,2073,2072,30,21.74,30,21.74,0
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,2087,2073,2072,458,21.95,477,22.86,19


In [89]:
# ------------------------------------------------------------------
# 9.4D
# Compare nominal D:A+PCE overlap with corrected exact overlap
# ------------------------------------------------------------------

corrected_overlap_comparison = (
    tier_global_overlap[
        [
            "file",
            "records_matching_global_DA_PCE",
            "percent_matching_global",
        ]
    ]
    .merge(
        tier_shared_overlap_corrected,
        on="file",
        how="inner"
    )
)


corrected_overlap_comparison[
    "nominal_minus_exact_percent"
] = (
    corrected_overlap_comparison[
        "percent_matching_global"
    ]
    -
    corrected_overlap_comparison[
        "exact_shared_record_percent"
    ]
)


display(
    corrected_overlap_comparison[
        [
            "file",
            "records",
            "shared_columns",
            "percent_matching_global",
            "exact_shared_record_percent",
            "shared_predictor_percent",
            "predictor_match_but_target_diff",
            "nominal_minus_exact_percent",
        ]
    ]
    .round(2)
)

,file,records,shared_columns,percent_matching_global,exact_shared_record_percent,shared_predictor_percent,predictor_match_but_target_diff,nominal_minus_exact_percent
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1767,2073,15.22,15.22,15.73,9,0.00
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,128,2075,8.59,8.59,9.38,1,0.00
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,407,2078,13.51,11.06,12.04,4,2.46
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,1132,2073,15.19,15.19,15.37,2,0.00
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,95,2073,6.32,6.32,8.42,2,0.00
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,213,2073,21.60,21.13,21.13,0,0.47
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,199,2073,31.16,31.16,32.16,2,0.00
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1600,2072,18.25,18.25,19.25,16,0.00
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,138,2073,21.74,21.74,21.74,0,0.00
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,2087,2073,22.04,21.95,22.86,19,0.10


## 9.5 Verification of Residual Cross-Tier Disagreements

Dtype-robust record matching resolves most apparent differences between the
global and lower-tier Wen–Zhang–Ma datasets.

Residual disagreements fall into two distinct categories.

First, a small number of records match the global dataset in normalized donor
identity, acceptor identity, and PCE but differ in one or more additional
shared variables.

Second, some records possess an identical complete shared predictor
representation but a different PCE target.

These cases are examined separately because the former reflects differences
in record representation or processing metadata, whereas the latter represents
target inconsistency for otherwise indistinguishable model inputs.

In [94]:
# ------------------------------------------------------------------
# 9.5A
# Nominal D:A+PCE matches that fail complete shared-column matching
# ------------------------------------------------------------------

corrected_overlap_comparison[
    "nominal_match_not_exact"
] = (
    corrected_overlap_comparison[
        "records_matching_global_DA_PCE"
    ]
    -
    corrected_overlap_comparison[
        "exact_shared_record_matches"
    ]
)


residual_nominal_summary = (
    corrected_overlap_comparison[
        [
            "file",
            "records",
            "records_matching_global_DA_PCE",
            "exact_shared_record_matches",
            "nominal_match_not_exact",
        ]
    ]
    .sort_values(
        "nominal_match_not_exact",
        ascending=False
    )
)


display(
    residual_nominal_summary
)

,file,records,records_matching_global_DA_PCE,exact_shared_record_matches,nominal_match_not_exact
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,407,55,45,10
10,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-solution-preparation.csv,3666,619,617,2
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,2087,460,458,2
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,213,46,45,1
11,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-spin-coating-film-formation.csv,532,340,339,1
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,1132,172,172,0
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1767,269,269,0
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,128,11,11,0
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1600,292,292,0
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,199,62,62,0


In [95]:
# ------------------------------------------------------------------
# 9.5B
# Diagnose remaining nominal-but-nonexact matches
# ------------------------------------------------------------------

def diagnose_residual_nominal_matches(
    tier_path,
    global_df,
    tier_tables
):

    tier_df = tier_tables[tier_path]

    shared_columns = [
        col
        for col in global_df.columns
        if col in tier_df.columns
    ]

    tier_cmp, global_cmp = (
        fast_normalize_shared_pair(
            tier_df,
            global_df,
            shared_columns
        )
    )


    tier_keys = make_da_pce_keys(tier_df)
    global_keys = make_da_pce_keys(global_df)


    global_key_map = {}

    for idx, key in enumerate(global_keys):

        global_key_map.setdefault(
            key,
            []
        ).append(idx)


    residual_rows = []
    column_counts = {}


    for tier_idx, key in enumerate(tier_keys):

        candidate_indices = (
            global_key_map.get(
                key,
                []
            )
        )

        if not candidate_indices:
            continue


        tier_row = tier_cmp.iloc[
            tier_idx
        ]

        best_columns = None
        best_count = None
        best_global_idx = None


        for global_idx in candidate_indices:

            global_row = global_cmp.iloc[
                global_idx
            ]

            equal = (
                tier_row.eq(global_row)
                |
                (
                    tier_row.isna()
                    &
                    global_row.isna()
                )
            )

            differences = (
                equal[
                    ~equal
                ]
                .index
                .tolist()
            )


            if (
                best_count is None
                or
                len(differences) < best_count
            ):

                best_count = len(
                    differences
                )

                best_columns = differences

                best_global_idx = (
                    global_idx
                )


        # Already truly exact
        if best_count == 0:
            continue


        residual_rows.append({

            "tier_row":
                tier_idx,

            "global_row":
                best_global_idx,

            "Name_Donor":
                tier_df.iloc[
                    tier_idx
                ]["Name_Donor"],

            "Name_Acceptor":
                tier_df.iloc[
                    tier_idx
                ]["Name_Acceptor"],

            "PCE (%)":
                tier_df.iloc[
                    tier_idx
                ]["PCE (%)"],

            "different_columns":
                best_count,

            "columns":
                best_columns,
        })


        for col in best_columns:

            column_counts[col] = (
                column_counts.get(
                    col,
                    0
                )
                + 1
            )


    detail = pd.DataFrame(
        residual_rows
    )

    frequency = pd.DataFrame(
        [
            {
                "column": col,
                "records": count,
            }
            for col, count
            in column_counts.items()
        ]
    )

    if len(frequency):

        frequency = (
            frequency
            .sort_values(
                "records",
                ascending=False
            )
            .reset_index(drop=True)
        )


    return detail, frequency



    # ------------------------------------------------------------------
# Run residual diagnosis only where nominal overlap exceeds exact overlap
# ------------------------------------------------------------------

residual_diagnostics = {}


for _, row in residual_nominal_summary.iterrows():

    if row["nominal_match_not_exact"] <= 0:
        continue

    tier_path = row["file"]

    detail, frequency = (
        diagnose_residual_nominal_matches(
            tier_path,
            global_clean,
            wen_tier_tables_clean
        )
    )

    residual_diagnostics[
        tier_path
    ] = {
        "detail": detail,
        "frequency": frequency,
    }


    print("\n" + "=" * 100)
    print(Path(tier_path).name)
    print("=" * 100)

    print(
        "Residual nominal-but-nonexact records:",
        len(detail)
    )

    display(
        frequency.head(20)
    )


single-parameter-03-additive-type.csv
Residual nominal-but-nonexact records: 10


,column,records
0,Additive_BoilingPoint (°C),10



stage-combined-solution-preparation.csv
Residual nominal-but-nonexact records: 2


,column,records
0,D_A_Weight_Ratio,2



stage-combined-post-processing.csv
Residual nominal-but-nonexact records: 2


,column,records
0,Annealing_Time (min),2



single-parameter-06-spin-coating-speed.csv
Residual nominal-but-nonexact records: 1


,column,records
0,Spin_Coating_Rate (rpm),1



stage-combined-spin-coating-film-formation.csv
Residual nominal-but-nonexact records: 1


,column,records
0,Spin_Coating_Rate (rpm),1


In [96]:
# ------------------------------------------------------------------
# 9.5C
# Same shared predictors, different PCE
# ------------------------------------------------------------------

cross_tier_target_conflict_summary = (
    tier_shared_overlap_corrected[
        [
            "file",
            "records",
            "shared_predictor_matches",
            "predictor_match_but_target_diff",
        ]
    ]
    .copy()
)


cross_tier_target_conflict_summary[
    "percent_of_predictor_matches_conflicting"
] = (
    100
    * cross_tier_target_conflict_summary[
        "predictor_match_but_target_diff"
    ]
    /
    cross_tier_target_conflict_summary[
        "shared_predictor_matches"
    ].replace(0, np.nan)
).round(2)


display(
    cross_tier_target_conflict_summary
    .sort_values(
        "predictor_match_but_target_diff",
        ascending=False
    )
)

,file,records,shared_predictor_matches,predictor_match_but_target_diff,percent_of_predictor_matches_conflicting
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,2087,477,19,3.98
10,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-solution-preparation.csv,3666,634,17,2.68
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1600,308,16,5.19
11,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-spin-coating-film-formation.csv,532,351,12,3.42
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1767,278,9,3.24
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,407,49,4,8.16
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,95,8,2,25.00
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,1132,174,2,1.15
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,199,64,2,3.12
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,128,12,1,8.33


## 9.6 Final Roles of the Wen–Zhang–Ma Modeling Tiers

Cross-tier analysis demonstrates that the thirteen Wen–Zhang–Ma modeling
tables are overlapping views of a common literature-derived experimental pool
rather than independent external datasets.

The global nine-parameter table shares exact high-dimensional records with
every major processing tier. Exact shared-record overlap ranges from modest
levels in several single-parameter datasets to 63.72% of the spin-coating /
film-formation stage dataset.

Residual donor–acceptor/PCE matches that fail complete shared-column matching
are rare and are localized to individual processing variables such as additive
boiling point, D:A ratio, annealing time, or spin-coating rate.

A small fraction of cross-tier shared predictor vectors also have different
reported PCE values, indicating residual target variability or curation
differences across tier construction.

Consequently, the lower-tier datasets are not treated as independent external
validation sets for models trained on the global dataset.

The primary processing benchmark is the physically harmonized global cohort
of 994 unique processing conditions. The larger single-parameter and
stage-combined datasets are retained as auxiliary resources for potential
processing-specific analyses, overlap-controlled pretraining, transfer
learning, and robustness studies. Any such experiment must explicitly remove
or group records overlapping with the target evaluation cohort before model
assessment.

In [97]:
# ------------------------------------------------------------------
# 9.6A
# Final quantitative summary of tier relationships
# ------------------------------------------------------------------

tier_role_summary = (
    wen_tier_chemistry[
        [
            "file",
            "records",
            "donor_names",
            "acceptor_names",
            "named_DA_pairs",
        ]
    ]
    .merge(
        tier_shared_overlap_corrected[
            [
                "file",
                "exact_shared_record_matches",
                "exact_shared_record_percent",
                "shared_predictor_matches",
                "shared_predictor_percent",
                "predictor_match_but_target_diff",
            ]
        ],
        on="file",
        how="left"
    )
)


tier_role_summary[
    "target_conflict_percent_of_shared_predictors"
] = (
    100
    * tier_role_summary[
        "predictor_match_but_target_diff"
    ]
    /
    tier_role_summary[
        "shared_predictor_matches"
    ].replace(0, np.nan)
).round(2)


def assign_tier_role(path):

    name = Path(path).name

    if name == "global-all-nine-parameters.csv":
        return "primary processing benchmark"

    if "stage-combined" in path:
        return "auxiliary stage-level / transfer resource"

    return "auxiliary single-parameter resource"


tier_role_summary["recommended_role"] = (
    tier_role_summary["file"]
    .apply(assign_tier_role)
)


display(
    tier_role_summary.round(2)
)

,file,records,donor_names,acceptor_names,named_DA_pairs,exact_shared_record_matches,exact_shared_record_percent,shared_predictor_matches,shared_predictor_percent,predictor_match_but_target_diff,target_conflict_percent_of_shared_predictors,recommended_role
0,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-01-d-a-weight-ratio.csv,1767,161,392,518,269.0,15.22,278.0,15.73,9.0,3.24,auxiliary single-parameter resource
1,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-02-solvent-type.csv,128,24,50,54,11.0,8.59,12.0,9.38,1.0,8.33,auxiliary single-parameter resource
2,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-03-additive-type.csv,407,40,108,126,45.0,11.06,49.0,12.04,4.0,8.16,auxiliary single-parameter resource
3,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-04-additive-volume-ratio.csv,1132,97,276,346,172.0,15.19,174.0,15.37,2.0,1.15,auxiliary single-parameter resource
4,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-05-blend-concentration.csv,95,16,24,30,6.0,6.32,8.0,8.42,2.0,25.00,auxiliary single-parameter resource
5,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-06-spin-coating-speed.csv,213,18,45,50,45.0,21.13,45.0,21.13,0.0,0.00,auxiliary single-parameter resource
6,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-07-active-layer-thickness.csv,199,32,34,51,62.0,31.16,64.0,32.16,2.0,3.12,auxiliary single-parameter resource
7,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-08-annealing-temperature.csv,1600,139,325,438,292.0,18.25,308.0,19.25,16.0,5.19,auxiliary single-parameter resource
8,opv-multi-tier-ml-database\01-single-parameter-models\single-parameter-09-annealing-time.csv,138,26,26,39,30.0,21.74,30.0,21.74,0.0,0.00,auxiliary single-parameter resource
9,opv-multi-tier-ml-database\02-stage-combined-models\stage-combined-post-processing.csv,2087,303,779,1138,458.0,21.95,477.0,22.86,19.0,3.98,auxiliary stage-level / transfer resource


In [98]:
# ------------------------------------------------------------------
# 9.6B
# Save final cross-tier audit outputs
# ------------------------------------------------------------------

tier_role_summary.to_csv(
    INTERIM_DIR / "wen_tier_role_summary.csv",
    index=False
)

tier_shared_overlap_corrected.to_csv(
    INTERIM_DIR / "wen_tier_shared_record_overlap.csv",
    index=False
)

cross_tier_target_conflict_summary.to_csv(
    INTERIM_DIR / "wen_cross_tier_target_conflicts.csv",
    index=False
)

residual_nominal_summary.to_csv(
    INTERIM_DIR / "wen_cross_tier_residual_nominal_matches.csv",
    index=False
)

print("Section 9 tier audit saved.")

Section 9 tier audit saved.


## 10.0 Final Audit Freeze and Modeling Cohorts

Following provenance, identity, processing-semantic, representation, duplicate,
and cross-tier audits, the datasets used for subsequent modeling are formally
frozen.

Two primary research cohorts are retained:

1. the provenance-clean OPV-DB molecular benchmark for large-scale
   structure–performance and generalization analysis; and

2. the physically harmonized Wen–Zhang–Ma global benchmark containing one
   observation per unique interpreted processing condition.

Raw names, SMILES, audit flags, replicate metadata, and target-derived fields
are retained as metadata but are not automatically included as numerical model
features.

The frozen cohorts and feature definitions are saved before model training so
that subsequent experiments operate on fixed data rather than changing
preprocessing decisions between models.

In [100]:
# ------------------------------------------------------------------
# 10.1
# Freeze primary research cohorts
# ------------------------------------------------------------------

opv_primary_cohort = provenance_clean.copy()

wen_primary_cohort = (
    wen_physical_condition_level.copy()
)


print("PRIMARY COHORTS")
print("-" * 60)

print(
    "OPV-DB provenance-clean cohort:",
    opv_primary_cohort.shape
)

print(
    "Wen/Ma physical-condition cohort:",
    wen_primary_cohort.shape
)

print(
    "\nOPV-DB unique records:",
    opv_primary_cohort["id"].nunique()
)

print(
    "OPV-DB source DOIs:",
    opv_primary_cohort["doi_norm"].nunique()
)

print(
    "\nWen/Ma unique physical conditions:",
    wen_primary_cohort[
        "physical_predictor_hash"
    ].nunique()
)

print(
    "Wen/Ma named D:A systems:",
    wen_primary_cohort[
        ["Name_Donor", "Name_Acceptor"]
    ].drop_duplicates().shape[0]
)

PRIMARY COHORTS
------------------------------------------------------------
OPV-DB provenance-clean cohort: (21590, 42)
Wen/Ma physical-condition cohort: (994, 2101)

OPV-DB unique records: 21590
OPV-DB source DOIs: 5439

Wen/Ma unique physical conditions: 994
Wen/Ma named D:A systems: 247


In [101]:
# ------------------------------------------------------------------
# 10.2A
# Retain only informative fingerprint bits
# ------------------------------------------------------------------

donor_fp_variable = [
    col
    for col in donor_fp_columns
    if wen_primary_cohort[col].nunique() > 1
]

acceptor_fp_variable = [
    col
    for col in acceptor_fp_columns
    if wen_primary_cohort[col].nunique() > 1
]


print(
    "Donor fingerprint bits:",
    len(donor_fp_columns),
    "->",
    len(donor_fp_variable)
)

print(
    "Acceptor fingerprint bits:",
    len(acceptor_fp_columns),
    "->",
    len(acceptor_fp_variable)
)

print(
    "Invariant donor bits removed:",
    len(donor_fp_columns)
    - len(donor_fp_variable)
)

print(
    "Invariant acceptor bits removed:",
    len(acceptor_fp_columns)
    - len(acceptor_fp_variable)
)

Donor fingerprint bits: 1024 -> 985
Acceptor fingerprint bits: 1024 -> 1004
Invariant donor bits removed: 39
Invariant acceptor bits removed: 20


In [104]:
# ------------------------------------------------------------------
# 10.2B
# Processing feature panel
# ------------------------------------------------------------------

processing_model_features = [
    "D_A_Weight_Ratio",
    "Blend_Concentration (mg/ml)",

    "Solvent_DipoleMoment (Debye)",
    "Solvent_EnergyGap (eV)",
    "Solvent_Polarizability (a.u.)",

    *additive_descriptor_columns,

    "Additive_Volume_Ratio (vol%)",
    "Spin_Coating_Rate (rpm)",

    # Keep original Wen/Ma cohort header
    "Annealing_Temperature(℃)",
    "Annealing_Time_Physical",

    "Active_Layer_Thickness (nm)",

    "Additive_Present",
    "Thermal_Annealing_Present",
    "Annealing_Duration_Unresolved",
]


print(
    "Processing features:",
    len(processing_model_features)
)

missing_processing_features = [
    col
    for col in processing_model_features
    if col not in wen_primary_cohort.columns
]

print(
    "Missing processing features:",
    missing_processing_features
)

Processing features: 20
Missing processing features: []


In [105]:
# ------------------------------------------------------------------
# 10.2C
# Molecular-structure feature panel
# ------------------------------------------------------------------

structure_model_features = (
    molecular_descriptor_columns
    + donor_fp_variable
    + acceptor_fp_variable
)


print(
    "Molecular descriptors:",
    len(molecular_descriptor_columns)
)

print(
    "Variable donor fingerprint bits:",
    len(donor_fp_variable)
)

print(
    "Variable acceptor fingerprint bits:",
    len(acceptor_fp_variable)
)

print(
    "Total structure features:",
    len(structure_model_features)
)

Molecular descriptors: 19
Variable donor fingerprint bits: 985
Variable acceptor fingerprint bits: 1004
Total structure features: 2008


In [106]:
# ------------------------------------------------------------------
# 10.2D
# Full structure + processing feature panel
# ------------------------------------------------------------------

full_model_features = (
    structure_model_features
    + processing_model_features
)


assert (
    len(full_model_features)
    ==
    len(set(full_model_features))
)


print(
    "Processing features:",
    len(processing_model_features)
)

print(
    "Structure features:",
    len(structure_model_features)
)

print(
    "Full features:",
    len(full_model_features)
)

Processing features: 20
Structure features: 2008
Full features: 2028


In [107]:
# ------------------------------------------------------------------
# 10.3
# Construct numerical Wen/Ma model matrices
# ------------------------------------------------------------------

X_wen_processing = (
    wen_primary_cohort[
        processing_model_features
    ]
    .copy()
)

X_wen_structure = (
    wen_primary_cohort[
        structure_model_features
    ]
    .copy()
)

X_wen_full = (
    wen_primary_cohort[
        full_model_features
    ]
    .copy()
)


# --------------------------------------------------------------
# Six annealed conditions have unresolved duration.
# Zero is used only as a numerical placeholder because the
# Annealing_Duration_Unresolved indicator explicitly distinguishes
# these rows from true zero-duration/no-annealing states.
# --------------------------------------------------------------

X_wen_processing[
    "Annealing_Time_Physical"
] = (
    X_wen_processing[
        "Annealing_Time_Physical"
    ].fillna(0.0)
)

X_wen_full[
    "Annealing_Time_Physical"
] = (
    X_wen_full[
        "Annealing_Time_Physical"
    ].fillna(0.0)
)


y_wen = (
    wen_primary_cohort[
        TARGET_COL
    ]
    .astype(float)
    .copy()
)


print(
    "Processing matrix:",
    X_wen_processing.shape
)

print(
    "Structure matrix:",
    X_wen_structure.shape
)

print(
    "Full matrix:",
    X_wen_full.shape
)

print(
    "Target:",
    y_wen.shape
)

Processing matrix: (994, 20)
Structure matrix: (994, 2008)
Full matrix: (994, 2028)
Target: (994,)


In [108]:
# ------------------------------------------------------------------
# 10.4
# Final model-matrix numerical integrity
# ------------------------------------------------------------------

matrix_integrity_rows = []


for name, X in [
    ("processing", X_wen_processing),
    ("structure", X_wen_structure),
    ("full", X_wen_full),
]:

    numeric_X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    array = numeric_X.to_numpy(
        dtype=float
    )

    matrix_integrity_rows.append({

        "matrix":
            name,

        "records":
            len(X),

        "features":
            X.shape[1],

        "missing_cells":
            int(
                numeric_X
                .isna()
                .sum()
                .sum()
            ),

        "infinite_cells":
            int(
                np.isinf(array)
                .sum()
            ),

        "constant_features":
            int(
                (
                    numeric_X
                    .nunique(dropna=False)
                    <= 1
                ).sum()
            ),
    })


matrix_integrity = pd.DataFrame(
    matrix_integrity_rows
)

display(matrix_integrity)

,matrix,records,features,missing_cells,infinite_cells,constant_features
0,processing,994,20,0,0,0
1,structure,994,2008,0,0,0
2,full,994,2028,0,0,0


### 10.5 Validation-group metadata

Material names are retained as grouping metadata for validation but are not
used as numerical predictive features.

This distinction is necessary because the Wen–Zhang–Ma audit identified
several cases in which supplied SMILES or molecular fingerprints collapse
nominally distinct materials. Consequently, prospective validation groups are
defined using normalized curated material identities rather than fingerprint
equality alone.

Donor, acceptor, and donor–acceptor group identifiers are frozen before model
training so that validation definitions remain independent of model choice.

In [109]:
# ------------------------------------------------------------------
# 10.5
# Freeze validation-group metadata
# ------------------------------------------------------------------

wen_primary_cohort[
    "donor_name_norm"
] = (
    wen_primary_cohort[
        "Name_Donor"
    ]
    .apply(
        normalize_material_label
    )
)

wen_primary_cohort[
    "acceptor_name_norm"
] = (
    wen_primary_cohort[
        "Name_Acceptor"
    ]
    .apply(
        normalize_material_label
    )
)


wen_primary_cohort[
    "nominal_DA_group"
] = list(
    zip(
        wen_primary_cohort[
            "donor_name_norm"
        ],
        wen_primary_cohort[
            "acceptor_name_norm"
        ]
    )
)


validation_group_summary = pd.DataFrame({

    "grouping_level": [
        "donor",
        "acceptor",
        "nominal D:A pair",
    ],

    "unique_groups": [
        wen_primary_cohort[
            "donor_name_norm"
        ].nunique(),

        wen_primary_cohort[
            "acceptor_name_norm"
        ].nunique(),

        wen_primary_cohort[
            "nominal_DA_group"
        ].nunique(),
    ],
})


display(validation_group_summary)

,grouping_level,unique_groups
0,donor,60
1,acceptor,181
2,nominal D:A pair,247


In [110]:
# ------------------------------------------------------------------
# 10.6A
# Create final feature manifest
# ------------------------------------------------------------------

all_unique_model_features = list(
    dict.fromkeys(
        processing_model_features
        + structure_model_features
    )
)


feature_manifest = pd.DataFrame({
    "feature":
        all_unique_model_features
})


feature_manifest[
    "in_processing_panel"
] = (
    feature_manifest["feature"]
    .isin(processing_model_features)
)


feature_manifest[
    "in_structure_panel"
] = (
    feature_manifest["feature"]
    .isin(structure_model_features)
)


feature_manifest[
    "in_full_panel"
] = True


print(
    "Unique model features:",
    len(feature_manifest)
)

print(
    "Processing:",
    feature_manifest[
        "in_processing_panel"
    ].sum()
)

print(
    "Structure:",
    feature_manifest[
        "in_structure_panel"
    ].sum()
)

print(
    "Full:",
    feature_manifest[
        "in_full_panel"
    ].sum()
)

display(
    feature_manifest.head(30)
)

Unique model features: 2028
Processing: 20
Structure: 2008
Full: 2028


,feature,in_processing_panel,in_structure_panel,in_full_panel
0,D_A_Weight_Ratio,True,False,True
1,Blend_Concentration (mg/ml),True,False,True
2,Solvent_DipoleMoment (Debye),True,False,True
3,Solvent_EnergyGap (eV),True,False,True
4,Solvent_Polarizability (a.u.),True,False,True
5,Additive_MeltingPoint (℃),True,False,True
6,Additive_BoilingPoint (℃),True,False,True
7,Additive_Density (g/cm3),True,False,True
8,Additive_MolecularWeight,True,False,True
9,Additive_DipoleMoment (Debye),True,False,True


In [111]:
# ------------------------------------------------------------------
# 10.6B
# Save frozen cohorts and model definitions
# ------------------------------------------------------------------

opv_primary_cohort.to_csv(
    INTERIM_DIR
    / "FINAL_opvdb_primary_cohort.csv",
    index=False
)


wen_primary_cohort.to_csv(
    INTERIM_DIR
    / "FINAL_wen_primary_physical_cohort.csv",
    index=False
)


feature_manifest.to_csv(
    INTERIM_DIR
    / "FINAL_wen_feature_manifest.csv",
    index=False
)


matrix_integrity.to_csv(
    INTERIM_DIR
    / "FINAL_wen_matrix_integrity.csv",
    index=False
)


validation_group_summary.to_csv(
    INTERIM_DIR
    / "FINAL_wen_validation_group_summary.csv",
    index=False
)


print("=" * 70)
print("FINAL AUDIT ARTIFACTS SAVED")
print("=" * 70)

print(
    "OPV-DB primary cohort:",
    len(opv_primary_cohort)
)

print(
    "Wen/Ma primary cohort:",
    len(wen_primary_cohort)
)

print(
    "Processing features:",
    len(processing_model_features)
)

print(
    "Structure features:",
    len(structure_model_features)
)

print(
    "Full features:",
    len(full_model_features)
)

FINAL AUDIT ARTIFACTS SAVED
OPV-DB primary cohort: 21590
Wen/Ma primary cohort: 994
Processing features: 20
Structure features: 2008
Full features: 2028


## 10.7 Data-audit conclusion

The data-audit stage is complete.

The primary OPV-DB benchmark contains 21,590 provenance-clean experimental
device records from 5,439 source publications. Its chemical-space and
validation audit demonstrates that conventional random row splitting produces
substantial publication and chemical overlap between training and test sets,
motivating DOI-, donor-, acceptor-, and donor–acceptor-pair-held-out
generalization experiments.

The primary Wen–Zhang–Ma processing benchmark contains 994 unique physically
interpreted processing conditions derived from 1,028 raw global records.
Duplicate observations, conflicting targets, contextual zero-additive
encodings, ambiguous annealing-duration states, molecular fingerprints,
electronic descriptors, processing-media representations, and cross-tier
record dependence have been explicitly audited.

Three frozen Wen–Zhang–Ma feature panels are retained for modeling:
20 processing features, 2,008 structure features, and a combined 2,028-feature
structure–processing representation. All final model matrices contain no
missing, infinite, or constant features.

The thirteen Wen–Zhang–Ma modeling tiers are not independent datasets.
Lower-tier tables are therefore retained only as auxiliary resources for
overlap-controlled robustness, processing-specific, or transfer-learning
experiments rather than naive external validation.

All subsequent modeling is performed on these frozen cohorts and feature
definitions.